# Tool-call verifier / ranker training notebook, production edition

This notebook trains a small model sidecar for tool-call guardrails. It is meant to classify or rank candidate tool calls after deterministic checks already handled syntax, schema, unknown tool names, required-step enforcement, and terminal-tool rules.

Main changes in this production edition:

- defaults to `LABEL_MODE = "production"`
- defaults tool-call input serialization to `toolcall-verifier-input/v2` with `serialize_state_v3`; v1/v2 serializer fixtures are still written for compatibility
- defaults `GPU_PROFILE` to `"auto"`; on T4 this selects `t4_proven`, while `t4_fast` remains diagnostic only
- keeps Hugging Face upload enabled by default, private by default
- uses group-safe train/validation/test splits and protected-slice balancing to reduce leakage and keep validation/test slices meaningful
- includes targeted error-recovery numeric semantic rows for fixed-width count strings such as `"0010"`
- includes tokenization diagnostics that verify whether `CANDIDATE_CALL` and `CANDIDATE_TOOL_SCHEMA` survive the selected `MAX_LENGTH`
- separates raw diagnostic gates from deployment-gated promotion policy
- exports ONNX and optional dynamic quantization for Rust-side inference

## Read the latest T4 runs correctly

The `t4_fast` diagnostic baseline failed badly: test macro F1 `0.3716`, test `valid` recall `0.528`, and test `wrong_tool_semantic` precision `0.387`. That was a data/profile/checkpoint-selection failure, not a promotion candidate.

The current T4 `serialize_state_v3` run is a major improvement, but should stay shadow/advisory only until replay proves it safe.

Latest test readout:

| Metric | Gate / note | Latest test |
| --- | ---: | ---: |
| Accuracy | diagnostic | `0.9562` |
| Present-label macro F1 | diagnostic | `0.9096` |
| `valid_recall` | `>= 0.94` | `0.9775` |
| `valid_false_objection @ 0.90` | `<= 0.005` raw diagnostic | `0.00624`, 10/1602 |
| `wrong_tool_semantic_precision` | `>= 0.90` | `0.9892` |
| `wrong_tool_semantic_recall` | diagnostic | `0.9198` |
| `wrong_arguments_semantic_precision` | diagnostic | `0.9745` |
| `wrong_arguments_semantic_recall` | diagnostic | `0.9758` |

Candidate/schema truncation is fixed at `MAX_LENGTH=1024`: `candidate_marker_truncated_rate=0.0` and `schema_marker_truncated_rate=0.0`.

Interpretation:

- raw `valid_false_objection @ 0.90` is still slightly above the diagnostic ceiling.
- the deployment rule at `0.98` confidence plus `0.25` margin over `valid` passes the false-objection target on validation and test.
- this should be frozen as `candidate-v6-shadow-advisory`, not enforced.

Production action policy under evaluation:

```json
{
  "mode": "advisory",
  "deployment_thresholds": {
    "wrong_tool_semantic": 0.98,
    "wrong_arguments_semantic": 0.98,
    "tool_not_needed": 0.98,
    "needs_clarification": 0.99
  },
  "min_margin_over_valid": 0.25,
  "valid_logit_bias": 0.0,
  "raw_0_90_false_objection": "diagnostic_only"
}
```

Promotion vocabulary:

- `raw_gate_status`: raw diagnostic gates, including `valid_false_objection @ 0.90`.
- `deployment_gate_status`: gates for the guarded rule production would actually use.
- `artifact_status`: `promotable_pending_replay` only when the configured blocking gates pass.
- `enforcement_eligible_status`: remains blocked until ONNX parity, quantized parity, shadow replay, advisory replay, and no-regression checks pass.

Dataset implication: do not spend the next iteration on longer context or broad valid duplication. The next data addendum should target the remaining confusion pairs: valid calls that look like `wrong_arguments_semantic`, corrected error-recovery positives, fixed-width numeric string positives, `needs_clarification` versus `valid`, Team-ACE `wrong_tool_semantic -> valid`, and Salesforce `wrong_arguments_semantic -> valid`.


In [ ]:
#@title Install dependencies
!pip -q install -U \
  "datasets>=2.20.0" \
  "transformers>=4.41.0" \
  "accelerate>=0.30.0" \
  "evaluate>=0.4.2" \
  "scikit-learn>=1.3.0" \
  "huggingface_hub>=0.23.0" \
  "jsonschema>=4.22.0" \
  "optimum[onnxruntime]>=1.20.0" \
  "onnxruntime>=1.18.0" \
  "onnx>=1.16.0" \
  "sentencepiece>=0.2.0"

In [ ]:
#@title Imports, paths, and GPU/profile selector
import os
import re
import gc
import ast
import json
import time
import math
import random
import shutil
import hashlib
import glob
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from datasets import Dataset, DatasetDict, load_dataset, Value
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit, train_test_split

import torch
from torch import nn
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

WORKDIR = Path("/content/toolcall-verifier")
DATA_DIR = WORKDIR / "data"
MODEL_DIR = WORKDIR / "model"
ONNX_DIR = WORKDIR / "onnx"
ARTIFACT_DIR = WORKDIR / "artifacts"
for p in [WORKDIR, DATA_DIR, MODEL_DIR, ONNX_DIR, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TOOLCALL_INPUT_SCHEMA_VERSION_V1 = "toolcall-verifier-input/v1"
TOOLCALL_INPUT_SCHEMA_VERSION_V2 = "toolcall-verifier-input/v2"
ARTIFACT_SCHEMA_VERSION = "toolcall-verifier-artifact/v1"
FINAL_RESPONSE_ARTIFACT_SCHEMA_VERSION = "final-response-verifier-artifact/v1"
FINAL_RESPONSE_INPUT_SCHEMA_VERSION = "final-response-verifier-input/v1"
FINAL_RESPONSE_THRESHOLDS_SCHEMA_VERSION = "final-response-verifier-thresholds/v1"
FINAL_RESPONSE_SERIALIZER_VERSION = "serialize_final_response_state_v1"
USE_SERIALIZER_V2 = True  #@param {type:"boolean"}
USE_SERIALIZER_V3 = True  #@param {type:"boolean"}  # Candidate-first layout; v1/v2 frozen for artifact compat
INPUT_SCHEMA_VERSION = TOOLCALL_INPUT_SCHEMA_VERSION_V2 if (USE_SERIALIZER_V2 or USE_SERIALIZER_V3) else TOOLCALL_INPUT_SCHEMA_VERSION_V1
if USE_SERIALIZER_V3:
    SERIALIZER_VERSION = "serialize_state_v3"
elif USE_SERIALIZER_V2:
    SERIALIZER_VERSION = "serialize_state_v2"
else:
    SERIALIZER_VERSION = "serialize_state_v1"

BASE_MODEL = "microsoft/deberta-v3-small"  #@param {type:"string"}

# Pick "auto" for normal use. On T4, auto selects the proven 4-epoch baseline.
GPU_PROFILE = "auto"  #@param ["auto", "debug_smoke", "t4_fast", "t4_proven", "l4_balanced", "a100_40gb", "high_vram_fast", "high_vram_quality", "high_vram_full_context"]
LABEL_MODE = "production"  #@param ["production", "diagnostic"]

# Optional manual overrides. Leave 0 / 0.0 to use the selected GPU profile.
MAX_PER_SOURCE_OVERRIDE = 0  #@param {type:"integer"}
MAX_LENGTH_OVERRIDE = 0  #@param {type:"integer"}
NUM_EPOCHS_OVERRIDE = 0  #@param {type:"integer"}
TRAIN_BATCH_SIZE_OVERRIDE = 0  #@param {type:"integer"}
EVAL_BATCH_SIZE_OVERRIDE = 0  #@param {type:"integer"}
GRAD_ACCUM_OVERRIDE = 0  #@param {type:"integer"}
LEARNING_RATE_OVERRIDE = 0.0  #@param {type:"number"}

# Keep class weights off by default. Previous weighted + synthetic rare-class runs regressed badly.
USE_CLASS_WEIGHTS = False  #@param {type:"boolean"}
MAX_CLASS_WEIGHT = 2.0  #@param {type:"number"}

# When True, deterministic_invalid rows are excluded from ML train/val/test.
# Rust deterministic rules are authoritative; ML label competition adds noise.
# The six-label artifact format is preserved — model learns near-zero logits for this class.
EXCLUDE_DETERMINISTIC_INVALID_FROM_ML_TRAINING = True  #@param {type:"boolean"}

# Forge-specific rows and the final-response verifier are part of the default production run.
# Disable either only for smoke checks or controlled ablations.
ENABLE_FORGE_AUGMENTATION = True  #@param {type:"boolean"}
FORGE_AUGMENTATION_PER_LABEL = 100  #@param {type:"integer"}
ENABLE_FINAL_RESPONSE_VERIFIER = True  #@param {type:"boolean"}
FINAL_RESPONSE_MAX_PER_LABEL = 5_000  #@param {type:"integer"}
FINAL_RESPONSE_BALANCE_LABELS = True  #@param {type:"boolean"}
FINAL_RESPONSE_MIN_PER_LABEL = 64  #@param {type:"integer"}
FINAL_RESPONSE_MAX_LENGTH_OVERRIDE = 0  #@param {type:"integer"}
FINAL_RESPONSE_NUM_EPOCHS_OVERRIDE = 0  #@param {type:"integer"}
FINAL_RESPONSE_TRAIN_BATCH_SIZE_OVERRIDE = 0  #@param {type:"integer"}
FINAL_RESPONSE_EVAL_BATCH_SIZE_OVERRIDE = 0  #@param {type:"integer"}
FINAL_RESPONSE_GRAD_ACCUM_OVERRIDE = 0  #@param {type:"integer"}
FINAL_RESPONSE_FORCE_RETRAIN = True  #@param {type:"boolean"}
FINAL_RESPONSE_EXPORT_CPU_ONLY = True  #@param {type:"boolean"}

# Personal agent logs are opt-in because they may contain private paths, prompts, code, or secrets.
INCLUDE_PRIVATE_AGENT_LOGS = False  #@param {type:"boolean"}

# Hugging Face upload is intentionally enabled by default, but private by default.
# For experimental v2/final-response artifacts, provenance files mark the artifact as shadow-first.
UPLOAD_TO_HUB = True  #@param {type:"boolean"}
HF_DATASET_REPO = "cowWhySo/toolcall-verifier-dataset"  #@param {type:"string"}
HF_MODEL_REPO = "cowWhySo/toolcall-verifier-classifier-production"  #@param {type:"string"}
HF_FINAL_RESPONSE_MODEL_REPO = "cowWhySo/final-response-verifier-classifier-production"  #@param {type:"string"}
PRIVATE = True  #@param {type:"boolean"}
if INCLUDE_PRIVATE_AGENT_LOGS and not PRIVATE:
    print("Forcing PRIVATE=True because private agent log training is enabled.")
    PRIVATE = True
if INCLUDE_PRIVATE_AGENT_LOGS and UPLOAD_TO_HUB:
    print("WARNING: private agent-derived rows are enabled. Upload only to a private dataset/model repo.")

# Baseline from the first completed T4 run. Used only for comparison in summaries.
T4_PROVEN_BASELINE = {
    "profile": "t4_proven",
    "gpu": "Tesla T4",
    "max_length": 768,
    "epochs_completed": 4,
    "best_epoch": 4,
    "best_accuracy": 0.839146,
    "best_macro_f1": 0.741871,
    "best_macro_f1_all_labels": 0.593497,
    "train_runtime_seconds": 5144.4353,
    "steps_per_second": 0.915,
}


def cuda_info() -> Dict[str, Any]:
    if not torch.cuda.is_available():
        return {"available": False, "name": "CPU", "capability": None, "total_gb": 0.0}
    props = torch.cuda.get_device_properties(0)
    return {
        "available": True,
        "name": torch.cuda.get_device_name(0),
        "capability": torch.cuda.get_device_capability(0),
        "total_gb": round(props.total_memory / (1024 ** 3), 1),
    }

GPU_INFO = cuda_info()

def cuda_memory_snapshot(label: str) -> Dict[str, Any]:
    snapshot: Dict[str, Any] = {"label": label, "cuda_available": bool(torch.cuda.is_available())}
    if torch.cuda.is_available():
        device = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(device)
        snapshot.update({
            "device": torch.cuda.get_device_name(device),
            "allocated_gb": round(torch.cuda.memory_allocated(device) / (1024 ** 3), 3),
            "reserved_gb": round(torch.cuda.memory_reserved(device) / (1024 ** 3), 3),
            "max_allocated_gb": round(torch.cuda.max_memory_allocated(device) / (1024 ** 3), 3),
            "total_gb": round(props.total_memory / (1024 ** 3), 3),
        })
    print("CUDA memory snapshot:", json.dumps(snapshot, indent=2))
    return snapshot

def move_torch_object_to_cpu(name: str, obj: Any) -> None:
    candidates = []
    model_obj = getattr(obj, "model", None)
    if model_obj is not None:
        candidates.append((f"{name}.model", model_obj))
    candidates.append((name, obj))
    for candidate_name, candidate in candidates:
        if hasattr(candidate, "to"):
            try:
                candidate.to("cpu")
            except Exception as exc:
                print(f"WARNING: could not move {candidate_name} to CPU before cleanup: {exc}")

    optimizer = getattr(obj, "optimizer", None)
    state = getattr(optimizer, "state", None)
    if isinstance(state, dict):
        for state_values in state.values():
            if not isinstance(state_values, dict):
                continue
            for key, value in list(state_values.items()):
                if torch.is_tensor(value):
                    try:
                        state_values[key] = value.detach().cpu()
                    except Exception as exc:
                        print(f"WARNING: could not move optimizer state for {name}.{key} to CPU: {exc}")

def cleanup_runtime_objects(names: Iterable[str], label: str) -> List[str]:
    released = []
    for name in names:
        if name in globals():
            obj = globals().get(name)
            move_torch_object_to_cpu(name, obj)
            globals().pop(name, None)
            released.append(name)
    obj = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass
    if released:
        print(f"Released {len(released)} objects for {label}:", ", ".join(released))
    else:
        print(f"No runtime objects released for {label}.")
    cuda_memory_snapshot(label)
    return released

PROFILE_CONFIGS = {
    # Fast correctness run. Use this to validate cells, schemas, export, upload, and ONNX parity.
    "debug_smoke": {
        "max_per_source": 500,
        "max_length": 512,
        "epochs": 2,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "grad_accum": 1,
        "learning_rate": 2e-5,
        "warmup_ratio": 0.06,
        "early_stopping_patience": 1,
        "max_tools_in_prompt": 16,
        "dataloader_num_workers": 2,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Cheaper T4 iteration. Use for notebook/debug cycles, not final artifacts.
    "t4_fast": {
        "max_per_source": 3_000,
        "max_length": 640,
        "epochs": 3,
        "train_batch_size": 12,
        "eval_batch_size": 24,
        "grad_accum": 3,
        "learning_rate": 1.2e-5,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 1,
        "max_tools_in_prompt": 20,
        "dataloader_num_workers": 2,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Proven T4 baseline — updated for candidate-first serializer experiment.
    # max_length raised 768->1024 to cover p95 token lengths from truncation diagnostics.
    # max_per_source reduced 5000->4000 and grad_accum raised 4->6 to stay within T4 VRAM budget.
    # Pre-update baseline: epoch 4 validation macro_f1=0.741871, accuracy=0.839146, ~1h25m on T4.
    "t4_proven": {
        "max_per_source": 4_000,
        "max_length": 1024,
        "epochs": 5,
        "train_batch_size": 8,
        "eval_batch_size": 16,
        "grad_accum": 6,
        "learning_rate": 1e-5,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 24,
        "dataloader_num_workers": 2,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Good profile for Colab L4/A100-40GB style runtimes.
    "l4_balanced": {
        "max_per_source": 10_000,
        "max_length": 1024,
        "epochs": 4,
        "train_batch_size": 24,
        "eval_batch_size": 48,
        "grad_accum": 1,
        "learning_rate": 9e-6,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 32,
        "dataloader_num_workers": 4,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Higher memory profile for A100 40GB. Keeps p95-ish context and larger batches.
    "a100_40gb": {
        "max_per_source": 15_000,
        "max_length": 1024,
        "epochs": 5,
        "train_batch_size": 32,
        "eval_batch_size": 64,
        "grad_accum": 1,
        "learning_rate": 8e-6,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 40,
        "dataloader_num_workers": 6,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # First profile to try on a 80-100GB GPU. Fast enough to iterate while using longer context.
    "high_vram_fast": {
        "max_per_source": 15_000,
        "max_length": 1024,
        "epochs": 4,
        "train_batch_size": 64,
        "eval_batch_size": 128,
        "grad_accum": 1,
        "learning_rate": 8e-6,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 40,
        "dataloader_num_workers": 8,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Recommended quality profile for your ~98GB runtime. 1280 covers close to p99 from your token sample.
    "high_vram_quality": {
        "max_per_source": 40_000,
        "max_length": 1280,
        "epochs": 5,
        "train_batch_size": 64,
        "eval_batch_size": 128,
        "grad_accum": 1,
        "learning_rate": 6e-6,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 48,
        "dataloader_num_workers": 8,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },

    # Expensive ablation for very long contexts. Use only after high_vram_quality works.
    "high_vram_full_context": {
        "max_per_source": 40_000,
        "max_length": 1536,
        "epochs": 5,
        "train_batch_size": 48,
        "eval_batch_size": 96,
        "grad_accum": 1,
        "learning_rate": 6e-6,
        "warmup_ratio": 0.08,
        "early_stopping_patience": 2,
        "max_tools_in_prompt": 56,
        "dataloader_num_workers": 8,
        "gradient_checkpointing": False,
        "optimizer": "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    },
}


def choose_auto_profile(info: Dict[str, Any]) -> str:
    if not info["available"]:
        return "debug_smoke"
    name = str(info["name"]).lower()
    gb = float(info["total_gb"])
    if "h100" in name or gb >= 85:
        return "high_vram_quality"
    if gb >= 70:
        return "high_vram_quality"
    if "a100" in name or gb >= 35:
        return "a100_40gb"
    if "l4" in name or gb >= 20:
        return "l4_balanced"
    if "t4" in name or gb >= 14:
        return "t4_proven"
    return "debug_smoke"

SELECTED_PROFILE = choose_auto_profile(GPU_INFO) if GPU_PROFILE == "auto" else GPU_PROFILE
if SELECTED_PROFILE not in PROFILE_CONFIGS:
    raise ValueError(f"Unknown GPU_PROFILE={GPU_PROFILE!r}. Valid: {sorted(PROFILE_CONFIGS)}")

CFG = dict(PROFILE_CONFIGS[SELECTED_PROFILE])

# Apply manual overrides last.
if MAX_PER_SOURCE_OVERRIDE > 0:
    CFG["max_per_source"] = int(MAX_PER_SOURCE_OVERRIDE)
if MAX_LENGTH_OVERRIDE > 0:
    CFG["max_length"] = int(MAX_LENGTH_OVERRIDE)
if NUM_EPOCHS_OVERRIDE > 0:
    CFG["epochs"] = int(NUM_EPOCHS_OVERRIDE)
if TRAIN_BATCH_SIZE_OVERRIDE > 0:
    CFG["train_batch_size"] = int(TRAIN_BATCH_SIZE_OVERRIDE)
if EVAL_BATCH_SIZE_OVERRIDE > 0:
    CFG["eval_batch_size"] = int(EVAL_BATCH_SIZE_OVERRIDE)
if GRAD_ACCUM_OVERRIDE > 0:
    CFG["grad_accum"] = int(GRAD_ACCUM_OVERRIDE)
if LEARNING_RATE_OVERRIDE > 0:
    CFG["learning_rate"] = float(LEARNING_RATE_OVERRIDE)

RUN_PROFILE = SELECTED_PROFILE  # kept for artifact manifests/backward compatibility
MAX_PER_SOURCE = CFG["max_per_source"]
MAX_LENGTH = CFG["max_length"]
NUM_EPOCHS = CFG["epochs"]
TRAIN_BATCH_SIZE = CFG["train_batch_size"]
EVAL_BATCH_SIZE = CFG["eval_batch_size"]
GRADIENT_ACCUMULATION_STEPS = CFG["grad_accum"]
LEARNING_RATE = CFG["learning_rate"]
WARMUP_RATIO = CFG["warmup_ratio"]
EARLY_STOPPING_PATIENCE = CFG["early_stopping_patience"]
MAX_TOOLS_IN_PROMPT = CFG["max_tools_in_prompt"]
DATALOADER_NUM_WORKERS = CFG["dataloader_num_workers"]
GRADIENT_CHECKPOINTING = CFG["gradient_checkpointing"]
OPTIMIZER = CFG["optimizer"]

AUTO_FIND_BATCH_SIZE = True
GROUP_BY_LENGTH = True
MAX_PER_LABEL = 50_000
VALID_SIZE = 0.10
TEST_SIZE = 0.10

print("Detected GPU:", json.dumps(GPU_INFO, indent=2))
print("Requested GPU_PROFILE:", GPU_PROFILE)
print("Selected RUN_PROFILE:", RUN_PROFILE)
print("Effective profile config:")
print(json.dumps(CFG, indent=2))
print("Effective train batch:", TRAIN_BATCH_SIZE * max(1, torch.cuda.device_count()), "x grad_accum", GRADIENT_ACCUMULATION_STEPS)
print("T4 proven baseline:", json.dumps(T4_PROVEN_BASELINE, indent=2))
print("Forge augmentation enabled:", ENABLE_FORGE_AUGMENTATION)
print("Final-response verifier enabled:", ENABLE_FINAL_RESPONSE_VERIFIER)
print("Final-response balance labels:", FINAL_RESPONSE_BALANCE_LABELS, "min per label:", FINAL_RESPONSE_MIN_PER_LABEL)
print("Final-response force retrain:", FINAL_RESPONSE_FORCE_RETRAIN)
print("Final-response CPU-only export:", FINAL_RESPONSE_EXPORT_CPU_ONLY)
print("Private agent log rows enabled:", INCLUDE_PRIVATE_AGENT_LOGS)
print("Hugging Face private upload:", PRIVATE)
cuda_memory_snapshot("after profile setup")

In [ ]:
#@title Private Forge agent-training dataset controls
# This private Hub dataset is generated by forge-dataset/generatetd and is tailored to Forge's
# tool-call verifier schema. Keep private uploads enabled when using it.
# Current default is the clean forge-eval 3k v2 addendum; root agent_training.notebook.jsonl
# remains the older OpenRouter/generatetd dataset.
# Keep public function-calling coverage as the backbone; private rows should tune Forge slices, not dominate them.
ENABLE_FORGE_AGENT_HF_DATASET = True  #@param {type:"boolean"}
FORGE_AGENT_HF_DATASET_REPO = "cowWhySo/forge-toolcall-verifier-openrouter-2650-v1"  #@param {type:"string"}
FORGE_AGENT_HF_DATASET_FILE = "addenda/forge-eval-3k-v2/agent_training.notebook.jsonl"  #@param {type:"string"}
FORGE_AGENT_HF_DATASET_REVISION = "01eedcb861324df5fe5b6584ed4f12995b103d0f"  #@param {type:"string"}
FORGE_AGENT_HF_DATASET_WEIGHT = 1  #@param {type:"integer"}
FORGE_AGENT_HF_TRAIN_FRACTION_TARGET = 0.25  #@param {type:"number"}
FORGE_AGENT_HF_PUBLIC_ONLY_TRAIN_CAP = 0  #@param {type:"integer"}
FORGE_AGENT_HF_DOWNSAMPLE_PUBLIC_FOR_TARGET = False  #@param {type:"boolean"}
PREFER_FORGE_AGENT_HF_DATASET = True  #@param {type:"boolean"}

if FORGE_AGENT_HF_DATASET_WEIGHT < 1:
    raise ValueError("FORGE_AGENT_HF_DATASET_WEIGHT must be >= 1")
if not 0.0 <= FORGE_AGENT_HF_TRAIN_FRACTION_TARGET < 1.0:
    raise ValueError("FORGE_AGENT_HF_TRAIN_FRACTION_TARGET must be in [0.0, 1.0)")
if FORGE_AGENT_HF_PUBLIC_ONLY_TRAIN_CAP < 0:
    raise ValueError("FORGE_AGENT_HF_PUBLIC_ONLY_TRAIN_CAP must be >= 0")
if ENABLE_FORGE_AGENT_HF_DATASET:
    PRIVATE = True
    print("Private Forge agent dataset enabled:", FORGE_AGENT_HF_DATASET_REPO)
    print("Dataset file:", FORGE_AGENT_HF_DATASET_FILE)
    print("Dataset revision:", FORGE_AGENT_HF_DATASET_REVISION or "default")
    print("Training weight:", FORGE_AGENT_HF_DATASET_WEIGHT)
    print("Target private HF train fraction for covered labels:", FORGE_AGENT_HF_TRAIN_FRACTION_TARGET)
    print("Public-only train cap per label:", FORGE_AGENT_HF_PUBLIC_ONLY_TRAIN_CAP)
    print("Downsample public rows to hit private target:", FORGE_AGENT_HF_DOWNSAMPLE_PUBLIC_FOR_TARGET)
    print("Prefer during class caps:", PREFER_FORGE_AGENT_HF_DATASET)



In [ ]:
#@title GPU capability detection and Hugging Face auth

# Reuse GPU_INFO from the profile selector cell.
def cuda_capability():
    if not torch.cuda.is_available():
        return None
    return torch.cuda.get_device_capability(0)

def is_ampere_or_newer() -> bool:
    cap = cuda_capability()
    return cap is not None and cap[0] >= 8

USE_CUDA = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if USE_CUDA else "CPU"
CAP = cuda_capability()

# T4 is compute capability 7.5: it supports fp16, but not true bf16 or tf32.
# Ampere+ supports tf32 and usually bf16. Use capability gates in addition to PyTorch feature flags.
USE_TF32 = bool(USE_CUDA and is_ampere_or_newer())
USE_BF16 = bool(USE_CUDA and is_ampere_or_newer() and torch.cuda.is_bf16_supported())
USE_FP16 = bool(USE_CUDA and not USE_BF16)

if USE_CUDA:
    torch.backends.cuda.matmul.allow_tf32 = USE_TF32
    torch.backends.cudnn.allow_tf32 = USE_TF32
    # Faster matmul path on Ampere/Hopper where available. Safe no-op on older runtimes.
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("Device:", GPU_NAME)
print("CUDA capability:", CAP)
print("GPU total VRAM GB:", GPU_INFO.get("total_gb"))
print("Precision flags:", {"fp16": USE_FP16, "bf16": USE_BF16, "tf32": USE_TF32})
print("Training profile:", RUN_PROFILE)

if USE_CUDA:
    try:
        !nvidia-smi
    except Exception:
        pass

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Authenticated to Hugging Face.")
else:
    print("No HF_TOKEN found. Public dataset loading works, but upload will fail unless you set HF_TOKEN.")


## 1. Label schema

Production mode collapses deterministic failures into `deterministic_invalid`. The Rust guardrails should still own deterministic blocks. The classifier should mainly help with semantic ambiguity, nudge specificity, and ranking.

In [ ]:
RAW_LABELS = [
    "valid",
    "wrong_tool_semantic",
    "wrong_arguments_semantic",
    "invalid_args_schema",
    "missing_required_args",
    "unknown_tool",
    "premature_terminal",
    "missing_prerequisite",
    "unsafe_parallel_batch",
    "tool_not_needed",
    "needs_clarification",
    "malformed_tool_call",
]

SEMANTIC_LABEL_MAP = {
    "valid": "valid",
    "wrong_tool_semantic": "wrong_tool_semantic",
    "wrong_arguments_semantic": "wrong_arguments_semantic",
    "tool_not_needed": "tool_not_needed",
    "needs_clarification": "needs_clarification",

    # Deterministic Rust guardrail failures. Keep them as a single non-authoritative bucket.
    "invalid_args_schema": "deterministic_invalid",
    "missing_required_args": "deterministic_invalid",
    "unknown_tool": "deterministic_invalid",
    "premature_terminal": "deterministic_invalid",
    "missing_prerequisite": "deterministic_invalid",
    "unsafe_parallel_batch": "deterministic_invalid",
    "malformed_tool_call": "deterministic_invalid",
}

FINAL_RESPONSE_LABELS = [
    "valid_final_response",
    "missing_tool_fact",
    "contradicts_tool_result",
    "unsupported_claim",
    "failed_to_acknowledge_data_gap",
]
final_response_label2id = {label: i for i, label in enumerate(FINAL_RESPONSE_LABELS)}
final_response_id2label = {i: label for label, i in final_response_label2id.items()}

def normalize_label(label: str) -> str:
    if LABEL_MODE == "production":
        return SEMANTIC_LABEL_MAP.get(label, label)
    return label

if LABEL_MODE == "production":
    LABELS = [
        "valid",
        "wrong_tool_semantic",
        "wrong_arguments_semantic",
        "tool_not_needed",
        "needs_clarification",
        "deterministic_invalid",
    ]
else:
    LABELS = RAW_LABELS

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

labels_doc = {
    "label_mode": LABEL_MODE,
    "raw_labels": RAW_LABELS,
    "labels": LABELS,
    "semantic_label_map": SEMANTIC_LABEL_MAP,
    "label2id": label2id,
    "id2label": id2label,
    "final_response_labels": FINAL_RESPONSE_LABELS,
    "final_response_label2id": final_response_label2id,
    "final_response_id2label": final_response_id2label,
}
(DATA_DIR / "labels.json").write_text(json.dumps(labels_doc, indent=2))
labels_doc


## 2. Canonical input schema and serializer

The Rust side should serialize the same structured context into the same text format before tokenization. Do not train on one format and infer with another.

In [ ]:
BASE_WORKFLOW_STATE_SCHEMA = {
    "type": "object",
    "required": ["required_steps", "completed_steps", "pending_steps", "terminal_tools"],
    "properties": {
        "required_steps": {"type": "array", "items": {"type": "string"}},
        "completed_steps": {"type": "array", "items": {"type": "string"}},
        "pending_steps": {"type": "array", "items": {"type": "string"}},
        "terminal_tools": {"type": "array", "items": {"type": "string"}},
        "recent_errors": {"type": "array", "items": {"type": "string"}},
    },
    "additionalProperties": False,
}

SCORING_METADATA_SCHEMA = {
    "type": "object",
    "properties": {
        "scenario_family": {"type": ["string", "null"]},
        "requires_transform": {"type": ["boolean", "null"]},
        "requires_synthesis": {"type": ["boolean", "null"]},
        "requires_all_tool_facts": {"type": ["boolean", "null"]},
        "must_acknowledge_missing_data": {"type": ["boolean", "null"]},
    },
    "additionalProperties": False,
}

BASE_TOOLCALL_INPUT_PROPERTIES = {
    "schema_version": {"type": "string"},
    "user_request": {"type": "string"},
    "workflow_state": BASE_WORKFLOW_STATE_SCHEMA,
    "available_tools": {
        "type": "array",
        "items": {
            "type": "object",
            "required": ["name", "description", "parameters"],
            "properties": {
                "name": {"type": "string"},
                "description": {"type": "string"},
                "parameters": {"type": "object"},
            },
            "additionalProperties": True,
        },
    },
    "candidate_call": {
        "oneOf": [
            {
                "type": "object",
                "required": ["name", "arguments"],
                "properties": {
                    "name": {"type": "string"},
                    "arguments": {"type": "object"},
                },
                "additionalProperties": True,
            },
            {"type": "array"},
            {"type": "object"},
        ]
    },
}

INPUT_SCHEMA_V1 = {
    "$id": TOOLCALL_INPUT_SCHEMA_VERSION_V1,
    "type": "object",
    "required": ["schema_version", "user_request", "workflow_state", "available_tools", "candidate_call"],
    "properties": dict(BASE_TOOLCALL_INPUT_PROPERTIES, schema_version={"const": TOOLCALL_INPUT_SCHEMA_VERSION_V1}),
    "additionalProperties": False,
}

INPUT_SCHEMA_V2 = {
    "$id": TOOLCALL_INPUT_SCHEMA_VERSION_V2,
    "type": "object",
    "required": ["schema_version", "user_request", "workflow_state", "available_tools", "candidate_call"],
    "properties": dict(
        BASE_TOOLCALL_INPUT_PROPERTIES,
        schema_version={"const": TOOLCALL_INPUT_SCHEMA_VERSION_V2},
        metadata=SCORING_METADATA_SCHEMA,
    ),
    "additionalProperties": False,
}

INPUT_SCHEMA = INPUT_SCHEMA_V2 if USE_SERIALIZER_V2 else INPUT_SCHEMA_V1

FINAL_RESPONSE_INPUT_SCHEMA = {
    "$id": FINAL_RESPONSE_INPUT_SCHEMA_VERSION,
    "type": "object",
    "required": [
        "schema_version", "user_request", "workflow_state", "required_facts",
        "tool_trace", "tool_results", "candidate_final_response",
    ],
    "properties": {
        "schema_version": {"const": FINAL_RESPONSE_INPUT_SCHEMA_VERSION},
        "user_request": {"type": "string"},
        "workflow_state": BASE_WORKFLOW_STATE_SCHEMA,
        "required_facts": {"type": "array", "items": {"type": "string"}},
        "tool_trace": {"type": "array", "items": {"type": "string"}},
        "tool_results": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["tool_name", "content"],
                "properties": {
                    "tool_name": {"type": "string"},
                    "content": {"type": "string"},
                },
                "additionalProperties": True,
            },
        },
        "candidate_final_response": {"type": "string"},
        "metadata": SCORING_METADATA_SCHEMA,
    },
    "additionalProperties": False,
}

(DATA_DIR / "input_schema.json").write_text(json.dumps(INPUT_SCHEMA, indent=2))
(DATA_DIR / "input_schema_v1.json").write_text(json.dumps(INPUT_SCHEMA_V1, indent=2))
(DATA_DIR / "input_schema_v2.json").write_text(json.dumps(INPUT_SCHEMA_V2, indent=2))
(DATA_DIR / "final_response_input_schema.json").write_text(json.dumps(FINAL_RESPONSE_INPUT_SCHEMA, indent=2))
INPUT_SCHEMA


In [ ]:
@dataclass
class VerifierRow:
    id: str
    source: str
    input_text: str
    label: str
    rank_score: float
    metadata: Dict[str, Any]

@dataclass
class FinalResponseVerifierRow:
    id: str
    source: str
    input_text: str
    label: str
    rank_score: float
    metadata: Dict[str, Any]

def stable_id(*parts: Any) -> str:
    h = hashlib.sha256(json.dumps(parts, sort_keys=True, default=str).encode()).hexdigest()[:16]
    return h

def compact_json(obj: Any, max_chars: int = 2500) -> str:
    try:
        s = json.dumps(obj, ensure_ascii=False, sort_keys=True)
    except TypeError:
        s = str(obj)
    if len(s) > max_chars:
        return s[:max_chars] + "...<truncated>"
    return s

def stable_toolset_hash(tools: List[Dict[str, Any]]) -> str:
    names = sorted([str(t.get("name", "")) for t in tools])
    return stable_id(names)

def normalize_tool_for_prompt(tool: Dict[str, Any]) -> Dict[str, Any]:
    if tool.get("type") == "function" and isinstance(tool.get("function"), dict):
        f = tool["function"]
        return {
            "name": str(f.get("name", "unknown_tool")),
            "description": str(f.get("description", "")),
            "parameters": f.get("parameters") or {"type": "object", "properties": {}},
        }
    return {
        "name": str(tool.get("name", tool.get("tool_name", "unknown_tool"))),
        "description": str(tool.get("description", tool.get("desc", ""))),
        "parameters": tool.get("parameters") or {"type": "object", "properties": {}},
    }

def candidate_tool_names(candidate: Any) -> List[str]:
    if isinstance(candidate, list):
        out = []
        for item in candidate:
            out.extend(candidate_tool_names(item))
        return out
    if isinstance(candidate, dict):
        if "name" in candidate:
            return [str(candidate["name"])]
        if isinstance(candidate.get("function"), dict) and "name" in candidate["function"]:
            return [str(candidate["function"]["name"])]
        if isinstance(candidate.get("tool"), dict):
            return candidate_tool_names(candidate["tool"])
    return []

def select_relevant_tools(
    tools: List[Dict[str, Any]],
    candidate: Any,
    pending_steps: Optional[List[str]] = None,
    terminal_tools: Optional[List[str]] = None,
    max_tools: int = MAX_TOOLS_IN_PROMPT,
) -> List[Dict[str, Any]]:
    normalized = [normalize_tool_for_prompt(t) for t in tools]
    by_name = {t.get("name"): t for t in normalized}
    priority_names = []
    priority_names.extend(candidate_tool_names(candidate))
    priority_names.extend(pending_steps or [])
    priority_names.extend(terminal_tools or [])

    selected = []
    seen = set()
    for name in priority_names:
        if name in by_name and name not in seen:
            selected.append(by_name[name])
            seen.add(name)
    for t in normalized:
        name = t.get("name")
        if name not in seen:
            selected.append(t)
            seen.add(name)
        if len(selected) >= max_tools:
            break
    return selected[:max_tools]

def serialize_tool(tool: Dict[str, Any]) -> str:
    t = normalize_tool_for_prompt(tool)
    return f"{t['name']}: {t['description']}\nPARAMETERS: {compact_json(t['parameters'], 1200)}"

def infer_scoring_metadata(scenario_family: Optional[str] = None, **overrides: Any) -> Dict[str, Any]:
    family = (scenario_family or "").replace("_stateful", "")
    metadata = {
        "scenario_family": scenario_family,
        "requires_transform": family in {"argument_transformation", "inconsistent_api_recovery"},
        "requires_synthesis": family in {"grounded_synthesis", "data_gap_recovery", "data_gap_recovery_extended"},
        "requires_all_tool_facts": family in {"grounded_synthesis", "data_gap_recovery", "data_gap_recovery_extended", "argument_transformation"},
        "must_acknowledge_missing_data": family in {"data_gap_recovery", "data_gap_recovery_extended"},
    }
    metadata.update({k: v for k, v in overrides.items() if v is not None})
    return metadata

def build_input_object(
    user_request: str,
    tools: List[Dict[str, Any]],
    candidate: Any,
    required_steps: Optional[List[str]] = None,
    completed_steps: Optional[List[str]] = None,
    pending_steps: Optional[List[str]] = None,
    terminal_tools: Optional[List[str]] = None,
    recent_errors: Optional[List[str]] = None,
    scoring_metadata: Optional[Dict[str, Any]] = None,
    schema_version: Optional[str] = None,
) -> Dict[str, Any]:
    schema_version = schema_version or INPUT_SCHEMA_VERSION
    relevant_tools = select_relevant_tools(tools, candidate, pending_steps, terminal_tools)
    obj = {
        "schema_version": schema_version,
        "user_request": user_request,
        "workflow_state": {
            "required_steps": required_steps or [],
            "completed_steps": completed_steps or [],
            "pending_steps": pending_steps or [],
            "terminal_tools": terminal_tools or [],
            "recent_errors": recent_errors or [],
        },
        "available_tools": relevant_tools,
        "candidate_call": candidate,
    }
    if schema_version == TOOLCALL_INPUT_SCHEMA_VERSION_V2:
        obj["metadata"] = scoring_metadata or infer_scoring_metadata(None)
    return obj

def serialize_state_v1(input_obj: Dict[str, Any]) -> str:
    ws = input_obj["workflow_state"]
    tool_text = "\n\n".join(serialize_tool(t) for t in input_obj["available_tools"])
    return f"""SCHEMA_VERSION:
{input_obj['schema_version']}

USER_REQUEST:
{input_obj['user_request']}

WORKFLOW_STATE:
required_steps={ws.get('required_steps', [])}
completed_steps={ws.get('completed_steps', [])}
pending_steps={ws.get('pending_steps', [])}
terminal_tools={ws.get('terminal_tools', [])}
recent_errors={ws.get('recent_errors', [])}

AVAILABLE_TOOLS:
{tool_text}

CANDIDATE_CALL:
{compact_json(input_obj['candidate_call'], 2400)}
""".strip()

def _json_or_null(value: Any) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "true" if value else "false"
    return json.dumps(value, ensure_ascii=False)

def serialize_state_v2(input_obj: Dict[str, Any]) -> str:
    metadata = input_obj.get("metadata") or {}
    base = serialize_state_v1(input_obj)
    return base + f"""

SCORING_METADATA:
scenario_family={_json_or_null(metadata.get('scenario_family'))}
requires_transform={_json_or_null(metadata.get('requires_transform'))}
requires_synthesis={_json_or_null(metadata.get('requires_synthesis'))}
requires_all_tool_facts={_json_or_null(metadata.get('requires_all_tool_facts'))}
must_acknowledge_missing_data={_json_or_null(metadata.get('must_acknowledge_missing_data'))}"""


def serialize_candidate_tool_schema(
    tools: List[Dict[str, Any]],
    candidate: Any,
) -> str:
    """Return the full parameter schema for the candidate tool only."""
    names = set(candidate_tool_names(candidate))
    for t in tools:
        normalized = normalize_tool_for_prompt(t)
        if normalized.get("name") in names:
            return f"{normalized['name']}: {normalized['description']}\nPARAMETERS: {compact_json(normalized['parameters'], 2400)}"
    return ""


def serialize_competing_tool_signatures(
    tools: List[Dict[str, Any]],
    candidate: Any,
    max_competing: int = 12,
) -> str:
    """Return name + description only (no parameters) for non-candidate tools."""
    names = set(candidate_tool_names(candidate))
    lines = []
    for t in tools:
        normalized = normalize_tool_for_prompt(t)
        n = normalized.get("name", "")
        if n not in names:
            lines.append(f"{n}: {normalized.get('description', '')}")
        if len(lines) >= max_competing:
            break
    return "\n".join(lines)


SERIALIZE_V3_CHARS_PER_TOKEN = 3.5  # SentencePiece on JSON-ish text runs ~3-4 chars/token.


def serialize_state_v3(input_obj: Dict[str, Any]) -> str:
    """Candidate-first layout: candidate call and its schema appear before the tool list.
    OTHER_AVAILABLE_TOOLS (the lowest-value section) is char-budget trimmed so the
    high-value front sections plus SCORING_METADATA fit within MAX_LENGTH at
    ~SERIALIZE_V3_CHARS_PER_TOKEN chars/token. The cell 25 hard guard backstops
    an optimistic estimate.
    NOTE: v1 and v2 remain byte-stable; this is a new layout, not a replacement."""
    ws = input_obj["workflow_state"]
    metadata = input_obj.get("metadata") or {}
    candidate_schema = serialize_candidate_tool_schema(
        input_obj["available_tools"], input_obj["candidate_call"]
    )
    competing_sigs = serialize_competing_tool_signatures(
        input_obj["available_tools"], input_obj["candidate_call"]
    )
    front = f"""SCHEMA_VERSION:
{input_obj['schema_version']}

USER_REQUEST:
{input_obj['user_request']}

CANDIDATE_CALL:
{compact_json(input_obj['candidate_call'], 2400)}

CANDIDATE_TOOL_SCHEMA:
{candidate_schema}

WORKFLOW_STATE:
required_steps={ws.get('required_steps', [])}
completed_steps={ws.get('completed_steps', [])}
pending_steps={ws.get('pending_steps', [])}
terminal_tools={ws.get('terminal_tools', [])}
recent_errors={ws.get('recent_errors', [])}"""
    tail = f"""SCORING_METADATA:
scenario_family={_json_or_null(metadata.get('scenario_family'))}
requires_transform={_json_or_null(metadata.get('requires_transform'))}
requires_synthesis={_json_or_null(metadata.get('requires_synthesis'))}
requires_all_tool_facts={_json_or_null(metadata.get('requires_all_tool_facts'))}
must_acknowledge_missing_data={_json_or_null(metadata.get('must_acknowledge_missing_data'))}"""
    other_header = "\n\nOTHER_AVAILABLE_TOOLS:\n"
    char_budget = int(float(globals().get("MAX_LENGTH", 1024)) * SERIALIZE_V3_CHARS_PER_TOKEN)
    remaining = char_budget - len(front) - len(tail) - len(other_header) - 2
    sig_lines = [line for line in competing_sigs.split("\n") if line]
    kept_lines = []
    used = 0
    for line in sig_lines:
        cost = len(line) + 1
        if used + cost > remaining:
            break
        kept_lines.append(line)
        used += cost
    dropped = len(sig_lines) - len(kept_lines)
    if dropped > 0:
        kept_lines.append(f"...<+{dropped} more tools>")
    competing_block = "\n".join(kept_lines)
    return f"{front}{other_header}{competing_block}\n\n{tail}".strip()


def serialize_state_from_object(input_obj: Dict[str, Any]) -> str:
    if SERIALIZER_VERSION == "serialize_state_v3":
        return serialize_state_v3(input_obj)
    if SERIALIZER_VERSION == "serialize_state_v2":
        return serialize_state_v2(input_obj)
    return serialize_state_v1(input_obj)

def serialize_state(
    user_request: str,
    tools: List[Dict[str, Any]],
    candidate: Any,
    required_steps: Optional[List[str]] = None,
    completed_steps: Optional[List[str]] = None,
    pending_steps: Optional[List[str]] = None,
    terminal_tools: Optional[List[str]] = None,
    recent_errors: Optional[List[str]] = None,
    scoring_metadata: Optional[Dict[str, Any]] = None,
) -> str:
    return serialize_state_from_object(build_input_object(
        user_request=user_request,
        tools=tools,
        candidate=candidate,
        required_steps=required_steps,
        completed_steps=completed_steps,
        pending_steps=pending_steps,
        terminal_tools=terminal_tools,
        recent_errors=recent_errors,
        scoring_metadata=scoring_metadata,
    ))

def serialize_final_response_state_v1(input_obj: Dict[str, Any]) -> str:
    ws = input_obj["workflow_state"]
    metadata = input_obj.get("metadata") or {}
    tool_results = "\n".join(
        f"{r.get('tool_name', '')}: {json.dumps(str(r.get('content', '')), ensure_ascii=False)}"
        for r in input_obj.get("tool_results", [])
    )
    return f"""SCHEMA_VERSION:
{input_obj['schema_version']}

USER_REQUEST:
{input_obj['user_request']}

WORKFLOW_STATE:
required_steps={ws.get('required_steps', [])}
completed_steps={ws.get('completed_steps', [])}
pending_steps={ws.get('pending_steps', [])}
terminal_tools={ws.get('terminal_tools', [])}
recent_errors={ws.get('recent_errors', [])}

REQUIRED_FACTS:
{input_obj.get('required_facts', [])}

TOOL_TRACE:
{input_obj.get('tool_trace', [])}

TOOL_RESULTS:
{tool_results}

CANDIDATE_FINAL_RESPONSE:
{input_obj.get('candidate_final_response', '')}

SCORING_METADATA:
scenario_family={_json_or_null(metadata.get('scenario_family'))}
requires_transform={_json_or_null(metadata.get('requires_transform'))}
requires_synthesis={_json_or_null(metadata.get('requires_synthesis'))}
requires_all_tool_facts={_json_or_null(metadata.get('requires_all_tool_facts'))}
must_acknowledge_missing_data={_json_or_null(metadata.get('must_acknowledge_missing_data'))}""".strip()

def make_row(
    source: str,
    label: str,
    user_request: str,
    tools: List[Dict[str, Any]],
    candidate: Any,
    rank_score: float,
    metadata: Optional[Dict[str, Any]] = None,
    required_steps: Optional[List[str]] = None,
    completed_steps: Optional[List[str]] = None,
    pending_steps: Optional[List[str]] = None,
    terminal_tools: Optional[List[str]] = None,
    recent_errors: Optional[List[str]] = None,
    group_id: Optional[str] = None,
    scoring_metadata: Optional[Dict[str, Any]] = None,
) -> VerifierRow:
    metadata = dict(metadata or {})
    scoring_metadata = scoring_metadata or metadata.get("scoring_metadata") or infer_scoring_metadata(metadata.get("scenario_family"))
    group_id = group_id or metadata.get("example_group_id") or stable_id(source, user_request, tools)
    metadata["example_group_id"] = group_id
    metadata["toolset_hash"] = stable_toolset_hash(tools)
    metadata["input_schema_version"] = INPUT_SCHEMA_VERSION
    metadata["serializer_version"] = SERIALIZER_VERSION
    metadata["scoring_metadata"] = scoring_metadata

    input_obj = build_input_object(
        user_request=user_request,
        tools=tools,
        candidate=candidate,
        required_steps=required_steps,
        completed_steps=completed_steps,
        pending_steps=pending_steps,
        terminal_tools=terminal_tools,
        recent_errors=recent_errors,
        scoring_metadata=scoring_metadata,
    )
    input_text = serialize_state_from_object(input_obj)
    rid = stable_id(source, label, user_request, candidate, group_id, required_steps, completed_steps, pending_steps, terminal_tools)
    return VerifierRow(rid, source, input_text, label, float(rank_score), metadata)

def make_final_response_row(
    source: str,
    label: str,
    user_request: str,
    candidate_final_response: str,
    rank_score: float,
    workflow_state: Optional[Dict[str, Any]] = None,
    required_facts: Optional[List[str]] = None,
    tool_trace: Optional[List[str]] = None,
    tool_results: Optional[List[Dict[str, str]]] = None,
    metadata: Optional[Dict[str, Any]] = None,
    group_id: Optional[str] = None,
) -> FinalResponseVerifierRow:
    metadata = dict(metadata or {})
    scoring_metadata = metadata.get("scoring_metadata") or infer_scoring_metadata(metadata.get("scenario_family"), requires_synthesis=True)
    group_id = group_id or metadata.get("example_group_id") or stable_id(source, user_request, candidate_final_response)
    metadata["example_group_id"] = group_id
    metadata["input_schema_version"] = FINAL_RESPONSE_INPUT_SCHEMA_VERSION
    metadata["serializer_version"] = FINAL_RESPONSE_SERIALIZER_VERSION
    metadata["scoring_metadata"] = scoring_metadata
    input_obj = {
        "schema_version": FINAL_RESPONSE_INPUT_SCHEMA_VERSION,
        "user_request": user_request,
        "workflow_state": workflow_state or {"required_steps": [], "completed_steps": [], "pending_steps": [], "terminal_tools": ["respond"], "recent_errors": []},
        "required_facts": required_facts or [],
        "tool_trace": tool_trace or [],
        "tool_results": tool_results or [],
        "candidate_final_response": candidate_final_response,
        "metadata": scoring_metadata,
    }
    input_text = serialize_final_response_state_v1(input_obj)
    rid = stable_id(source, label, user_request, candidate_final_response, group_id, required_facts, tool_trace)
    return FinalResponseVerifierRow(rid, source, input_text, label, float(rank_score), metadata)

# Save fixed serializer fixtures for Rust and artifact consumers.
SERIALIZER_FIXTURE = build_input_object(
    user_request="Generate a sales report from the Q4 2024 dataset.",
    tools=[
        {"name": "fetch_sales_data", "description": "Fetch sales data.", "parameters": {"type": "object", "properties": {"quarter": {"type": "integer"}, "year": {"type": "integer"}}, "required": ["quarter", "year"]}},
        {"name": "analyze_sales", "description": "Analyze sales.", "parameters": {"type": "object", "properties": {}}},
        {"name": "report", "description": "Produce final report.", "parameters": {"type": "object", "properties": {"summary": {"type": "string"}}, "required": ["summary"]}},
    ],
    candidate={"name": "report", "arguments": {"summary": "Done."}},
    required_steps=["fetch_sales_data", "analyze_sales"],
    completed_steps=[],
    pending_steps=["fetch_sales_data", "analyze_sales"],
    terminal_tools=["report"],
    schema_version=TOOLCALL_INPUT_SCHEMA_VERSION_V1,
)
(DATA_DIR / "serializer_fixture_v1.json").write_text(json.dumps({
    "input": SERIALIZER_FIXTURE,
    "serialized": serialize_state_v1(SERIALIZER_FIXTURE),
}, indent=2))

SERIALIZER_FIXTURE_V2 = build_input_object(
    user_request=SERIALIZER_FIXTURE["user_request"],
    tools=SERIALIZER_FIXTURE["available_tools"],
    candidate=SERIALIZER_FIXTURE["candidate_call"],
    required_steps=SERIALIZER_FIXTURE["workflow_state"]["required_steps"],
    completed_steps=SERIALIZER_FIXTURE["workflow_state"]["completed_steps"],
    pending_steps=SERIALIZER_FIXTURE["workflow_state"]["pending_steps"],
    terminal_tools=SERIALIZER_FIXTURE["workflow_state"]["terminal_tools"],
    scoring_metadata=infer_scoring_metadata("argument_transformation"),
    schema_version=TOOLCALL_INPUT_SCHEMA_VERSION_V2,
)
(DATA_DIR / "serializer_fixture_v2.json").write_text(json.dumps({
    "input": SERIALIZER_FIXTURE_V2,
    "serialized": serialize_state_v2(SERIALIZER_FIXTURE_V2),
}, indent=2))
SERIALIZER_FIXTURE_V3 = build_input_object(
    user_request=SERIALIZER_FIXTURE["user_request"],
    tools=SERIALIZER_FIXTURE["available_tools"],
    candidate=SERIALIZER_FIXTURE["candidate_call"],
    required_steps=SERIALIZER_FIXTURE["workflow_state"]["required_steps"],
    completed_steps=SERIALIZER_FIXTURE["workflow_state"]["completed_steps"],
    pending_steps=SERIALIZER_FIXTURE["workflow_state"]["pending_steps"],
    terminal_tools=SERIALIZER_FIXTURE["workflow_state"]["terminal_tools"],
    scoring_metadata=infer_scoring_metadata("argument_transformation"),
    schema_version=TOOLCALL_INPUT_SCHEMA_VERSION_V2,
)
(DATA_DIR / "serializer_fixture_v3.json").write_text(json.dumps({
    "input": SERIALIZER_FIXTURE_V3,
    "serialized": serialize_state_v3(SERIALIZER_FIXTURE_V3),
}, indent=2))

if SERIALIZER_VERSION == "serialize_state_v3":
    ACTIVE_SERIALIZER_FIXTURE = SERIALIZER_FIXTURE_V3
elif USE_SERIALIZER_V2:
    ACTIVE_SERIALIZER_FIXTURE = SERIALIZER_FIXTURE_V2
else:
    ACTIVE_SERIALIZER_FIXTURE = SERIALIZER_FIXTURE
(DATA_DIR / "serializer_fixture.json").write_text(json.dumps({
    "input": ACTIVE_SERIALIZER_FIXTURE,
    "serialized": serialize_state_from_object(ACTIVE_SERIALIZER_FIXTURE),
}, indent=2))
print(serialize_state_from_object(ACTIVE_SERIALIZER_FIXTURE))

## 3. Dataset loaders

These loaders normalize several public function-calling datasets into:

```python
{
  "source": str,
  "user_request": str,
  "tools": list[dict],
  "gold_calls": list[dict],
  "group_id": str
}
```

In [ ]:
def try_json_loads(x: Any) -> Any:
    if isinstance(x, (dict, list)):
        return x
    if x is None:
        return None
    if not isinstance(x, str):
        return x
    s = x.strip()
    if not s:
        return None
    try:
        return json.loads(s)
    except Exception:
        pass
    try:
        return ast.literal_eval(s)
    except Exception:
        return x

def get_first_text(record: Dict[str, Any], keys: List[str]) -> str:
    for k in keys:
        v = record.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

def balanced_json_objects(text: str, start_pos: int = 0) -> List[str]:
    objects = []
    i = start_pos
    while i < len(text):
        if text[i] != "{":
            i += 1
            continue
        start = i
        depth = 0
        in_str = False
        escape = False
        while i < len(text):
            ch = text[i]
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = not in_str
            elif not in_str:
                if ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        objects.append(text[start:i + 1])
                        break
            i += 1
        i += 1
    return objects

def extract_json_after_marker(text: str, marker: str) -> List[Any]:
    out = []
    if not isinstance(text, str) or marker not in text:
        return out
    cursor = 0
    while True:
        idx = text.find(marker, cursor)
        if idx < 0:
            break
        start = idx + len(marker)
        objs = balanced_json_objects(text, start)
        if objs:
            parsed = try_json_loads(objs[0])
            out.append(parsed)
            cursor = text.find(objs[0], start) + len(objs[0])
        else:
            cursor = start
    return out

def extract_first_user_from_chat(chat: Any) -> str:
    chat = try_json_loads(chat)
    if isinstance(chat, list):
        for m in chat:
            if isinstance(m, dict) and m.get("from") in ("user", "human"):
                return str(m.get("value", "")).strip()
    if not isinstance(chat, str):
        return ""
    m = re.search(r"USER:\s*(.*?)(?:\s+ASSISTANT:|<\|endoftext\|>|$)", chat, flags=re.S)
    if m:
        return m.group(1).strip()
    return ""

def extract_messages_text(messages: Any) -> str:
    messages = try_json_loads(messages)
    if isinstance(messages, list):
        parts = []
        for m in messages:
            if not isinstance(m, dict):
                continue
            role = m.get("role", m.get("from", ""))
            content = m.get("content", m.get("value", ""))
            if isinstance(content, list):
                content = " ".join(str(c) for c in content)
            if role in ("user", "human") and content:
                parts.append(str(content))
        return "\n".join(parts).strip()
    return extract_first_user_from_chat(messages)

def normalize_tool_object(t: Any) -> Optional[Dict[str, Any]]:
    t = try_json_loads(t)
    if not isinstance(t, dict):
        return None
    if t.get("type") == "function" and isinstance(t.get("function"), dict):
        f = t["function"]
        return {
            "name": f.get("name", "unknown_tool"),
            "description": f.get("description", ""),
            "parameters": f.get("parameters", {"type": "object", "properties": {}}),
        }
    name = t.get("name") or t.get("tool_name") or t.get("api_name") or t.get("function_name")
    if not name and isinstance(t.get("function"), dict):
        name = t["function"].get("name")
    if not name:
        return None
    params = t.get("parameters") or t.get("parameter") or t.get("args") or t.get("arguments_schema") or t.get("input_schema") or {"type": "object", "properties": {}}
    params = try_json_loads(params)
    if not isinstance(params, dict):
        params = {"type": "object", "properties": {}}
    return {"name": str(name), "description": str(t.get("description") or t.get("desc") or ""), "parameters": params}

def extract_tools_from_text(text: Any) -> List[Dict[str, Any]]:
    if not isinstance(text, str):
        return []
    tools = []
    for obj_text in balanced_json_objects(text):
        obj = try_json_loads(obj_text)
        if isinstance(obj, list):
            for item in obj:
                tool = normalize_tool_object(item)
                if tool:
                    tools.append(tool)
        else:
            tool = normalize_tool_object(obj)
            if tool:
                tools.append(tool)
    seen = set()
    deduped = []
    for t in tools:
        name = t.get("name")
        if name and name not in seen:
            seen.add(name)
            deduped.append(t)
    return deduped

def extract_tools(record: Dict[str, Any]) -> List[Dict[str, Any]]:
    candidate_keys = ["tools", "functions", "function", "tool", "apis", "api_list", "function_list", "available_tools", "function_definition"]
    for k in candidate_keys:
        if k not in record:
            continue
        raw = try_json_loads(record.get(k))
        if isinstance(raw, dict):
            if any(key in raw for key in ["name", "function", "tool_name", "api_name"]):
                maybe = normalize_tool_object(raw)
                return [maybe] if maybe else []
            for nested_key in ["tools", "functions", "apis"]:
                if isinstance(raw.get(nested_key), list):
                    raw = raw[nested_key]
                    break
        if isinstance(raw, list):
            out = []
            for item in raw:
                t = normalize_tool_object(item)
                if t:
                    out.append(t)
            if out:
                return out
    for text_key in ["system", "chat", "conversation", "conversations"]:
        out = extract_tools_from_text(record.get(text_key))
        if out:
            return out
    return []

def normalize_call_object(c: Any) -> Optional[Dict[str, Any]]:
    c = try_json_loads(c)
    if isinstance(c, list) and c:
        return normalize_call_object(c[0])
    if not isinstance(c, dict):
        return None
    if isinstance(c.get("function"), dict):
        f = c["function"]
        args = try_json_loads(f.get("arguments", {}))
        if not isinstance(args, dict):
            args = {"_raw": args}
        return {"name": f.get("name", "unknown_tool"), "arguments": args}
    name = c.get("name") or c.get("tool_name") or c.get("function_name") or c.get("api_name")
    args = c.get("arguments", c.get("args", c.get("parameters", c.get("input", {}))))
    args = try_json_loads(args)
    if not isinstance(args, dict):
        args = {"_raw": args}
    if name:
        return {"name": str(name), "arguments": args}
    if isinstance(c.get("tool"), dict):
        return normalize_call_object(c["tool"])
    return None

def parse_toolace_call_text(value: str) -> Optional[Dict[str, Any]]:
    if not isinstance(value, str):
        return None
    m = re.search(r"\[([A-Za-z0-9_ ./-]+)\((.*?)\)\]", value)
    if not m:
        return None
    name = m.group(1).strip()
    raw_args = m.group(2).strip()
    args = {}
    if raw_args:
        for part in re.split(r",\s*(?=[A-Za-z_][A-Za-z0-9_]*\s*=)", raw_args):
            if "=" not in part:
                continue
            k, v = part.split("=", 1)
            k = k.strip()
            v = v.strip()
            try:
                args[k] = ast.literal_eval(v)
            except Exception:
                args[k] = v.strip('"').strip("'")
    return {"name": name, "arguments": args}

def extract_gold_calls(record: Dict[str, Any]) -> List[Dict[str, Any]]:
    candidate_keys = ["tool_calls", "function_call", "function_calls", "answers", "answer", "assistant", "calls", "api_calls", "gold_calls"]
    calls = []
    for k in candidate_keys:
        if k not in record:
            continue
        raw = try_json_loads(record.get(k))
        if isinstance(raw, list):
            for item in raw:
                call = normalize_call_object(item)
                if call:
                    calls.append(call)
        else:
            call = normalize_call_object(raw)
            if call:
                calls.append(call)
        if calls:
            return calls
    for text_key in ["chat", "conversation", "conversations", "messages"]:
        text = record.get(text_key)
        if isinstance(text, str):
            for marker in ["<functioncall>", "<tool_call>"]:
                for obj in extract_json_after_marker(text, marker):
                    call = normalize_call_object(obj)
                    if call:
                        calls.append(call)
        conv = try_json_loads(text)
        if isinstance(conv, list):
            for m in conv:
                if not isinstance(m, dict):
                    continue
                role = m.get("role", m.get("from", ""))
                content = m.get("content", m.get("value", ""))
                if role in ("assistant", "gpt"):
                    call = normalize_call_object(m.get("tool_calls") or m.get("function_call"))
                    if call:
                        calls.append(call)
                    parsed = parse_toolace_call_text(str(content))
                    if parsed:
                        calls.append(parsed)
    return calls

def normalize_record(record: Dict[str, Any], source: str, index: int) -> Optional[Dict[str, Any]]:
    user_request = (
        get_first_text(record, ["query", "question", "instruction", "user", "prompt", "input"])
        or extract_messages_text(record.get("messages"))
        or extract_messages_text(record.get("conversation"))
        or extract_messages_text(record.get("conversations"))
        or extract_first_user_from_chat(record.get("chat"))
    )
    tools = extract_tools(record)
    gold_calls = extract_gold_calls(record)
    if not tools and gold_calls:
        tools = [
            {"name": c["name"], "description": "Tool parsed from conversation.", "parameters": {"type": "object", "properties": {}}}
            for c in gold_calls if c.get("name")
        ]
    if not user_request or not tools or not gold_calls:
        return None
    return {
        "source": source,
        "user_request": user_request,
        "tools": tools,
        "gold_calls": gold_calls,
        "group_id": stable_id(source, index, user_request, tools, gold_calls),
    }

def load_source_dataset(name: str, split: str = "train", max_rows: int = MAX_PER_SOURCE, **kwargs) -> List[Dict[str, Any]]:
    print(f"Loading {name}...")
    try:
        ds = load_dataset(name, split=split, streaming=False, **kwargs)
    except Exception as e:
        print(f"  load_dataset failed for {name}: {type(e).__name__}: {e}")
        return []
    if max_rows:
        ds = ds.select(range(min(max_rows, len(ds))))
    rows = []
    for i, rec in enumerate(tqdm(ds, desc=f"normalizing {name}")):
        norm = normalize_record(dict(rec), source=name, index=i)
        if norm:
            rows.append(norm)
    print(f"  normalized {len(rows)} usable rows out of {len(ds)}")
    if len(rows) == 0 and len(ds) > 0:
        sample = dict(ds[0])
        print("  No rows parsed. Sample keys:", list(sample.keys()))
        print("  Sample preview:", json.dumps({k: str(v)[:500] for k, v in sample.items()}, indent=2))
    return rows

In [ ]:
#@title Load public function-calling sources
# Some datasets change schema or require auth. Failed datasets are skipped.
PUBLIC_SOURCES = [
    ("glaiveai/glaive-function-calling-v2", {"split": "train"}),
    ("Team-ACE/ToolACE", {"split": "train"}),
    ("Salesforce/xlam-function-calling-60k", {"split": "train"}),
    # BFCL is better used as a separate eval harness from its JSONL files.
]

normalized_examples = []
source_counts = {}
for dataset_name, kwargs in PUBLIC_SOURCES:
    rows = load_source_dataset(dataset_name, max_rows=MAX_PER_SOURCE, **kwargs)
    normalized_examples.extend(rows)
    source_counts[dataset_name] = len(rows)

print("Usable rows by source:")
for k, v in source_counts.items():
    print(f"  {k}: {v}")
print("Total normalized examples:", len(normalized_examples))

if normalized_examples:
    print(json.dumps(normalized_examples[0], indent=2)[:4000])
else:
    print("No public rows parsed. Training will be a smoke test unless Forge traces are added.")

## 4. Forge-specific synthetic and trace rows

Public datasets do not know your workflow contract concepts: required steps, terminal tools, prerequisites, unsafe terminal-plus-work batches, or respond/final tools. Keep these rows modest unless a separate ablation proves they help.

In [ ]:
FORGE_WORKFLOWS = [
    {
        "source": "forge_synthetic",
        "user_request": "Generate a sales report from the Q4 2024 dataset.",
        "tools": [
            {"name": "fetch_sales_data", "description": "Fetch sales data for a given quarter and year.", "parameters": {"type": "object", "properties": {"quarter": {"type": "integer"}, "year": {"type": "integer"}}, "required": ["quarter", "year"]}},
            {"name": "analyze_sales", "description": "Analyze the loaded sales data and produce findings.", "parameters": {"type": "object", "properties": {}}},
            {"name": "report", "description": "Produce the final report from findings.", "parameters": {"type": "object", "properties": {"summary": {"type": "string"}}, "required": ["summary"]}},
        ],
        "required_steps": ["fetch_sales_data", "analyze_sales"],
        "terminal_tools": ["report"],
        "valid_sequence": [
            {"name": "fetch_sales_data", "arguments": {"quarter": 4, "year": 2024}},
            {"name": "analyze_sales", "arguments": {}},
            {"name": "report", "arguments": {"summary": "Revenue grew 23% YoY. Top product: Widget Pro. Weakest region: APAC."}},
        ],
    },
    {
        "source": "forge_synthetic",
        "user_request": "Fetch 10 records and summarize them.",
        "tools": [
            {"name": "fetch", "description": "Fetch a count of records. Count must be a four digit string.", "parameters": {"type": "object", "properties": {"count": {"type": "string"}}, "required": ["count"]}},
            {"name": "summarize", "description": "Summarize fetched records.", "parameters": {"type": "object", "properties": {"summary": {"type": "string"}}, "required": ["summary"]}},
        ],
        "required_steps": ["fetch"],
        "terminal_tools": ["summarize"],
        "valid_sequence": [
            {"name": "fetch", "arguments": {"count": "0010"}},
            {"name": "summarize", "arguments": {"summary": "Fetched 10 records."}},
        ],
    },
]

ARG_TRANSFORMATION_TOOLS = [
    {"name": "list_transactions", "description": "List all expense transactions for a given fiscal quarter and year.", "parameters": {"type": "object", "properties": {"quarter": {"type": "string"}, "year": {"type": "integer"}}, "required": ["quarter", "year"]}},
    {"name": "get_approved_vendors", "description": "Return the canonical list of approved vendor names.", "parameters": {"type": "object", "properties": {}}},
    {"name": "get_vendor_details", "description": "Look up vendor details, status, legal entity, and aliases.", "parameters": {"type": "object", "properties": {"vendor_name": {"type": "string"}}, "required": ["vendor_name"]}},
    {"name": "currency_convert", "description": "Convert an amount between currencies.", "parameters": {"type": "object", "properties": {"amount": {"type": "number"}, "from_currency": {"type": "string"}, "to_currency": {"type": "string"}}, "required": ["amount", "from_currency", "to_currency"]}},
    {"name": "submit_audit_report", "description": "Submit transaction IDs, total USD amount, and largest vendor.", "parameters": {"type": "object", "properties": {"transaction_ids": {"type": "string"}, "total_flagged_usd": {"type": "string"}, "top_vendor": {"type": "string"}}, "required": ["transaction_ids", "total_flagged_usd", "top_vendor"]}},
]
ARG_TRANSFORMATION_USER = "Run our Q4 2024 expense audit. Flag any transaction of $5,000 or more from vendors NOT on our approved list. Submit the audit report with: comma-separated transaction IDs, total flagged amount in USD, and the vendor of the single largest flagged transaction."
ARG_TRANSFORMATION_CANONICAL_CALL = {"name": "submit_audit_report", "arguments": {"transaction_ids": "TX-1001, TX-1005, TX-1006, TX-1008", "total_flagged_usd": "$28,980.00", "top_vendor": "Wonka Industries"}}
ARG_TRANSFORMATION_BAD_CALLS = [
    ("skip_currency_conversion", {"transaction_ids": "TX-1001, TX-1005, TX-1008", "total_flagged_usd": "$23,700", "top_vendor": "Wonka Industries"}),
    ("strict_gt_threshold", {"transaction_ids": "TX-1001, TX-1006, TX-1008", "total_flagged_usd": "$23,980", "top_vendor": "Wonka Industries"}),
    ("vendor_alias_overflag", {"transaction_ids": "TX-1001, TX-1005, TX-1006, TX-1008, TX-1009", "total_flagged_usd": "$35,480", "top_vendor": "Wonka Industries"}),
    ("wrong_total", {"transaction_ids": "TX-1001, TX-1005, TX-1006, TX-1008", "total_flagged_usd": "$76,620.00", "top_vendor": "Wonka Industries"}),
    ("wrong_top_vendor", {"transaction_ids": "TX-1001, TX-1005, TX-1006, TX-1008", "total_flagged_usd": "$28,980.00", "top_vendor": "Cyberdyne LLC"}),
]

def build_forge_synthetic_rows() -> List[VerifierRow]:
    rows = []
    for wf_idx, wf in enumerate(FORGE_WORKFLOWS):
        tools = wf["tools"]
        required = wf["required_steps"]
        terminal = wf["terminal_tools"]
        group_id = stable_id("forge_synthetic", wf_idx, wf["user_request"])
        completed = []
        for step_idx, call in enumerate(wf["valid_sequence"]):
            pending = [s for s in required if s not in completed]
            rows.append(make_row("forge_synthetic", "valid", wf["user_request"], tools, call, 1.0, {"generator": "forge_synthetic", "step_idx": step_idx}, required, completed.copy(), pending, terminal, group_id=group_id))
            if call["name"] in required:
                completed.append(call["name"])
        rows.append(make_row("forge_synthetic", "premature_terminal", wf["user_request"], tools, wf["valid_sequence"][-1], 0.05, {"generator": "forge_synthetic", "negative_type": "premature_terminal"}, required, [], required, terminal, group_id=group_id))
        if len(wf["valid_sequence"]) > 1:
            rows.append(make_row("forge_synthetic", "unsafe_parallel_batch", wf["user_request"], tools, [wf["valid_sequence"][0], wf["valid_sequence"][-1]], 0.02, {"generator": "forge_synthetic", "negative_type": "unsafe_parallel_batch"}, required, [], required, terminal, group_id=group_id))
    return rows

def build_argument_semantic_rows() -> List[VerifierRow]:
    rows = []
    metadata = {"generator": "forge_argument_semantic", "scenario_family": "argument_transformation"}
    group_id = stable_id("forge_argument_semantic", ARG_TRANSFORMATION_USER)
    rows.append(make_row("forge_argument_semantic", "valid", ARG_TRANSFORMATION_USER, ARG_TRANSFORMATION_TOOLS, ARG_TRANSFORMATION_CANONICAL_CALL, 1.0, metadata, ["list_transactions", "get_approved_vendors"], ["list_transactions", "get_approved_vendors"], [], ["submit_audit_report"], group_id=group_id))
    for negative_type, args in ARG_TRANSFORMATION_BAD_CALLS:
        rows.append(make_row("forge_argument_semantic", "wrong_arguments_semantic", ARG_TRANSFORMATION_USER, ARG_TRANSFORMATION_TOOLS, {"name": "submit_audit_report", "arguments": args}, 0.05, dict(metadata, negative_type=negative_type), ["list_transactions", "get_approved_vendors"], ["list_transactions", "get_approved_vendors"], [], ["submit_audit_report"], group_id=group_id))
    return rows


ERROR_RECOVERY_NUMERIC_SEMANTIC_REQUESTS = list(range(1, 61))


def error_recovery_numeric_tools() -> List[Dict[str, Any]]:
    return [
        {
            "name": "fetch",
            "description": "Fetch a count of records. Count must be a zero-padded four-digit string.",
            "parameters": {
                "type": "object",
                "properties": {"count": {"type": "string", "description": "Four digit zero-padded record count."}},
                "required": ["count"],
            },
        },
        {
            "name": "summarize",
            "description": "Summarize fetched records.",
            "parameters": {
                "type": "object",
                "properties": {"content": {"type": "string"}},
                "required": ["content"],
            },
        },
    ]


def error_recovery_wrong_count_variants(requested_count: int) -> List[Tuple[str, Any]]:
    return [
        ("unpadded_string", str(requested_count)),
        ("off_by_one_low", f"{max(0, requested_count - 1):04d}"),
        ("off_by_one_high", f"{min(9999, requested_count + 1):04d}"),
        ("zero_for_positive_count", "0000"),
        ("over_padded", f"{requested_count:06d}"),
        ("json_integer", requested_count),
    ]


def build_error_recovery_numeric_semantic_rows() -> List[VerifierRow]:
    rows: List[VerifierRow] = []
    tools = error_recovery_numeric_tools()
    required = ["fetch"]
    terminal = ["summarize"]
    family_meta = infer_scoring_metadata("error_recovery")
    for requested_count in ERROR_RECOVERY_NUMERIC_SEMANTIC_REQUESTS:
        user_request = f"Fetch {requested_count} records and summarize them."
        valid_count = f"{requested_count:04d}"
        group_id = stable_id("forge_error_recovery_numeric", requested_count)
        valid_metadata = {
            "generator": "forge_hard_negative_corrected_positive",
            "scenario_family": "error_recovery",
            "negative_type": "corrected_positive",
            "source_kind": "synthetic_numeric_semantic",
            "corrected_positive": True,
        }
        rows.append(make_row(
            "forge_error_recovery_numeric",
            "valid",
            user_request,
            tools,
            {"name": "fetch", "arguments": {"count": valid_count}},
            1.0,
            valid_metadata,
            required_steps=required,
            completed_steps=[],
            pending_steps=required,
            terminal_tools=terminal,
            group_id=group_id,
            scoring_metadata=family_meta,
        ))

        seen_values = {json.dumps(valid_count, sort_keys=True)}
        for negative_type, count_value in error_recovery_wrong_count_variants(requested_count):
            value_key = json.dumps(count_value, sort_keys=True, default=str)
            if value_key in seen_values:
                continue
            seen_values.add(value_key)
            rows.append(make_row(
                "forge_error_recovery_numeric",
                "wrong_arguments_semantic",
                user_request,
                tools,
                {"name": "fetch", "arguments": {"count": count_value}},
                0.05,
                {
                    "generator": "forge_error_recovery_numeric",
                    "scenario_family": "error_recovery",
                    "negative_type": f"numeric_{negative_type}",
                    "source_kind": "synthetic_numeric_semantic",
                    "valid_counterpart": valid_count,
                },
                required_steps=required,
                completed_steps=[],
                pending_steps=required,
                terminal_tools=terminal,
                group_id=group_id,
                scoring_metadata=family_meta,
            ))
    return rows

def build_guardrail_augmented_rows(n_per_label: int = FORGE_AUGMENTATION_PER_LABEL) -> List[VerifierRow]:
    rows = []
    for i in range(n_per_label):
        base = random.choice(FORGE_WORKFLOWS)
        tools = base["tools"]
        required = base["required_steps"]
        terminal = base["terminal_tools"]
        clear_user = base["user_request"]
        missing_user = "Do the thing with the dataset, but I do not know which quarter, year, or count."
        fetch = required[0]
        fetch_tool = next(t for t in tools if t["name"] == fetch)
        req_keys = fetch_tool.get("parameters", {}).get("required", []) or []
        required_args = {k: (4 if k == "quarter" else 2024 if k == "year" else "0010") for k in req_keys}
        terminal_tool = terminal[0]
        group_id = stable_id("forge_augmented", i, clear_user)
        rows.append(make_row("forge_augmented", "premature_terminal", clear_user, tools, {"name": terminal_tool, "arguments": {"summary": "Done."}}, 0.05, {"generator": "forge_augmented", "negative_type": "premature_terminal"}, required, [], required, terminal, group_id=group_id))
        rows.append(make_row("forge_augmented", "unsafe_parallel_batch", clear_user, tools, [{"name": fetch, "arguments": required_args}, {"name": terminal_tool, "arguments": {"summary": "Done."}}], 0.02, {"generator": "forge_augmented", "negative_type": "unsafe_parallel_batch"}, required, [], required, terminal, group_id=group_id))
        guessed_args = dict(required_args)
        if guessed_args:
            guessed_args[list(guessed_args.keys())[0]] = "UNKNOWN"
        rows.append(make_row("forge_augmented", "needs_clarification", missing_user, tools, {"name": fetch, "arguments": guessed_args}, 0.25, {"generator": "forge_augmented", "negative_type": "needs_clarification"}, required, [], required, terminal, group_id=group_id))
    return rows


# ---------------------------------------------------------------------------
# C3: Contrastive wrong_tool_semantic pairs
# Each pair shares a group_id so valid + wrong_tool rows stay together in splits.
# ---------------------------------------------------------------------------
FORGE_CONTRASTIVE_WRONG_TOOL_PAIRS = [
    # fetch/update pairs
    {"user_request": "Retrieve the order details for order ID 00812.",
     "tools": [
         {"name": "get_order", "description": "Fetch details for a specific order.", "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}},
         {"name": "update_order", "description": "Modify an existing order.", "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}, "status": {"type": "string"}}, "required": ["order_id", "status"]}},
         {"name": "report", "description": "Produce final report.", "parameters": {"type": "object", "properties": {"findings": {"type": "string"}}, "required": ["findings"]}},
     ],
     "valid_candidate": {"name": "get_order", "arguments": {"order_id": "00812"}},
     "wrong_tool_candidate": {"name": "update_order", "arguments": {"order_id": "00812", "status": "pending"}},
     "required_steps": ["get_order"], "completed_steps": [], "pending_steps": ["get_order"], "terminal_tools": ["report"]},
    {"user_request": "Look up the inventory count for SKU-4421.",
     "tools": [
         {"name": "get_inventory", "description": "Look up stock levels for a given SKU.", "parameters": {"type": "object", "properties": {"sku": {"type": "string"}}, "required": ["sku"]}},
         {"name": "set_inventory", "description": "Update stock levels for a given SKU.", "parameters": {"type": "object", "properties": {"sku": {"type": "string"}, "count": {"type": "integer"}}, "required": ["sku", "count"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "get_inventory", "arguments": {"sku": "SKU-4421"}},
     "wrong_tool_candidate": {"name": "set_inventory", "arguments": {"sku": "SKU-4421", "count": 0}},
     "required_steps": ["get_inventory"], "completed_steps": [], "pending_steps": ["get_inventory"], "terminal_tools": ["respond"]},
    {"user_request": "Show me the customer record for customer 10045.",
     "tools": [
         {"name": "get_customer", "description": "Retrieve a customer record by ID.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "integer"}}, "required": ["customer_id"]}},
         {"name": "update_customer", "description": "Update fields on a customer record.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "integer"}, "email": {"type": "string"}}, "required": ["customer_id"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "get_customer", "arguments": {"customer_id": 10045}},
     "wrong_tool_candidate": {"name": "update_customer", "arguments": {"customer_id": 10045}},
     "required_steps": ["get_customer"], "completed_steps": [], "pending_steps": ["get_customer"], "terminal_tools": ["respond"]},
    # scoped vs global search
    {"user_request": "Search for running shoes in the footwear category.",
     "tools": [
         {"name": "search_category", "description": "Search products within a specific category.", "parameters": {"type": "object", "properties": {"query": {"type": "string"}, "category": {"type": "string"}}, "required": ["query", "category"]}},
         {"name": "search_all", "description": "Search all products across all categories.", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "search_category", "arguments": {"query": "running shoes", "category": "footwear"}},
     "wrong_tool_candidate": {"name": "search_all", "arguments": {"query": "running shoes"}},
     "required_steps": ["search_category"], "completed_steps": [], "pending_steps": ["search_category"], "terminal_tools": ["respond"]},
    {"user_request": "Find the Q4 2024 invoice for account ACC-982.",
     "tools": [
         {"name": "search_invoices", "description": "Search invoices for a specific account.", "parameters": {"type": "object", "properties": {"account_id": {"type": "string"}, "quarter": {"type": "string"}}, "required": ["account_id"]}},
         {"name": "search_all_records", "description": "Global record search across accounts and invoices.", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "search_invoices", "arguments": {"account_id": "ACC-982", "quarter": "Q4-2024"}},
     "wrong_tool_candidate": {"name": "search_all_records", "arguments": {"query": "Q4 2024 invoice ACC-982"}},
     "required_steps": ["search_invoices"], "completed_steps": [], "pending_steps": ["search_invoices"], "terminal_tools": ["respond"]},
    # premature terminal tool
    {"user_request": "Fetch the sales data for Q3 2024 and then generate a report.",
     "tools": [
         {"name": "fetch_sales", "description": "Fetch sales data for a given period.", "parameters": {"type": "object", "properties": {"quarter": {"type": "string"}}, "required": ["quarter"]}},
         {"name": "generate_report", "description": "Generate the final sales report.", "parameters": {"type": "object", "properties": {"summary": {"type": "string"}}, "required": ["summary"]}},
     ],
     "valid_candidate": {"name": "fetch_sales", "arguments": {"quarter": "Q3-2024"}},
     "wrong_tool_candidate": {"name": "generate_report", "arguments": {"summary": "Sales data for Q3 2024."}},
     "required_steps": ["fetch_sales", "generate_report"], "completed_steps": [], "pending_steps": ["fetch_sales", "generate_report"], "terminal_tools": ["generate_report"]},
    {"user_request": "First pull the user activity log, then summarize it.",
     "tools": [
         {"name": "get_activity_log", "description": "Retrieve user activity log for a period.", "parameters": {"type": "object", "properties": {"user_id": {"type": "string"}, "days": {"type": "integer"}}, "required": ["user_id"]}},
         {"name": "summarize", "description": "Summarize content into a report.", "parameters": {"type": "object", "properties": {"content": {"type": "string"}}, "required": ["content"]}},
     ],
     "valid_candidate": {"name": "get_activity_log", "arguments": {"user_id": "usr-77", "days": 30}},
     "wrong_tool_candidate": {"name": "summarize", "arguments": {"content": "Activity log for user usr-77."}},
     "required_steps": ["get_activity_log", "summarize"], "completed_steps": [], "pending_steps": ["get_activity_log", "summarize"], "terminal_tools": ["summarize"]},
    # completed-step repeat
    {"user_request": "Analyze the dataset and produce a report.",
     "tools": [
         {"name": "analyze_data", "description": "Run statistical analysis on a dataset.", "parameters": {"type": "object", "properties": {"dataset_id": {"type": "string"}}, "required": ["dataset_id"]}},
         {"name": "produce_report", "description": "Write and deliver the final report.", "parameters": {"type": "object", "properties": {"findings": {"type": "string"}}, "required": ["findings"]}},
     ],
     "valid_candidate": {"name": "produce_report", "arguments": {"findings": "Dataset shows 15% growth in Q4."}},
     "wrong_tool_candidate": {"name": "analyze_data", "arguments": {"dataset_id": "ds-001"}},
     "required_steps": ["analyze_data", "produce_report"], "completed_steps": ["analyze_data"], "pending_steps": ["produce_report"], "terminal_tools": ["produce_report"]},
    {"user_request": "Pull the transaction history and generate the audit report.",
     "tools": [
         {"name": "get_transactions", "description": "Fetch transaction history.", "parameters": {"type": "object", "properties": {"account": {"type": "string"}}, "required": ["account"]}},
         {"name": "audit_report", "description": "Generate audit report from transactions.", "parameters": {"type": "object", "properties": {"summary": {"type": "string"}}, "required": ["summary"]}},
     ],
     "valid_candidate": {"name": "audit_report", "arguments": {"summary": "No anomalies found in transaction history for account-22."}},
     "wrong_tool_candidate": {"name": "get_transactions", "arguments": {"account": "account-22"}},
     "required_steps": ["get_transactions", "audit_report"], "completed_steps": ["get_transactions"], "pending_steps": ["audit_report"], "terminal_tools": ["audit_report"]},
    # list vs delete
    {"user_request": "List all pending tickets for project FORGE-7.",
     "tools": [
         {"name": "list_tickets", "description": "List tickets for a project filtered by status.", "parameters": {"type": "object", "properties": {"project": {"type": "string"}, "status": {"type": "string"}}, "required": ["project"]}},
         {"name": "delete_ticket", "description": "Delete a ticket by ID.", "parameters": {"type": "object", "properties": {"ticket_id": {"type": "string"}}, "required": ["ticket_id"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "list_tickets", "arguments": {"project": "FORGE-7", "status": "pending"}},
     "wrong_tool_candidate": {"name": "delete_ticket", "arguments": {"ticket_id": "FORGE-7"}},
     "required_steps": ["list_tickets"], "completed_steps": [], "pending_steps": ["list_tickets"], "terminal_tools": ["respond"]},
    # create vs fetch
    {"user_request": "Get the profile for user ID 5514.",
     "tools": [
         {"name": "get_profile", "description": "Retrieve an existing user profile.", "parameters": {"type": "object", "properties": {"user_id": {"type": "integer"}}, "required": ["user_id"]}},
         {"name": "create_profile", "description": "Create a new user profile.", "parameters": {"type": "object", "properties": {"user_id": {"type": "integer"}, "name": {"type": "string"}}, "required": ["user_id", "name"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "get_profile", "arguments": {"user_id": 5514}},
     "wrong_tool_candidate": {"name": "create_profile", "arguments": {"user_id": 5514, "name": "Unknown"}},
     "required_steps": ["get_profile"], "completed_steps": [], "pending_steps": ["get_profile"], "terminal_tools": ["respond"]},
    # approve vs reject
    {"user_request": "Approve the purchase request PR-2041.",
     "tools": [
         {"name": "approve_request", "description": "Approve a purchase request.", "parameters": {"type": "object", "properties": {"request_id": {"type": "string"}}, "required": ["request_id"]}},
         {"name": "reject_request", "description": "Reject a purchase request.", "parameters": {"type": "object", "properties": {"request_id": {"type": "string"}, "reason": {"type": "string"}}, "required": ["request_id", "reason"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "approve_request", "arguments": {"request_id": "PR-2041"}},
     "wrong_tool_candidate": {"name": "reject_request", "arguments": {"request_id": "PR-2041", "reason": "Budget exceeded"}},
     "required_steps": ["approve_request"], "completed_steps": [], "pending_steps": ["approve_request"], "terminal_tools": ["respond"]},
    # balance read vs transfer (write)
    {"user_request": "Check the balance on account ACC-1144.",
     "tools": [
         {"name": "get_balance", "description": "Get the current balance of an account.", "parameters": {"type": "object", "properties": {"account_id": {"type": "string"}}, "required": ["account_id"]}},
         {"name": "transfer_funds", "description": "Transfer funds between accounts.", "parameters": {"type": "object", "properties": {"from_account": {"type": "string"}, "to_account": {"type": "string"}, "amount": {"type": "number"}}, "required": ["from_account", "to_account", "amount"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "get_balance", "arguments": {"account_id": "ACC-1144"}},
     "wrong_tool_candidate": {"name": "transfer_funds", "arguments": {"from_account": "ACC-1144", "to_account": "ACC-0000", "amount": 0}},
     "required_steps": ["get_balance"], "completed_steps": [], "pending_steps": ["get_balance"], "terminal_tools": ["respond"]},
    # compute vs fetch
    {"user_request": "Calculate the tax owed for invoice INV-9901.",
     "tools": [
         {"name": "compute_tax", "description": "Calculate tax for a given invoice.", "parameters": {"type": "object", "properties": {"invoice_id": {"type": "string"}}, "required": ["invoice_id"]}},
         {"name": "get_invoice", "description": "Fetch the raw invoice data.", "parameters": {"type": "object", "properties": {"invoice_id": {"type": "string"}}, "required": ["invoice_id"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "compute_tax", "arguments": {"invoice_id": "INV-9901"}},
     "wrong_tool_candidate": {"name": "get_invoice", "arguments": {"invoice_id": "INV-9901"}},
     "required_steps": ["compute_tax"], "completed_steps": [], "pending_steps": ["compute_tax"], "terminal_tools": ["respond"]},
    # send vs schedule
    {"user_request": "Send the welcome email to user usr-42 now.",
     "tools": [
         {"name": "send_email", "description": "Send an email immediately.", "parameters": {"type": "object", "properties": {"user_id": {"type": "string"}, "template": {"type": "string"}}, "required": ["user_id", "template"]}},
         {"name": "schedule_email", "description": "Schedule an email for future delivery.", "parameters": {"type": "object", "properties": {"user_id": {"type": "string"}, "template": {"type": "string"}, "send_at": {"type": "string"}}, "required": ["user_id", "template", "send_at"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "send_email", "arguments": {"user_id": "usr-42", "template": "welcome"}},
     "wrong_tool_candidate": {"name": "schedule_email", "arguments": {"user_id": "usr-42", "template": "welcome", "send_at": "2025-01-01T09:00:00Z"}},
     "required_steps": ["send_email"], "completed_steps": [], "pending_steps": ["send_email"], "terminal_tools": ["respond"]},
    # lock vs unlock
    {"user_request": "Lock the account for user usr-99 due to suspicious activity.",
     "tools": [
         {"name": "lock_account", "description": "Lock a user account to prevent access.", "parameters": {"type": "object", "properties": {"user_id": {"type": "string"}, "reason": {"type": "string"}}, "required": ["user_id", "reason"]}},
         {"name": "unlock_account", "description": "Restore access to a locked user account.", "parameters": {"type": "object", "properties": {"user_id": {"type": "string"}}, "required": ["user_id"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "lock_account", "arguments": {"user_id": "usr-99", "reason": "suspicious activity"}},
     "wrong_tool_candidate": {"name": "unlock_account", "arguments": {"user_id": "usr-99"}},
     "required_steps": ["lock_account"], "completed_steps": [], "pending_steps": ["lock_account"], "terminal_tools": ["respond"]},
    # publish vs archive
    {"user_request": "Publish the draft report RPT-555 to the portal.",
     "tools": [
         {"name": "publish_report", "description": "Publish a report draft to the live portal.", "parameters": {"type": "object", "properties": {"report_id": {"type": "string"}}, "required": ["report_id"]}},
         {"name": "archive_report", "description": "Move a report to archive storage.", "parameters": {"type": "object", "properties": {"report_id": {"type": "string"}}, "required": ["report_id"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "publish_report", "arguments": {"report_id": "RPT-555"}},
     "wrong_tool_candidate": {"name": "archive_report", "arguments": {"report_id": "RPT-555"}},
     "required_steps": ["publish_report"], "completed_steps": [], "pending_steps": ["publish_report"], "terminal_tools": ["respond"]},
    # escalate vs resolve
    {"user_request": "Escalate ticket TKT-1022 to tier-2 support.",
     "tools": [
         {"name": "escalate_ticket", "description": "Escalate a ticket to a higher support tier.", "parameters": {"type": "object", "properties": {"ticket_id": {"type": "string"}, "tier": {"type": "integer"}}, "required": ["ticket_id", "tier"]}},
         {"name": "resolve_ticket", "description": "Mark a ticket as resolved.", "parameters": {"type": "object", "properties": {"ticket_id": {"type": "string"}, "resolution": {"type": "string"}}, "required": ["ticket_id", "resolution"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "escalate_ticket", "arguments": {"ticket_id": "TKT-1022", "tier": 2}},
     "wrong_tool_candidate": {"name": "resolve_ticket", "arguments": {"ticket_id": "TKT-1022", "resolution": "Escalated automatically"}},
     "required_steps": ["escalate_ticket"], "completed_steps": [], "pending_steps": ["escalate_ticket"], "terminal_tools": ["respond"]},
    # enable vs disable
    {"user_request": "Enable two-factor authentication for tenant ACME.",
     "tools": [
         {"name": "enable_feature", "description": "Enable a feature flag for a tenant.", "parameters": {"type": "object", "properties": {"tenant": {"type": "string"}, "feature": {"type": "string"}}, "required": ["tenant", "feature"]}},
         {"name": "disable_feature", "description": "Disable a feature flag for a tenant.", "parameters": {"type": "object", "properties": {"tenant": {"type": "string"}, "feature": {"type": "string"}}, "required": ["tenant", "feature"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "enable_feature", "arguments": {"tenant": "ACME", "feature": "two_factor_auth"}},
     "wrong_tool_candidate": {"name": "disable_feature", "arguments": {"tenant": "ACME", "feature": "two_factor_auth"}},
     "required_steps": ["enable_feature"], "completed_steps": [], "pending_steps": ["enable_feature"], "terminal_tools": ["respond"]},
    # subscribe vs unsubscribe
    {"user_request": "Subscribe user usr-301 to the weekly digest newsletter.",
     "tools": [
         {"name": "subscribe_newsletter", "description": "Subscribe a user to a newsletter.", "parameters": {"type": "object", "properties": {"user_id": {"type": "string"}, "newsletter": {"type": "string"}}, "required": ["user_id", "newsletter"]}},
         {"name": "unsubscribe_newsletter", "description": "Unsubscribe a user from a newsletter.", "parameters": {"type": "object", "properties": {"user_id": {"type": "string"}, "newsletter": {"type": "string"}}, "required": ["user_id", "newsletter"]}},
         {"name": "respond", "description": "Send final response.", "parameters": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
     ],
     "valid_candidate": {"name": "subscribe_newsletter", "arguments": {"user_id": "usr-301", "newsletter": "weekly_digest"}},
     "wrong_tool_candidate": {"name": "unsubscribe_newsletter", "arguments": {"user_id": "usr-301", "newsletter": "weekly_digest"}},
     "required_steps": ["subscribe_newsletter"], "completed_steps": [], "pending_steps": ["subscribe_newsletter"], "terminal_tools": ["respond"]},
]


def build_contrastive_wrong_tool_rows() -> List[VerifierRow]:
    rows: List[VerifierRow] = []
    for i, pair in enumerate(FORGE_CONTRASTIVE_WRONG_TOOL_PAIRS):
        user_request = pair["user_request"]
        tools = pair["tools"]
        required = pair.get("required_steps", [])
        completed = pair.get("completed_steps", [])
        pending = pair.get("pending_steps", required)
        terminal = pair.get("terminal_tools", [])
        group_id = stable_id("forge_contrastive_wts", i, user_request)
        group_id = f"contrastive_wts_{i:03d}_{group_id[:8]}"
        base_meta = {
            "generator": "contrastive_pair",
            "scenario_family": "wrong_tool_contrastive",
            "source_kind": "forge_contrastive",
            "contrastive_pair_index": i,
        }
        sm = infer_scoring_metadata("wrong_tool_contrastive")
        rows.append(make_row(
            "forge_contrastive_wts", "valid",
            user_request, tools, pair["valid_candidate"], 1.0,
            dict(base_meta, pair_role="contrastive_valid"),
            required_steps=required, completed_steps=completed,
            pending_steps=pending, terminal_tools=terminal,
            group_id=group_id, scoring_metadata=sm,
        ))
        rows.append(make_row(
            "forge_contrastive_wts", "wrong_tool_semantic",
            user_request, tools, pair["wrong_tool_candidate"], 0.05,
            dict(base_meta, negative_type="wrong_tool_contrastive"),
            required_steps=required, completed_steps=completed,
            pending_steps=pending, terminal_tools=terminal,
            group_id=group_id, scoring_metadata=sm,
        ))
    return rows


# ---------------------------------------------------------------------------
# C4: Protected valid slice expansion — fixed-width numeric strings
# ---------------------------------------------------------------------------
FORGE_FIXED_WIDTH_VALID_CASES = [
    ("Fetch 42 records.", "fetch", "Fetch records. Count must be zero-padded 4-digit string.", "count", "4-digit zero-padded count.", "0042", 42),
    ("Fetch 7 records.", "fetch", "Fetch records. Count must be zero-padded 4-digit string.", "count", "4-digit zero-padded count.", "0007", 7),
    ("Fetch 100 records.", "fetch", "Fetch records. Count must be zero-padded 4-digit string.", "count", "4-digit zero-padded count.", "0100", 100),
    ("Fetch 999 records.", "fetch", "Fetch records. Count must be zero-padded 4-digit string.", "count", "4-digit zero-padded count.", "0999", 999),
    ("Fetch 1 record.", "fetch", "Fetch records. Count must be zero-padded 4-digit string.", "count", "4-digit zero-padded count.", "0001", 1),
    ("Retrieve order 00045.", "get_order", "Fetch order. ID must be zero-padded 5-digit string.", "order_id", "5-digit zero-padded order ID.", "00045", 45),
    ("Retrieve order 00001.", "get_order", "Fetch order. ID must be zero-padded 5-digit string.", "order_id", "5-digit zero-padded order ID.", "00001", 1),
    ("Retrieve order 09999.", "get_order", "Fetch order. ID must be zero-padded 5-digit string.", "order_id", "5-digit zero-padded order ID.", "09999", 9999),
    ("Look up zip code 07030.", "lookup_zip", "Look up a ZIP code. Must be 5-digit string.", "zip_code", "5-digit ZIP code string.", "07030", 7030),
    ("Look up zip code 00501.", "lookup_zip", "Look up a ZIP code. Must be 5-digit string.", "zip_code", "5-digit ZIP code string.", "00501", 501),
    ("Look up zip code 90210.", "lookup_zip", "Look up a ZIP code. Must be 5-digit string.", "zip_code", "5-digit ZIP code string.", "90210", 90210),
    ("Route payment via routing number 021000021.", "route_payment", "Route payment. Routing number must be string.", "routing_number", "9-digit routing number string.", "021000021", 21000021),
    ("Route payment via routing number 111000025.", "route_payment", "Route payment. Routing number must be string.", "routing_number", "9-digit routing number string.", "111000025", 111000025),
    ("Lookup account 000123.", "lookup_account", "Lookup account. ID must be zero-padded 6-digit string.", "account_id", "6-digit zero-padded account ID.", "000123", 123),
    ("Lookup account 001000.", "lookup_account", "Lookup account. ID must be zero-padded 6-digit string.", "account_id", "6-digit zero-padded account ID.", "001000", 1000),
    ("Fetch batch 0050.", "fetch_batch", "Fetch batch. ID must be zero-padded 4-digit string.", "batch_id", "4-digit zero-padded batch ID.", "0050", 50),
    ("Fetch batch 0001.", "fetch_batch", "Fetch batch. ID must be zero-padded 4-digit string.", "batch_id", "4-digit zero-padded batch ID.", "0001", 1),
    ("Fetch batch 0099.", "fetch_batch", "Fetch batch. ID must be zero-padded 4-digit string.", "batch_id", "4-digit zero-padded batch ID.", "0099", 99),
    ("Send SMS to +14155551234.", "send_sms", "Send SMS. Phone must be E.164 format string.", "phone", "E.164 phone number string.", "+14155551234", 14155551234),
    ("Retrieve order 00100.", "get_order", "Fetch order. ID must be zero-padded 5-digit string.", "order_id", "5-digit zero-padded order ID.", "00100", 100),
]


def _fw_tools(tool_name: str, tool_description: str, param_name: str, param_description: str) -> List[Dict[str, Any]]:
    return [
        {"name": tool_name, "description": tool_description, "parameters": {
            "type": "object", "properties": {param_name: {"type": "string", "description": param_description}}, "required": [param_name]}},
        {"name": "respond", "description": "Send final answer.", "parameters": {
            "type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
    ]


def build_fixed_width_numeric_rows() -> List[VerifierRow]:
    rows: List[VerifierRow] = []
    fm = infer_scoring_metadata("fixed_width_numeric")
    for (user_request, tool_name, tool_desc, param_name, param_desc, valid_value, wrong_int) in FORGE_FIXED_WIDTH_VALID_CASES:
        tools = _fw_tools(tool_name, tool_desc, param_name, param_desc)
        required = [tool_name]
        terminal = ["respond"]
        group_id = stable_id("forge_fixed_width", user_request, tool_name, param_name, valid_value)
        valid_meta = {
            "generator": "forge_fixed_width_numeric",
            "scenario_family": "fixed_width_numeric",
            "source_kind": "synthetic_numeric_string",
            "corrected_positive": True,
            "valid_protection_fixed_width_numeric_string": True,
        }
        rows.append(make_row(
            "forge_fixed_width_numeric", "valid",
            user_request, tools, {"name": tool_name, "arguments": {param_name: valid_value}}, 1.0,
            valid_meta, required_steps=required, completed_steps=[], pending_steps=required,
            terminal_tools=terminal, group_id=group_id, scoring_metadata=fm,
        ))
        seen = {json.dumps(valid_value, sort_keys=True)}
        for neg_type, neg_val in [("integer_instead_of_string", wrong_int), ("unpadded_string", str(wrong_int))]:
            nk = json.dumps(neg_val, default=str, sort_keys=True)
            if nk in seen:
                continue
            seen.add(nk)
            rows.append(make_row(
                "forge_fixed_width_numeric", "wrong_arguments_semantic",
                user_request, tools, {"name": tool_name, "arguments": {param_name: neg_val}}, 0.05,
                {"generator": "forge_fixed_width_numeric", "scenario_family": "fixed_width_numeric",
                 "source_kind": "synthetic_numeric_string", "negative_type": f"fixed_width_{neg_type}",
                 "valid_counterpart": valid_value},
                required_steps=required, completed_steps=[], pending_steps=required,
                terminal_tools=terminal, group_id=group_id, scoring_metadata=fm,
            ))
    return rows


# C4: Protected valid slice expansion — corrected error recovery
FORGE_ERROR_RECOVERY_SCENARIOS = [
    {"user_request": "Fetch records for account ACC-88. Previous fetch failed with invalid account ID format.",
     "tool_name": "fetch_records", "tool_desc": "Fetch records for an account. Account ID must be string in ACC-NNN format.",
     "param_name": "account_id", "param_desc": "Account ID string in ACC-NNN format.", "valid_value": "ACC-88",
     "wrong_args": [("integer_id", 88), ("wrong_prefix", "account-88"), ("bare_number_str", "88")],
     "recent_errors": ["Error: Invalid account_id format. Got: 88. Expected string like ACC-NNN."],
     "wrong_tool": "lookup_account", "wrong_tool_desc": "Look up account metadata by numeric ID.",
     "wrong_tool_params": {"type": "object", "properties": {"id": {"type": "integer"}}, "required": ["id"]},
     "wrong_tool_call": {"name": "lookup_account", "arguments": {"id": 88}}},
    {"user_request": "Retrieve order ORD-0042. Prior attempt failed because ID was numeric instead of string.",
     "tool_name": "get_order", "tool_desc": "Fetch order. Order ID must be string in ORD-NNNN format.",
     "param_name": "order_id", "param_desc": "Order ID string like ORD-0042.", "valid_value": "ORD-0042",
     "wrong_args": [("integer_id", 42), ("bare_string", "0042"), ("wrong_prefix", "order-0042")],
     "recent_errors": ["Error: order_id must be string like ORD-NNNN, got integer 42."],
     "wrong_tool": "search_orders", "wrong_tool_desc": "Search orders by keyword query.",
     "wrong_tool_params": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
     "wrong_tool_call": {"name": "search_orders", "arguments": {"query": "ORD-0042"}}},
    {"user_request": "Fetch the first 5 records. Last call used integer 5 but the API needs zero-padded string.",
     "tool_name": "fetch", "tool_desc": "Fetch records by count. Count must be zero-padded 4-digit string.",
     "param_name": "count", "param_desc": "4-digit zero-padded count string.", "valid_value": "0005",
     "wrong_args": [("integer", 5), ("unpadded_string", "5"), ("over_padded", "000005")],
     "recent_errors": ["Error: count must be 4-digit zero-padded string like '0005', got 5."],
     "wrong_tool": "summarize", "wrong_tool_desc": "Summarize previously fetched records.",
     "wrong_tool_params": {"type": "object", "properties": {"content": {"type": "string"}}, "required": ["content"]},
     "wrong_tool_call": {"name": "summarize", "arguments": {"content": "5 records"}}},
    {"user_request": "Look up ZIP code 07040. Previous lookup failed because ZIP was passed as integer.",
     "tool_name": "lookup_zip", "tool_desc": "Look up a ZIP code. Must be 5-digit string.",
     "param_name": "zip_code", "param_desc": "5-digit ZIP code string.", "valid_value": "07040",
     "wrong_args": [("integer", 7040), ("unpadded_string", "7040"), ("over_padded", "007040")],
     "recent_errors": ["Error: zip_code must be string '07040', got integer 7040."],
     "wrong_tool": "get_city", "wrong_tool_desc": "Get city name by city ID.",
     "wrong_tool_params": {"type": "object", "properties": {"city_id": {"type": "integer"}}, "required": ["city_id"]},
     "wrong_tool_call": {"name": "get_city", "arguments": {"city_id": 7040}}},
    {"user_request": "Submit audit report for transaction batch 0033. Previous attempt failed: batch_id was integer.",
     "tool_name": "submit_audit", "tool_desc": "Submit audit report. batch_id must be zero-padded 4-digit string.",
     "param_name": "batch_id", "param_desc": "4-digit zero-padded batch ID string.", "valid_value": "0033",
     "wrong_args": [("integer", 33), ("unpadded", "33"), ("over_padded", "000033")],
     "recent_errors": ["Error: batch_id must be string '0033', got integer 33."],
     "wrong_tool": "list_batches", "wrong_tool_desc": "List all available batches.",
     "wrong_tool_params": {"type": "object", "properties": {}, "required": []},
     "wrong_tool_call": {"name": "list_batches", "arguments": {}}},
]


def _er_tools(scen: Dict[str, Any]) -> List[Dict[str, Any]]:
    return [
        {"name": scen["tool_name"], "description": scen["tool_desc"], "parameters": {
            "type": "object", "properties": {scen["param_name"]: {"type": "string", "description": scen["param_desc"]}}, "required": [scen["param_name"]]}},
        {"name": scen["wrong_tool"], "description": scen["wrong_tool_desc"], "parameters": scen["wrong_tool_params"]},
        {"name": "respond", "description": "Send final answer.", "parameters": {
            "type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
    ]


def build_error_recovery_protected_rows() -> List[VerifierRow]:
    rows: List[VerifierRow] = []
    fm = infer_scoring_metadata("error_recovery")
    for scen in FORGE_ERROR_RECOVERY_SCENARIOS:
        tools = _er_tools(scen)
        required = [scen["tool_name"]]
        terminal = ["respond"]
        recent_errors = scen.get("recent_errors", [])
        group_id = stable_id("forge_er_protected", scen["user_request"], scen["tool_name"])
        valid_meta = {
            "generator": "forge_error_recovery_protected",
            "scenario_family": "error_recovery",
            "source_kind": "synthetic_error_recovery",
            "corrected_positive": True,
            "valid_protection_corrected_error_recovery": True,
        }
        rows.append(make_row(
            "forge_error_recovery_protected", "valid",
            scen["user_request"], tools, {"name": scen["tool_name"], "arguments": {scen["param_name"]: scen["valid_value"]}}, 1.0,
            valid_meta, required_steps=required, completed_steps=[], pending_steps=required,
            terminal_tools=terminal, recent_errors=recent_errors, group_id=group_id, scoring_metadata=fm,
        ))
        seen = {json.dumps(scen["valid_value"], sort_keys=True)}
        for neg_type, neg_val in scen.get("wrong_args", []):
            nk = json.dumps(neg_val, default=str, sort_keys=True)
            if nk in seen:
                continue
            seen.add(nk)
            rows.append(make_row(
                "forge_error_recovery_protected", "wrong_arguments_semantic",
                scen["user_request"], tools, {"name": scen["tool_name"], "arguments": {scen["param_name"]: neg_val}}, 0.05,
                {"generator": "forge_error_recovery_protected", "scenario_family": "error_recovery",
                 "source_kind": "synthetic_error_recovery", "negative_type": f"er_{neg_type}", "valid_counterpart": scen["valid_value"]},
                required_steps=required, completed_steps=[], pending_steps=required,
                terminal_tools=terminal, recent_errors=recent_errors, group_id=group_id, scoring_metadata=fm,
            ))
        rows.append(make_row(
            "forge_error_recovery_protected", "wrong_tool_semantic",
            scen["user_request"], tools, scen["wrong_tool_call"], 0.05,
            {"generator": "forge_error_recovery_protected", "scenario_family": "error_recovery",
             "source_kind": "synthetic_error_recovery", "negative_type": "er_wrong_tool"},
            required_steps=required, completed_steps=[], pending_steps=required,
            terminal_tools=terminal, recent_errors=recent_errors, group_id=group_id, scoring_metadata=fm,
        ))
    return rows


# ---------------------------------------------------------------------------
# v5b C3: Protected valid slices at scale (seeded generators).
# The hardcoded v5 lists above stay as-is; these add 500+ distinct valid rows per
# slice plus paired one-difference hard negatives so the slice gates measure real
# variation instead of duplicated rows.
# ---------------------------------------------------------------------------
FWNS_SCALED_SCENARIOS = 520  #@param {type:"integer"}
ER_SCALED_SCENARIOS = 520  #@param {type:"integer"}

_FWNS_DOMAINS = [
    ("order", "order", "order_id", "order ID"),
    ("invoice", "invoice", "invoice_id", "invoice ID"),
    ("shipment", "shipment", "shipment_id", "shipment ID"),
    ("ticket", "support ticket", "ticket_id", "ticket ID"),
    ("account", "account", "account_id", "account ID"),
    ("employee", "employee record", "employee_id", "employee ID"),
    ("batch", "transaction batch", "batch_id", "batch ID"),
    ("meter", "meter reading", "meter_id", "meter ID"),
    ("case", "case file", "case_id", "case ID"),
    ("device", "device", "device_id", "device ID"),
    ("bin", "warehouse bin", "bin_id", "bin ID"),
    ("policy", "insurance policy", "policy_id", "policy number"),
    ("loan", "loan application", "application_id", "application ID"),
    ("chart", "patient chart", "chart_id", "chart ID"),
    ("registration", "vehicle registration", "registration_id", "registration ID"),
]
_FWNS_VERBS = ["get", "lookup", "fetch", "retrieve", "load"]
_FWNS_PHRASES = [
    "Retrieve {np} {value}.",
    "Pull up {np} number {value} please.",
    "Get the details for {np} {value}.",
    "I need {np} {value}.",
    "Open {np} {value} for me.",
    "Show me {np} {value}.",
    "Look up {np} {value} in the system.",
    "Can you fetch {np} {value}?",
]
# Generalization extras: formats that are valid stringly-typed values but do NOT
# satisfy the zero-padded slice predicate; they broaden coverage without being
# counted on for slice support.
_FWNS_FORMAT_EXTRAS = [
    ("e164_phone", "customer phone", "phone_number", "E.164 phone number string with + prefix",
     lambda rng: "+1415555" + str(rng.randint(0, 9999)).zfill(4),
     lambda v: [("integer_instead_of_string", int(v[1:])), ("missing_plus_prefix", v[1:])]),
    ("iban", "bank account", "iban", "IBAN string including country prefix",
     lambda rng: "DE89" + str(rng.randint(0, 10 ** 16 - 1)).zfill(16),
     lambda v: [("missing_country_prefix", v[4:]), ("lowercase_prefix", v[:4].lower() + v[4:])]),
    ("sku_code", "product SKU", "sku", "SKU string in SKU-NNNNNN format",
     lambda rng: "SKU-" + str(rng.randint(1, 999999)).zfill(6),
     lambda v: [("bare_number_int", int(v.split("-", 1)[1])), ("missing_prefix", v.split("-", 1)[1])]),
    ("routing_number", "bank routing number", "routing_number", "9-digit routing number string",
     lambda rng: "0" + str(rng.randint(10 ** 7, 10 ** 8 - 1)),
     lambda v: [("integer_instead_of_string", int(v)), ("unpadded_string", str(int(v)))]),
]


def build_fixed_width_numeric_rows_scaled(n_scenarios: int = FWNS_SCALED_SCENARIOS) -> List[VerifierRow]:
    """Scaled fixed-width protected-valid coverage. Every core valid value is sampled
    below 10**(width-1) and zfilled, so it is guaranteed to start with '0' and fire
    has_fixed_width_numeric_string / the training protection flags."""
    rng = random.Random(SEED + 4242)
    rows: List[VerifierRow] = []
    fm = infer_scoring_metadata("fixed_width_numeric")

    def emit_scenario(i: int, user_request: str, tool_name: str, tool_desc: str,
                      param_name: str, param_desc: str, valid_value: Any,
                      negatives: List[Tuple[str, Any]]) -> None:
        tools = _fw_tools(tool_name, tool_desc, param_name, param_desc)
        required = [tool_name]
        terminal = ["respond"]
        group_id = stable_id("forge_fwns_v2", i, user_request, tool_name, param_name, valid_value)
        valid_meta = {
            "generator": "forge_fixed_width_numeric_v2",
            "scenario_family": "fixed_width_numeric",
            "source_kind": "synthetic_numeric_string",
            "corrected_positive": True,
            "valid_protection_fixed_width_numeric_string": True,
        }
        rows.append(make_row(
            "forge_fixed_width_numeric", "valid",
            user_request, tools, {"name": tool_name, "arguments": {param_name: valid_value}}, 1.0,
            valid_meta, required_steps=required, completed_steps=[], pending_steps=required,
            terminal_tools=terminal, group_id=group_id, scoring_metadata=fm,
        ))
        seen = {json.dumps(valid_value, default=str, sort_keys=True)}
        for neg_type, neg_val in negatives:
            nk = json.dumps(neg_val, default=str, sort_keys=True)
            if nk in seen:
                continue
            seen.add(nk)
            rows.append(make_row(
                "forge_fixed_width_numeric", "wrong_arguments_semantic",
                user_request, tools, {"name": tool_name, "arguments": {param_name: neg_val}}, 0.05,
                {"generator": "forge_fixed_width_numeric_v2", "scenario_family": "fixed_width_numeric",
                 "source_kind": "synthetic_numeric_string", "negative_type": f"fixed_width_{neg_type}",
                 "valid_counterpart": valid_value},
                required_steps=required, completed_steps=[], pending_steps=required,
                terminal_tools=terminal, group_id=group_id, scoring_metadata=fm,
            ))

    for i in range(int(n_scenarios)):
        noun, np_, param_name, param_noun = _FWNS_DOMAINS[i % len(_FWNS_DOMAINS)]
        verb = rng.choice(_FWNS_VERBS)
        width = rng.randint(3, 8)
        number = rng.randint(1, 10 ** (width - 1) - 1)
        valid_value = str(number).zfill(width)
        tool_name = f"{verb}_{noun}"
        tool_desc = f"{verb.capitalize()} a {np_}. The {param_noun} must be a zero-padded {width}-digit string."
        param_desc = f"{width}-digit zero-padded {param_noun} string."
        user_request = rng.choice(_FWNS_PHRASES).format(np=np_, value=valid_value)
        negatives = [("integer_instead_of_string", number), ("unpadded_string", str(number))]
        if rng.random() < 0.3:
            negatives.append(("over_padded", valid_value.zfill(width + 2)))
        emit_scenario(i, user_request, tool_name, tool_desc, param_name, param_desc, valid_value, negatives)
        if i % 5 == 4:
            fmt_name, fmt_np, fmt_param, fmt_param_desc, make_value, make_negs = (
                _FWNS_FORMAT_EXTRAS[(i // 5) % len(_FWNS_FORMAT_EXTRAS)]
            )
            extra_value = make_value(rng)
            extra_tool = f"verify_{fmt_param}"
            extra_desc = f"Verify a {fmt_np}. The value must be a {fmt_param_desc}."
            extra_request = rng.choice(_FWNS_PHRASES).format(np=fmt_np, value=extra_value)
            emit_scenario(10_000 + i, extra_request, extra_tool, extra_desc, fmt_param,
                          fmt_param_desc, extra_value, make_negs(extra_value))
    return rows


_ER_PREFIXES = [None, "ACC", "ORD", "REQ", "INV", "TKT"]
_ER_PHRASES = [
    "Retry: fetch {np} {value}. The previous call failed with an invalid {param} format.",
    "Fetch {np} {value} again; the last attempt passed {param} as a number and was rejected.",
    "The earlier {param} was malformed. Get {np} {value} with the corrected format.",
    "Previous request errored on {param}. Retrieve {np} {value} now.",
    "That failed because {param} was numeric. Please pull {np} {value} using the string form.",
]
_ER_DISTRACTORS = [
    ("search_records", "Search records by keyword query.",
     {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
     lambda v: {"query": str(v)}),
    ("list_items", "List all available items.",
     {"type": "object", "properties": {}, "required": []},
     lambda v: {}),
    ("summarize", "Summarize previously fetched data.",
     {"type": "object", "properties": {"content": {"type": "string"}}, "required": ["content"]},
     lambda v: {"content": f"records for {v}"}),
    ("delete_record", "Delete a record by numeric ID.",
     {"type": "object", "properties": {"id": {"type": "integer"}}, "required": ["id"]},
     lambda v: {"id": 7}),
]


def build_error_recovery_protected_rows_scaled(n_scenarios: int = ER_SCALED_SCENARIOS) -> List[VerifierRow]:
    """Scaled corrected-error-recovery protected-valid coverage: a prior call failed on
    argument format (recent_errors populated), the candidate is the corrected retry."""
    rng = random.Random(SEED + 7311)
    rows: List[VerifierRow] = []
    fm = infer_scoring_metadata("error_recovery")
    for i in range(int(n_scenarios)):
        noun, np_, param_name, param_noun = _FWNS_DOMAINS[i % len(_FWNS_DOMAINS)]
        verb = rng.choice(_FWNS_VERBS)
        prefix = rng.choice(_ER_PREFIXES)
        width = rng.randint(3, 6)
        number = rng.randint(1, 10 ** (width - 1) - 1)
        core = str(number).zfill(width)
        if prefix:
            valid_value = f"{prefix}-{core}"
            fmt_desc = f"{prefix}-" + "N" * width
            wrong_args = [
                ("integer_id", number),
                ("bare_number_str", core),
                ("wrong_prefix", f"{prefix.lower()}-{core}"),
            ]
        else:
            valid_value = core
            fmt_desc = f"zero-padded {width}-digit string"
            wrong_args = [("integer", number), ("unpadded_string", str(number))]
            if rng.random() < 0.3:
                wrong_args.append(("over_padded", core.zfill(width + 2)))
        tool_name = f"{verb}_{noun}"
        tool_desc = f"{verb.capitalize()} a {np_}. The {param_noun} must be a string in {fmt_desc} format."
        param_desc = f"{param_noun} string in {fmt_desc} format."
        user_request = rng.choice(_ER_PHRASES).format(np=np_, value=valid_value, param=param_name)
        recent_errors = [f"Error: {param_name} must be a string like '{valid_value}', got {number}."]
        d_name, d_desc, d_params, d_args = _ER_DISTRACTORS[i % len(_ER_DISTRACTORS)]
        tools = [
            {"name": tool_name, "description": tool_desc, "parameters": {
                "type": "object",
                "properties": {param_name: {"type": "string", "description": param_desc}},
                "required": [param_name]}},
            {"name": d_name, "description": d_desc, "parameters": d_params},
            {"name": "respond", "description": "Send final answer.", "parameters": {
                "type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
        ]
        required = [tool_name]
        terminal = ["respond"]
        group_id = stable_id("forge_er_v2", i, user_request, tool_name, param_name, valid_value)
        valid_meta = {
            "generator": "forge_error_recovery_protected_v2",
            "scenario_family": "error_recovery_scaled",
            "source_kind": "synthetic_error_recovery",
            "corrected_positive": True,
            "valid_protection_corrected_error_recovery": True,
        }
        rows.append(make_row(
            "forge_error_recovery_protected", "valid",
            user_request, tools, {"name": tool_name, "arguments": {param_name: valid_value}}, 1.0,
            valid_meta, required_steps=required, completed_steps=[], pending_steps=required,
            terminal_tools=terminal, recent_errors=recent_errors, group_id=group_id, scoring_metadata=fm,
        ))
        seen = {json.dumps(valid_value, default=str, sort_keys=True)}
        for neg_type, neg_val in wrong_args:
            nk = json.dumps(neg_val, default=str, sort_keys=True)
            if nk in seen:
                continue
            seen.add(nk)
            rows.append(make_row(
                "forge_error_recovery_protected", "wrong_arguments_semantic",
                user_request, tools, {"name": tool_name, "arguments": {param_name: neg_val}}, 0.05,
                {"generator": "forge_error_recovery_protected_v2", "scenario_family": "error_recovery_scaled",
                 "source_kind": "synthetic_error_recovery", "negative_type": f"er_{neg_type}",
                 "valid_counterpart": valid_value},
                required_steps=required, completed_steps=[], pending_steps=required,
                terminal_tools=terminal, recent_errors=recent_errors, group_id=group_id, scoring_metadata=fm,
            ))
        rows.append(make_row(
            "forge_error_recovery_protected", "wrong_tool_semantic",
            user_request, tools, {"name": d_name, "arguments": d_args(valid_value)}, 0.05,
            {"generator": "forge_error_recovery_protected_v2", "scenario_family": "error_recovery_scaled",
             "source_kind": "synthetic_error_recovery", "negative_type": "er_wrong_tool"},
            required_steps=required, completed_steps=[], pending_steps=required,
            terminal_tools=terminal, recent_errors=recent_errors, group_id=group_id, scoring_metadata=fm,
        ))
    return rows


forge_rows = (
    build_forge_synthetic_rows()
    + build_argument_semantic_rows()
    + build_error_recovery_numeric_semantic_rows()
    + build_contrastive_wrong_tool_rows()
    + build_fixed_width_numeric_rows()
    + build_error_recovery_protected_rows()
    + build_fixed_width_numeric_rows_scaled()
    + build_error_recovery_protected_rows_scaled()
)
if ENABLE_FORGE_AUGMENTATION:
    forge_rows += build_guardrail_augmented_rows()

print("Forge rows:", len(forge_rows))
print(pd.Series([r.label for r in forge_rows]).value_counts())

In [ ]:
#@title Optional: load Forge eval/proxy trace JSONL and hard-negative files
# In Colab, upload python_oracle_colab_trace.jsonl and optional *hard_negatives.jsonl siblings to /content.
FORGE_TRACE_JSONL = "/content/python_oracle_colab_trace.jsonl"  #@param {type:"string"}
FORGE_HARD_NEGATIVE_GLOB = "/content/*hard_negatives.jsonl"  #@param {type:"string"}
FORGE_AGENT_TRAINING_GLOB = "/content/*agent_training*.jsonl"  #@param {type:"string"}

GROUNDED_SYNTHESIS_USER = "We have an open senior backend engineering role and four candidates in the pool. Review the role, evaluate the candidates, and submit your hiring decision with a brief rationale explaining why you selected (and did not select) the candidates you considered."
GROUNDED_SYNTHESIS_FACTS = ["Aisha Nakamura", "Sarah blocked by non-compete", "James blocked by visa or sponsorship", "Diana role-track mismatch"]
GROUNDED_SYNTHESIS_POSITIVE = "Selected Aisha Nakamura. Sarah Chen is blocked by the Stripe non-compete timeline, James Patel is blocked by visa sponsorship constraints, and Diana Kim is misaligned with the IC role track. Aisha has relevant payments experience and can start immediately."

SCENARIO_USER_MESSAGES = {
    "relevance_detection": "What is the square root of 144?",
    "tool_selection": "What permissions does Alice have?",
    "argument_fidelity": "Look up the entity 'Widget Pro' and get its details.",
    "sequential_reasoning": "Check drug interactions for patient John Doe's current medication.",
    "conditional_routing": "We got a P1 alert on the payments service. Diagnose the root cause and recommend an action.",
    "data_gap_recovery": "Pull together a complete profile for Sarah Chen — we need her clearance level and emergency contact for the onboarding audit.",
    "data_gap_recovery_extended": "Pull together a complete profile for Sarah Chen — we need her clearance level, current emergency contact, latest salary band, and active access groups for the onboarding audit and access review.",
    "argument_transformation": "Run our Q4 2024 expense audit. Flag any transaction of $5,000 or more from vendors NOT on our approved list. Submit the audit report with: comma-separated transaction IDs, total flagged amount in USD, and the vendor of the single largest flagged transaction.",
    "inconsistent_api_recovery": "Run a Q4 2024 (Oct 1 - Dec 31) compliance audit on Acme Corp (account ACC-12345). Pull the account balance and Q4 transactions, categorize at least one transaction, run a US-region compliance check for the period, calculate the subtotal of all transaction amounts in USD, and submit the audit report with keys: flagged_count (int), total_usd (str), compliance_status (str), txn_count (int).",
    "grounded_synthesis": GROUNDED_SYNTHESIS_USER,
    "basic_2step": "What is the capital of France?",
    "sequential_3step": "Generate a sales report from the Q4 2024 dataset.",
    "error_recovery": "Fetch 10 records and summarize them.",
}

def base_scenario_name(name: str) -> str:
    return str(name or "").removesuffix("_stateful")

def trace_user_request(rec: Dict[str, Any], scenario: str) -> str:
    for key in ["user_message", "prompt", "request", "user_request", "initial_user_request"]:
        value = rec.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return SCENARIO_USER_MESSAGES.get(base_scenario_name(scenario), "")

def tools_from_trace(tool_names: List[str]) -> List[Dict[str, Any]]:
    seen = []
    for name in tool_names:
        if name and name not in seen:
            seen.append(name)
    return [{"name": str(n), "description": "Tool observed in Forge eval trace.", "parameters": {"type": "object", "properties": {}}} for n in seen]

def trace_terminal_candidate(rec: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    tool_names = rec.get("tool_sequence") or []
    tool_args = rec.get("tool_args") or []
    if not tool_names:
        return None
    name = tool_names[-1]
    args = tool_args[-1] if tool_args and len(tool_args) >= len(tool_names) else {}
    return {"name": name, "arguments": args if isinstance(args, dict) else {"_raw": args}}

def load_forge_trace_rows(path: str) -> Tuple[List[VerifierRow], List[FinalResponseVerifierRow]]:
    if not path or not Path(path).exists():
        return [], []
    tool_rows: List[VerifierRow] = []
    final_rows: List[FinalResponseVerifierRow] = []
    with open(path) as f:
        for idx, line in enumerate(f):
            if not line.strip():
                continue
            rec = json.loads(line)
            scenario = str(rec.get("scenario") or "")
            user_request = trace_user_request(rec, scenario)
            tool_names = rec.get("tool_sequence") or []
            tool_args = rec.get("tool_args") or []
            tools = tools_from_trace(tool_names)
            if not user_request or not tools:
                continue
            group_id = stable_id("forge_trace", scenario, rec.get("run"), user_request)
            failed_accuracy = rec.get("accuracy") is False or rec.get("success") is False
            terminal = trace_terminal_candidate(rec)
            family_meta = infer_scoring_metadata(scenario)
            if failed_accuracy and terminal and "argument_transformation" in scenario:
                tool_rows.append(make_row(
                    "forge_trace", "wrong_arguments_semantic", user_request, tools, terminal, 0.05,
                    {"generator": "forge_trace", "trace_index": idx, "scenario_family": scenario, "failure_kind": "accuracy_false"},
                    required_steps=["list_transactions", "get_approved_vendors"], completed_steps=["list_transactions", "get_approved_vendors"], pending_steps=[], terminal_tools=[terminal["name"]], group_id=group_id, scoring_metadata=family_meta,
                ))
                tool_rows.append(make_row(
                    "forge_trace", "valid", ARG_TRANSFORMATION_USER, ARG_TRANSFORMATION_TOOLS, ARG_TRANSFORMATION_CANONICAL_CALL, 1.0,
                    {"generator": "forge_trace_canonical_positive", "scenario_family": scenario},
                    required_steps=["list_transactions", "get_approved_vendors"], completed_steps=["list_transactions", "get_approved_vendors"], pending_steps=[], terminal_tools=["submit_audit_report"], group_id=group_id, scoring_metadata=family_meta,
                ))
            else:
                for call_idx, name in enumerate(tool_names):
                    args = tool_args[call_idx] if call_idx < len(tool_args) else {}
                    label = "valid"
                    if bool(rec.get("required_step_mismatch")) or rec.get("missing_required_steps"):
                        label = "missing_prerequisite"
                    error_kind = str(rec.get("error_kind") or rec.get("error_type") or "")
                    if "unknown" in error_kind.lower():
                        label = "unknown_tool"
                    tool_rows.append(make_row(
                        "forge_trace", label, user_request, tools, {"name": name, "arguments": args if isinstance(args, dict) else {"_raw": args}},
                        1.0 if label == "valid" else 0.2,
                        {"generator": "forge_trace", "trace_index": idx, "error_kind": error_kind, "scenario_family": scenario},
                        group_id=group_id, scoring_metadata=family_meta,
                    ))
            if failed_accuracy and "grounded_synthesis" in scenario:
                candidate_final = str(rec.get("final_text") or "")
                final_rows.append(make_final_response_row(
                    "forge_trace", "contradicts_tool_result", user_request, candidate_final, 0.05,
                    workflow_state={"required_steps": ["get_open_role", "get_candidate_pool"], "completed_steps": ["get_open_role", "get_candidate_pool"], "pending_steps": [], "terminal_tools": ["submit_hiring_decision"], "recent_errors": []},
                    required_facts=GROUNDED_SYNTHESIS_FACTS,
                    tool_trace=tool_names,
                    tool_results=[{"tool_name": n, "content": compact_json(a, 500)} for n, a in zip(tool_names, tool_args)],
                    metadata={"generator": "forge_trace", "scenario_family": scenario, "failure_kind": "accuracy_false"},
                    group_id=group_id,
                ))
                final_rows.append(make_final_response_row(
                    "forge_trace", "valid_final_response", user_request, GROUNDED_SYNTHESIS_POSITIVE, 1.0,
                    workflow_state={"required_steps": ["get_open_role", "get_candidate_pool"], "completed_steps": ["get_open_role", "get_candidate_pool"], "pending_steps": [], "terminal_tools": ["submit_hiring_decision"], "recent_errors": []},
                    required_facts=GROUNDED_SYNTHESIS_FACTS,
                    tool_trace=tool_names,
                    tool_results=[{"tool_name": n, "content": compact_json(a, 500)} for n, a in zip(tool_names, tool_args)],
                    metadata={"generator": "forge_trace_canonical_positive", "scenario_family": scenario},
                    group_id=group_id,
                ))
    return tool_rows, final_rows

def hard_negative_list(value: Any) -> List[Any]:
    return value if isinstance(value, list) else []

def hard_negative_string_list(value: Any) -> List[str]:
    return [str(item) for item in hard_negative_list(value) if item is not None]

def hard_negative_workflow_state(context: Dict[str, Any]) -> Dict[str, List[str]]:
    raw = context.get("workflow_state") if isinstance(context, dict) else {}
    raw = raw if isinstance(raw, dict) else {}
    return {
        "required_steps": hard_negative_string_list(raw.get("required_steps")),
        "completed_steps": hard_negative_string_list(raw.get("completed_steps")),
        "pending_steps": hard_negative_string_list(raw.get("pending_steps")),
        "terminal_tools": hard_negative_string_list(raw.get("terminal_tools")),
        "recent_errors": hard_negative_string_list(raw.get("recent_errors")),
    }

def normalize_candidate_arguments(args: Any) -> Dict[str, Any]:
    if isinstance(args, dict):
        return args
    if isinstance(args, str):
        try:
            loaded = json.loads(args)
            if isinstance(loaded, dict):
                return loaded
        except Exception:
            pass
    if args is None:
        return {}
    return {"_raw": args}

def normalize_candidate_call(call: Any) -> Optional[Dict[str, Any]]:
    if not isinstance(call, dict):
        return None
    if isinstance(call.get("function"), dict):
        function = call["function"]
        name = function.get("name") or call.get("name") or "unknown_tool"
        args = function.get("arguments", call.get("arguments", {}))
        return {"name": str(name), "arguments": normalize_candidate_arguments(args)}
    if isinstance(call.get("tool"), dict):
        nested = normalize_candidate_call(call["tool"])
        if nested:
            return nested
    name = call.get("name") or call.get("tool_name") or call.get("tool") or "unknown_tool"
    args = call.get("arguments")
    if args is None:
        args = call.get("args", {})
    return {"name": str(name), "arguments": normalize_candidate_arguments(args)}

def candidate_calls_from_sequence(seq: List[Any], args: List[Any]) -> List[Dict[str, Any]]:
    calls = []
    for call_idx, name in enumerate(seq):
        if name is None:
            continue
        call_args = args[call_idx] if call_idx < len(args) else {}
        calls.append({"name": str(name), "arguments": normalize_candidate_arguments(call_args)})
    return calls

def candidate_calls_from_candidate(candidate: Dict[str, Any]) -> List[Dict[str, Any]]:
    seq = hard_negative_list(candidate.get("tool_sequence"))
    args = hard_negative_list(candidate.get("tool_args"))
    calls = []
    for call in hard_negative_list(candidate.get("candidate_calls")):
        normalized = normalize_candidate_call(call)
        if normalized:
            calls.append(normalized)
    single = normalize_candidate_call(candidate.get("candidate_call"))
    if single and single not in calls:
        calls.append(single)
    if not calls:
        calls = candidate_calls_from_sequence(seq, args)
    return calls

def corrected_candidate_calls_from_outcome(outcome: Dict[str, Any]) -> List[Dict[str, Any]]:
    calls = []
    for call in hard_negative_list(outcome.get("corrected_candidate_calls")):
        normalized = normalize_candidate_call(call)
        if normalized:
            calls.append(normalized)
    single = normalize_candidate_call(outcome.get("corrected_candidate_call"))
    if single and single not in calls:
        calls.append(single)
    corrected = outcome.get("corrected_positive")
    if not calls and isinstance(corrected, dict):
        for call in hard_negative_list(corrected.get("candidate_calls")):
            normalized = normalize_candidate_call(call)
            if normalized:
                calls.append(normalized)
        single = normalize_candidate_call(corrected.get("candidate_call"))
        if single and single not in calls:
            calls.append(single)
    return calls

def hard_negative_tools(context: Dict[str, Any], candidates: List[Dict[str, Any]], seq: List[Any]) -> List[Dict[str, Any]]:
    tools = context.get("available_tools") if isinstance(context, dict) else None
    if isinstance(tools, list):
        normalized_tools = [tool for tool in tools if isinstance(tool, dict)]
        if normalized_tools:
            return normalized_tools
    names = [str(call.get("name")) for call in candidates if call.get("name")]
    if not names:
        names = [str(name) for name in seq if name]
    return tools_from_trace(names)

def first_call_index_named(calls: List[Dict[str, Any]], names: List[str]) -> Optional[int]:
    wanted = {str(name) for name in names if name}
    if not wanted:
        return None
    for call_idx, call in enumerate(calls):
        if str(call.get("name")) in wanted:
            return call_idx
    return None

def last_call_index_named(calls: List[Dict[str, Any]], names: List[str]) -> Optional[int]:
    wanted = {str(name) for name in names if name}
    if not wanted:
        return None
    for call_idx in range(len(calls) - 1, -1, -1):
        if str(calls[call_idx].get("name")) in wanted:
            return call_idx
    return None

def selected_hard_negative_call(
    scenario: str,
    candidates: List[Dict[str, Any]],
    classifier_scores: List[Any],
    workflow_state: Dict[str, List[str]],
) -> Tuple[Dict[str, Any], int]:
    if not candidates:
        return {"name": "unknown_tool", "arguments": {}}, -1
    lower = scenario.lower()
    if "error_recovery" in lower:
        idx = first_call_index_named(candidates, workflow_state.get("required_steps", []))
        if idx is None:
            terminal_names = set(workflow_state.get("terminal_tools", []))
            idx = next((i for i, call in enumerate(candidates) if str(call.get("name")) not in terminal_names), None)
        if idx is not None:
            return candidates[idx], idx
    for score in classifier_scores:
        if not isinstance(score, dict):
            continue
        label = str(score.get("label") or "")
        if label and label != "valid":
            tool_name = score.get("tool") or score.get("name") or score.get("candidate_tool")
            idx = first_call_index_named(candidates, [str(tool_name)] if tool_name else [])
            if idx is not None:
                return candidates[idx], idx
    if "argument_transformation" in lower or "inconsistent_api_recovery" in lower:
        idx = last_call_index_named(candidates, workflow_state.get("terminal_tools", []))
        if idx is not None:
            return candidates[idx], idx
    idx = len(candidates) - 1
    return candidates[idx], idx

def workflow_state_for_call(
    base_state: Dict[str, List[str]],
    calls: List[Dict[str, Any]],
    call_index: int,
) -> Dict[str, List[str]]:
    state = {key: list(base_state.get(key, [])) for key in ["required_steps", "completed_steps", "pending_steps", "terminal_tools", "recent_errors"]}
    if call_index < 0:
        return state
    required = state["required_steps"]
    if required:
        prior_names = [str(call.get("name")) for call in calls[:call_index]]
        completed = [step for step in required if step in prior_names]
        state["completed_steps"] = completed
        state["pending_steps"] = [step for step in required if step not in completed]
    return state

def hard_negative_call_key(call: Dict[str, Any]) -> str:
    return json.dumps({
        "name": str(call.get("name") or ""),
        "arguments": call.get("arguments") if isinstance(call.get("arguments"), dict) else {},
    }, sort_keys=True, default=str)

def hard_negative_tool_label(
    scenario: str,
    candidate_call: Dict[str, Any],
    corrected_calls: List[Dict[str, Any]],
    workflow_state: Dict[str, List[str]],
) -> str:
    lower = scenario.lower()
    candidate_name = str(candidate_call.get("name") or "")
    candidate_key = hard_negative_call_key(candidate_call)
    corrected_keys = {hard_negative_call_key(call) for call in corrected_calls}
    if candidate_key in corrected_keys:
        return "valid"

    # error_recovery has a known numeric string invariant: the valid value is zero-padded.
    # Prior classifier telemetry falsely objected to count="0010" and allowed count="10".
    if "error_recovery" in lower and candidate_name == "fetch":
        args = candidate_call.get("arguments") if isinstance(candidate_call.get("arguments"), dict) else {}
        count = str(args.get("count") or "")
        if count == "0010":
            return "valid"
        if count in {"10", "0000"}:
            return "wrong_arguments_semantic"

    if "argument_transformation" in lower or "inconsistent_api_recovery" in lower:
        return "wrong_arguments_semantic"
    corrected_names = {str(call.get("name")) for call in corrected_calls if call.get("name")}
    terminal_names = {str(name) for name in workflow_state.get("terminal_tools", [])}
    if candidate_name in corrected_names or candidate_name in terminal_names:
        return "wrong_arguments_semantic"
    return "wrong_tool_semantic"

def hard_negative_final_label(scenario: str) -> str:
    lower = scenario.lower()
    if "data_gap" in lower:
        return "failed_to_acknowledge_data_gap"
    if "grounded_synthesis" in lower:
        return "contradicts_tool_result"
    return "missing_tool_fact"

def hard_negative_user_request(rec: Dict[str, Any], context: Dict[str, Any], outcome: Dict[str, Any], fallback: str) -> str:
    return str(
        context.get("user_request")
        or outcome.get("user_request")
        or rec.get("user_request")
        or rec.get("initial_user_request")
        or fallback
    )

def hard_negative_required_facts(context: Dict[str, Any], scenario: str) -> List[str]:
    facts = hard_negative_string_list(context.get("required_facts") if isinstance(context, dict) else [])
    if facts:
        return facts
    return GROUNDED_SYNTHESIS_FACTS if "grounded_synthesis" in scenario else []

def hard_negative_tool_trace(rec: Dict[str, Any], context: Dict[str, Any], candidate: Dict[str, Any]) -> List[str]:
    for value in [context.get("tool_trace") if isinstance(context, dict) else None, rec.get("tool_trace"), candidate.get("tool_sequence")]:
        trace = hard_negative_string_list(value)
        if trace:
            return trace
    calls = candidate_calls_from_candidate(candidate)
    return [str(call.get("name")) for call in calls if call.get("name")]

def hard_negative_tool_results(rec: Dict[str, Any], context: Dict[str, Any]) -> List[Dict[str, str]]:
    for value in [context.get("tool_results") if isinstance(context, dict) else None, rec.get("tool_results")]:
        if isinstance(value, list):
            rows = []
            for item in value:
                if isinstance(item, dict):
                    rows.append({"tool_name": str(item.get("tool_name") or item.get("name") or ""), "content": str(item.get("content") or item.get("result") or "")})
            if rows:
                return rows
    return []

def expand_jsonl_glob_patterns(pattern: str) -> List[str]:
    files = []
    for raw_pattern in str(pattern or "").split(","):
        raw_pattern = raw_pattern.strip()
        if raw_pattern:
            files.extend(glob.glob(raw_pattern))
    return sorted({str(Path(file_path)) for file_path in files})

def load_hard_negative_rows(pattern: str) -> Tuple[List[VerifierRow], List[FinalResponseVerifierRow]]:
    tool_rows: List[VerifierRow] = []
    final_rows: List[FinalResponseVerifierRow] = []
    files = expand_jsonl_glob_patterns(pattern)
    if files:
        print("Hard-negative files matched:", [Path(file_path).name for file_path in files])
    else:
        likely_files = sorted(glob.glob("/content/*hard_negatives.jsonl"))
        if likely_files:
            print(
                "WARNING: hard-negative files exist in /content but FORGE_HARD_NEGATIVE_GLOB matched none:",
                [Path(file_path).name for file_path in likely_files],
            )
    for file_path in files:
        with open(file_path) as f:
            for idx, line in enumerate(f):
                if not line.strip():
                    continue
                rec = json.loads(line)
                outcome = rec.get("outcome") or {}
                context = rec.get("context") or {}
                outcome = outcome if isinstance(outcome, dict) else {}
                context = context if isinstance(context, dict) else {}
                scenario = str(outcome.get("scenario_family") or outcome.get("scenario") or rec.get("scenario_family") or rec.get("scenario") or "")
                group_id = stable_id("hard_negative", file_path, idx, scenario, outcome.get("run"))
                workflow_state = hard_negative_workflow_state(context)
                user_request = hard_negative_user_request(rec, context, outcome, "Forge eval hard negative")
                family_meta = infer_scoring_metadata(scenario)
                metadata = {"generator": "forge_hard_negative", "scenario_family": scenario, "source_file": file_path, "source_index": idx}
                if rec.get("kind") == "tool_call":
                    candidate = rec.get("candidate") or {}
                    candidate = candidate if isinstance(candidate, dict) else {}
                    seq = hard_negative_list(candidate.get("tool_sequence"))
                    candidates = candidate_calls_from_candidate(candidate)
                    corrected_calls = corrected_candidate_calls_from_outcome(outcome)
                    terminal, terminal_idx = selected_hard_negative_call(
                        scenario,
                        candidates,
                        hard_negative_list(rec.get("classifier_scores")),
                        workflow_state,
                    )
                    state = workflow_state_for_call(workflow_state, candidates, terminal_idx)
                    tools = hard_negative_tools(context, candidates or corrected_calls, seq)
                    label = hard_negative_tool_label(scenario, terminal, corrected_calls, state)
                    row_metadata = dict(metadata)
                    row_metadata["candidate_index"] = terminal_idx
                    tool_rows.append(make_row(
                        "forge_hard_negative", label, user_request, tools, terminal, 0.05,
                        row_metadata,
                        required_steps=state["required_steps"], completed_steps=state["completed_steps"], pending_steps=state["pending_steps"], terminal_tools=state["terminal_tools"], recent_errors=state["recent_errors"],
                        group_id=group_id, scoring_metadata=family_meta,
                    ))
                    for corrected_idx, corrected_call in enumerate(corrected_calls):
                        corrected_state = workflow_state_for_call(workflow_state, corrected_calls, corrected_idx)
                        tool_rows.append(make_row(
                            "forge_hard_negative", "valid", user_request, tools, corrected_call, 1.0,
                            {**metadata, "generator": "forge_hard_negative_corrected_positive", "candidate_index": corrected_idx},
                            required_steps=corrected_state["required_steps"], completed_steps=corrected_state["completed_steps"], pending_steps=corrected_state["pending_steps"], terminal_tools=corrected_state["terminal_tools"], recent_errors=corrected_state["recent_errors"],
                            group_id=group_id, scoring_metadata=family_meta,
                        ))
                elif rec.get("kind") == "final_response":
                    candidate = rec.get("candidate") or {}
                    candidate = candidate if isinstance(candidate, dict) else {}
                    required_facts = hard_negative_required_facts(context, scenario)
                    final_state = {key: workflow_state[key] for key in ["required_steps", "completed_steps", "pending_steps", "terminal_tools", "recent_errors"]}
                    tool_trace = hard_negative_tool_trace(rec, context, candidate)
                    tool_results = hard_negative_tool_results(rec, context)
                    final_rows.append(make_final_response_row(
                        "forge_hard_negative", hard_negative_final_label(scenario), user_request, str(candidate.get("final_text") or ""), 0.05,
                        workflow_state=final_state,
                        required_facts=required_facts,
                        tool_trace=tool_trace,
                        tool_results=tool_results,
                        metadata=metadata,
                        group_id=group_id,
                    ))
                    corrected_final = outcome.get("corrected_final_response")
                    corrected_positive = outcome.get("corrected_positive")
                    if not isinstance(corrected_final, str) and isinstance(corrected_positive, dict):
                        corrected_final = corrected_positive.get("final_text")
                    if isinstance(corrected_final, str) and corrected_final.strip():
                        final_rows.append(make_final_response_row(
                            "forge_hard_negative", "valid_final_response", user_request, corrected_final, 1.0,
                            workflow_state=final_state,
                            required_facts=required_facts,
                            tool_trace=tool_trace,
                            tool_results=tool_results,
                            metadata={**metadata, "generator": "forge_hard_negative_corrected_positive"},
                            group_id=group_id,
                        ))
    return tool_rows, final_rows

def normalized_workflow_state(raw: Any) -> Dict[str, List[str]]:
    raw = raw if isinstance(raw, dict) else {}
    return {
        "required_steps": hard_negative_string_list(raw.get("required_steps")),
        "completed_steps": hard_negative_string_list(raw.get("completed_steps")),
        "pending_steps": hard_negative_string_list(raw.get("pending_steps")),
        "terminal_tools": hard_negative_string_list(raw.get("terminal_tools")),
        "recent_errors": hard_negative_string_list(raw.get("recent_errors")),
    }

def normalized_agent_tools(rec: Dict[str, Any], context: Dict[str, Any], candidate: Optional[Dict[str, Any]]) -> List[Dict[str, Any]]:
    for value in [rec.get("available_tools"), rec.get("tools"), context.get("available_tools")]:
        if isinstance(value, list):
            tools = [tool for tool in value if isinstance(tool, dict)]
            if tools:
                return tools
    if candidate and candidate.get("name"):
        return tools_from_trace([str(candidate["name"])])
    return []

def load_agent_training_rows(pattern: str) -> Tuple[List[VerifierRow], List[FinalResponseVerifierRow]]:
    files = sorted(glob.glob(pattern or ""))
    if files and not INCLUDE_PRIVATE_AGENT_LOGS:
        print(f"Skipping {len(files)} agent-training files because INCLUDE_PRIVATE_AGENT_LOGS=False")
        return [], []
    if files and not PRIVATE:
        raise RuntimeError("Agent-derived training rows require PRIVATE=True before upload.")

    tool_rows: List[VerifierRow] = []
    final_rows: List[FinalResponseVerifierRow] = []
    skipped = 0
    for file_path in files:
        with open(file_path) as f:
            for idx, line in enumerate(f):
                if not line.strip():
                    continue
                rec = json.loads(line)
                kind = str(rec.get("kind") or "tool_call")
                context = rec.get("context") if isinstance(rec.get("context"), dict) else {}
                metadata = rec.get("metadata") if isinstance(rec.get("metadata"), dict) else {}
                metadata = {**metadata, "generator": metadata.get("generator", "agent_training"), "source_file": file_path, "source_index": idx, "private_agent_log": True}
                scenario = str(metadata.get("scenario_family") or rec.get("scenario_family") or rec.get("scenario") or "agent_training")
                user_request = str(rec.get("user_request") or context.get("user_request") or metadata.get("user_request") or "").strip()
                workflow_state = normalized_workflow_state(rec.get("workflow_state") or context.get("workflow_state"))
                group_id = str(rec.get("example_group_id") or metadata.get("example_group_id") or stable_id("agent_training", file_path, idx, user_request, scenario))
                rank_score = float(rec.get("rank_score", 1.0))
                family_meta = rec.get("scoring_metadata") if isinstance(rec.get("scoring_metadata"), dict) else infer_scoring_metadata(scenario)

                if kind == "final_response":
                    label = str(rec.get("label") or "")
                    if label not in FINAL_RESPONSE_LABELS or not user_request:
                        skipped += 1
                        continue
                    candidate_text = rec.get("candidate_final_response")
                    if candidate_text is None and isinstance(rec.get("candidate"), dict):
                        candidate_text = rec["candidate"].get("final_text")
                    tool_results = rec.get("tool_results") or context.get("tool_results") or []
                    if not isinstance(tool_results, list):
                        tool_results = []
                    final_rows.append(make_final_response_row(
                        "agent_training", label, user_request, str(candidate_text or ""), rank_score,
                        workflow_state=workflow_state,
                        required_facts=hard_negative_string_list(rec.get("required_facts") or context.get("required_facts")),
                        tool_trace=hard_negative_string_list(rec.get("tool_trace") or context.get("tool_trace")),
                        tool_results=tool_results,
                        metadata={**metadata, "scenario_family": scenario, "scoring_metadata": family_meta},
                        group_id=group_id,
                    ))
                    continue

                raw_candidate = rec.get("candidate_call") or rec.get("candidate")
                if isinstance(raw_candidate, dict) and "candidate_call" in raw_candidate:
                    raw_candidate = raw_candidate.get("candidate_call")
                candidate = normalize_candidate_call(raw_candidate)
                label_raw = str(rec.get("label") or "")
                label = normalize_label(label_raw) if label_raw else ""
                tools = normalized_agent_tools(rec, context, candidate)
                if label not in LABELS or not user_request or not candidate or not tools:
                    skipped += 1
                    continue
                tool_rows.append(make_row(
                    "agent_training", label, user_request, tools, candidate, rank_score,
                    {**metadata, "scenario_family": scenario, "scoring_metadata": family_meta},
                    required_steps=workflow_state["required_steps"],
                    completed_steps=workflow_state["completed_steps"],
                    pending_steps=workflow_state["pending_steps"],
                    terminal_tools=workflow_state["terminal_tools"],
                    recent_errors=workflow_state["recent_errors"],
                    group_id=group_id,
                    scoring_metadata=family_meta,
                ))
    if skipped:
        print(f"Skipped {skipped} malformed or unsupported agent-training rows.")
    return tool_rows, final_rows

trace_tool_rows, final_response_trace_rows = load_forge_trace_rows(FORGE_TRACE_JSONL)
hard_negative_rows, final_response_hard_negative_rows = load_hard_negative_rows(FORGE_HARD_NEGATIVE_GLOB)
agent_training_rows, final_response_agent_training_rows = load_agent_training_rows(FORGE_AGENT_TRAINING_GLOB)
trace_rows = trace_tool_rows + hard_negative_rows + agent_training_rows
final_response_rows = final_response_trace_rows + final_response_hard_negative_rows + final_response_agent_training_rows
print("Forge trace rows:", len(trace_tool_rows))
print("Forge tool hard-negative rows:", len(hard_negative_rows))
print("Agent-training tool rows:", len(agent_training_rows))
print("Total tool rows from trace/hard-negative/agent loaders:", len(trace_rows))
print("Final-response trace rows:", len(final_response_trace_rows))
print("Final-response hard-negative rows:", len(final_response_hard_negative_rows))
print("Final-response agent-training rows:", len(final_response_agent_training_rows))
print("Total final-response rows:", len(final_response_rows))


In [ ]:
#@title Load private Forge agent-training dataset from Hugging Face
# The uploaded generatetd dataset is private and already sanitized/reviewed for the
# tool-call verifier. It is loaded separately from local agent logs so local log
# ingestion remains opt-in.

def _as_dict(value: Any) -> Dict[str, Any]:
    return value if isinstance(value, dict) else {}

def _coerce_float(value: Any, default: float = 1.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return float(default)

def private_agent_metadata(
    base: Optional[Dict[str, Any]],
    source_repo: str,
    source_file: str,
    source_split: str,
    source_index: int,
) -> Dict[str, Any]:
    metadata = dict(base or {})
    metadata["generator"] = metadata.get("generator", "agent_training")
    metadata["source_repo"] = source_repo
    metadata["source_file"] = source_file
    metadata["source_split"] = source_split
    metadata["source_index"] = source_index
    metadata["private_agent_log"] = True
    metadata["public_export_allowed"] = False
    return metadata

def canonical_tool_training_row_to_rows(
    rec: Dict[str, Any],
    source_repo: str,
    source_file: str,
    source_split: str,
    source_index: int,
) -> Tuple[List[VerifierRow], List[FinalResponseVerifierRow]]:
    input_obj = _as_dict(rec.get("input"))
    review = _as_dict(rec.get("review"))
    input_metadata = _as_dict(input_obj.get("metadata"))
    metadata = private_agent_metadata({**review, **input_metadata}, source_repo, source_file, source_split, source_index)
    scenario = str(metadata.get("scenario_family") or rec.get("scenario_family") or "agent_training")
    family_meta = _as_dict(input_metadata.get("scoring_metadata")) or infer_scoring_metadata(scenario)
    user_request = str(input_obj.get("user_request") or "").strip()
    workflow_state = normalized_workflow_state(input_obj.get("workflow_state"))
    candidate = normalize_candidate_call(input_obj.get("candidate_call"))
    tools = [tool for tool in input_obj.get("available_tools", []) if isinstance(tool, dict)]
    label = normalize_label(str(rec.get("label") or ""))
    if label not in LABELS or not user_request or not candidate or not tools:
        return [], []
    group_id = str(
        metadata.get("task_group_id")
        or metadata.get("example_group_id")
        or rec.get("id")
        or stable_id("agent_training_hf", source_repo, source_index, user_request, scenario)
    )
    rank_score = _coerce_float(review.get("verifier_confidence", review.get("confidence", rec.get("rank_score", 1.0))), 1.0)
    return [make_row(
        "agent_training_hf", label, user_request, tools, candidate, rank_score,
        {**metadata, "scenario_family": scenario, "scoring_metadata": family_meta},
        required_steps=workflow_state["required_steps"],
        completed_steps=workflow_state["completed_steps"],
        pending_steps=workflow_state["pending_steps"],
        terminal_tools=workflow_state["terminal_tools"],
        recent_errors=workflow_state["recent_errors"],
        group_id=group_id,
        scoring_metadata=family_meta,
    )], []

def canonical_final_training_row_to_rows(
    rec: Dict[str, Any],
    source_repo: str,
    source_file: str,
    source_split: str,
    source_index: int,
) -> Tuple[List[VerifierRow], List[FinalResponseVerifierRow]]:
    input_obj = _as_dict(rec.get("input"))
    review = _as_dict(rec.get("review"))
    input_metadata = _as_dict(input_obj.get("metadata"))
    metadata = private_agent_metadata({**review, **input_metadata}, source_repo, source_file, source_split, source_index)
    scenario = str(metadata.get("scenario_family") or rec.get("scenario_family") or "agent_training")
    label = str(rec.get("label") or "")
    user_request = str(input_obj.get("user_request") or "").strip()
    candidate_text = str(input_obj.get("candidate_final_response") or "")
    if label not in FINAL_RESPONSE_LABELS or not user_request:
        return [], []
    workflow_state = normalized_workflow_state(input_obj.get("workflow_state"))
    group_id = str(
        metadata.get("task_group_id")
        or metadata.get("example_group_id")
        or rec.get("id")
        or stable_id("agent_training_hf_final", source_repo, source_index, user_request, scenario)
    )
    rank_score = _coerce_float(review.get("verifier_confidence", review.get("confidence", rec.get("rank_score", 1.0))), 1.0)
    return [], [make_final_response_row(
        "agent_training_hf", label, user_request, candidate_text, rank_score,
        workflow_state=workflow_state,
        required_facts=hard_negative_string_list(input_obj.get("required_facts")),
        tool_trace=hard_negative_string_list(input_obj.get("tool_trace")),
        tool_results=input_obj.get("tool_results") if isinstance(input_obj.get("tool_results"), list) else [],
        metadata={**metadata, "scenario_family": scenario},
        group_id=group_id,
    )]

def adapter_agent_training_record_to_rows(
    rec: Dict[str, Any],
    source_repo: str,
    source_file: str,
    source_split: str,
    source_index: int,
) -> Tuple[List[VerifierRow], List[FinalResponseVerifierRow]]:
    kind = str(rec.get("kind") or "tool_call")
    context = _as_dict(rec.get("context"))
    raw_metadata = _as_dict(rec.get("metadata"))
    metadata = private_agent_metadata(raw_metadata, source_repo, source_file, source_split, source_index)
    scenario = str(metadata.get("scenario_family") or rec.get("scenario_family") or rec.get("scenario") or "agent_training")
    user_request = str(rec.get("user_request") or context.get("user_request") or metadata.get("user_request") or "").strip()
    workflow_state = normalized_workflow_state(rec.get("workflow_state") or context.get("workflow_state"))
    group_id = str(rec.get("example_group_id") or metadata.get("example_group_id") or stable_id("agent_training_hf", source_repo, source_index, user_request, scenario))
    rank_score = _coerce_float(rec.get("rank_score", 1.0), 1.0)
    family_meta = _as_dict(rec.get("scoring_metadata")) or _as_dict(metadata.get("scoring_metadata")) or infer_scoring_metadata(scenario)

    if kind == "final_response":
        label = str(rec.get("label") or "")
        if label not in FINAL_RESPONSE_LABELS or not user_request:
            return [], []
        candidate_text = rec.get("candidate_final_response")
        if candidate_text is None and isinstance(rec.get("candidate"), dict):
            candidate_text = rec["candidate"].get("final_text")
        tool_results = rec.get("tool_results") or context.get("tool_results") or []
        if not isinstance(tool_results, list):
            tool_results = []
        return [], [make_final_response_row(
            "agent_training_hf", label, user_request, str(candidate_text or ""), rank_score,
            workflow_state=workflow_state,
            required_facts=hard_negative_string_list(rec.get("required_facts") or context.get("required_facts")),
            tool_trace=hard_negative_string_list(rec.get("tool_trace") or context.get("tool_trace")),
            tool_results=tool_results,
            metadata={**metadata, "scenario_family": scenario, "scoring_metadata": family_meta},
            group_id=group_id,
        )]

    raw_candidate = rec.get("candidate_call") or rec.get("candidate")
    if isinstance(raw_candidate, dict) and "candidate_call" in raw_candidate:
        raw_candidate = raw_candidate.get("candidate_call")
    candidate = normalize_candidate_call(raw_candidate)
    label_raw = str(rec.get("label") or "")
    label = normalize_label(label_raw) if label_raw else ""
    tools = normalized_agent_tools(rec, context, candidate)
    if label not in LABELS or not user_request or not candidate or not tools:
        return [], []
    return [make_row(
        "agent_training_hf", label, user_request, tools, candidate, rank_score,
        {**metadata, "scenario_family": scenario, "scoring_metadata": family_meta},
        required_steps=workflow_state["required_steps"],
        completed_steps=workflow_state["completed_steps"],
        pending_steps=workflow_state["pending_steps"],
        terminal_tools=workflow_state["terminal_tools"],
        recent_errors=workflow_state["recent_errors"],
        group_id=group_id,
        scoring_metadata=family_meta,
    )], []

def agent_training_record_to_rows(
    rec: Dict[str, Any],
    source_repo: str,
    source_file: str,
    source_split: str,
    source_index: int,
) -> Tuple[List[VerifierRow], List[FinalResponseVerifierRow]]:
    schema_version = str(rec.get("schema_version") or "")
    if schema_version == "toolcall-verifier-training/v1":
        return canonical_tool_training_row_to_rows(rec, source_repo, source_file, source_split, source_index)
    if schema_version == "final-response-verifier-training/v1":
        return canonical_final_training_row_to_rows(rec, source_repo, source_file, source_split, source_index)
    return adapter_agent_training_record_to_rows(rec, source_repo, source_file, source_split, source_index)

def load_hf_agent_training_dataset(repo_id: str, data_file: str, revision: Optional[str] = None) -> Tuple[List[VerifierRow], List[FinalResponseVerifierRow]]:
    if not repo_id or not data_file:
        return [], []
    data_files = {"train": data_file}
    dataset_kwargs: Dict[str, Any] = {"data_files": data_files}
    if revision:
        dataset_kwargs["revision"] = revision
    try:
        dataset_dict = load_dataset(repo_id, **dataset_kwargs, token=True)
    except TypeError:
        dataset_dict = load_dataset(repo_id, **dataset_kwargs, use_auth_token=True)
    except Exception as exc:
        print(f"WARNING: could not load private Forge agent dataset {repo_id}: {type(exc).__name__}: {exc}")
        return [], []

    tool_rows: List[VerifierRow] = []
    final_rows: List[FinalResponseVerifierRow] = []
    skipped = 0
    split_names = list(dataset_dict.keys()) if isinstance(dataset_dict, DatasetDict) else ["train"]
    for split_name in split_names:
        split_rows = dataset_dict[split_name] if isinstance(dataset_dict, DatasetDict) else dataset_dict
        for idx, rec in enumerate(split_rows):
            try:
                next_tool_rows, next_final_rows = agent_training_record_to_rows(dict(rec), repo_id, data_file, split_name, idx)
            except Exception as exc:
                skipped += 1
                if skipped <= 5:
                    print(f"WARNING: skipped HF agent row {split_name}/{idx}: {type(exc).__name__}: {exc}")
                continue
            if not next_tool_rows and not next_final_rows:
                skipped += 1
                continue
            tool_rows.extend(next_tool_rows)
            final_rows.extend(next_final_rows)
    if skipped:
        print(f"Skipped {skipped} unsupported private Forge agent dataset rows.")
    return tool_rows, final_rows

hf_agent_training_rows: List[VerifierRow] = []
hf_final_response_rows: List[FinalResponseVerifierRow] = []
if globals().get("ENABLE_FORGE_AGENT_HF_DATASET", False):
    hf_agent_training_rows, hf_final_response_rows = load_hf_agent_training_dataset(
        FORGE_AGENT_HF_DATASET_REPO,
        FORGE_AGENT_HF_DATASET_FILE,
        globals().get("FORGE_AGENT_HF_DATASET_REVISION") or None,
    )
    trace_rows += hf_agent_training_rows
    final_response_rows += hf_final_response_rows

print("Private HF Forge agent rows:", len(hf_agent_training_rows))
print("Private HF final-response rows:", len(hf_final_response_rows))
print("Trace rows after private HF dataset:", len(trace_rows))
print("Final-response rows after private HF dataset:", len(final_response_rows))


In [ ]:
#@title Telemetry-only eval diagnostics
# Upload release eval files to /content to summarize failure slices. These rows are diagnostics only;
# this cell never appends them to trace_rows, final_response_rows, or any training split.
from collections import Counter

ENABLE_FORGE_TELEMETRY_DIAGNOSTICS = True  #@param {type:"boolean"}
FORGE_TELEMETRY_EVAL_GLOBS = "/content/proxy_classifier*.jsonl,/content/rust_smoke.jsonl,/content/*/proxy_classifier*.jsonl,/content/*/rust_smoke.jsonl"  #@param {type:"string"}

TERMINALISH_TOOL_NAMES = {"respond", "summarize", "report", "present", "recommend", "diagnose"}


def telemetry_paths_from_globs(globs: str) -> List[Path]:
    paths: List[Path] = []
    for pattern in [part.strip() for part in str(globs or "").split(",") if part.strip()]:
        paths.extend(Path(p) for p in glob.glob(pattern))
    unique = sorted({p.resolve() for p in paths if p.exists() and p.is_file()})
    return unique


def telemetry_load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with path.open() as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                print(f"WARNING: skipped malformed telemetry JSONL row {path}:{line_no}: {exc}")
    return rows


def telemetry_counter(counter: Counter) -> Dict[str, int]:
    return {str(k): int(v) for k, v in counter.most_common()}


def telemetry_tool_is_terminalish(tool_name: Any) -> bool:
    name = str(tool_name or "")
    lower = name.lower()
    return lower in TERMINALISH_TOOL_NAMES or lower.startswith("submit_") or "summar" in lower or "report" in lower


def telemetry_walk_values(value: Any) -> Iterable[Any]:
    if isinstance(value, dict):
        for child in value.values():
            yield from telemetry_walk_values(child)
    elif isinstance(value, list):
        for child in value:
            yield from telemetry_walk_values(child)
    else:
        yield value


def telemetry_has_numeric_string(value: Any) -> bool:
    return any(isinstance(item, str) and item.isdigit() for item in telemetry_walk_values(value))


def telemetry_has_fixed_width_numeric(value: Any) -> bool:
    for item in telemetry_walk_values(value):
        if isinstance(item, str) and item.isdigit() and len(item) > 1 and item.startswith("0"):
            return True
    return False


def telemetry_valid_probability(score: Dict[str, Any]) -> Optional[float]:
    for item in score.get("top_k") or []:
        if item.get("label") == "valid":
            try:
                return float(item.get("confidence"))
            except (TypeError, ValueError):
                return None
    return None


def summarize_proxy_classifier_telemetry(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    terminal_rows = []
    numeric_rows = []
    fixed_width_rows = []
    close_valid_margin = []
    for row in rows:
        workflow_terminal = set((row.get("workflow_state") or {}).get("terminal_tools") or [])
        tool = row.get("tool")
        if tool in workflow_terminal or telemetry_tool_is_terminalish(tool):
            terminal_rows.append(row)
        if telemetry_has_numeric_string(row.get("candidate_call")):
            numeric_rows.append(row)
        if telemetry_has_fixed_width_numeric(row.get("candidate_call")):
            fixed_width_rows.append(row)
        valid_prob = telemetry_valid_probability(row)
        conf = row.get("confidence")
        if row.get("label") != "valid" and valid_prob is not None and conf is not None:
            margin = abs(float(conf) - valid_prob)
            if margin <= 0.05:
                close_valid_margin.append({
                    "tool": tool,
                    "label": row.get("label"),
                    "confidence": float(conf),
                    "valid_confidence": valid_prob,
                    "margin": margin,
                    "user_request": row.get("initial_user_request") or row.get("user_request"),
                    "candidate_call": row.get("candidate_call"),
                })
    return {
        "kind": "proxy_classifier",
        "rows": len(rows),
        "predicted_labels": telemetry_counter(Counter(row.get("label") for row in rows)),
        "actions": telemetry_counter(Counter(row.get("action") for row in rows)),
        "terminal_or_terminal_like": {
            "rows": len(terminal_rows),
            "predicted_labels": telemetry_counter(Counter(row.get("label") for row in terminal_rows)),
            "tools": telemetry_counter(Counter(row.get("tool") for row in terminal_rows)),
        },
        "numeric_string_candidates": {
            "rows": len(numeric_rows),
            "predicted_labels": telemetry_counter(Counter(row.get("label") for row in numeric_rows)),
            "tools": telemetry_counter(Counter(row.get("tool") for row in numeric_rows)),
        },
        "fixed_width_numeric_candidates": {
            "rows": len(fixed_width_rows),
            "predicted_labels": telemetry_counter(Counter(row.get("label") for row in fixed_width_rows)),
            "tools": telemetry_counter(Counter(row.get("tool") for row in fixed_width_rows)),
        },
        "non_valid_top_label_with_close_valid_margin": sorted(close_valid_margin, key=lambda item: item["margin"])[:25],
    }


def summarize_rust_smoke_telemetry(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    all_scores = []
    successful_valid_proxy = []
    false_objection_tools = Counter()
    false_objection_scenarios = Counter()
    terminal_false_objections = Counter()
    numeric_false_objections = Counter()
    fixed_width_false_objections = Counter()
    for row in rows:
        row_numeric = telemetry_has_numeric_string(row.get("tool_args"))
        row_fixed_width = telemetry_has_fixed_width_numeric(row.get("tool_args"))
        for score in row.get("classifier_scores") or []:
            item = {
                "scenario": row.get("scenario"),
                "run": row.get("run"),
                "tool": score.get("tool"),
                "label": score.get("label"),
                "confidence": score.get("confidence"),
                "valid_confidence": telemetry_valid_probability(score),
                "accuracy": row.get("accuracy"),
                "success": row.get("success"),
            }
            all_scores.append(item)
            if row.get("success") is True and row.get("accuracy") is True:
                successful_valid_proxy.append(item)
                if score.get("label") != "valid":
                    false_objection_tools[score.get("tool")] += 1
                    false_objection_scenarios[row.get("scenario")] += 1
                    if telemetry_tool_is_terminalish(score.get("tool")):
                        terminal_false_objections[score.get("tool")] += 1
                    if row_numeric:
                        numeric_false_objections[score.get("tool")] += 1
                    if row_fixed_width:
                        fixed_width_false_objections[score.get("tool")] += 1
    valid_predictions = sum(1 for item in successful_valid_proxy if item.get("label") == "valid")
    valid_proxy_count = len(successful_valid_proxy)
    return {
        "kind": "rust_smoke",
        "rows": len(rows),
        "classifier_scores": len(all_scores),
        "predicted_labels": telemetry_counter(Counter(item.get("label") for item in all_scores)),
        "successful_accurate_scores_as_valid_proxy": valid_proxy_count,
        "valid_recall_proxy": float(valid_predictions / valid_proxy_count) if valid_proxy_count else None,
        "valid_proxy_predicted_labels": telemetry_counter(Counter(item.get("label") for item in successful_valid_proxy)),
        "false_objection_tools": telemetry_counter(false_objection_tools),
        "false_objection_scenarios": telemetry_counter(false_objection_scenarios),
        "terminal_false_objection_tools": telemetry_counter(terminal_false_objections),
        "numeric_string_false_objection_tools": telemetry_counter(numeric_false_objections),
        "fixed_width_numeric_false_objection_tools": telemetry_counter(fixed_width_false_objections),
    }


def build_telemetry_diagnostics() -> Dict[str, Any]:
    paths = telemetry_paths_from_globs(FORGE_TELEMETRY_EVAL_GLOBS) if ENABLE_FORGE_TELEMETRY_DIAGNOSTICS else []
    report: Dict[str, Any] = {
        "schema_version": "toolcall-verifier-telemetry-diagnostics/v1",
        "telemetry_only": True,
        "used_for_training": False,
        "notes": [
            "Use these rows to diagnose false objections and dataset gaps.",
            "Do not treat top-k telemetry from the current artifact as promotion evidence.",
        ],
        "files": [],
    }
    for path in paths:
        rows = telemetry_load_jsonl(path)
        name = path.name
        if "proxy_classifier" in name:
            summary = summarize_proxy_classifier_telemetry(rows)
        elif name == "rust_smoke.jsonl":
            summary = summarize_rust_smoke_telemetry(rows)
        else:
            summary = {"kind": "unknown_jsonl", "rows": len(rows)}
        report["files"].append({"path": str(path), **summary})
    return report


telemetry_diagnostics = build_telemetry_diagnostics()
(DATA_DIR / "telemetry_diagnostics.json").write_text(json.dumps(telemetry_diagnostics, indent=2, ensure_ascii=False))
print("Telemetry diagnostics files:", len(telemetry_diagnostics["files"]))
for item in telemetry_diagnostics["files"]:
    print(item["kind"], item["rows"], item["path"])


## 5. Convert public generation rows into verifier/ranker rows

For each gold call, generate a valid candidate plus hard negatives. All candidates derived from the same original example share `example_group_id` so split leakage is reduced.

In [ ]:
def required_args_for_tool(tool: Dict[str, Any]) -> List[str]:
    params = tool.get("parameters") or {}
    req = params.get("required") or []
    return [str(x) for x in req] if isinstance(req, list) else []


def tool_by_name(tools: List[Dict[str, Any]], name: str) -> Optional[Dict[str, Any]]:
    for t in tools:
        if t.get("name") == name:
            return t
    return None


# v5b: schema-compatible wrong-tool selection replaces the random distractor.
# Random distractors with verbatim gold args let the model shortcut on schema mismatch
# instead of intent, and near-duplicate tools on xLAM/ToolACE made many "wrong tool"
# labels noisy (several tools were semantically plausible).
WRONG_TOOL_AMBIGUITY_SKIP_THRESHOLD = 0.60  #@param {type:"number"}
WRONG_TOOL_MIN_COMPAT_SCORE = 0.15  #@param {type:"number"}
_WRONG_TOOL_RNG = random.Random(SEED + 9173)
_DESC_STOPWORDS = {
    "the", "a", "an", "of", "to", "for", "in", "on", "by", "with", "and", "or",
    "is", "are", "be", "this", "that", "it", "as", "at", "from", "into", "you", "your",
}


def _tool_props(tool: Dict[str, Any]) -> Dict[str, Any]:
    params = tool.get("parameters") or {}
    props = params.get("properties") or {}
    return props if isinstance(props, dict) else {}


def _tool_required_set(tool: Dict[str, Any]) -> set:
    return set(required_args_for_tool(tool))


def _desc_tokens(text: Any) -> set:
    cleaned = "".join(ch if ch.isalnum() else " " for ch in str(text or "").lower())
    return {t for t in cleaned.split() if t not in _DESC_STOPWORDS}


def _desc_overlap(a: Any, b: Any) -> float:
    ta, tb = _desc_tokens(a), _desc_tokens(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


def _json_type_of(value: Any) -> str:
    if isinstance(value, bool):
        return "boolean"
    if isinstance(value, int):
        return "integer"
    if isinstance(value, float):
        return "number"
    if isinstance(value, str):
        return "string"
    if isinstance(value, list):
        return "array"
    if isinstance(value, dict):
        return "object"
    return "null"


def _schema_type_compatible(value: Any, spec: Any) -> bool:
    if not isinstance(spec, dict):
        return True
    declared = spec.get("type")
    if not declared:
        return True
    declared = [str(d) for d in (declared if isinstance(declared, list) else [declared])]
    vt = _json_type_of(value)
    if vt == "integer" and "number" in declared:
        return True
    return vt in declared


def _fill_value_for_schema(spec: Any, rng: random.Random) -> Any:
    spec = spec if isinstance(spec, dict) else {}
    enum = spec.get("enum")
    if isinstance(enum, list) and enum:
        return rng.choice(enum)
    if "default" in spec:
        return spec["default"]
    declared = spec.get("type")
    if isinstance(declared, list) and declared:
        declared = declared[0]
    if declared == "integer":
        return rng.randint(1, 9)
    if declared == "number":
        return float(rng.randint(1, 9))
    if declared == "boolean":
        return True
    if declared == "array":
        return []
    if declared == "object":
        return {}
    return "example"


def _ambiguity_score(gold_tool: Optional[Dict[str, Any]], cand_tool: Dict[str, Any]) -> float:
    """High score = the candidate tool is plausibly interchangeable with the gold tool,
    so labeling it wrong_tool_semantic would inject noise rather than signal."""
    if not gold_tool:
        return 0.0
    score = _desc_overlap(gold_tool.get("description"), cand_tool.get("description"))
    same_schema = (
        bool(_tool_props(cand_tool))
        and set(_tool_props(gold_tool)) == set(_tool_props(cand_tool))
        and _tool_required_set(gold_tool) == _tool_required_set(cand_tool)
    )
    if same_schema:
        score += 0.25
    return score


def _schema_compat_score(gold_args: Dict[str, Any], cand_tool: Dict[str, Any]) -> float:
    props = _tool_props(cand_tool)
    if not props:
        return 0.05
    carried = [k for k in gold_args if k in props]
    score = len(carried) / max(1, len(gold_args))
    if carried:
        typed = sum(1 for k in carried if _schema_type_compatible(gold_args[k], props.get(k)))
        score += 0.25 * (typed / len(carried))
    score -= 0.10 * len(_tool_required_set(cand_tool) - set(gold_args))
    return score


def _remap_args_onto_tool(gold_args: Dict[str, Any], cand_tool: Dict[str, Any], rng: random.Random) -> Dict[str, Any]:
    props = _tool_props(cand_tool)
    if not props:
        return dict(gold_args)
    carried = {k: v for k, v in gold_args.items() if k in props}
    for req in sorted(_tool_required_set(cand_tool) - set(carried)):
        carried[req] = _fill_value_for_schema(props.get(req), rng)
    return carried


def schema_compatible_wrong_tool_candidate(
    tools: List[Dict[str, Any]], gold_name: str, gold_args: Dict[str, Any]
) -> Optional[Tuple[Dict[str, Any], Dict[str, Any]]]:
    """Pick the most schema-compatible non-gold tool so the only mismatch is intent.

    Near-duplicate tools (high description overlap or identical schemas) are skipped.
    Name similarity is deliberately NOT filtered: get_/set_ and approve_/reject_
    pairs are exactly the hard negatives the classifier needs.

    Returns None only when every distractor is ambiguity-filtered (or none exist).
    WRONG_TOOL_MIN_COMPAT_SCORE decides the metadata tag (schema_compatible vs
    low_overlap), not whether a negative is emitted."""
    gold_tool = tool_by_name(tools, gold_name)
    gold_args = gold_args or {}
    scored = []
    for t in tools:
        name = t.get("name")
        if not name or name == gold_name:
            continue
        ambiguity = _ambiguity_score(gold_tool, t)
        if ambiguity >= WRONG_TOOL_AMBIGUITY_SKIP_THRESHOLD:
            continue
        compat = _schema_compat_score(gold_args, t)
        scored.append((compat, -ambiguity, str(name), t, ambiguity))
    if not scored:
        return None
    scored.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
    compat, _, _, tool, ambiguity = scored[0]
    # Low compat must not suppress emission: on arg-name-disjoint tool sets (most of
    # xLAM/glaive) a hard gate zeroed the wrong_tool class in the first v5b run.
    # The best non-ambiguous distractor is still a legitimate wrong tool; remapped
    # args keep it schema-plausible so the model cannot shortcut on schema mismatch.
    candidate = {
        "name": tool.get("name", "unknown_tool"),
        "arguments": _remap_args_onto_tool(gold_args, tool, _WRONG_TOOL_RNG),
    }
    info = {
        "ambiguity_score": round(float(ambiguity), 4),
        "compat_score": round(float(compat), 4),
        "distractor_tool": tool.get("name", "unknown_tool"),
        "schema_compatible": bool(compat >= WRONG_TOOL_MIN_COMPAT_SCORE),
    }
    return candidate, info


def missing_arg_candidate(tool: Dict[str, Any], call: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    req = required_args_for_tool(tool)
    if not req:
        return None
    args = dict(call.get("arguments") or {})
    removed = False
    for k in req:
        if k in args:
            args.pop(k)
            removed = True
            break
    if not removed:
        args = {}
    return {"name": call["name"], "arguments": args}


def invalid_shape_candidate(call: Dict[str, Any]) -> Dict[str, Any]:
    return {"name": call["name"], "arguments": {"_raw": "not an object or wrong shape"}}


def wrong_argument_candidate(call: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    args = dict(call.get("arguments") or {})
    if not args:
        return None
    key = next(iter(args.keys()))
    value = args[key]
    mutated = dict(args)
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        mutated[key] = value + 1 if value != 0 else 999
    elif isinstance(value, str):
        mutated[key] = value[::-1] if len(value) > 1 else value + "_wrong"
    elif isinstance(value, bool):
        mutated[key] = not value
    else:
        mutated[key] = "wrong_value"
    if mutated == args:
        return None
    return {"name": call["name"], "arguments": mutated}


# Public valid/wrong-tool/wrong-argument coverage is the backbone. Deterministic failures are
# still represented, but Rust owns them, so do not let synthetic deterministic negatives dominate.
PUBLIC_MISSING_REQUIRED_ARGS_KEEP_PROB = 0.35  #@param {type:"number"}
PUBLIC_INVALID_ARG_SHAPE_KEEP_PROB = 0.08  #@param {type:"number"}
PUBLIC_UNKNOWN_TOOL_KEEP_PROB = 0.08  #@param {type:"number"}
PUBLIC_MALFORMED_TOOL_CALL_KEEP_PROB = 0.05  #@param {type:"number"}
PUBLIC_TOOL_NOT_NEEDED_KEEP_PROB = 0.20  #@param {type:"number"}


def keep_public_deterministic_negative(probability: float) -> bool:
    return random.random() < max(0.0, min(1.0, float(probability)))


# v5b C4: needs_clarification support from underspecified public-request variants.
# The gold call's key argument value is replaced with a vague reference, so calling
# the gold tool with the original concrete arguments is no longer justified.
NEEDS_CLARIFICATION_TARGET_ROWS = 1000  #@param {type:"integer"}

_NC_GENERIC_VAGUE = ["that one", "the usual one", "the one I mentioned", "the one from before"]
_NC_PARAM_VAGUE = {
    "id": "one of my records",
    "name": "that one",
    "city": "my city",
    "location": "around here",
    "date": "whenever it was",
    "time": "sometime soon",
    "count": "some of them",
    "number": "a few",
    "amount": "some amount",
    "query": "that thing",
    "symbol": "that stock",
    "email": "my usual address",
}


def _vague_replacement(param_name: str, rng: random.Random) -> str:
    key = str(param_name or "").lower()
    for token, phrase in _NC_PARAM_VAGUE.items():
        if token in key:
            return phrase
    return rng.choice(_NC_GENERIC_VAGUE)


def _droppable_argument(call: Dict[str, Any], user_request: str) -> Optional[Tuple[str, str]]:
    for k, v in (call.get("arguments") or {}).items():
        if isinstance(v, bool):
            continue
        if not isinstance(v, (str, int, float)):
            continue
        text = str(v)
        if len(text) >= 3 and text in user_request:
            return str(k), text
    return None


def build_needs_clarification_rows(
    examples: List[Dict[str, Any]], target_rows: int = NEEDS_CLARIFICATION_TARGET_ROWS
) -> List[VerifierRow]:
    """Underspecified variants of public requests. Shares the example group_id so each
    variant co-splits with the gold valid row emitted by make_public_rows, giving the
    model paired contrast between justified and unjustified concrete arguments.

    ~15% of the budget is spent on borderline cases distinguishing needs_clarification
    from valid-with-inferable-context and from tool_not_needed."""
    rng = random.Random(SEED + 5519)
    eligible = []
    for ex in examples:
        gold_calls = ex.get("gold_calls") or []
        if not gold_calls:
            continue
        call = gold_calls[0]
        if not call.get("name"):
            continue
        hit = _droppable_argument(call, ex.get("user_request") or "")
        if hit:
            eligible.append((ex, call, hit))
    if len(eligible) > int(target_rows):
        eligible = rng.sample(eligible, int(target_rows))
    rows: List[VerifierRow] = []
    nc_label_counts = Counter()
    for ex, call, (param_name, surface) in eligible:
        source = ex["source"]
        user_request = ex["user_request"]
        tools = ex["tools"]
        group_id = ex.get("group_id") or stable_id(source, user_request, tools, ex.get("gold_calls"))
        vague = _vague_replacement(param_name, rng)
        vague_request = user_request.replace(surface, vague, 1)
        roll = rng.random()
        if roll < 0.85:
            rows.append(make_row(
                source=source,
                label="needs_clarification",
                user_request=vague_request,
                tools=tools,
                candidate=call,
                rank_score=0.20,
                metadata={"generator": "underspecified_variant", "negative_type": "needs_clarification_entity_drop",
                          "dropped_param": param_name, "gold_tool": call["name"]},
                group_id=group_id,
            ))
            nc_label_counts["needs_clarification"] += 1
        elif roll < 0.90:
            # Borderline valid: vague phrasing, but the concrete value is restated, so
            # the gold call stays fully inferable. No negative_type, so the suspicious
            # valid hard-negative audit in the dataframe cell passes.
            inferable_request = f"{vague_request} To be specific, I mean {surface}."
            rows.append(make_row(
                source=source,
                label="valid",
                user_request=inferable_request,
                tools=tools,
                candidate=call,
                rank_score=0.90,
                metadata={"generator": "nc_borderline_inferable", "dropped_param": param_name,
                          "gold_tool": call["name"]},
                group_id=group_id,
            ))
            nc_label_counts["valid"] += 1
        elif roll < 0.95:
            # Borderline tool_not_needed: explanation-only request answerable without tools.
            explain_request = (
                f"Briefly explain what information you would need before you could look up {vague}."
            )
            rows.append(make_row(
                source=source,
                label="tool_not_needed",
                user_request=explain_request,
                tools=tools,
                candidate={"text_response": "I would need the specific identifier before any lookup."},
                rank_score=0.30,
                metadata={"generator": "nc_borderline_explanation", "negative_type": "text_instead_of_tool",
                          "gold_tool": call["name"]},
                group_id=group_id,
            ))
            nc_label_counts["tool_not_needed"] += 1
        else:
            # Borderline needs_clarification: a hedge that still names no concrete entity.
            hedged_request = f"{vague_request} Whichever makes sense."
            rows.append(make_row(
                source=source,
                label="needs_clarification",
                user_request=hedged_request,
                tools=tools,
                candidate=call,
                rank_score=0.20,
                metadata={"generator": "underspecified_variant", "negative_type": "needs_clarification_hedged",
                          "dropped_param": param_name, "gold_tool": call["name"]},
                group_id=group_id,
            ))
            nc_label_counts["needs_clarification"] += 1
    print("needs_clarification builder label counts:", dict(nc_label_counts))
    return rows


def make_public_rows(examples: List[Dict[str, Any]]) -> List[VerifierRow]:
    rows = []
    deterministic_counts = Counter()
    wrong_tool_counts: Dict[str, Counter] = {}
    for ex in examples:
        source = ex["source"]
        user_request = ex["user_request"]
        tools = ex["tools"]
        gold_calls = ex["gold_calls"]
        group_id = ex.get("group_id") or stable_id(source, user_request, tools, gold_calls)
        for call in gold_calls:
            if not call.get("name"):
                continue
            call_tool = tool_by_name(tools, call["name"])
            rows.append(make_row(
                source=source,
                label="valid",
                user_request=user_request,
                tools=tools,
                candidate=call,
                rank_score=1.0,
                metadata={"generator": "public_gold", "gold_tool": call["name"]},
                group_id=group_id,
            ))
            wrong_pair = schema_compatible_wrong_tool_candidate(tools, call["name"], call.get("arguments") or {})
            if wrong_pair:
                wrong, wrong_info = wrong_pair
                if wrong_info.pop("schema_compatible"):
                    wt_negative_type = "wrong_tool_schema_compatible"
                    wrong_tool_counts.setdefault(source, Counter())["kept_schema_compatible"] += 1
                else:
                    wt_negative_type = "wrong_tool_low_overlap"
                    wrong_tool_counts.setdefault(source, Counter())["kept_low_overlap"] += 1
                rows.append(make_row(
                    source=source,
                    label="wrong_tool_semantic",
                    user_request=user_request,
                    tools=tools,
                    candidate=wrong,
                    rank_score=0.15,
                    metadata={"generator": "hard_negative", "negative_type": wt_negative_type, "gold_tool": call["name"], **wrong_info},
                    group_id=group_id,
                ))
            else:
                wrong_tool_counts.setdefault(source, Counter())["skipped"] += 1
            if call_tool:
                missing = missing_arg_candidate(call_tool, call)
                if missing and keep_public_deterministic_negative(PUBLIC_MISSING_REQUIRED_ARGS_KEEP_PROB):
                    deterministic_counts["missing_required_args"] += 1
                    rows.append(make_row(
                        source=source,
                        label="missing_required_args",
                        user_request=user_request,
                        tools=tools,
                        candidate=missing,
                        rank_score=0.10,
                        metadata={"generator": "hard_negative", "negative_type": "missing_required_args", "gold_tool": call["name"]},
                        group_id=group_id,
                    ))
                wrong_args = wrong_argument_candidate(call)
                if wrong_args:
                    rows.append(make_row(
                        source=source,
                        label="wrong_arguments_semantic",
                        user_request=user_request,
                        tools=tools,
                        candidate=wrong_args,
                        rank_score=0.08,
                        metadata={"generator": "hard_negative", "negative_type": "wrong_argument_value", "gold_tool": call["name"]},
                        group_id=group_id,
                    ))
            if keep_public_deterministic_negative(PUBLIC_INVALID_ARG_SHAPE_KEEP_PROB):
                deterministic_counts["invalid_args_schema"] += 1
                rows.append(make_row(
                    source=source,
                    label="invalid_args_schema",
                    user_request=user_request,
                    tools=tools,
                    candidate=invalid_shape_candidate(call),
                    rank_score=0.05,
                    metadata={"generator": "hard_negative", "negative_type": "invalid_arg_shape", "gold_tool": call["name"]},
                    group_id=group_id,
                ))
            if keep_public_deterministic_negative(PUBLIC_UNKNOWN_TOOL_KEEP_PROB):
                deterministic_counts["unknown_tool"] += 1
                rows.append(make_row(
                    source=source,
                    label="unknown_tool",
                    user_request=user_request,
                    tools=tools,
                    candidate={"name": "unknown_tool", "arguments": call.get("arguments") or {}},
                    rank_score=0.01,
                    metadata={"generator": "hard_negative", "negative_type": "unknown_tool", "gold_tool": call["name"]},
                    group_id=group_id,
                ))
            if keep_public_deterministic_negative(PUBLIC_MALFORMED_TOOL_CALL_KEEP_PROB):
                deterministic_counts["malformed_tool_call"] += 1
                rows.append(make_row(
                    source=source,
                    label="malformed_tool_call",
                    user_request=user_request,
                    tools=tools,
                    candidate={"malformed": "missing name/function fields"},
                    rank_score=0.01,
                    metadata={"generator": "hard_negative", "negative_type": "malformed_tool_call", "gold_tool": call["name"]},
                    group_id=group_id,
                ))
            if random.random() < PUBLIC_TOOL_NOT_NEEDED_KEEP_PROB:
                rows.append(make_row(
                    source=source,
                    label="tool_not_needed",
                    user_request=user_request,
                    tools=tools,
                    candidate={"text_response": "I can answer this directly without tools."},
                    rank_score=0.30,
                    metadata={"generator": "hard_negative", "negative_type": "text_instead_of_tool", "gold_tool": call["name"]},
                    group_id=group_id,
                ))
    print("Wrong-tool distractor selection by source (ambiguity-filtered):")
    for src_name in sorted(wrong_tool_counts):
        counts = wrong_tool_counts[src_name]
        total = sum(counts.values())
        skip_rate = counts["skipped"] / total if total else 0.0
        print(
            f"  {src_name}: schema_compatible={counts['kept_schema_compatible']} "
            f"low_overlap={counts['kept_low_overlap']} skipped={counts['skipped']} "
            f"skip_rate={skip_rate:.1%}"
        )
    print("Public deterministic negative rows kept:", dict(deterministic_counts))
    return rows


public_rows = make_public_rows(normalized_examples)
needs_clarification_rows = build_needs_clarification_rows(normalized_examples)
all_rows = public_rows + needs_clarification_rows + forge_rows + trace_rows
print("public rows:", len(public_rows))
print("needs_clarification rows:", len(needs_clarification_rows))
print("forge rows:", len(forge_rows))
print("trace rows:", len(trace_rows))
print("all rows:", len(all_rows))

In [ ]:
#@title Build dataframe, audit labels, and cap oversized classes with groups
# The previous run failed because broad valid-call recall collapsed. Before training,
# audit label/source leakage and keep generated pairs grouped when trimming large classes.

FAIL_ON_SUSPICIOUS_VALID_HARD_NEGATIVES = True  #@param {type:"boolean"}
MAX_SUSPICIOUS_VALID_SAMPLE_ROWS = 50  #@param {type:"integer"}
MIN_SOURCE_ROWS_FOR_TRAIN = 25  #@param {type:"integer"}
AUTO_QUARANTINE_LOW_SUPPORT_SOURCES = True  #@param {type:"boolean"}
QUARANTINE_LOW_SUPPORT_SOURCES = {
    "forge_argument_semantic",
    "forge_contrastive_wts",
}

# Reflect the config-cell flag for filtering deterministic_invalid from ML training.
_EXCLUDE_DI = bool(globals().get("EXCLUDE_DETERMINISTIC_INVALID_FROM_ML_TRAINING", True))



# deterministic_invalid filtering happens after normalize_label maps raw labels onto the
# collapsed bucket (a previous filter here matched the post-mapping name against raw
# labels such as missing_required_args, so it silently removed nothing).

def metadata_value(metadata: Any, key: str, default: Any = None) -> Any:
    return metadata.get(key, default) if isinstance(metadata, dict) else default


def metadata_dict(value: Any) -> Dict[str, Any]:
    return value if isinstance(value, dict) else {}


def print_source_label_summary(frame: pd.DataFrame, title: str) -> None:
    print(f"\n{title} by source and label:")
    print(pd.crosstab(frame["source"], frame["label"]).to_string())
    generators = frame["metadata"].map(lambda m: metadata_value(m, "generator", "unknown"))
    print(f"\n{title} by generator:")
    print(generators.value_counts(dropna=False).to_string())
    private_rows = int(frame["metadata"].map(lambda m: bool(metadata_value(m, "private_agent_log", False))).sum())
    if private_rows:
        print(f"\nPrivate agent-derived rows: {private_rows}")
        if not PRIVATE:
            raise RuntimeError("Private agent-derived rows require PRIVATE=True before upload.")


def rows_to_df(rows: List[VerifierRow]) -> pd.DataFrame:
    return pd.DataFrame([asdict(r) for r in rows])


def is_preferred_private_hf_agent(metadata: Any) -> bool:
    if not globals().get("PREFER_FORGE_AGENT_HF_DATASET", False):
        return False
    if not isinstance(metadata, dict):
        return False
    return metadata.get("source_repo") == globals().get("FORGE_AGENT_HF_DATASET_REPO")


def ensure_group_ids(frame: pd.DataFrame, group_col: str = "example_group_id") -> pd.DataFrame:
    out = frame.copy()
    if group_col not in out.columns:
        out[group_col] = None
    missing = out[group_col].isna() | (out[group_col].astype(str).str.len() == 0)
    if missing.any():
        out.loc[missing, group_col] = [
            stable_id("row_group", out.loc[idx, "id"] if "id" in out.columns else int(idx))
            for idx in out.index[missing]
        ]
    return out


def crosstab_to_dict(frame: pd.DataFrame, row_col: str, col_col: str) -> Dict[str, Dict[str, int]]:
    if frame.empty or row_col not in frame.columns or col_col not in frame.columns:
        return {}
    table = pd.crosstab(frame[row_col].fillna("<missing>").astype(str), frame[col_col].fillna("<missing>").astype(str))
    return {
        str(index): {str(column): int(value) for column, value in row.items()}
        for index, row in table.iterrows()
    }


def metadata_series(frame: pd.DataFrame, key: str, default: str = "<missing>") -> pd.Series:
    return frame["metadata"].map(lambda m: str(metadata_value(m, key, default)))


def is_corrected_positive_metadata(metadata: Any) -> bool:
    md = metadata_dict(metadata)
    text = " ".join(str(md.get(k, "")) for k in ["generator", "negative_type", "scenario_family", "source_kind", "correction_kind", "rationale"]).lower()
    if any(token in text for token in ["corrected_positive", "corrected valid", "recovery_positive", "positive_pair"]):
        return True
    if bool(md.get("corrected_positive")) or bool(md.get("valid_counterpart")):
        return True
    return False


def suspicious_valid_hard_negative_reason(row: pd.Series) -> Optional[str]:
    if str(row.get("label")) != "valid":
        return None
    md = metadata_dict(row.get("metadata"))
    if is_corrected_positive_metadata(md):
        return None
    generator = str(md.get("generator", "")).lower()
    negative_type = str(md.get("negative_type", "")).lower()
    if generator == "hard_negative":
        return "valid row uses generator=hard_negative"
    if negative_type and not any(token in negative_type for token in ["corrected", "positive", "valid_counterpart"]):
        return f"valid row has negative_type={negative_type}"
    return None


def extract_candidate_payload_for_audit(input_text: Any) -> Any:
    text = str(input_text or "")
    marker = "CANDIDATE_CALL:\n"
    if marker not in text:
        return None
    raw = text.split(marker, 1)[1].strip()
    # v3 layout: CANDIDATE_TOOL_SCHEMA follows the candidate call; strip it first or
    # json.loads receives trailing sections and every v3 row loses its payload flags.
    schema_marker = "\n\nCANDIDATE_TOOL_SCHEMA:"
    if schema_marker in raw:
        raw = raw.split(schema_marker, 1)[0].strip()
    metadata_marker = "\n\nSCORING_METADATA:"
    if metadata_marker in raw:
        raw = raw.split(metadata_marker, 1)[0].strip()
    try:
        return json.loads(raw)
    except Exception:
        return None


def candidate_payloads_for_audit(value: Any) -> List[Dict[str, Any]]:
    if isinstance(value, list):
        return [item for item in value if isinstance(item, dict)]
    if isinstance(value, dict):
        return [value]
    return []


def candidate_tool_names_for_audit(value: Any) -> List[str]:
    names = []
    for payload in candidate_payloads_for_audit(value):
        if isinstance(payload.get("function"), dict):
            name = payload["function"].get("name")
        else:
            name = payload.get("name") or payload.get("tool_name") or payload.get("tool")
        if name:
            names.append(str(name))
    return names


def candidate_arguments_for_audit(value: Any) -> List[Dict[str, Any]]:
    args = []
    for payload in candidate_payloads_for_audit(value):
        raw_args = payload.get("arguments")
        if raw_args is None and isinstance(payload.get("function"), dict):
            raw_args = payload["function"].get("arguments")
        if isinstance(raw_args, str):
            try:
                raw_args = json.loads(raw_args)
            except Exception:
                raw_args = {"_raw": raw_args}
        args.append(raw_args if isinstance(raw_args, dict) else {})
    return args


def walk_audit_values(value: Any) -> Iterable[Any]:
    if isinstance(value, dict):
        for child in value.values():
            yield from walk_audit_values(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_audit_values(child)
    else:
        yield value


def has_fixed_width_numeric_string_for_audit(value: Any) -> bool:
    for item in walk_audit_values(value):
        if isinstance(item, str) and item.isdigit() and len(item) > 1 and item.startswith("0"):
            return True
    return False


def is_terminal_like_tool_name_for_audit(name: Any) -> bool:
    lower = str(name or "").lower()
    terminalish = globals().get("TERMINALISH_TOOL_NAMES", {"respond", "summarize", "report", "present", "recommend", "diagnose"})
    return lower in terminalish or lower.startswith("submit_") or "summar" in lower or "report" in lower


def add_training_valid_protection_flags(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    payloads = out["input_text"].map(extract_candidate_payload_for_audit)
    names = payloads.map(candidate_tool_names_for_audit)
    args = payloads.map(candidate_arguments_for_audit)
    out["valid_protection_terminal_like_tool"] = names.map(lambda xs: any(is_terminal_like_tool_name_for_audit(x) for x in xs))
    out["valid_protection_fixed_width_numeric_string"] = args.map(has_fixed_width_numeric_string_for_audit)
    out["valid_protection_noop_call"] = args.map(lambda xs: bool(xs) and all(x == {} for x in xs))
    out["valid_protection_corrected_error_recovery"] = out.apply(
        lambda row: (
            str(row.get("label")) == "valid"
            and "error_recovery" in str(metadata_value(row.get("metadata"), "scenario_family", ""))
            and is_corrected_positive_metadata(row.get("metadata"))
        ),
        axis=1,
    )
    protection_cols = [
        "valid_protection_terminal_like_tool",
        "valid_protection_fixed_width_numeric_string",
        "valid_protection_noop_call",
        "valid_protection_corrected_error_recovery",
    ]
    out["valid_protection_any"] = out[protection_cols].any(axis=1)
    return out


def dataframe_json_sample(frame: pd.DataFrame, max_rows: int) -> List[Dict[str, Any]]:
    rows = []
    columns = ["id", "source", "raw_label", "label", "example_group_id", "metadata"]
    for item in frame.head(max_rows)[[c for c in columns if c in frame.columns]].to_dict(orient="records"):
        md = metadata_dict(item.get("metadata"))
        item["metadata"] = {
            "generator": md.get("generator"),
            "negative_type": md.get("negative_type"),
            "scenario_family": md.get("scenario_family"),
            "source_repo": md.get("source_repo"),
            "source_file": md.get("source_file"),
            "source_index": md.get("source_index"),
        }
        rows.append(item)
    return rows


def build_dataset_audit_report(frame: pd.DataFrame, suspicious: pd.DataFrame) -> Dict[str, Any]:
    generator = metadata_series(frame, "generator")
    negative_type = metadata_series(frame, "negative_type")
    private_mask = frame["metadata"].map(lambda m: bool(metadata_value(m, "private_agent_log", False)))
    valid_rows = frame[frame["label"] == "valid"]
    report = {
        "schema_version": "toolcall-verifier-dataset-audit/v1",
        "baseline_note": "Added after run with validation valid_recall ~= 0.41 and test valid_recall ~= 0.38.",
        "rows": int(len(frame)),
        "groups": int(frame["example_group_id"].nunique()),
        "label_counts": {str(k): int(v) for k, v in frame["label"].value_counts(dropna=False).items()},
        "raw_label_counts": {str(k): int(v) for k, v in frame["raw_label"].value_counts(dropna=False).items()},
        "source_label_counts": crosstab_to_dict(frame, "source", "label"),
        "generator_label_counts": crosstab_to_dict(frame.assign(_generator=generator), "_generator", "label"),
        "negative_type_label_counts": crosstab_to_dict(frame.assign(_negative_type=negative_type), "_negative_type", "label"),
        "private_agent_rows": int(private_mask.sum()),
        "private_agent_label_counts": {str(k): int(v) for k, v in frame.loc[private_mask, "label"].value_counts(dropna=False).items()},
        "valid_fraction": float(len(valid_rows) / len(frame)) if len(frame) else 0.0,
        "valid_protection_counts": {
            "terminal_like_tool_valid": int(((frame["label"] == "valid") & frame["valid_protection_terminal_like_tool"]).sum()),
            "fixed_width_numeric_string_valid": int(((frame["label"] == "valid") & frame["valid_protection_fixed_width_numeric_string"]).sum()),
            "noop_valid_call_valid": int(((frame["label"] == "valid") & frame["valid_protection_noop_call"]).sum()),
            "corrected_error_recovery_valid": int(((frame["label"] == "valid") & frame["valid_protection_corrected_error_recovery"]).sum()),
        },
        "suspicious_valid_hard_negative_rows": int(len(suspicious)),
        "suspicious_valid_hard_negative_sample": dataframe_json_sample(suspicious, MAX_SUSPICIOUS_VALID_SAMPLE_ROWS),
        "public_gold_nonvalid_rows": int(((frame["label"] != "valid") & (generator == "public_gold")).sum()),
        "public_gold_nonvalid_sample": dataframe_json_sample(frame[(frame["label"] != "valid") & (generator == "public_gold")], MAX_SUSPICIOUS_VALID_SAMPLE_ROWS),
        "fail_on_suspicious_valid_hard_negatives": bool(FAIL_ON_SUSPICIOUS_VALID_HARD_NEGATIVES),
    }
    return report


def quarantine_low_support_sources(frame: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    source_counts = {str(k): int(v) for k, v in frame["source"].astype(str).value_counts(dropna=False).items()}
    configured_sources = {str(source) for source in QUARANTINE_LOW_SUPPORT_SOURCES}
    low_support_sources = {
        source
        for source, rows in source_counts.items()
        if int(rows) < int(MIN_SOURCE_ROWS_FOR_TRAIN)
    }
    quarantine_sources = set(configured_sources)
    if AUTO_QUARANTINE_LOW_SUPPORT_SOURCES:
        quarantine_sources |= low_support_sources
    quarantine_sources = {source for source in quarantine_sources if source in source_counts}
    if not quarantine_sources:
        report = {
            "schema_version": "toolcall-verifier-source-quality/v1",
            "status": "no_sources_quarantined",
            "min_source_rows_for_train": int(MIN_SOURCE_ROWS_FOR_TRAIN),
            "auto_quarantine_low_support_sources": bool(AUTO_QUARANTINE_LOW_SUPPORT_SOURCES),
            "configured_quarantine_sources": sorted(configured_sources),
            "low_support_sources": sorted(low_support_sources),
            "source_counts_before": source_counts,
            "rows_before": int(len(frame)),
            "rows_after": int(len(frame)),
            "rows_quarantined": 0,
        }
        (DATA_DIR / "source_quality_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
        return frame.reset_index(drop=True), report

    mask = frame["source"].astype(str).isin(quarantine_sources)
    quarantined = frame[mask].copy()
    kept = frame[~mask].copy()
    if kept.empty:
        raise RuntimeError(
            "Source-quality quarantine removed every row. Lower MIN_SOURCE_ROWS_FOR_TRAIN or narrow "
            "QUARANTINE_LOW_SUPPORT_SOURCES before training."
        )

    quarantine_path = DATA_DIR / "source_quality_quarantine.jsonl"
    quarantined.to_json(quarantine_path, orient="records", lines=True, force_ascii=False)
    report = {
        "schema_version": "toolcall-verifier-source-quality/v1",
        "status": "sources_quarantined",
        "min_source_rows_for_train": int(MIN_SOURCE_ROWS_FOR_TRAIN),
        "auto_quarantine_low_support_sources": bool(AUTO_QUARANTINE_LOW_SUPPORT_SOURCES),
        "configured_quarantine_sources": sorted(configured_sources),
        "low_support_sources": sorted(low_support_sources),
        "quarantined_sources": sorted(quarantine_sources),
        "source_counts_before": source_counts,
        "source_counts_quarantined": {str(k): int(v) for k, v in quarantined["source"].astype(str).value_counts(dropna=False).items()},
        "label_counts_quarantined": {str(k): int(v) for k, v in quarantined["label"].value_counts(dropna=False).items()},
        "rows_before": int(len(frame)),
        "rows_after": int(len(kept)),
        "rows_quarantined": int(len(quarantined)),
        "quarantine_file": str(quarantine_path),
        "note": "Quarantined rows are exported for manual review or separate eval; they are not used for this training split.",
    }
    (DATA_DIR / "source_quality_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
    print("Source-quality quarantine:")
    print(json.dumps({k: report[k] for k in ["status", "quarantined_sources", "rows_quarantined", "rows_after"]}, indent=2))
    return kept.reset_index(drop=True), report


def cap_dataset_by_label_preserving_groups(frame: pd.DataFrame, max_per_label: int) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    frame = ensure_group_ids(frame)
    if max_per_label <= 0 or frame.empty:
        return frame.reset_index(drop=True), {"status": "disabled"}
    before_counts = {str(k): int(v) for k, v in frame["label"].value_counts().items()}
    if all(count <= max_per_label for count in before_counts.values()):
        return frame.reset_index(drop=True), {"status": "no_cap_needed", "max_per_label": int(max_per_label), "label_counts_before": before_counts}

    group_rows = []
    for group_id, group in frame.groupby("example_group_id", sort=False):
        counts = {str(k): int(v) for k, v in group["label"].value_counts().items()}
        preferred = bool(group.get("preferred_private_hf_agent", pd.Series(False, index=group.index)).fillna(False).astype(bool).any())
        valid_protection = bool(group.get("valid_protection_any", pd.Series(False, index=group.index)).fillna(False).astype(bool).any())
        has_valid = bool((group["label"] == "valid").any())
        group_rows.append({
            "group_id": group_id,
            "rows": int(len(group)),
            "counts": counts,
            "preferred": preferred,
            "valid_protection": valid_protection,
            "has_valid": has_valid,
        })

    rng = random.Random(SEED)
    rng.shuffle(group_rows)
    selected_groups = set()
    selected_counts: Counter[str] = Counter()

    def select_group(info: Dict[str, Any]) -> None:
        selected_groups.add(info["group_id"])
        for label, count in info["counts"].items():
            selected_counts[label] += int(count)

    for info in [item for item in group_rows if item["preferred"]]:
        select_group(info)

    remaining_groups = [item for item in group_rows if item["group_id"] not in selected_groups]
    remaining_groups.sort(key=lambda item: (not item["valid_protection"], not item["has_valid"], item["rows"]))
    for info in remaining_groups:
        labels = list(info["counts"].keys())
        if any(selected_counts[label] < max_per_label for label in labels):
            select_group(info)

    capped = frame[frame["example_group_id"].isin(selected_groups)].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    after_counts = {str(k): int(v) for k, v in capped["label"].value_counts().items()}
    report = {
        "status": "group_preserving_cap_applied",
        "max_per_label": int(max_per_label),
        "rows_before": int(len(frame)),
        "rows_after": int(len(capped)),
        "groups_before": int(frame["example_group_id"].nunique()),
        "groups_after": int(capped["example_group_id"].nunique()),
        "label_counts_before": before_counts,
        "label_counts_after": after_counts,
        "selected_preferred_private_groups": int(sum(1 for item in group_rows if item["preferred"] and item["group_id"] in selected_groups)),
    }
    return capped, report


df = rows_to_df(all_rows)
if df.empty:
    raise RuntimeError("No training rows were created. Inspect dataset schemas or add Forge traces.")

df["raw_label"] = df["label"]
df["label"] = df["label"].map(normalize_label)
if _EXCLUDE_DI:
    _di_rows = int((df["label"] == "deterministic_invalid").sum())
    if _di_rows:
        df = df[df["label"] != "deterministic_invalid"].reset_index(drop=True)
        print(f"[EXCLUDE_DI] Removed {_di_rows} deterministic_invalid rows from ML training/eval (deterministic rules own them).")
df["example_group_id"] = df["metadata"].map(lambda m: m.get("example_group_id") if isinstance(m, dict) else None)
df["toolset_hash"] = df["metadata"].map(lambda m: m.get("toolset_hash") if isinstance(m, dict) else None)
df["preferred_private_hf_agent"] = df["metadata"].map(is_preferred_private_hf_agent)
df = ensure_group_ids(df)
df = add_training_valid_protection_flags(df)

suspicious_reasons = df.apply(suspicious_valid_hard_negative_reason, axis=1)
df["dataset_audit_warning"] = suspicious_reasons.fillna("")
suspicious_valid_hard_negatives = df[suspicious_reasons.notna()].copy()
public_gold_nonvalid_rows = df[(df["label"] != "valid") & (metadata_series(df, "generator") == "public_gold")].copy()

print("Raw labels:")
print(df["raw_label"].value_counts(dropna=False))
print("\nTraining labels:")
print(df["label"].value_counts(dropna=False))
print("\nGroups:", df["example_group_id"].nunique())
preferred_rows = int(df["preferred_private_hf_agent"].sum())
if preferred_rows:
    print("Preferred private HF agent rows:", preferred_rows)
    print(df.loc[df["preferred_private_hf_agent"], "label"].value_counts().to_string())
print_source_label_summary(df, "Raw training rows")

if len(suspicious_valid_hard_negatives):
    print("\nSuspicious valid rows carrying hard-negative metadata:", len(suspicious_valid_hard_negatives))
    display(suspicious_valid_hard_negatives[["id", "source", "label", "dataset_audit_warning", "metadata"]].head(MAX_SUSPICIOUS_VALID_SAMPLE_ROWS))
if len(public_gold_nonvalid_rows):
    print("\nNon-valid rows carrying public_gold metadata:", len(public_gold_nonvalid_rows))
    display(public_gold_nonvalid_rows[["id", "source", "raw_label", "label", "metadata"]].head(MAX_SUSPICIOUS_VALID_SAMPLE_ROWS))

dataset_audit_report = build_dataset_audit_report(df, suspicious_valid_hard_negatives)
(DATA_DIR / "dataset_audit_report.json").write_text(json.dumps(dataset_audit_report, indent=2, ensure_ascii=False))
if len(suspicious_valid_hard_negatives) and FAIL_ON_SUSPICIOUS_VALID_HARD_NEGATIVES:
    raise RuntimeError(
        "Suspicious valid rows carry hard-negative metadata. "
        "Inspect dataset_audit_report.json before retraining."
    )
if len(public_gold_nonvalid_rows):
    raise RuntimeError(
        "Non-valid rows carry public_gold metadata. This indicates label/metadata corruption before training. "
        "Inspect dataset_audit_report.json before retraining."
    )

df, source_quality_report = quarantine_low_support_sources(df)
dataset_audit_report["source_quality_report"] = source_quality_report
(DATA_DIR / "dataset_audit_report.json").write_text(json.dumps(dataset_audit_report, indent=2, ensure_ascii=False))

balanced, label_cap_report = cap_dataset_by_label_preserving_groups(df, MAX_PER_LABEL)
dataset_audit_report["label_cap_report"] = label_cap_report
(DATA_DIR / "dataset_audit_report.json").write_text(json.dumps(dataset_audit_report, indent=2, ensure_ascii=False))

print("\nCapped labels:")
print(balanced["label"].value_counts())
if preferred_rows:
    kept_preferred = int(balanced["preferred_private_hf_agent"].sum())
    print("Preferred private HF rows kept after caps:", kept_preferred)
print("Label cap report:")
print(json.dumps(label_cap_report, indent=2)[:4000])
print_source_label_summary(balanced, "Capped training rows")
balanced.head()

## 6. Group-safe train/validation/test split

Rows generated from the same source example share a group. The split is by group, not by row, so gold calls and hard negatives from one example do not leak across train and test.

In [ ]:
#@title Group-safe split, private mix, and train-only valid protection rebalance
from collections import Counter

LOW_RESOURCE_TOOLCALL_PROFILE = RUN_PROFILE in {"debug_smoke", "t4_fast", "t4_proven"}
APPLY_T4_REBALANCE_SAFETY_DEFAULTS = LOW_RESOURCE_TOOLCALL_PROFILE  #@param {type:"boolean"}
ENABLE_VALID_PROTECTION_TRAIN_REBALANCE = True  #@param {type:"boolean"}
VALID_TRAIN_FRACTION_TARGET = 0.40  #@param {type:"number"}
VALID_TRAIN_MAX_DUPLICATION_FACTOR = 2  #@param {type:"integer"}
MAX_DETERMINISTIC_INVALID_TO_VALID_TRAIN_RATIO = 0.35  #@param {type:"number"}
MAX_WRONG_TOOL_TO_VALID_TRAIN_RATIO = 0.75  #@param {type:"number"}
MAX_WRONG_ARGUMENTS_TO_VALID_TRAIN_RATIO = 0.90  #@param {type:"number"}
MAX_TOOL_NOT_NEEDED_TO_VALID_TRAIN_RATIO = 0.30  #@param {type:"number"}
MAX_NEEDS_CLARIFICATION_TO_VALID_TRAIN_RATIO = 0.15  #@param {type:"number"}
ENABLE_SEMANTIC_NEGATIVE_TRAIN_REBALANCE = False  #@param {type:"boolean"}
WRONG_TOOL_TRAIN_TO_VALID_RATIO_TARGET = 0.90  #@param {type:"number"}
WRONG_ARGUMENTS_TRAIN_TO_VALID_RATIO_TARGET = 0.75  #@param {type:"number"}
MAX_SEMANTIC_NEGATIVE_DUPLICATION_FACTOR = 4  #@param {type:"integer"}
ENABLE_VALID_PROTECTION_EXTRA_TRAIN_REBALANCE = False  #@param {type:"boolean"}
VALID_PROTECTION_EXTRA_COPY_FACTOR = 2  #@param {type:"integer"}
VALID_PROTECTION_EXTRA_COPY_ROWS_CAP = 5000  #@param {type:"integer"}

if APPLY_T4_REBALANCE_SAFETY_DEFAULTS:
    VALID_TRAIN_FRACTION_TARGET = 0.40
    ENABLE_SEMANTIC_NEGATIVE_TRAIN_REBALANCE = False
    WRONG_TOOL_TRAIN_TO_VALID_RATIO_TARGET = 0.55
    WRONG_ARGUMENTS_TRAIN_TO_VALID_RATIO_TARGET = 0.70
    MAX_SEMANTIC_NEGATIVE_DUPLICATION_FACTOR = 2
    ENABLE_VALID_PROTECTION_EXTRA_TRAIN_REBALANCE = False
    print(
        "Applied T4 rebalance safety defaults: valid target=0.40, semantic-negative rebalance off, "
        "protected-valid extra duplication off."
    )


def group_split(
    frame: pd.DataFrame,
    group_col: str = "example_group_id",
    valid_size: float = VALID_SIZE,
    test_size: float = TEST_SIZE,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    frame = frame.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    if group_col not in frame or frame[group_col].isna().all() or frame[group_col].nunique() < 5:
        print("WARNING: insufficient groups; falling back to stratified row split.")
        strat = frame["label"] if frame["label"].value_counts().min() >= 2 else None
        train_df, tmp_df = train_test_split(frame, test_size=valid_size + test_size, random_state=SEED, stratify=strat)
        tmp_strat = tmp_df["label"] if tmp_df["label"].value_counts().min() >= 2 else None
        valid_df, test_df = train_test_split(tmp_df, test_size=test_size / (valid_size + test_size), random_state=SEED, stratify=tmp_strat)
        return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)

    splitter = GroupShuffleSplit(n_splits=1, test_size=valid_size + test_size, random_state=SEED)
    train_idx, tmp_idx = next(splitter.split(frame, groups=frame[group_col]))
    train_df = frame.iloc[train_idx].copy()
    tmp_df = frame.iloc[tmp_idx].copy()

    tmp_test_fraction = test_size / (valid_size + test_size)
    if tmp_df[group_col].nunique() < 2:
        valid_df = tmp_df.copy()
        test_df = tmp_df.copy()
    else:
        splitter2 = GroupShuffleSplit(n_splits=1, test_size=tmp_test_fraction, random_state=SEED + 1)
        valid_idx, test_idx = next(splitter2.split(tmp_df, groups=tmp_df[group_col]))
        valid_df = tmp_df.iloc[valid_idx].copy()
        test_df = tmp_df.iloc[test_idx].copy()

    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)


def mark_weighted_metadata(metadata: Any, copy_idx: int, weight: int) -> Dict[str, Any]:
    out = dict(metadata) if isinstance(metadata, dict) else {}
    out["weight_copy"] = copy_idx
    out["weight_total"] = weight
    out["weighted_split"] = "train"
    return out


def private_hf_mask(frame: pd.DataFrame) -> pd.Series:
    if "preferred_private_hf_agent" not in frame.columns:
        return pd.Series(False, index=frame.index)
    return frame["preferred_private_hf_agent"].fillna(False).astype(bool)


def ensure_group_ids(frame: pd.DataFrame, group_col: str = "example_group_id") -> pd.DataFrame:
    out = frame.copy()
    if group_col not in out.columns:
        out[group_col] = None
    missing = out[group_col].isna() | (out[group_col].astype(str).str.len() == 0)
    if missing.any():
        out.loc[missing, group_col] = [
            stable_id("row_group", out.loc[idx, "id"] if "id" in out.columns else int(idx))
            for idx in out.index[missing]
        ]
    return out


def proportional_label_row_targets(frame: pd.DataFrame, target_rows: int, label_col: str = "label") -> Dict[str, int]:
    counts = frame[label_col].astype(str).value_counts()
    total = int(counts.sum())
    if target_rows <= 0 or total <= 0:
        return {}
    quotas: Dict[str, int] = {}
    fractional: List[Tuple[float, str]] = []
    remaining = int(target_rows)
    for label, count in counts.items():
        raw = target_rows * (int(count) / total)
        quota = min(int(count), int(math.floor(raw)))
        quotas[str(label)] = quota
        remaining -= quota
        fractional.append((raw - quota, str(label)))
    for _, label in sorted(fractional, reverse=True):
        if remaining <= 0:
            break
        if quotas[label] < int(counts[label]):
            quotas[label] += 1
            remaining -= 1
    return quotas


def sample_groups_for_row_budget(
    frame: pd.DataFrame,
    target_rows: int,
    group_col: str = "example_group_id",
    label_col: str = "label",
) -> pd.DataFrame:
    if target_rows <= 0 or frame.empty:
        return frame.iloc[0:0]
    frame = ensure_group_ids(frame, group_col)
    if target_rows >= len(frame):
        return frame

    group_rows = []
    for group_id, group in frame.groupby(group_col, sort=False):
        labels = group[label_col].astype(str).value_counts()
        primary_label = str(labels.index[0]) if len(labels) else ""
        protected = bool(group.get("valid_protection_any", pd.Series(False, index=group.index)).fillna(False).astype(bool).any())
        group_rows.append({"group_id": group_id, "rows": len(group), "primary_label": primary_label, "protected": protected})
    groups = pd.DataFrame(group_rows).sample(frac=1.0, random_state=SEED).to_dict(orient="records")
    quotas = proportional_label_row_targets(frame, target_rows, label_col)
    selected_groups = set()
    selected_rows = 0
    selected_by_label: Counter[str] = Counter()
    remaining = list(groups)

    while remaining and selected_rows < target_rows:
        remaining_budget = target_rows - selected_rows
        fitting = [group for group in remaining if int(group["rows"]) <= remaining_budget]
        candidates = fitting if fitting else remaining

        def group_score(group: Dict[str, Any]) -> Tuple[int, int, int]:
            label = str(group["primary_label"])
            deficit = int(quotas.get(label, 0) - selected_by_label[label])
            protected_bonus = 1 if group.get("protected") else 0
            return (deficit, protected_bonus, -int(group["rows"]))

        chosen = max(candidates, key=group_score)
        selected_groups.add(chosen["group_id"])
        selected_rows += int(chosen["rows"])
        selected_by_label[str(chosen["primary_label"])] += int(chosen["rows"])
        remaining = [group for group in remaining if group["group_id"] != chosen["group_id"]]
        if not fitting:
            break

    sampled = frame[frame[group_col].isin(selected_groups)]
    return sampled.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


def cap_public_only_train_rows(frame: pd.DataFrame) -> pd.DataFrame:
    cap = int(globals().get("FORGE_AGENT_HF_PUBLIC_ONLY_TRAIN_CAP", 0))
    if cap <= 0 or frame.empty:
        return frame
    pieces = []
    removed = 0
    for label, group in frame.groupby("label", sort=False):
        if len(group) > cap:
            sampled = sample_groups_for_row_budget(group, cap)
            pieces.append(sampled)
            removed += len(group) - len(sampled)
        else:
            pieces.append(group)
    capped = pd.concat(pieces, ignore_index=False) if pieces else frame.iloc[0:0]
    if removed:
        print(f"Downsampled public-only train labels: -{removed} rows using group-preserving cap {cap} per label.")
    return capped.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


private_mix_report: Dict[str, Any] = {
    "schema_version": "toolcall-verifier-private-mix/v1",
    "target_private_fraction": float(globals().get("FORGE_AGENT_HF_TRAIN_FRACTION_TARGET", 0.0)),
    "group_preserving": True,
    "max_row_overshoot_groups_per_sample": 1,
}


def limit_nonprivate_rows_for_private_target(frame: pd.DataFrame) -> pd.DataFrame:
    target = float(globals().get("FORGE_AGENT_HF_TRAIN_FRACTION_TARGET", 0.0))
    frame = ensure_group_ids(frame)
    if target <= 0.0 or target >= 1.0:
        private_mix_report["status"] = "disabled"
        return frame.reset_index(drop=True)
    mask = private_hf_mask(frame)
    private = frame[mask]
    if private.empty:
        private_mix_report["status"] = "no_private_rows"
        return frame.reset_index(drop=True)

    covered_labels = set(private["label"].dropna().astype(str))
    if not covered_labels:
        private_mix_report["status"] = "no_private_covered_labels"
        return frame.reset_index(drop=True)

    label_values = frame["label"].astype(str)
    covered = label_values.isin(covered_labels)
    private_covered = frame[mask & covered]
    nonprivate_covered = frame[(~mask) & covered]
    protected = frame[~covered]
    covered_total = len(private_covered) + len(nonprivate_covered)
    current_fraction = len(private_covered) / covered_total if covered_total else 0.0
    private_mix_report.update({
        "covered_labels": sorted(covered_labels),
        "private_covered_rows_before": int(len(private_covered)),
        "nonprivate_covered_rows_before": int(len(nonprivate_covered)),
        "private_fraction_before": float(current_fraction),
        "downsample_public_for_target": bool(globals().get("FORGE_AGENT_HF_DOWNSAMPLE_PUBLIC_FOR_TARGET", False)),
    })
    if not bool(globals().get("FORGE_AGENT_HF_DOWNSAMPLE_PUBLIC_FOR_TARGET", False)):
        private_mix_report.update({
            "status": "public_downsample_disabled",
            "private_fraction_after": float(current_fraction),
            "nonprivate_covered_rows_after": int(len(nonprivate_covered)),
            "removed_nonprivate_covered_rows": 0,
            "reason": "Public valid/wrong-tool/wrong-argument coverage is the backbone; private rows tune slices only.",
        })
        print(
            f"Public downsampling for private target disabled; covered-label private fraction remains {current_fraction:.3f} "
            f"against target {target:.3f}."
        )
        return frame.reset_index(drop=True)
    if current_fraction >= target:
        print(f"Private HF train fraction on covered labels already {current_fraction:.3f}; target {target:.3f}.")
        private_mix_report.update({
            "status": "already_at_or_above_target",
            "private_fraction_after": float(current_fraction),
            "nonprivate_covered_rows_after": int(len(nonprivate_covered)),
        })
        return frame.reset_index(drop=True)

    max_nonprivate = int(math.floor(len(private_covered) * (1.0 - target) / target))
    if max_nonprivate >= len(nonprivate_covered):
        private_mix_report.update({
            "status": "no_downsample_needed",
            "private_fraction_after": float(current_fraction),
            "nonprivate_covered_rows_after": int(len(nonprivate_covered)),
        })
        return frame.reset_index(drop=True)

    sampled_nonprivate = sample_groups_for_row_budget(nonprivate_covered, max_nonprivate)
    protected = cap_public_only_train_rows(protected)
    mixed = pd.concat([private_covered, sampled_nonprivate, protected], ignore_index=False)
    new_covered_total = len(private_covered) + len(sampled_nonprivate)
    new_fraction = len(private_covered) / new_covered_total if new_covered_total else 0.0
    private_mix_report.update({
        "status": "downsampled_nonprivate_covered_rows",
        "private_fraction_after": float(new_fraction),
        "nonprivate_covered_rows_after": int(len(sampled_nonprivate)),
        "removed_nonprivate_covered_rows": int(len(nonprivate_covered) - len(sampled_nonprivate)),
        "sampled_nonprivate_groups": int(sampled_nonprivate["example_group_id"].nunique()) if len(sampled_nonprivate) else 0,
    })
    print(
        "Downsampled non-private train rows on private-covered labels with group preservation: "
        f"-{len(nonprivate_covered) - len(sampled_nonprivate)} rows; "
        f"private fraction {current_fraction:.3f} -> {new_fraction:.3f}."
    )
    if len(protected):
        print("Preserved capped non-private-only labels:", sorted(protected["label"].dropna().astype(str).unique()))
    return mixed.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


def with_weighted_private_hf_train_rows(frame: pd.DataFrame) -> pd.DataFrame:
    weight = int(globals().get("FORGE_AGENT_HF_DATASET_WEIGHT", 1))
    if weight <= 1 or "preferred_private_hf_agent" not in frame.columns:
        return limit_nonprivate_rows_for_private_target(frame)
    preferred = frame[private_hf_mask(frame)].copy()
    if preferred.empty:
        return frame.reset_index(drop=True)
    pieces = [frame]
    for copy_idx in range(1, weight):
        copy = preferred.copy()
        if "id" in copy.columns:
            copy["id"] = copy["id"].map(lambda value: stable_id("weighted_private_hf_train", value, copy_idx))
        copy["metadata"] = copy["metadata"].map(lambda metadata: mark_weighted_metadata(metadata, copy_idx, weight))
        pieces.append(copy)
    weighted = pd.concat(pieces, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    print(f"Upweighted private HF training rows: +{len(weighted) - len(frame)} rows ({weight}x train only)")
    return limit_nonprivate_rows_for_private_target(weighted)


def train_label_counts(frame: pd.DataFrame) -> Dict[str, int]:
    return {str(k): int(v) for k, v in frame["label"].value_counts(dropna=False).items()}


def valid_fraction(frame: pd.DataFrame) -> float:
    return float((frame["label"] == "valid").sum() / len(frame)) if len(frame) else 0.0


def cap_train_nonvalid_ratios(frame: pd.DataFrame) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    valid_rows = int((frame["label"] == "valid").sum())
    if valid_rows <= 0:
        return frame, []
    ratio_by_label = {
        "deterministic_invalid": float(MAX_DETERMINISTIC_INVALID_TO_VALID_TRAIN_RATIO),
        "wrong_tool_semantic": float(MAX_WRONG_TOOL_TO_VALID_TRAIN_RATIO),
        "wrong_arguments_semantic": float(MAX_WRONG_ARGUMENTS_TO_VALID_TRAIN_RATIO),
        "tool_not_needed": float(MAX_TOOL_NOT_NEEDED_TO_VALID_TRAIN_RATIO),
        "needs_clarification": float(MAX_NEEDS_CLARIFICATION_TO_VALID_TRAIN_RATIO),
    }
    pieces = [frame[frame["label"] == "valid"]]
    actions = []
    handled = {"valid"}
    for label, ratio in ratio_by_label.items():
        handled.add(label)
        group = frame[frame["label"] == label]
        if group.empty:
            continue
        target = max(1, int(math.ceil(valid_rows * ratio)))
        if len(group) > target:
            sampled = sample_groups_for_row_budget(group, target)
            actions.append({
                "label": label,
                "ratio_limit": ratio,
                "target_rows": int(target),
                "rows_before": int(len(group)),
                "rows_after": int(len(sampled)),
                "removed_rows": int(len(group) - len(sampled)),
            })
            pieces.append(sampled)
        else:
            actions.append({
                "label": label,
                "ratio_limit": ratio,
                "target_rows": int(target),
                "rows_before": int(len(group)),
                "rows_after": int(len(group)),
                "removed_rows": 0,
            })
            pieces.append(group)
    remainder = frame[~frame["label"].isin(handled)]
    if len(remainder):
        pieces.append(remainder)
    out = pd.concat(pieces, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return out, actions


def mark_train_rebalance_metadata(metadata: Any, copy_idx: int, reason: str) -> Dict[str, Any]:
    out = dict(metadata) if isinstance(metadata, dict) else {}
    out["train_rebalance_copy"] = copy_idx
    out["train_rebalance_reason"] = reason
    out["weighted_split"] = "train"
    return out


def duplicate_label_train_rows(
    frame: pd.DataFrame,
    label: str,
    target_rows: int,
    max_factor: int,
    reason: str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    group = frame[frame["label"] == label].copy()
    current = int(len(group))
    target_rows = max(0, int(target_rows))
    max_factor = max(1, int(max_factor))
    if current <= 0 or target_rows <= current:
        return frame, {
            "label": label,
            "status": "already_at_or_above_target" if current else "no_rows",
            "target_rows": int(target_rows),
            "rows_before": current,
            "added_rows": 0,
            "rows_after": current,
        }

    needed = target_rows - current
    max_extra = max(0, (max_factor - 1) * current)
    needed = min(needed, max_extra)
    if needed <= 0:
        return frame, {
            "label": label,
            "status": "max_duplication_factor_prevents_target",
            "target_rows": int(target_rows),
            "rows_before": current,
            "added_rows": 0,
            "rows_after": current,
            "max_duplication_factor": int(max_factor),
        }

    protected = group[group.get("valid_protection_any", pd.Series(False, index=group.index)).fillna(False).astype(bool)]
    ordinary = group.drop(index=protected.index)
    candidates = pd.concat([
        protected.sample(frac=1.0, random_state=SEED) if len(protected) else protected,
        ordinary.sample(frac=1.0, random_state=SEED + 11) if len(ordinary) else ordinary,
    ], ignore_index=False)
    candidates = candidates.drop_duplicates(subset=["id"] if "id" in candidates.columns else None)

    copies = []
    remaining = needed
    copy_idx = 1
    while remaining > 0 and copy_idx < max_factor:
        selected = candidates.sample(frac=1.0, random_state=SEED + copy_idx + 101).head(min(remaining, len(candidates))).copy()
        if selected.empty:
            break
        if "id" in selected.columns:
            selected["id"] = selected["id"].map(lambda value: stable_id(f"{label}_train_rebalance", value, copy_idx))
        selected["metadata"] = selected["metadata"].map(lambda metadata: mark_train_rebalance_metadata(metadata, copy_idx, reason))
        selected["semantic_negative_train_rebalance_copy"] = copy_idx
        copies.append(selected)
        remaining -= len(selected)
        copy_idx += 1

    if not copies:
        return frame, {
            "label": label,
            "status": "no_candidates_selected",
            "target_rows": int(target_rows),
            "rows_before": current,
            "added_rows": 0,
            "rows_after": current,
        }
    out = pd.concat([frame] + copies, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    added = int(len(out) - len(frame))
    return out, {
        "label": label,
        "status": "duplicated",
        "target_rows": int(target_rows),
        "rows_before": current,
        "added_rows": added,
        "rows_after": int(current + added),
        "max_duplication_factor": int(max_factor),
        "reason": reason,
    }


def duplicate_semantic_negative_train_rows(frame: pd.DataFrame) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    if not ENABLE_SEMANTIC_NEGATIVE_TRAIN_REBALANCE:
        return frame, []
    valid_rows = int((frame["label"] == "valid").sum())
    if valid_rows <= 0:
        return frame, []
    targets = {
        "wrong_tool_semantic": int(math.ceil(valid_rows * float(WRONG_TOOL_TRAIN_TO_VALID_RATIO_TARGET))),
        "wrong_arguments_semantic": int(math.ceil(valid_rows * float(WRONG_ARGUMENTS_TRAIN_TO_VALID_RATIO_TARGET))),
    }
    out = frame
    actions = []
    for label, target_rows in targets.items():
        out, action = duplicate_label_train_rows(
            out,
            label=label,
            target_rows=target_rows,
            max_factor=MAX_SEMANTIC_NEGATIVE_DUPLICATION_FACTOR,
            reason=f"semantic_negative_{label}_protection",
        )
        actions.append(action)
    return out.sample(frac=1.0, random_state=SEED).reset_index(drop=True), actions


def duplicate_valid_train_rows(frame: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    target = float(VALID_TRAIN_FRACTION_TARGET)
    max_factor = max(1, int(VALID_TRAIN_MAX_DUPLICATION_FACTOR))
    valid_rows_df = frame[frame["label"] == "valid"].copy()
    if not ENABLE_VALID_PROTECTION_TRAIN_REBALANCE or target <= 0 or target >= 1 or valid_rows_df.empty:
        return frame, {"status": "disabled_or_no_valid_rows"}
    current_valid = len(valid_rows_df)
    current_total = len(frame)
    current_fraction = current_valid / current_total if current_total else 0.0
    if current_fraction >= target:
        return frame, {
            "status": "already_at_or_above_target",
            "target_valid_fraction": target,
            "valid_fraction_before": float(current_fraction),
            "valid_fraction_after": float(current_fraction),
            "added_valid_rows": 0,
        }

    needed = int(math.ceil((target * current_total - current_valid) / max(1e-9, 1.0 - target)))
    max_extra = max(0, (max_factor - 1) * current_valid)
    needed = min(needed, max_extra)
    if needed <= 0:
        return frame, {
            "status": "max_duplication_factor_prevents_target",
            "target_valid_fraction": target,
            "valid_fraction_before": float(current_fraction),
            "valid_fraction_after": float(current_fraction),
            "added_valid_rows": 0,
            "max_duplication_factor": max_factor,
        }

    protected = valid_rows_df[valid_rows_df.get("valid_protection_any", pd.Series(False, index=valid_rows_df.index)).fillna(False).astype(bool)]
    ordinary = valid_rows_df.drop(index=protected.index)
    base_candidates = pd.concat([
        protected.sample(frac=1.0, random_state=SEED) if len(protected) else protected,
        ordinary.sample(frac=1.0, random_state=SEED + 1) if len(ordinary) else ordinary,
    ], ignore_index=False)
    base_candidates = base_candidates.drop_duplicates(subset=["id"] if "id" in base_candidates.columns else None)

    copies = []
    remaining = needed
    copy_idx = 1
    while remaining > 0 and copy_idx < max_factor:
        round_candidates = base_candidates.sample(frac=1.0, random_state=SEED + copy_idx).reset_index(drop=True)
        if len(protected):
            protected_round = round_candidates[round_candidates.get("valid_protection_any", pd.Series(False, index=round_candidates.index)).fillna(False).astype(bool)]
            ordinary_round = round_candidates.drop(index=protected_round.index)
            round_candidates = pd.concat([protected_round, ordinary_round], ignore_index=True)
        selected = round_candidates.head(min(remaining, len(round_candidates))).copy()
        if selected.empty:
            break
        if "id" in selected.columns:
            selected["id"] = selected["id"].map(lambda value: stable_id("valid_train_rebalance", value, copy_idx))
        selected["metadata"] = selected["metadata"].map(lambda metadata: mark_train_rebalance_metadata(metadata, copy_idx, "valid_recall_protection"))
        selected["valid_train_rebalance_copy"] = copy_idx
        copies.append(selected)
        remaining -= len(selected)
        copy_idx += 1

    if not copies:
        return frame, {"status": "no_candidates_selected", "target_valid_fraction": target, "valid_fraction_before": float(current_fraction)}
    out = pd.concat([frame] + copies, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    after_fraction = valid_fraction(out)
    return out, {
        "status": "valid_rows_duplicated",
        "target_valid_fraction": target,
        "valid_fraction_before": float(current_fraction),
        "valid_fraction_after": float(after_fraction),
        "valid_rows_before": int(current_valid),
        "rows_before": int(current_total),
        "rows_after": int(len(out)),
        "added_valid_rows": int(len(out) - len(frame)),
        "protected_valid_candidates": int(len(protected)),
        "max_duplication_factor": int(max_factor),
    }


def valid_protection_counts(frame: pd.DataFrame) -> Dict[str, int]:
    valid_mask = frame["label"] == "valid"
    columns = {
        "terminal_like_tool_valid": "valid_protection_terminal_like_tool",
        "fixed_width_numeric_string_valid": "valid_protection_fixed_width_numeric_string",
        "noop_valid_call_valid": "valid_protection_noop_call",
        "corrected_error_recovery_valid": "valid_protection_corrected_error_recovery",
        "any_valid_protection_valid": "valid_protection_any",
    }
    return {
        output_name: int(
            (
                valid_mask
                & frame.get(column, pd.Series(False, index=frame.index)).fillna(False).astype(bool)
            ).sum()
        )
        for output_name, column in columns.items()
    }


def duplicate_valid_protection_train_rows(frame: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if not ENABLE_VALID_PROTECTION_EXTRA_TRAIN_REBALANCE:
        return frame, {"status": "disabled"}
    max_factor = max(1, int(VALID_PROTECTION_EXTRA_COPY_FACTOR))
    cap = int(VALID_PROTECTION_EXTRA_COPY_ROWS_CAP)
    protected_mask = frame.get("valid_protection_any", pd.Series(False, index=frame.index)).fillna(False).astype(bool)
    protected = frame[(frame["label"] == "valid") & protected_mask].copy()
    protected = protected.drop_duplicates(subset=["id"] if "id" in protected.columns else None)
    if protected.empty:
        return frame, {
            "status": "no_protected_valid_rows",
            "added_valid_rows": 0,
            "protected_valid_candidates": 0,
        }

    if cap > 0 and len(protected) > cap:
        protected = protected.sample(n=cap, random_state=SEED + 313)

    copies = []
    for copy_idx in range(1, max_factor):
        selected = protected.sample(frac=1.0, random_state=SEED + 313 + copy_idx).copy()
        if selected.empty:
            continue
        if "id" in selected.columns:
            selected["id"] = selected["id"].map(lambda value: stable_id("valid_protection_extra_rebalance", value, copy_idx))
        selected["metadata"] = selected["metadata"].map(
            lambda metadata: mark_train_rebalance_metadata(metadata, copy_idx, "valid_protection_slice_extra")
        )
        selected["valid_protection_extra_rebalance_copy"] = copy_idx
        copies.append(selected)

    if not copies:
        return frame, {
            "status": "max_copy_factor_prevents_extra_rows",
            "added_valid_rows": 0,
            "protected_valid_candidates": int(len(protected)),
            "max_copy_factor": int(max_factor),
            "rows_cap": int(cap),
        }

    out = pd.concat([frame] + copies, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return out, {
        "status": "protected_valid_rows_duplicated",
        "added_valid_rows": int(len(out) - len(frame)),
        "protected_valid_candidates": int(len(protected)),
        "max_copy_factor": int(max_factor),
        "rows_cap": int(cap),
        "valid_fraction_before": valid_fraction(frame),
        "valid_fraction_after": valid_fraction(out),
        "protected_counts_before": valid_protection_counts(frame),
        "protected_counts_after": valid_protection_counts(out),
    }


def rebalance_train_for_valid_protection(frame: pd.DataFrame) -> pd.DataFrame:
    global train_rebalance_report
    before_counts = train_label_counts(frame)
    if not ENABLE_VALID_PROTECTION_TRAIN_REBALANCE:
        train_rebalance_report = {
            "schema_version": "toolcall-verifier-train-rebalance/v1",
            "enabled": False,
            "label_counts_before": before_counts,
            "label_counts_after": before_counts,
            "valid_fraction_before": valid_fraction(frame),
            "valid_fraction_after": valid_fraction(frame),
        }
        (DATA_DIR / "train_rebalance_report.json").write_text(json.dumps(train_rebalance_report, indent=2))
        return frame.reset_index(drop=True)

    capped, cap_actions = cap_train_nonvalid_ratios(frame)
    semantic_rebalanced, semantic_duplicate_actions = duplicate_semantic_negative_train_rows(capped)
    valid_rebalanced, duplicate_report = duplicate_valid_train_rows(semantic_rebalanced)
    rebalanced, protection_duplicate_report = duplicate_valid_protection_train_rows(valid_rebalanced)
    train_rebalance_report = {
        "schema_version": "toolcall-verifier-train-rebalance/v1",
        "enabled": True,
        "baseline_note": (
            "Current serialize_state_v3 T4 run passes valid recall, wrong-tool precision, "
            "and protected-slice gates; keep semantic-negative rebalance off and avoid more protected-valid "
            "duplication as the primary fix. Focus the next data addendum on contrastive valid positives."
        ),
        "controls": {
            "valid_train_fraction_target": float(VALID_TRAIN_FRACTION_TARGET),
            "valid_train_max_duplication_factor": int(VALID_TRAIN_MAX_DUPLICATION_FACTOR),
            "max_deterministic_invalid_to_valid_ratio": float(MAX_DETERMINISTIC_INVALID_TO_VALID_TRAIN_RATIO),
            "max_wrong_tool_to_valid_ratio": float(MAX_WRONG_TOOL_TO_VALID_TRAIN_RATIO),
            "max_wrong_arguments_to_valid_ratio": float(MAX_WRONG_ARGUMENTS_TO_VALID_TRAIN_RATIO),
            "max_tool_not_needed_to_valid_ratio": float(MAX_TOOL_NOT_NEEDED_TO_VALID_TRAIN_RATIO),
            "max_needs_clarification_to_valid_ratio": float(MAX_NEEDS_CLARIFICATION_TO_VALID_TRAIN_RATIO),
            "semantic_negative_train_rebalance_enabled": bool(ENABLE_SEMANTIC_NEGATIVE_TRAIN_REBALANCE),
            "wrong_tool_train_to_valid_ratio_target": float(WRONG_TOOL_TRAIN_TO_VALID_RATIO_TARGET),
            "wrong_arguments_train_to_valid_ratio_target": float(WRONG_ARGUMENTS_TRAIN_TO_VALID_RATIO_TARGET),
            "max_semantic_negative_duplication_factor": int(MAX_SEMANTIC_NEGATIVE_DUPLICATION_FACTOR),
            "valid_protection_extra_train_rebalance_enabled": bool(ENABLE_VALID_PROTECTION_EXTRA_TRAIN_REBALANCE),
            "valid_protection_extra_copy_factor": int(VALID_PROTECTION_EXTRA_COPY_FACTOR),
            "valid_protection_extra_copy_rows_cap": int(VALID_PROTECTION_EXTRA_COPY_ROWS_CAP),
            "run_profile": str(RUN_PROFILE),
            "low_resource_toolcall_profile": bool(LOW_RESOURCE_TOOLCALL_PROFILE),
            "t4_rebalance_safety_defaults_applied": bool(APPLY_T4_REBALANCE_SAFETY_DEFAULTS),
        },
        "label_counts_before": before_counts,
        "label_counts_after_nonvalid_caps": train_label_counts(capped),
        "label_counts_after_semantic_negative_rebalance": train_label_counts(semantic_rebalanced),
        "label_counts_after_valid_rebalance": train_label_counts(valid_rebalanced),
        "label_counts_after": train_label_counts(rebalanced),
        "valid_fraction_before": valid_fraction(frame),
        "valid_fraction_after_nonvalid_caps": valid_fraction(capped),
        "valid_fraction_after_semantic_negative_rebalance": valid_fraction(semantic_rebalanced),
        "valid_fraction_after_valid_rebalance": valid_fraction(valid_rebalanced),
        "valid_fraction_after": valid_fraction(rebalanced),
        "valid_protection_counts_before": valid_protection_counts(frame),
        "valid_protection_counts_after_nonvalid_caps": valid_protection_counts(capped),
        "valid_protection_counts_after_valid_rebalance": valid_protection_counts(valid_rebalanced),
        "valid_protection_counts_after": valid_protection_counts(rebalanced),
        "nonvalid_cap_actions": cap_actions,
        "semantic_negative_duplicate_actions": semantic_duplicate_actions,
        "duplicate_valid_rows": duplicate_report,
        "duplicate_valid_protection_rows": protection_duplicate_report,
        "train_rows_before": int(len(frame)),
        "train_rows_after": int(len(rebalanced)),
    }
    (DATA_DIR / "train_rebalance_report.json").write_text(json.dumps(train_rebalance_report, indent=2))
    print("Train valid-protection rebalance report:")
    print(json.dumps(train_rebalance_report, indent=2)[:6000])
    return rebalanced.reset_index(drop=True)



PROTECTED_SPLIT_MIN_VALID_ROWS = 25
PROTECTED_SPLIT_TARGETS = ["validation", "test"]
PROTECTED_SPLIT_COLUMNS = {
    "terminal_like_tool": "valid_protection_terminal_like_tool",
    "corrected_error_recovery_positive": "valid_protection_corrected_error_recovery",
    "fixed_width_numeric_string": "valid_protection_fixed_width_numeric_string",
    "noop_valid_call": "valid_protection_noop_call",
}


def protected_valid_count(frame: pd.DataFrame, column: str) -> int:
    if frame.empty or column not in frame.columns:
        return 0
    return int(((frame["label"] == "valid") & frame[column].fillna(False).astype(bool)).sum())


def protected_counts_by_split(splits: Dict[str, pd.DataFrame]) -> Dict[str, Dict[str, int]]:
    return {
        split_name: {
            slice_name: protected_valid_count(frame, column)
            for slice_name, column in PROTECTED_SPLIT_COLUMNS.items()
        }
        for split_name, frame in splits.items()
    }


def best_protected_donor_group(
    splits: Dict[str, pd.DataFrame],
    donor_names: List[str],
    column: str,
) -> Optional[Tuple[str, Any, int, int]]:
    best: Optional[Tuple[Tuple[int, int, int], str, Any, int, int]] = None
    for donor_order, donor_name in enumerate(donor_names):
        frame = splits.get(donor_name)
        if frame is None or frame.empty or column not in frame.columns:
            continue
        candidate_groups = frame.loc[
            (frame["label"] == "valid") & frame[column].fillna(False).astype(bool),
            "example_group_id",
        ].dropna().unique()
        for group_id in candidate_groups:
            group = frame[frame["example_group_id"] == group_id]
            valid_rows = protected_valid_count(group, column)
            if valid_rows <= 0:
                continue
            rows = int(len(group))
            score = (valid_rows, -rows, -donor_order)
            if best is None or score > best[0]:
                best = (score, donor_name, group_id, valid_rows, rows)
    if best is None:
        return None
    _, donor_name, group_id, valid_rows, rows = best
    return donor_name, group_id, valid_rows, rows


def move_group_between_splits(
    splits: Dict[str, pd.DataFrame],
    donor_name: str,
    target_name: str,
    group_id: Any,
) -> int:
    donor = splits[donor_name]
    mask = donor["example_group_id"] == group_id
    moved = donor[mask].copy()
    if moved.empty:
        return 0
    splits[donor_name] = donor[~mask].reset_index(drop=True)
    splits[target_name] = pd.concat([splits[target_name], moved], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return int(len(moved))


def balance_protected_slice_splits(
    train: pd.DataFrame,
    calibration: pd.DataFrame,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    min_valid_rows: int = PROTECTED_SPLIT_MIN_VALID_ROWS,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    splits = {
        "train": train.reset_index(drop=True),
        "calibration": calibration.reset_index(drop=True),
        "validation": validation.reset_index(drop=True),
        "test": test.reset_index(drop=True),
    }
    report: Dict[str, Any] = {
        "schema_version": "toolcall-verifier-protected-split-balance/v1",
        "group_preserving": True,
        "min_valid_rows_per_target": int(min_valid_rows),
        "target_splits": PROTECTED_SPLIT_TARGETS,
        "counts_before": protected_counts_by_split(splits),
        "actions": [],
        "deficits": [],
    }

    donor_order = ["train", "calibration"]
    for slice_name, column in PROTECTED_SPLIT_COLUMNS.items():
        total_support = sum(protected_valid_count(frame, column) for frame in splits.values())
        report.setdefault("total_support", {})[slice_name] = int(total_support)
        if total_support < min_valid_rows * len(PROTECTED_SPLIT_TARGETS):
            report["deficits"].append({
                "slice": slice_name,
                "reason": "insufficient_total_support",
                "total_valid_rows": int(total_support),
                "needed_valid_rows": int(min_valid_rows * len(PROTECTED_SPLIT_TARGETS)),
            })
            continue

        for target_name in PROTECTED_SPLIT_TARGETS:
            while protected_valid_count(splits[target_name], column) < min_valid_rows:
                donor = best_protected_donor_group(splits, donor_order, column)
                if donor is None:
                    break
                donor_name, group_id, valid_rows, rows = donor
                moved_rows = move_group_between_splits(splits, donor_name, target_name, group_id)
                report["actions"].append({
                    "slice": slice_name,
                    "from": donor_name,
                    "to": target_name,
                    "group_id": str(group_id),
                    "protected_valid_rows": int(valid_rows),
                    "moved_rows": int(moved_rows),
                })
            final_count = protected_valid_count(splits[target_name], column)
            if final_count < min_valid_rows:
                report["deficits"].append({
                    "slice": slice_name,
                    "split": target_name,
                    "reason": "no_donor_group_available",
                    "valid_rows": int(final_count),
                    "needed_valid_rows": int(min_valid_rows),
                })

    report["counts_after"] = protected_counts_by_split(splits)
    report["split_rows_after"] = {name: int(len(frame)) for name, frame in splits.items()}
    return splits["train"], splits["calibration"], splits["validation"], splits["test"], report

train_df, valid_full_df, test_df = group_split(balanced)

if valid_full_df["example_group_id"].nunique() >= 4:
    cal_splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED + 7)
    calibration_idx, valid_idx = next(cal_splitter.split(valid_full_df, groups=valid_full_df["example_group_id"]))
    calibration_df = valid_full_df.iloc[calibration_idx].copy().reset_index(drop=True)
    valid_df = valid_full_df.iloc[valid_idx].copy().reset_index(drop=True)
else:
    calibration_df = valid_full_df.copy().reset_index(drop=True)
    valid_df = valid_full_df.copy().reset_index(drop=True)

train_df, calibration_df, valid_df, test_df, split_balance_report = balance_protected_slice_splits(
    train_df,
    calibration_df,
    valid_df,
    test_df,
)
(DATA_DIR / "split_balance_report.json").write_text(json.dumps(split_balance_report, indent=2, ensure_ascii=False))
print("Protected split balance report:")
print(json.dumps(split_balance_report, indent=2)[:6000])

train_df = with_weighted_private_hf_train_rows(train_df)
train_df = rebalance_train_for_valid_protection(train_df)
(DATA_DIR / "private_mix_report.json").write_text(json.dumps(private_mix_report, indent=2))
print("Private/public train mix report:")
print(json.dumps(private_mix_report, indent=2))

for name, split_df in [("train", train_df), ("calibration", calibration_df), ("valid", valid_df), ("test", test_df)]:
    path = DATA_DIR / f"{name}.jsonl"
    with path.open("w") as f:
        for row in split_df.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("\n", name, len(split_df), "groups", split_df["example_group_id"].nunique(), path)
    print(split_df["label"].value_counts().to_string())

# Verify no train/test leakage. Calibration and validation are split from the same validation holdout.
train_groups = set(train_df["example_group_id"])
calibration_groups = set(calibration_df["example_group_id"])
valid_groups = set(valid_df["example_group_id"])
test_groups = set(test_df["example_group_id"])
assert train_groups.isdisjoint(calibration_groups)
assert train_groups.isdisjoint(valid_groups)
assert train_groups.isdisjoint(test_groups)
assert test_groups.isdisjoint(calibration_groups)
assert test_groups.isdisjoint(valid_groups)
if calibration_groups and valid_groups:
    assert calibration_groups.isdisjoint(valid_groups)
print("No train/test/calibration group leakage detected.")

full_dataset_path = DATA_DIR / "toolcall_verifier_dataset.jsonl"
with full_dataset_path.open("w") as f:
    for row in balanced.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("Dataset written:", full_dataset_path)


## 7. Tokenization diagnostics

The old `t4_fast` run used `MAX_LENGTH=640`, and an earlier `t4_proven` diagnostic still showed meaningful `CANDIDATE_CALL` truncation. That is no longer the current blocker.

The current T4 `serialize_state_v3` run uses the candidate-first layout at `MAX_LENGTH=1024`. Current diagnostics show `candidate_marker_truncated_rate=0.0` and `schema_marker_truncated_rate=0.0`, so candidate visibility and candidate tool schema visibility are fixed for this run.

Keep this section as a guardrail: if tokenization diagnostics regress, stop and fix serializer/profile alignment. Otherwise, treat longer contexts such as `1280` or `1536` as later quality experiments, not the main next iteration. The active failure is calibration plus hard valid/argument contrast, especially valid calls that look close to `wrong_arguments_semantic` negatives.


In [ ]:
ds = load_dataset(
    "json",
    data_files={
        "train": str(DATA_DIR / "train.jsonl"),
        "calibration": str(DATA_DIR / "calibration.jsonl"),
        "validation": str(DATA_DIR / "valid.jsonl"),
        "test": str(DATA_DIR / "test.jsonl"),
    },
)

def add_label_id(ex):
    ex["label"] = normalize_label(ex["label"])
    ex["labels"] = label2id[ex["label"]]
    return ex

ds = ds.map(add_label_id)
ds


In [ ]:
#@title Tokenizer load, token-length sample, and truncation diagnostics
# DeBERTa v3 uses SentencePiece. The slow tokenizer warning about byte fallback is expected.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

TOKEN_DIAGNOSTIC_SAMPLE_ROWS = 5000  #@param {type:"integer"}
CANDIDATE_MARKER = "CANDIDATE_CALL:\n"


def token_count(text: Any) -> int:
    return len(tokenizer.encode(str(text or ""), add_special_tokens=True))


def candidate_marker_token_position(text: Any) -> Optional[int]:
    raw = str(text or "")
    if CANDIDATE_MARKER not in raw:
        return None
    prefix = raw.split(CANDIDATE_MARKER, 1)[0] + CANDIDATE_MARKER
    return token_count(prefix)


def numeric_percentiles(values: pd.Series, percentiles: List[float]) -> Dict[str, Optional[float]]:
    clean = pd.to_numeric(values, errors="coerce").dropna()
    if clean.empty:
        return {str(p): None for p in percentiles}
    return {str(p): float(clean.quantile(p)) for p in percentiles}


def split_token_sample() -> pd.DataFrame:
    pieces = []
    for split_name, frame in [
        ("train", train_df),
        ("calibration", calibration_df),
        ("validation", valid_df),
        ("test", test_df),
    ]:
        if frame.empty:
            continue
        tmp = frame.copy()
        tmp["split"] = split_name
        pieces.append(tmp)
    if not pieces:
        return pd.DataFrame()
    combined = pd.concat(pieces, ignore_index=True)
    sample_rows = int(TOKEN_DIAGNOSTIC_SAMPLE_ROWS)
    if sample_rows > 0 and len(combined) > sample_rows:
        protected = combined[combined.get("valid_protection_any", pd.Series(False, index=combined.index)).fillna(False).astype(bool)]
        ordinary = combined.drop(index=protected.index)
        protected_sample = protected.sample(n=min(len(protected), sample_rows // 2), random_state=SEED) if len(protected) else protected
        remaining = max(0, sample_rows - len(protected_sample))
        ordinary_sample = ordinary.sample(n=min(len(ordinary), remaining), random_state=SEED + 17) if len(ordinary) else ordinary
        combined = pd.concat([protected_sample, ordinary_sample], ignore_index=True)
    return combined.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


token_diag_df = split_token_sample()
if token_diag_df.empty:
    raise RuntimeError("No rows available for tokenization diagnostics.")

token_diag_df["token_length"] = [token_count(t) for t in tqdm(token_diag_df["input_text"].tolist(), desc="token lengths")]
token_diag_df["candidate_marker_token_position"] = token_diag_df["input_text"].map(candidate_marker_token_position)
token_diag_df["candidate_marker_present"] = token_diag_df["candidate_marker_token_position"].notna()
token_diag_df["candidate_marker_survives_truncation"] = token_diag_df["candidate_marker_token_position"].map(
    lambda value: bool(pd.notna(value) and int(value) <= int(MAX_LENGTH))
)
token_diag_df["candidate_payload_token_budget"] = token_diag_df["candidate_marker_token_position"].map(
    lambda value: int(MAX_LENGTH - int(value)) if pd.notna(value) else None
)

lengths = token_diag_df["token_length"]
print("Token length sample:")
print(lengths.describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]))

p95 = float(lengths.quantile(0.95)) if len(lengths) else 0
if p95 > MAX_LENGTH:
    print(f"WARNING: p95 token length {p95:.0f} exceeds MAX_LENGTH={MAX_LENGTH}. Consider t4_proven/high_vram_quality or more aggressive tool filtering.")
else:
    print("MAX_LENGTH covers p95 in this sample.")

marker_present = token_diag_df["candidate_marker_present"]
marker_truncated = marker_present & ~token_diag_df["candidate_marker_survives_truncation"]
payload_budget = pd.to_numeric(token_diag_df.loc[marker_present, "candidate_payload_token_budget"], errors="coerce")

tokenization_diagnostics = {
    "schema_version": "toolcall-verifier-tokenization-diagnostics/v1",
    "base_model": BASE_MODEL,
    "run_profile": RUN_PROFILE,
    "max_length": int(MAX_LENGTH),
    "sample_rows": int(len(token_diag_df)),
    "token_length_percentiles": numeric_percentiles(lengths, [0.5, 0.75, 0.90, 0.95, 0.99]),
    "candidate_marker_missing_rows": int((~marker_present).sum()),
    "candidate_marker_truncated_rows": int(marker_truncated.sum()),
    "candidate_marker_truncated_rate": float(marker_truncated.sum() / marker_present.sum()) if int(marker_present.sum()) else None,
    "candidate_payload_token_budget_percentiles": numeric_percentiles(payload_budget, [0.01, 0.05, 0.10, 0.50]),
    "by_split": {},
    "by_label": {},
}

for split_name, group in token_diag_df.groupby("split"):
    split_marker_present = group["candidate_marker_present"]
    split_marker_truncated = split_marker_present & ~group["candidate_marker_survives_truncation"]
    tokenization_diagnostics["by_split"][str(split_name)] = {
        "rows": int(len(group)),
        "token_length_percentiles": numeric_percentiles(group["token_length"], [0.5, 0.90, 0.95, 0.99]),
        "candidate_marker_truncated_rows": int(split_marker_truncated.sum()),
        "candidate_marker_truncated_rate": float(split_marker_truncated.sum() / split_marker_present.sum()) if int(split_marker_present.sum()) else None,
    }

for label, group in token_diag_df.groupby("label"):
    label_marker_present = group["candidate_marker_present"]
    label_marker_truncated = label_marker_present & ~group["candidate_marker_survives_truncation"]
    tokenization_diagnostics["by_label"][str(label)] = {
        "rows": int(len(group)),
        "token_length_p95": float(group["token_length"].quantile(0.95)) if len(group) else None,
        "candidate_marker_truncated_rows": int(label_marker_truncated.sum()),
        "candidate_marker_truncated_rate": float(label_marker_truncated.sum() / label_marker_present.sum()) if int(label_marker_present.sum()) else None,
    }

(DATA_DIR / "tokenization_diagnostics.json").write_text(json.dumps(tokenization_diagnostics, indent=2, ensure_ascii=False))
print("Tokenization diagnostics:")
print(json.dumps(tokenization_diagnostics, indent=2)[:6000])

# v5b: truncation hard guard. Under the v3 candidate-first layout the candidate call
# must essentially never be truncated; treat anything above the threshold as a data
# layout bug, not a tolerable diagnostic.
CANDIDATE_TRUNCATION_HARD_FAIL_RATE = 0.005  #@param {type:"number"}
SCHEMA_MARKER = "CANDIDATE_TOOL_SCHEMA:\n"


def schema_marker_token_position(text: Any) -> Optional[int]:
    raw = str(text or "")
    if SCHEMA_MARKER not in raw:
        return None
    prefix = raw.split(SCHEMA_MARKER, 1)[0] + SCHEMA_MARKER
    return token_count(prefix)


if SERIALIZER_VERSION == "serialize_state_v3":
    schema_positions = token_diag_df["input_text"].map(schema_marker_token_position)
    schema_present = schema_positions.notna()
    schema_truncated = schema_present & schema_positions.map(
        lambda value: bool(pd.notna(value) and int(value) > int(MAX_LENGTH))
    )
    schema_truncated_rate = (
        float(schema_truncated.sum() / schema_present.sum()) if int(schema_present.sum()) else None
    )
    tokenization_diagnostics["schema_marker_truncated_rows"] = int(schema_truncated.sum())
    tokenization_diagnostics["schema_marker_truncated_rate"] = schema_truncated_rate
    tokenization_diagnostics["candidate_truncation_hard_fail_rate"] = float(CANDIDATE_TRUNCATION_HARD_FAIL_RATE)
    (DATA_DIR / "tokenization_diagnostics.json").write_text(
        json.dumps(tokenization_diagnostics, indent=2, ensure_ascii=False)
    )
    candidate_truncated_rate = float(tokenization_diagnostics.get("candidate_marker_truncated_rate") or 0.0)
    if candidate_truncated_rate > CANDIDATE_TRUNCATION_HARD_FAIL_RATE:
        raise RuntimeError(
            f"CANDIDATE_CALL marker truncated in {candidate_truncated_rate:.2%} of sampled rows "
            f"(hard limit {CANDIDATE_TRUNCATION_HARD_FAIL_RATE:.2%}, MAX_LENGTH={MAX_LENGTH}). "
            "Raise MAX_LENGTH or lower SERIALIZE_V3_CHARS_PER_TOKEN so the v3 char budget "
            "trims more of OTHER_AVAILABLE_TOOLS."
        )
    if schema_truncated_rate is not None and schema_truncated_rate > 0.02:
        print(
            f"WARNING: CANDIDATE_TOOL_SCHEMA truncated in {schema_truncated_rate:.2%} of sampled rows; "
            "the candidate schema is high-value context."
        )
    print(
        f"Truncation hard guard passed: candidate marker truncated rate "
        f"{candidate_truncated_rate:.3%} <= {CANDIDATE_TRUNCATION_HARD_FAIL_RATE:.3%}."
    )
else:
    print("Truncation hard guard skipped (serializer is not v3).")

In [ ]:
#@title Tokenize datasets safely for Trainer

def tokenize_batch(batch):
    encoded = tokenizer(
        batch["input_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,  # dynamic padding happens in DataCollatorWithPadding
    )
    # Keep only numeric labels. Drop the original string `label` column via remove_columns below.
    encoded["labels"] = [int(x) for x in batch["labels"]]
    return encoded

# Remove all non-model columns so DataCollatorWithPadding never tries to tensorize strings/dicts.
tokenized = ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=ds["train"].column_names,
    desc="Tokenizing",
)

# Ensure labels are int64 and torch-compatible.
tokenized = tokenized.cast_column("labels", Value("int64"))
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized)
print("Train feature keys:", tokenized["train"][0].keys())
print({k: type(v) for k, v in tokenized["train"][0].items()})

# Fail fast before Trainer/DataLoader workers hide the root cause.
sample_batch = [tokenized["train"][i] for i in range(min(4, len(tokenized["train"])))]
collated = data_collator(sample_batch)
print("Collated batch keys:", collated.keys())
print("input_ids:", collated["input_ids"].shape, collated["input_ids"].dtype)
print("attention_mask:", collated["attention_mask"].shape, collated["attention_mask"].dtype)
print("labels:", collated["labels"].shape, collated["labels"].dtype)
assert collated["labels"].dtype in (torch.int64, torch.long)


## 8. Train classifier

The `t4_fast` diagnostic run remains the failed baseline: test macro F1 `0.3716`, `valid` recall `0.528`, `wrong_tool_semantic` precision `0.387`, and broad collapse into `wrong_arguments_semantic`.

The current T4 `serialize_state_v3` run is the first strong candidate deployment-rule artifact:

- Accuracy: 0.9562.
- Present-label macro F1: 0.9096.
- `valid_recall`: 0.9775 on test, passing the 0.94 gate.
- `valid_false_objection @ 0.90`: 10/1602 = 0.00624, still above the 0.005 raw diagnostic ceiling.
- `wrong_tool_semantic_precision`: 0.9892 on test, passing the 0.90 gate.
- `wrong_tool_semantic_recall`: 0.9198.
- `wrong_arguments_semantic_precision`: 0.9745.
- `wrong_arguments_semantic_recall`: 0.9758.
- Candidate and schema truncation are fixed at `MAX_LENGTH=1024`.

The deployment sweep validates the conservative policy: confidence near `0.98` plus margin `0.25` over `valid` gets valid false objections under the gate. Keep raw `@0.90` visible as a stress diagnostic, but do not let it contradict deployment-rule interpretation unless `BLOCK_ON_RAW_0_90_GATE=True`.

Do not enforce this artifact yet. Freeze it as `candidate-v6-shadow-advisory`, run shadow and advisory replay, then decide whether advisory promotion is safe. Keep semantic-negative rebalance off and avoid broad valid duplication. Add targeted reviewed rows for the remaining confusion pairs instead.


In [ ]:
#@title Model, metrics, class weights, and Trainer
if "cuda_memory_snapshot" not in globals():
    def cuda_memory_snapshot(label):
        snapshot = {"label": label, "cuda_available": bool(torch.cuda.is_available())}
        if torch.cuda.is_available():
            device = torch.cuda.current_device()
            props = torch.cuda.get_device_properties(device)
            snapshot.update({
                "device": torch.cuda.get_device_name(device),
                "allocated_gb": round(torch.cuda.memory_allocated(device) / (1024 ** 3), 3),
                "reserved_gb": round(torch.cuda.memory_reserved(device) / (1024 ** 3), 3),
                "max_allocated_gb": round(torch.cuda.max_memory_allocated(device) / (1024 ** 3), 3),
                "total_gb": round(props.total_memory / (1024 ** 3), 3),
            })
        print("CUDA memory snapshot:", json.dumps(snapshot, indent=2))
        return snapshot

cuda_memory_snapshot("before tool-call model load")
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label={int(k): v for k, v in id2label.items()},
    label2id=label2id,
)

# Align model config with tokenizer special tokens when needed.
if tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id
if tokenizer.eos_token_id is not None:
    model.config.eos_token_id = tokenizer.eos_token_id
if tokenizer.bos_token_id is not None:
    model.config.bos_token_id = tokenizer.bos_token_id

ALL_LABEL_IDS = list(range(len(LABELS)))
VALID_LABEL_ID = label2id["valid"]
WRONG_TOOL_LABEL_ID = label2id["wrong_tool_semantic"]
WRONG_ARGUMENTS_LABEL_ID = label2id["wrong_arguments_semantic"]
CHECKPOINT_VALID_RECALL_GATE = 0.94
CHECKPOINT_FALSE_OBJECTION_90_GATE = 0.005
CHECKPOINT_WRONG_TOOL_PRECISION_GATE = 0.90
CHECKPOINT_FALSE_OBJECTION_DISCARD_CEILING = 2.5 * CHECKPOINT_FALSE_OBJECTION_90_GATE
# Competence floor: a checkpoint below either bound is the undertrained "everything is
# valid" pathology — its high valid_recall and near-zero false objections are vacuous.
CHECKPOINT_MIN_WRONG_ARGS_RECALL = 0.50
CHECKPOINT_MIN_PRESENT_MACRO_F1 = 0.70


def checkpoint_selection_score(
    valid_recall: float,
    wrong_tool_precision: float,
    wrong_tool_recall: float,
    present_f1: float,
    wrong_args_recall: float,
    valid_false_objection_90: float,
) -> Tuple[float, bool]:
    """Tiered checkpoint selection, replacing the -inf hard discard that picked a
    degenerate epoch-1 model and starved EarlyStoppingCallback of finite improvements
    while false objection was still trending down across epochs.

    Tier 0, fails competence floor: score in [-100, -99].
    Tier 1, competent but false objection above the discard ceiling: score < ~1.12,
        rising as quality improves and false objection falls.
    Tier 2, competent and within the ceiling: score >= 100, lexicographic ordering
        (valid_recall, then wrong_tool precision, then wrong_tool recall, then F1).
    Tiers cannot overlap, so any competent checkpoint beats any degenerate one and any
    within-ceiling checkpoint beats any over-ceiling one."""
    lexicographic = (
        valid_recall
        + 0.1 * wrong_tool_precision
        + 0.01 * wrong_tool_recall
        + 0.001 * present_f1
    )
    competent = bool(
        wrong_args_recall >= CHECKPOINT_MIN_WRONG_ARGS_RECALL
        and present_f1 >= CHECKPOINT_MIN_PRESENT_MACRO_F1
    )
    fo_ok = bool(valid_false_objection_90 <= CHECKPOINT_FALSE_OBJECTION_DISCARD_CEILING)
    if not competent:
        score = -100.0 + present_f1
    elif not fo_ok:
        score = lexicographic - 10.0 * (
            valid_false_objection_90 - CHECKPOINT_FALSE_OBJECTION_DISCARD_CEILING
        )
    else:
        score = 100.0 + lexicographic - 10.0 * max(
            0.0, valid_false_objection_90 - CHECKPOINT_FALSE_OBJECTION_90_GATE
        )
    # The tiered score keeps the 2.5x discard ceiling so checkpoint selection and
    # early stopping are unchanged; the boolean uses the strict gate set so a per-eval
    # flag can never claim promotability that promotion_gate_report would deny.
    strict_promotable = bool(
        valid_false_objection_90 <= CHECKPOINT_FALSE_OBJECTION_90_GATE
        and valid_recall >= CHECKPOINT_VALID_RECALL_GATE
        and wrong_tool_precision >= CHECKPOINT_WRONG_TOOL_PRECISION_GATE
    )
    return float(score), strict_promotable


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    confidences = probs.max(axis=-1)
    present_precision, present_recall, present_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    full_precision, full_recall, full_f1, _ = precision_recall_fscore_support(
        labels, preds, labels=ALL_LABEL_IDS, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    valid_mask = labels == VALID_LABEL_ID
    valid_count = int(valid_mask.sum())
    valid_recall = float((preds[valid_mask] == VALID_LABEL_ID).mean()) if valid_count else 0.0
    valid_precision_mask = preds == VALID_LABEL_ID
    valid_precision = float((labels[valid_precision_mask] == VALID_LABEL_ID).mean()) if int(valid_precision_mask.sum()) else 0.0
    valid_false_objection_90 = 0.0
    if valid_count:
        valid_false_objection_90 = float(((preds[valid_mask] != VALID_LABEL_ID) & (confidences[valid_mask] >= 0.90)).mean())

    wrong_tool_pred_mask = preds == WRONG_TOOL_LABEL_ID
    wrong_tool_precision = float((labels[wrong_tool_pred_mask] == WRONG_TOOL_LABEL_ID).mean()) if int(wrong_tool_pred_mask.sum()) else 0.0
    wrong_tool_true_mask = labels == WRONG_TOOL_LABEL_ID
    wrong_tool_recall = float((preds[wrong_tool_true_mask] == WRONG_TOOL_LABEL_ID).mean()) if int(wrong_tool_true_mask.sum()) else 0.0
    wrong_args_true_mask = labels == WRONG_ARGUMENTS_LABEL_ID
    wrong_args_recall = float((preds[wrong_args_true_mask] == WRONG_ARGUMENTS_LABEL_ID).mean()) if int(wrong_args_true_mask.sum()) else 0.0
    wrong_args_pred_mask = preds == WRONG_ARGUMENTS_LABEL_ID
    wrong_args_precision = float((labels[wrong_args_pred_mask] == WRONG_ARGUMENTS_LABEL_ID).mean()) if int(wrong_args_pred_mask.sum()) else 0.0
    valid_to_wrong_args_rate = float((preds[valid_mask] == WRONG_ARGUMENTS_LABEL_ID).mean()) if valid_count else 0.0
    wrong_tool_to_wrong_args_rate = float((preds[wrong_tool_true_mask] == WRONG_ARGUMENTS_LABEL_ID).mean()) if int(wrong_tool_true_mask.sum()) else 0.0

    valid_recall_deficit = max(0.0, CHECKPOINT_VALID_RECALL_GATE - valid_recall) / CHECKPOINT_VALID_RECALL_GATE
    wrong_tool_precision_deficit = max(0.0, CHECKPOINT_WRONG_TOOL_PRECISION_GATE - wrong_tool_precision) / CHECKPOINT_WRONG_TOOL_PRECISION_GATE
    false_objection_excess = max(0.0, valid_false_objection_90 - CHECKPOINT_FALSE_OBJECTION_90_GATE) / CHECKPOINT_FALSE_OBJECTION_90_GATE
    # Keep legacy gate_deficit for telemetry backward-compat.
    gate_deficit = float(
        valid_recall_deficit
        + wrong_tool_precision_deficit
        + 5.0 * false_objection_excess
        + 0.5 * valid_to_wrong_args_rate
    )
    gate_deficit_score, constrained_promotable = checkpoint_selection_score(
        valid_recall=valid_recall,
        wrong_tool_precision=wrong_tool_precision,
        wrong_tool_recall=wrong_tool_recall,
        present_f1=present_f1,
        wrong_args_recall=wrong_args_recall,
        valid_false_objection_90=valid_false_objection_90,
    )

    return {
        "accuracy": acc,
        "macro_precision": present_precision,
        "macro_recall": present_recall,
        "macro_f1": present_f1,
        "macro_precision_all_labels": full_precision,
        "macro_recall_all_labels": full_recall,
        "macro_f1_all_labels": full_f1,
        "valid_recall": valid_recall,
        "valid_precision": valid_precision,
        "valid_false_objection_0_90": valid_false_objection_90,
        "wrong_tool_semantic_precision": wrong_tool_precision,
        "wrong_tool_semantic_recall": wrong_tool_recall,
        "wrong_arguments_semantic_recall": wrong_args_recall,
        "wrong_arguments_semantic_precision": wrong_args_precision,
        "valid_to_wrong_arguments_semantic_rate": valid_to_wrong_args_rate,
        "wrong_tool_to_wrong_arguments_semantic_rate": wrong_tool_to_wrong_args_rate,
        "gate_deficit": gate_deficit,
        "gate_deficit_score": gate_deficit_score,
        "checkpoint_constrained_promotable": constrained_promotable,
    }


def build_class_weights(train_frame: pd.DataFrame) -> Optional[torch.Tensor]:
    if not USE_CLASS_WEIGHTS:
        return None
    counts = train_frame["label"].value_counts()
    weights = []
    for label in LABELS:
        count = counts.get(label, 0)
        if count <= 0:
            weights.append(0.0)
        else:
            raw = len(train_frame) / (len(LABELS) * count)
            weights.append(min(float(raw), MAX_CLASS_WEIGHT))
    weights = np.array(weights, dtype=np.float32)
    nonzero = weights[weights > 0]
    if len(nonzero):
        weights[weights > 0] = weights[weights > 0] / nonzero.mean()
    return torch.tensor(weights, dtype=torch.float)


class WeightedTrainer(Trainer):
    def __init__(self, class_weights: Optional[torch.Tensor] = None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weights is None:
            loss = outputs.get("loss")
        else:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


class_weights = build_class_weights(train_df)
if class_weights is None:
    print("Class weights: disabled")
else:
    print("Class weights:")
    for label, weight in zip(LABELS, class_weights.tolist()):
        print(f"  {label:28s} {weight:.3f}")

print("Training profile:", RUN_PROFILE)
print("Training size:", len(tokenized["train"]), "validation size:", len(tokenized["validation"]))
print("Max length:", MAX_LENGTH)
print("Per-device batch:", TRAIN_BATCH_SIZE, "grad accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Optimizer:", OPTIMIZER)
print("Gradient checkpointing:", GRADIENT_CHECKPOINTING)
print("Best-checkpoint metric: gate_deficit_score")

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="gate_deficit_score",
    greater_is_better=True,
    fp16=USE_FP16,
    bf16=USE_BF16,
    tf32=USE_TF32,
    optim=OPTIMIZER,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
    dataloader_num_workers=DATALOADER_NUM_WORKERS if USE_CUDA else 0,
    dataloader_pin_memory=USE_CUDA,
    group_by_length=GROUP_BY_LENGTH,
    save_total_limit=3,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

train_result = trainer.train()
trainer.save_state()
cuda_memory_snapshot("after tool-call training")

# Build a compact run summary for artifact selection, Hub upload, and Rust-side provenance.
metric_key = f"eval_{training_args.metric_for_best_model}"
eval_history = [dict(row) for row in trainer.state.log_history if metric_key in row]
best_eval = max(eval_history, key=lambda row: row.get(metric_key, float("-inf")), default={})
training_run_summary = {
    "schema_version": "toolcall-verifier-training-run/v1",
    "base_model": BASE_MODEL,
    "label_mode": LABEL_MODE,
    "requested_gpu_profile": GPU_PROFILE,
    "run_profile": RUN_PROFILE,
    "profile_config": CFG,
    "gpu_info": GPU_INFO,
    "precision_flags": {"fp16": USE_FP16, "bf16": USE_BF16, "tf32": USE_TF32},
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "max_per_source": MAX_PER_SOURCE,
    "forge_agent_hf_dataset_enabled": bool(globals().get("ENABLE_FORGE_AGENT_HF_DATASET", False)),
    "forge_agent_hf_dataset_repo": globals().get("FORGE_AGENT_HF_DATASET_REPO"),
    "forge_agent_hf_dataset_file": globals().get("FORGE_AGENT_HF_DATASET_FILE"),
    "forge_agent_hf_dataset_revision": globals().get("FORGE_AGENT_HF_DATASET_REVISION"),
    "forge_agent_hf_dataset_weight": int(globals().get("FORGE_AGENT_HF_DATASET_WEIGHT", 1)),
    "forge_agent_hf_train_fraction_target": float(globals().get("FORGE_AGENT_HF_TRAIN_FRACTION_TARGET", 0.0)),
    "forge_agent_hf_downsample_public_for_target": bool(globals().get("FORGE_AGENT_HF_DOWNSAMPLE_PUBLIC_FOR_TARGET", False)),
    "enable_valid_protection_train_rebalance": bool(globals().get("ENABLE_VALID_PROTECTION_TRAIN_REBALANCE", False)),
    "valid_train_fraction_target": float(globals().get("VALID_TRAIN_FRACTION_TARGET", 0.0)),
    "enable_forge_augmentation": ENABLE_FORGE_AUGMENTATION,
    "enable_final_response_verifier": ENABLE_FINAL_RESPONSE_VERIFIER,
    "train_rows": len(tokenized["train"]),
    "validation_rows": len(tokenized["validation"]),
    "test_rows": len(tokenized["test"]),
    "num_epochs_requested": NUM_EPOCHS,
    "train_result_metrics": train_result.metrics,
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric_name": training_args.metric_for_best_model,
    "best_metric": trainer.state.best_metric,
    "best_eval_log": best_eval,
    "eval_history": eval_history,
    "t4_proven_baseline": T4_PROVEN_BASELINE,
    "dataset_audit_report_file": "dataset_audit_report.json",
    "train_rebalance_report_file": "train_rebalance_report.json",
    "private_mix_report_file": "private_mix_report.json",
    "split_balance_report_file": "split_balance_report.json",
    "tokenization_diagnostics_file": "tokenization_diagnostics.json",
    "dataset_audit_summary": {
        "rows": int(dataset_audit_report.get("rows", 0)) if "dataset_audit_report" in globals() else None,
        "suspicious_valid_hard_negative_rows": int(dataset_audit_report.get("suspicious_valid_hard_negative_rows", 0)) if "dataset_audit_report" in globals() else None,
        "valid_fraction": float(dataset_audit_report.get("valid_fraction", 0.0)) if "dataset_audit_report" in globals() else None,
    },
    "train_rebalance_summary": train_rebalance_report if "train_rebalance_report" in globals() else None,
    "split_balance_summary": split_balance_report if "split_balance_report" in globals() else None,
    "tokenization_diagnostics_summary": tokenization_diagnostics if "tokenization_diagnostics" in globals() else None,
    "created_unix": int(time.time()),
}
(DATA_DIR / "training_run_summary.json").write_text(json.dumps(training_run_summary, indent=2))
print("Training run summary:")
print(json.dumps(training_run_summary, indent=2))

In [ ]:
#@title Evaluate calibration, validation, and test splits
from collections import Counter

ENFORCE_TOOLCALL_PROMOTION_GATES = False  #@param {type:"boolean"}
BLOCK_ON_RAW_0_90_GATE = False  #@param {type:"boolean"}
BLOCK_ON_DEPLOYMENT_GATE = True  #@param {type:"boolean"}
VALID_RECALL_GATE = 0.94
PROMOTION_DIAGNOSTIC_THRESHOLD = 0.90
VALID_FALSE_OBJECTION_DIAGNOSTIC_GATE = 0.005
VALID_FALSE_OBJECTION_DEPLOYMENT_GATE = 0.005
VALID_FALSE_OBJECTION_90_GATE = VALID_FALSE_OBJECTION_DIAGNOSTIC_GATE  # backward-compatible report key
WRONG_TOOL_PRECISION_GATE = 0.90
PROMOTION_DEPLOYMENT_THRESHOLDS = {
    "wrong_tool_semantic": 0.98,
    "wrong_arguments_semantic": 0.98,
    "tool_not_needed": 0.98,
    "needs_clarification": 0.99,
}
PROMOTION_MIN_MARGIN_OVER_VALID = 0.25
PROMOTION_VALID_LOGIT_BIAS = 0.0
PROTECTED_SLICE_RATE_GATE_MIN_VALID_ROWS = 100
PROTECTED_SLICE_FALSE_OBJECTION_MAX_COUNT = 1
ENFORCEMENT_REPLAY_REQUIREMENTS = [
    "ONNX parity passes",
    "quantized ONNX parity passes",
    "shadow replay passes",
    "advisory replay passes",
    "no regression vs no-classifier baseline",
]
NEEDS_CLARIFICATION_MIN_SUPPORT = 50
# needs_clarification is telemetry-only until test support >= NEEDS_CLARIFICATION_MIN_SUPPORT.
# Current artifact thresholds set advisory_min_confidence=1.01 (effectively no-op for enforcement).
DETERMINISTIC_INVALID_EXCLUDED = bool(globals().get("EXCLUDE_DETERMINISTIC_INVALID_FROM_ML_TRAINING", True))
VALID_PROTECTION_SLICE_MIN_VALID_ROWS = 25

split_logits = {}
split_label_ids = {}


def predict_split_sequential(split_name: str) -> Tuple[np.ndarray, np.ndarray]:
    """Predict in dataset row order for metadata-safe slice diagnostics."""
    split_dataset = tokenized[split_name]
    model.eval()
    device = next(model.parameters()).device
    logits_chunks: List[np.ndarray] = []
    label_chunks: List[np.ndarray] = []
    batch_size = max(1, int(EVAL_BATCH_SIZE))
    with torch.no_grad():
        for start_idx in range(0, len(split_dataset), batch_size):
            end_idx = min(start_idx + batch_size, len(split_dataset))
            examples = [split_dataset[i] for i in range(start_idx, end_idx)]
            batch = data_collator(examples)
            labels = batch.pop("labels", None)
            if labels is None:
                raise RuntimeError(f"Tokenized split {split_name} is missing labels")
            model_inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if torch.is_tensor(value)
            }
            outputs = model(**model_inputs)
            logits_chunks.append(outputs.logits.detach().cpu().numpy())
            label_chunks.append(labels.detach().cpu().numpy().astype(int))
    logits = np.concatenate(logits_chunks, axis=0) if logits_chunks else np.empty((0, len(LABELS)), dtype=np.float32)
    labels_arr = np.concatenate(label_chunks, axis=0) if label_chunks else np.empty((0,), dtype=np.int64)
    return logits, labels_arr


def guarded_prediction_columns(
    logits: np.ndarray,
    nonvalid_conf_thresholds: Optional[Dict[str, float]] = None,
    min_nonvalid_valid_margin: float = PROMOTION_MIN_MARGIN_OVER_VALID,
    valid_logit_bias: float = PROMOTION_VALID_LOGIT_BIAS,
) -> pd.DataFrame:
    thresholds = dict(nonvalid_conf_thresholds or PROMOTION_DEPLOYMENT_THRESHOLDS)
    if logits.size == 0:
        return pd.DataFrame({
            "deployment_raw_pred_label": [],
            "deployment_pred_label": [],
            "deployment_confidence": [],
            "deployment_valid_confidence": [],
            "deployment_threshold": [],
            "deployment_margin_over_valid": [],
            "deployment_objected": [],
        })
    adjusted = logits.astype(np.float64, copy=True)
    adjusted[:, VALID_LABEL_ID] += float(valid_logit_bias)
    probs = torch.softmax(torch.tensor(adjusted), dim=-1).numpy()
    raw_preds = np.argmax(probs, axis=-1)
    row_idx = np.arange(len(raw_preds))
    raw_labels = [id2label[int(pred)] for pred in raw_preds]
    pred_conf = probs[row_idx, raw_preds]
    valid_conf = probs[:, VALID_LABEL_ID]
    margin_over_valid = adjusted[row_idx, raw_preds] - adjusted[:, VALID_LABEL_ID]
    threshold_by_pred = np.array([
        0.0 if label == "valid" else float(thresholds.get(label, 1.01))
        for label in raw_labels
    ])
    should_object = (
        (raw_preds != VALID_LABEL_ID)
        & (pred_conf >= threshold_by_pred)
        & (margin_over_valid >= float(min_nonvalid_valid_margin))
    )
    guarded_preds = np.where(should_object, raw_preds, VALID_LABEL_ID)
    return pd.DataFrame({
        "deployment_raw_pred_label": raw_labels,
        "deployment_pred_label": [id2label[int(pred)] for pred in guarded_preds],
        "deployment_confidence": pred_conf,
        "deployment_valid_confidence": valid_conf,
        "deployment_threshold": threshold_by_pred,
        "deployment_margin_over_valid": margin_over_valid,
        "deployment_objected": should_object,
    })


def guarded_predictions(
    logits: np.ndarray,
    nonvalid_conf_threshold: float = 0.98,
    min_nonvalid_valid_margin: float = 0.0,
    valid_logit_bias: float = 0.0,
) -> np.ndarray:
    thresholds = {label: float(nonvalid_conf_threshold) for label in PROMOTION_DEPLOYMENT_THRESHOLDS}
    details = guarded_prediction_columns(
        logits,
        nonvalid_conf_thresholds=thresholds,
        min_nonvalid_valid_margin=min_nonvalid_valid_margin,
        valid_logit_bias=valid_logit_bias,
    )
    return np.array([label2id[label] for label in details["deployment_pred_label"]], dtype=np.int64)


def score_split(split_name: str) -> pd.DataFrame:
    logits, labels_arr = predict_split_sequential(split_name)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(logits, axis=-1)
    split_logits[split_name] = logits
    split_label_ids[split_name] = labels_arr
    raw = ds[split_name].to_pandas()
    if len(raw) != len(labels_arr):
        raise RuntimeError(f"Split {split_name} row count mismatch: raw={len(raw)} predictions={len(labels_arr)}")
    if "labels" in raw.columns:
        raw_label_ids = raw["labels"].astype(int).to_numpy()
        mismatch = raw_label_ids != labels_arr.astype(int)
        if bool(mismatch.any()):
            mismatch_rows = raw.loc[mismatch, [col for col in ["id", "source", "label", "labels", "metadata"] if col in raw.columns]].head(10).copy()
            mismatch_rows["sequential_label_id"] = labels_arr[mismatch][: len(mismatch_rows)]
            mismatch_rows["sequential_label"] = [id2label[int(i)] for i in labels_arr[mismatch][: len(mismatch_rows)]]
            print("Label order mismatch between raw and sequential tokenized split:", split_name)
            display(mismatch_rows)
            raise RuntimeError(f"Raw/tokenized label mismatch in split {split_name}: {int(mismatch.sum())} rows")
    raw["true_label"] = [id2label[int(i)] for i in labels_arr]
    raw["pred_label"] = [id2label[int(i)] for i in preds]
    raw["confidence"] = probs.max(axis=-1)
    _sorted_probs = np.sort(probs, axis=-1)
    raw["margin_top1_top2"] = _sorted_probs[:, -1] - _sorted_probs[:, -2]
    for idx, label in id2label.items():
        raw[f"prob_{label}"] = probs[:, int(idx)]
        raw[f"logit_{label}"] = logits[:, int(idx)]
    raw["correct"] = raw["true_label"] == raw["pred_label"]
    deployment_cols = guarded_prediction_columns(logits)
    raw = pd.concat([raw.reset_index(drop=True), deployment_cols.reset_index(drop=True)], axis=1)
    raw["deployment_correct"] = raw["true_label"] == raw["deployment_pred_label"]
    return add_valid_protection_slice_columns(raw)


def metadata_generator_for_diagnostics(value: Any) -> str:
    return str(value.get("generator", "unknown")) if isinstance(value, dict) else "unknown"


import math

def source_balanced_eval_summary(scored: pd.DataFrame, split_name: str) -> Dict[str, Any]:
    """Per-source breakdown of key promotion metrics with forge-weighted aggregate score."""
    if scored.empty:
        return {}
    forge_sources = {
        "forge_eval", "forge_synthetic", "forge_contrastive_wts",
        "forge_argument_semantic", "forge_error_recovery_numeric",
        "forge_fixed_width_numeric", "forge_error_recovery_protected",
        "forge_augmented",
    }
    sources = scored["source"].unique().tolist()
    rows_by_source: Dict[str, Any] = {}
    forge_correct, forge_total, public_correct, public_total = 0, 0, 0, 0
    for src in sources:
        src_df = scored[scored["source"] == src]
        if src_df.empty:
            continue
        valid_mask = src_df["true_label"] == "valid"
        valid_count = int(valid_mask.sum())
        valid_recall_src = float((src_df.loc[valid_mask, "pred_label"] == "valid").mean()) if valid_count else float("nan")
        fo_mask = valid_mask & (src_df["pred_label"] != "valid") & (src_df["confidence"] >= 0.90)
        false_obj_src = float(fo_mask.sum() / valid_count) if valid_count else float("nan")
        wts_pred = src_df["pred_label"] == "wrong_tool_semantic"
        wts_prec = float((src_df.loc[wts_pred, "true_label"] == "wrong_tool_semantic").mean()) if int(wts_pred.sum()) else float("nan")
        correct = int((src_df["pred_label"] == src_df["true_label"]).sum())
        total = len(src_df)
        rows_by_source[src] = {
            "total": total, "correct": correct,
            "accuracy": round(correct / total, 4) if total else float("nan"),
            "valid_count": valid_count,
            "valid_recall": round(valid_recall_src, 4),
            "false_objection_90": round(false_obj_src, 4),
            "wrong_tool_semantic_precision": round(wts_prec, 4),
        }
        if src in forge_sources:
            forge_correct += correct; forge_total += total
        else:
            public_correct += correct; public_total += total
    forge_acc = forge_correct / forge_total if forge_total else float("nan")
    public_acc = public_correct / public_total if public_total else float("nan")
    forge_weight = 0.70
    if forge_total and public_total:
        weighted_score = forge_weight * forge_acc + (1 - forge_weight) * public_acc
    elif forge_total:
        weighted_score = forge_acc
    else:
        weighted_score = public_acc
    summary = {
        "split": split_name,
        "per_source": rows_by_source,
        "forge_accuracy": round(forge_acc, 4) if not math.isnan(forge_acc) else None,
        "public_accuracy": round(public_acc, 4) if not math.isnan(public_acc) else None,
        "forge_weighted_score": round(weighted_score, 4) if not math.isnan(weighted_score) else None,
    }
    print(f"\n=== Source-balanced eval summary ({split_name}) ===")
    for src, info in sorted(rows_by_source.items(), key=lambda kv: -kv[1]["total"]):
        flag = " [FORGE]" if src in forge_sources else ""
        print(f"  {src}{flag}: n={info['total']}, acc={info.get('accuracy', 'nan'):.3f}, "
              f"valid_recall={info.get('valid_recall', 'nan'):.3f}, "
              f"false_obj_90={info.get('false_objection_90', 'nan'):.3f}, "
              f"wts_prec={info.get('wrong_tool_semantic_precision', 'nan'):.3f}")
    print(f"  Forge-weighted aggregate: {weighted_score:.4f} "
          f"(forge={forge_acc:.3f}x0.70 + public={public_acc:.3f}x0.30)")
    return summary


def confusion_pair_diagnostics(scored: pd.DataFrame, max_rows: int = 100) -> List[Dict[str, Any]]:
    if scored.empty:
        return []
    wrong = scored[scored["true_label"] != scored["pred_label"]].copy()
    if wrong.empty:
        return []
    wrong["generator"] = wrong["metadata"].map(metadata_generator_for_diagnostics) if "metadata" in wrong.columns else "unknown"
    grouped = (
        wrong.groupby(["true_label", "pred_label", "source", "generator"], dropna=False)
        .size()
        .reset_index(name="rows")
        .sort_values("rows", ascending=False)
        .head(max_rows)
    )
    return [
        {
            "true_label": str(row["true_label"]),
            "pred_label": str(row["pred_label"]),
            "source": str(row["source"]),
            "generator": str(row["generator"]),
            "rows": int(row["rows"]),
        }
        for row in grouped.to_dict(orient="records")
    ]


def print_report(scored: pd.DataFrame, name: str):
    y_true = [label2id[x] for x in scored["true_label"]]
    y_pred = [label2id[x] for x in scored["pred_label"]]
    print(f"\n{name} classification report")
    print(classification_report(y_true, y_pred, labels=ALL_LABEL_IDS, target_names=LABELS, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=ALL_LABEL_IDS)
    display(pd.DataFrame(cm, index=LABELS, columns=LABELS))
    print("Per-source accuracy:")
    display(scored.groupby("source").agg(rows=("correct", "size"), accuracy=("correct", "mean"), avg_confidence=("confidence", "mean")).sort_values("rows", ascending=False))
    print("Per-label accuracy:")
    display(scored.groupby("true_label").agg(rows=("correct", "size"), accuracy=("correct", "mean"), avg_confidence=("confidence", "mean")).sort_values("rows", ascending=False))
    pairs = confusion_pair_diagnostics(scored, max_rows=25)
    if pairs:
        print("Top confusion pairs by source/generator:")
        display(pd.DataFrame(pairs))


def extract_candidate_payload(input_text: Any) -> Any:
    text = str(input_text or "")
    marker = "CANDIDATE_CALL:\n"
    if marker not in text:
        return None
    raw = text.split(marker, 1)[1].strip()
    # v3 layout: CANDIDATE_TOOL_SCHEMA follows the candidate call; strip it first or
    # json.loads receives trailing sections and every v3 row loses its payload flags.
    schema_marker = "\n\nCANDIDATE_TOOL_SCHEMA:"
    if schema_marker in raw:
        raw = raw.split(schema_marker, 1)[0].strip()
    metadata_marker = "\n\nSCORING_METADATA:"
    if metadata_marker in raw:
        raw = raw.split(metadata_marker, 1)[0].strip()
    try:
        return json.loads(raw)
    except Exception:
        return None


def candidate_payloads(value: Any) -> List[Dict[str, Any]]:
    if isinstance(value, list):
        return [item for item in value if isinstance(item, dict)]
    if isinstance(value, dict):
        return [value]
    return []


def candidate_tool_names(value: Any) -> List[str]:
    names = []
    for payload in candidate_payloads(value):
        if isinstance(payload.get("function"), dict):
            name = payload["function"].get("name")
        else:
            name = payload.get("name") or payload.get("tool_name") or payload.get("tool")
        if name:
            names.append(str(name))
    return names


def candidate_arguments(value: Any) -> List[Dict[str, Any]]:
    args = []
    for payload in candidate_payloads(value):
        raw_args = payload.get("arguments")
        if raw_args is None and isinstance(payload.get("function"), dict):
            raw_args = payload["function"].get("arguments")
        if isinstance(raw_args, str):
            try:
                raw_args = json.loads(raw_args)
            except Exception:
                raw_args = {"_raw": raw_args}
        args.append(raw_args if isinstance(raw_args, dict) else {})
    return args


def walk_values(value: Any) -> Iterable[Any]:
    if isinstance(value, dict):
        for child in value.values():
            yield from walk_values(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_values(child)
    else:
        yield value


def has_fixed_width_numeric_string(value: Any) -> bool:
    for item in walk_values(value):
        if isinstance(item, str) and item.isdigit() and len(item) > 1 and item.startswith("0"):
            return True
    return False


def is_terminal_like_tool_name(name: Any) -> bool:
    lower = str(name or "").lower()
    terminalish = globals().get("TERMINALISH_TOOL_NAMES", {"respond", "summarize", "report", "present", "recommend", "diagnose"})
    return lower in terminalish or lower.startswith("submit_") or "summar" in lower or "report" in lower


def metadata_dict(value: Any) -> Dict[str, Any]:
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
        except Exception:
            parsed = None
        if isinstance(parsed, dict):
            return parsed
    return {}


def bool_value(value: Any) -> bool:
    if isinstance(value, str):
        return value.strip().lower() in {"1", "true", "yes", "y"}
    if pd.isna(value):
        return False
    return bool(value)


def boolean_series(frame: pd.DataFrame, column: str) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(False, index=frame.index)
    return frame[column].map(bool_value).fillna(False).astype(bool)


def add_valid_protection_slice_columns(scored: pd.DataFrame) -> pd.DataFrame:
    scored = scored.copy()
    payloads = scored["input_text"].map(extract_candidate_payload)
    scored["candidate_tool_names"] = payloads.map(candidate_tool_names)
    scored["candidate_arguments"] = payloads.map(candidate_arguments)

    terminal_from_payload = scored["candidate_tool_names"].map(lambda names: any(is_terminal_like_tool_name(name) for name in names))
    fixed_width_from_payload = scored["candidate_arguments"].map(has_fixed_width_numeric_string)
    noop_from_payload = scored["candidate_arguments"].map(lambda args: bool(args) and all(arg == {} for arg in args))
    corrected_from_metadata = scored.apply(
        lambda row: (
            row.get("true_label") == "valid"
            and "error_recovery" in str(metadata_dict(row.get("metadata")).get("scenario_family", ""))
            and is_corrected_positive_metadata(row.get("metadata"))
        ),
        axis=1,
    )

    scored["slice_terminal_like_tool"] = terminal_from_payload | boolean_series(scored, "valid_protection_terminal_like_tool")
    scored["slice_fixed_width_numeric_string"] = fixed_width_from_payload | boolean_series(scored, "valid_protection_fixed_width_numeric_string")
    scored["slice_noop_valid_call"] = noop_from_payload | boolean_series(scored, "valid_protection_noop_call")
    scored["slice_corrected_error_recovery_positive"] = corrected_from_metadata | boolean_series(scored, "valid_protection_corrected_error_recovery")
    return scored


def split_gate_metrics_for_predictions(
    scored: pd.DataFrame,
    prediction_col: str,
    confidence_col: str,
    false_objection_threshold: Optional[float],
    false_objection_key: str,
) -> Dict[str, Any]:
    valid_rows = scored[scored["true_label"] == "valid"]
    pred = scored[prediction_col]
    if false_objection_threshold is None:
        valid_false_objections = valid_rows[valid_rows[prediction_col] != "valid"]
    else:
        valid_false_objections = valid_rows[
            (valid_rows[prediction_col] != "valid")
            & (valid_rows[confidence_col] >= float(false_objection_threshold))
        ]
    wrong_tool_predictions = scored[pred == "wrong_tool_semantic"]
    wrong_tool_precision = None
    if len(wrong_tool_predictions):
        wrong_tool_precision = float((wrong_tool_predictions["true_label"] == "wrong_tool_semantic").mean())
    wrong_arguments_predictions = scored[pred == "wrong_arguments_semantic"]
    wrong_arguments_precision = None
    if len(wrong_arguments_predictions):
        wrong_arguments_precision = float((wrong_arguments_predictions["true_label"] == "wrong_arguments_semantic").mean())
    wrong_tool_rows = scored[scored["true_label"] == "wrong_tool_semantic"]
    valid_to_wrong_arguments = valid_rows[valid_rows[prediction_col] == "wrong_arguments_semantic"]
    wrong_tool_to_wrong_arguments = wrong_tool_rows[wrong_tool_rows[prediction_col] == "wrong_arguments_semantic"]
    return {
        "rows": int(len(scored)),
        "prediction_column": prediction_col,
        "confidence_column": confidence_col,
        "false_objection_threshold": false_objection_threshold,
        "valid_rows": int(len(valid_rows)),
        "valid_recall": float((valid_rows[prediction_col] == "valid").mean()) if len(valid_rows) else None,
        false_objection_key: {
            "false_objections": int(len(valid_false_objections)),
            "valid_rows": int(len(valid_rows)),
            "rate": float(len(valid_false_objections) / len(valid_rows)) if len(valid_rows) else None,
        },
        "wrong_tool_semantic_precision": wrong_tool_precision,
        "wrong_tool_semantic_predicted_rows": int(len(wrong_tool_predictions)),
        "wrong_arguments_semantic_precision": wrong_arguments_precision,
        "wrong_arguments_semantic_predicted_rows": int(len(wrong_arguments_predictions)),
        "valid_to_wrong_arguments_semantic": {
            "rows": int(len(valid_to_wrong_arguments)),
            "valid_rows": int(len(valid_rows)),
            "rate": float(len(valid_to_wrong_arguments) / len(valid_rows)) if len(valid_rows) else None,
        },
        "wrong_tool_to_wrong_arguments_semantic": {
            "rows": int(len(wrong_tool_to_wrong_arguments)),
            "wrong_tool_rows": int(len(wrong_tool_rows)),
            "rate": float(len(wrong_tool_to_wrong_arguments) / len(wrong_tool_rows)) if len(wrong_tool_rows) else None,
        },
        "needs_clarification_support": int((scored["true_label"] == "needs_clarification").sum()),
    }


def split_gate_metrics(scored: pd.DataFrame) -> Dict[str, Any]:
    return split_gate_metrics_for_predictions(
        scored,
        prediction_col="pred_label",
        confidence_col="confidence",
        false_objection_threshold=PROMOTION_DIAGNOSTIC_THRESHOLD,
        false_objection_key="valid_false_objection_at_0_90",
    )


def deployment_gate_metrics(scored: pd.DataFrame) -> Dict[str, Any]:
    return split_gate_metrics_for_predictions(
        scored,
        prediction_col="deployment_pred_label",
        confidence_col="deployment_confidence",
        false_objection_threshold=None,
        false_objection_key="valid_false_objection_deployment",
    )

def valid_protection_slice_metrics(
    scored: pd.DataFrame,
    prediction_col: str = "pred_label",
    confidence_col: str = "confidence",
    false_objection_threshold: Optional[float] = PROMOTION_DIAGNOSTIC_THRESHOLD,
    false_objection_key: str = "valid_false_objection_at_0_90",
) -> Dict[str, Any]:
    slice_columns = {
        "terminal_like_tool": "slice_terminal_like_tool",
        "corrected_error_recovery_positive": "slice_corrected_error_recovery_positive",
        "fixed_width_numeric_string": "slice_fixed_width_numeric_string",
        "noop_valid_call": "slice_noop_valid_call",
    }
    metrics: Dict[str, Any] = {}
    for slice_name, column in slice_columns.items():
        if column not in scored:
            continue
        subset = scored[scored[column].fillna(False).astype(bool)]
        valid_subset = subset[subset["true_label"] == "valid"]
        if false_objection_threshold is None:
            false_objections = valid_subset[valid_subset[prediction_col] != "valid"]
        else:
            false_objections = valid_subset[
                (valid_subset[prediction_col] != "valid")
                & (valid_subset[confidence_col] >= float(false_objection_threshold))
            ]
        metrics[slice_name] = {
            "rows": int(len(subset)),
            "valid_rows": int(len(valid_subset)),
            "prediction_column": prediction_col,
            "valid_recall": float((valid_subset[prediction_col] == "valid").mean()) if len(valid_subset) else None,
            false_objection_key: {
                "false_objections": int(len(false_objections)),
                "rate": float(len(false_objections) / len(valid_subset)) if len(valid_subset) else None,
            },
            "predicted_labels": {str(k): int(v) for k, v in Counter(subset[prediction_col]).most_common()},
        }
    return metrics


THRESHOLD_SWEEP_POINTS = [0.80, 0.90, 0.95, 0.98, 0.99]
CONFIDENCE_MARGIN_GRID = [(conf, margin) for conf in (0.95, 0.98, 0.99) for margin in (0.0, 0.10, 0.15)]
PER_SOURCE_DIAG_MIN_ROWS = 100
PER_SOURCE_DIAG_ACCURACY_GATE = 0.90
PER_SOURCE_DIAG_WRONG_TOOL_RECALL_GATE = 0.80


def threshold_sweep_diagnostics(scored: pd.DataFrame, split_name: str) -> Dict[str, Any]:
    """Diagnostic-only sweep of valid false-block rates and per-label objection
    precision per confidence threshold. Never added to the blocking gate list; the
    0.90 gate in evaluate_promotion_gates stays the only blocking false-objection
    threshold."""
    valid_rows = scored[scored["true_label"] == "valid"]
    non_valid_labels = [label for label in LABELS if label != "valid"]
    thresholds_out: Dict[str, Any] = {}
    for threshold in THRESHOLD_SWEEP_POINTS:
        false_blocks = valid_rows[(valid_rows["pred_label"] != "valid") & (valid_rows["confidence"] >= threshold)]
        per_label_objections: Dict[str, Any] = {}
        for label in non_valid_labels:
            predicted = scored[(scored["pred_label"] == label) & (scored["confidence"] >= threshold)]
            per_label_objections[label] = {
                "predicted_rows": int(len(predicted)),
                "objection_precision": float((predicted["true_label"] == label).mean()) if len(predicted) else None,
            }
        thresholds_out[str(threshold)] = {
            "valid_false_block": {
                "false_blocks": int(len(false_blocks)),
                "valid_rows": int(len(valid_rows)),
                "rate": float(len(false_blocks) / len(valid_rows)) if len(valid_rows) else None,
            },
            "per_label_objections": per_label_objections,
        }
    return {"split": split_name, "thresholds": thresholds_out}


def confidence_margin_diagnostics(scored: pd.DataFrame, split_name: str) -> Dict[str, Any]:
    """Diagnostic-only valid false-objection rates when an objection additionally
    requires a top1-top2 probability margin. Uses uncalibrated confidence, matching
    how the FO@0.90 gate measures."""
    valid_rows = scored[scored["true_label"] == "valid"]
    has_margin = "margin_top1_top2" in scored.columns
    grid: Dict[str, Any] = {}
    if has_margin:
        for conf, margin in CONFIDENCE_MARGIN_GRID:
            objections = valid_rows[
                (valid_rows["pred_label"] != "valid")
                & (valid_rows["confidence"] >= conf)
                & (valid_rows["margin_top1_top2"] >= margin)
            ]
            grid[f"conf_{conf:.2f}_margin_{margin:.2f}"] = {
                "false_objections": int(len(objections)),
                "valid_rows": int(len(valid_rows)),
                "rate": float(len(objections) / len(valid_rows)) if len(valid_rows) else None,
            }
    return {"split": split_name, "margin_column_present": bool(has_margin), "grid": grid}


def per_source_diagnostic_gates(scored: pd.DataFrame, split_name: str) -> Dict[str, Any]:
    """Diagnostic-only per-source gates; never added to the blocking promotion gates.
    Sources under PER_SOURCE_DIAG_MIN_ROWS rows are reported as insufficient_support
    instead of being judged on noise-level metrics."""
    sources_out: Dict[str, Any] = {}
    for source_name, src_df in scored.groupby("source"):
        total = int(len(src_df))
        if total < PER_SOURCE_DIAG_MIN_ROWS:
            sources_out[str(source_name)] = {"status": "insufficient_support", "rows": total}
            continue
        accuracy = float((src_df["pred_label"] == src_df["true_label"]).mean())
        valid_rows_src = src_df[src_df["true_label"] == "valid"]
        fo_rate = None
        if len(valid_rows_src):
            fo_rows = valid_rows_src[(valid_rows_src["pred_label"] != "valid") & (valid_rows_src["confidence"] >= 0.90)]
            fo_rate = float(len(fo_rows) / len(valid_rows_src))
        wrong_tool_rows_src = src_df[src_df["true_label"] == "wrong_tool_semantic"]
        wrong_tool_recall = None
        if len(wrong_tool_rows_src):
            wrong_tool_recall = float((wrong_tool_rows_src["pred_label"] == "wrong_tool_semantic").mean())
        gates = {
            "accuracy": {
                "value": accuracy,
                "threshold": f">= {PER_SOURCE_DIAG_ACCURACY_GATE}",
                "passed": accuracy >= PER_SOURCE_DIAG_ACCURACY_GATE,
            },
            "valid_false_objection_at_0_90": {
                "value": fo_rate,
                "threshold": f"<= {VALID_FALSE_OBJECTION_90_GATE}",
                "passed": None if fo_rate is None else fo_rate <= VALID_FALSE_OBJECTION_90_GATE,
            },
            "wrong_tool_semantic_recall": {
                "value": wrong_tool_recall,
                "threshold": f">= {PER_SOURCE_DIAG_WRONG_TOOL_RECALL_GATE}",
                "passed": None if wrong_tool_recall is None else wrong_tool_recall >= PER_SOURCE_DIAG_WRONG_TOOL_RECALL_GATE,
            },
        }
        sources_out[str(source_name)] = {
            "status": "evaluated",
            "rows": total,
            "gates": gates,
            "passed": all(gate["passed"] for gate in gates.values() if gate["passed"] is not None),
        }
    return {
        "split": split_name,
        "note": "diagnostic_only; never added to blocking promotion gates",
        "min_rows": PER_SOURCE_DIAG_MIN_ROWS,
        "sources": sources_out,
    }


def protected_slice_false_objection_gate(
    slice_metrics: Dict[str, Any],
    false_objection_key: str,
    rate_gate: float,
) -> Tuple[bool, str, Dict[str, Any]]:
    valid_rows = int(slice_metrics.get("valid_rows", 0))
    false_info = slice_metrics.get(false_objection_key, {})
    false_objections = int(false_info.get("false_objections", 0))
    false_rate = false_info.get("rate")
    rate_passed = false_rate is not None and float(false_rate) <= float(rate_gate)
    count_passed = false_objections <= int(PROTECTED_SLICE_FALSE_OBJECTION_MAX_COUNT)
    passed = bool(rate_passed or count_passed)
    threshold = (
        f"<= {rate_gate} rate or <= {PROTECTED_SLICE_FALSE_OBJECTION_MAX_COUNT} false objection(s)"
    )
    details = {
        "valid_rows": valid_rows,
        "false_objections": false_objections,
        "rate_gate": float(rate_gate),
        "count_gate": int(PROTECTED_SLICE_FALSE_OBJECTION_MAX_COUNT),
        "rate_gate_min_valid_rows": int(PROTECTED_SLICE_RATE_GATE_MIN_VALID_ROWS),
        "rate_passed": bool(rate_passed),
        "count_passed": bool(count_passed),
        "count_gate_primary": valid_rows < int(PROTECTED_SLICE_RATE_GATE_MIN_VALID_ROWS),
    }
    return passed, threshold, details


def build_raw_gate_report(
    split_name: str,
    diagnostic_metrics: Dict[str, Any],
    diagnostic_slices: Dict[str, Any],
) -> Dict[str, Any]:
    gates = []

    def add_raw_gate(name: str, passed: bool, value: Any, threshold: Any, details: Optional[Dict[str, Any]] = None) -> None:
        gates.append({
            "name": name,
            "passed": bool(passed),
            "value": value,
            "threshold": threshold,
            "details": details or {},
        })

    raw_valid_recall = diagnostic_metrics["valid_recall"]
    add_raw_gate("valid_recall", raw_valid_recall is not None and raw_valid_recall >= VALID_RECALL_GATE, raw_valid_recall, f">= {VALID_RECALL_GATE}")
    raw_false_rate = diagnostic_metrics["valid_false_objection_at_0_90"]["rate"]
    add_raw_gate(
        "valid_false_objection_at_0_90",
        raw_false_rate is not None and raw_false_rate <= VALID_FALSE_OBJECTION_DIAGNOSTIC_GATE,
        raw_false_rate,
        f"<= {VALID_FALSE_OBJECTION_DIAGNOSTIC_GATE}",
        {"threshold": PROMOTION_DIAGNOSTIC_THRESHOLD},
    )
    raw_wrong_tool_precision = diagnostic_metrics["wrong_tool_semantic_precision"]
    add_raw_gate(
        "wrong_tool_semantic_precision",
        raw_wrong_tool_precision is not None and raw_wrong_tool_precision >= WRONG_TOOL_PRECISION_GATE,
        raw_wrong_tool_precision,
        f">= {WRONG_TOOL_PRECISION_GATE}",
    )

    for slice_name, slice_metrics in diagnostic_slices.items():
        valid_rows = int(slice_metrics["valid_rows"])
        if valid_rows < VALID_PROTECTION_SLICE_MIN_VALID_ROWS:
            continue
        slice_recall = slice_metrics["valid_recall"]
        add_raw_gate(
            f"{slice_name}.valid_recall",
            slice_recall is not None and slice_recall >= VALID_RECALL_GATE,
            slice_recall,
            f">= {VALID_RECALL_GATE}",
            {"valid_rows": valid_rows},
        )
        slice_false_info = slice_metrics["valid_false_objection_at_0_90"]
        slice_false_passed, slice_false_threshold, slice_false_details = protected_slice_false_objection_gate(
            slice_metrics,
            "valid_false_objection_at_0_90",
            VALID_FALSE_OBJECTION_DIAGNOSTIC_GATE,
        )
        add_raw_gate(
            f"{slice_name}.valid_false_objection_at_0_90",
            slice_false_passed,
            slice_false_info.get("rate"),
            slice_false_threshold,
            slice_false_details,
        )

    return {
        "split": split_name,
        "status": "passed" if all(gate["passed"] for gate in gates) else "blocked",
        "passed": all(gate["passed"] for gate in gates),
        "gates": gates,
        "metrics": diagnostic_metrics,
        "valid_protection_slices": diagnostic_slices,
    }


def derive_promotion_status(gate_report: Dict[str, Any]) -> Tuple[str, List[Dict[str, Any]], bool]:
    """Artifact status is policy-configurable, but enforcement still requires replay proof."""
    raw_gate_passed = bool(all(gate_report[split]["raw_gate_report"]["passed"] for split in ("validation", "test")))
    deployment_gate_passed = bool(all(gate_report[split]["passed"] for split in ("validation", "test")))
    blocked_reasons: List[Dict[str, Any]] = []

    if BLOCK_ON_RAW_0_90_GATE and not raw_gate_passed:
        blocked_reasons.extend([
            {
                "policy": "raw_0_90",
                "split": split,
                "gate": gate["name"],
                "value": gate["value"],
                "threshold": gate["threshold"],
            }
            for split in ("validation", "test")
            for gate in gate_report[split]["raw_gate_report"]["gates"]
            if not gate["passed"]
        ])
    if BLOCK_ON_DEPLOYMENT_GATE and not deployment_gate_passed:
        blocked_reasons.extend([
            {
                "policy": "deployment",
                "split": split,
                "gate": gate["name"],
                "value": gate["value"],
                "threshold": gate["threshold"],
            }
            for split in ("validation", "test")
            for gate in gate_report[split]["gates"]
            if gate.get("blocking", True) and not gate["passed"]
        ])

    artifact_promotable = not blocked_reasons
    status = "promotable_pending_replay" if artifact_promotable else "blocked"
    gate_report["raw_gate_status"] = "passed" if raw_gate_passed else "blocked"
    gate_report["deployment_gate_status"] = "passed" if deployment_gate_passed else "blocked"
    gate_report["artifact_status"] = status
    gate_report["status_policy"] = {
        "block_on_raw_0_90_gate": bool(BLOCK_ON_RAW_0_90_GATE),
        "block_on_deployment_gate": bool(BLOCK_ON_DEPLOYMENT_GATE),
        "raw_0_90_false_objection": "blocking" if BLOCK_ON_RAW_0_90_GATE else "diagnostic_only",
    }
    gate_report["enforcement_eligible"] = False
    gate_report["enforcement_eligible_status"] = "blocked_pending_replay"
    gate_report["enforcement_replay_requirements"] = ENFORCEMENT_REPLAY_REQUIREMENTS
    return status, blocked_reasons, artifact_promotable


def evaluate_promotion_gates(scored: pd.DataFrame, split_name: str) -> Dict[str, Any]:
    diagnostic_metrics = split_gate_metrics(scored)
    deployment_metrics = deployment_gate_metrics(scored)
    diagnostic_slices = valid_protection_slice_metrics(scored)
    raw_gate_report = build_raw_gate_report(split_name, diagnostic_metrics, diagnostic_slices)
    deployment_slices = valid_protection_slice_metrics(
        scored,
        prediction_col="deployment_pred_label",
        confidence_col="deployment_confidence",
        false_objection_threshold=None,
        false_objection_key="valid_false_objection_deployment",
    )
    gates = []

    def add_gate(
        name: str,
        passed: bool,
        value: Any,
        threshold: Any,
        details: Optional[Dict[str, Any]] = None,
        blocking: bool = True,
    ) -> None:
        gates.append({
            "name": name,
            "passed": bool(passed),
            "blocking": bool(blocking),
            "value": value,
            "threshold": threshold,
            "details": details or {},
        })

    valid_recall = deployment_metrics["valid_recall"]
    add_gate(
        "valid_recall",
        valid_recall is not None and valid_recall >= VALID_RECALL_GATE,
        valid_recall,
        f">= {VALID_RECALL_GATE}",
        {"prediction_rule": "deployment_guarded"},
    )
    diagnostic_false_rate = diagnostic_metrics["valid_false_objection_at_0_90"]["rate"]
    add_gate(
        "diagnostic_valid_false_objection_at_0_90",
        diagnostic_false_rate is not None and diagnostic_false_rate <= VALID_FALSE_OBJECTION_DIAGNOSTIC_GATE,
        diagnostic_false_rate,
        f"<= {VALID_FALSE_OBJECTION_DIAGNOSTIC_GATE}",
        {"threshold": PROMOTION_DIAGNOSTIC_THRESHOLD, "prediction_rule": "raw_model_top_label"},
        blocking=False,
    )
    deployment_false_rate = deployment_metrics["valid_false_objection_deployment"]["rate"]
    add_gate(
        "deployment_valid_false_objection",
        deployment_false_rate is not None and deployment_false_rate <= VALID_FALSE_OBJECTION_DEPLOYMENT_GATE,
        deployment_false_rate,
        f"<= {VALID_FALSE_OBJECTION_DEPLOYMENT_GATE}",
        {
            "thresholds": PROMOTION_DEPLOYMENT_THRESHOLDS,
            "min_margin_over_valid": PROMOTION_MIN_MARGIN_OVER_VALID,
            "valid_logit_bias": PROMOTION_VALID_LOGIT_BIAS,
        },
    )
    wrong_tool_precision = deployment_metrics["wrong_tool_semantic_precision"]
    add_gate(
        "wrong_tool_semantic_precision",
        wrong_tool_precision is not None and wrong_tool_precision >= WRONG_TOOL_PRECISION_GATE,
        wrong_tool_precision,
        f">= {WRONG_TOOL_PRECISION_GATE}",
        {"prediction_rule": "deployment_guarded"},
    )

    needs_support = deployment_metrics["needs_clarification_support"]
    add_gate(
        "needs_clarification_support_gate_enabled",
        True,
        needs_support,
        f"ignored unless support >= {NEEDS_CLARIFICATION_MIN_SUPPORT}",
        {"ignored": needs_support < NEEDS_CLARIFICATION_MIN_SUPPORT},
    )

    for slice_name, slice_metrics in deployment_slices.items():
        valid_rows = int(slice_metrics["valid_rows"])
        if valid_rows < VALID_PROTECTION_SLICE_MIN_VALID_ROWS:
            continue
        slice_recall = slice_metrics["valid_recall"]
        add_gate(
            f"{slice_name}.valid_recall",
            slice_recall is not None and slice_recall >= VALID_RECALL_GATE,
            slice_recall,
            f">= {VALID_RECALL_GATE}",
            {"valid_rows": valid_rows, "prediction_rule": "deployment_guarded"},
        )
        slice_false_info = slice_metrics["valid_false_objection_deployment"]
        slice_false_passed, slice_false_threshold, slice_false_details = protected_slice_false_objection_gate(
            slice_metrics,
            "valid_false_objection_deployment",
            VALID_FALSE_OBJECTION_DEPLOYMENT_GATE,
        )
        add_gate(
            f"{slice_name}.deployment_valid_false_objection",
            slice_false_passed,
            slice_false_info.get("rate"),
            slice_false_threshold,
            slice_false_details,
        )

    return {
        "split": split_name,
        "raw_gate_report": raw_gate_report,
        "diagnostic_metrics": diagnostic_metrics,
        "deployment_metrics": deployment_metrics,
        "metrics": diagnostic_metrics,
        "valid_protection_slices": diagnostic_slices,
        "deployment_valid_protection_slices": deployment_slices,
        "gates": gates,
        "passed": all(gate["passed"] for gate in gates if gate.get("blocking", True)),
    }


test_metrics = trainer.evaluate(tokenized["test"])
print("Test metrics:")
print(json.dumps(test_metrics, indent=2))

calibration_scored = score_split("calibration")
valid_scored = score_split("validation")
test_scored = score_split("test")

print_report(test_scored, "test")

mistakes = test_scored[test_scored["correct"] == False].sort_values("confidence", ascending=False)[
    ["true_label", "pred_label", "confidence", "source", "input_text", "metadata"]
]
print("High-confidence mistakes:")
display(mistakes.head(25))

# Export wrong predictions at very high confidence across validation+test for manual
# audit (forge_argument_semantic showed >= 0.99 confidence on mispredicted rows).
HIGH_CONF_MISTAKE_THRESHOLD = 0.99
high_confidence_mistakes = pd.concat(
    [valid_scored.assign(eval_split="validation"), test_scored.assign(eval_split="test")],
    ignore_index=True,
)
high_confidence_mistakes = high_confidence_mistakes[
    (~high_confidence_mistakes["correct"].astype(bool))
    & (high_confidence_mistakes["confidence"] >= HIGH_CONF_MISTAKE_THRESHOLD)
][["eval_split", "source", "true_label", "pred_label", "confidence"]]
high_confidence_mistakes.to_json(DATA_DIR / "high_confidence_mistakes.jsonl", orient="records", lines=True, force_ascii=False)
print(f"High-confidence (>= {HIGH_CONF_MISTAKE_THRESHOLD}) mistakes exported: {len(high_confidence_mistakes)} rows")


def export_valid_false_objections(eval_frame: pd.DataFrame, logits: np.ndarray, split_name: str) -> pd.DataFrame:
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=-1)
    conf = probs[np.arange(len(preds)), preds]
    valid_p = probs[:, VALID_LABEL_ID]
    margin = logits[np.arange(len(preds)), preds] - logits[:, VALID_LABEL_ID]

    out = eval_frame.copy()
    out["pred_label"] = [id2label[int(x)] for x in preds]
    out["pred_confidence"] = conf
    out["valid_confidence"] = valid_p
    out["nonvalid_valid_margin"] = margin
    if "deployment_pred_label" not in out.columns:
        deployment_cols = guarded_prediction_columns(logits)
        out = pd.concat([out.reset_index(drop=True), deployment_cols.reset_index(drop=True)], axis=1)

    false_obj = out[
        (out["true_label"] == "valid")
        & (out["pred_label"] != "valid")
    ].copy()
    false_obj["deployment_would_block"] = false_obj["deployment_pred_label"] != "valid"
    false_obj = false_obj.sort_values(
        ["deployment_would_block", "pred_confidence", "nonvalid_valid_margin"],
        ascending=[False, False, False],
    )

    path = DATA_DIR / f"{split_name}_valid_false_objections.jsonl"
    false_obj.to_json(path, orient="records", lines=True, force_ascii=False)
    print(f"Exported {len(false_obj)} valid false objections to {path}")
    return false_obj


validation_valid_false_objections = export_valid_false_objections(valid_scored, split_logits["validation"], "validation")
test_valid_false_objections = export_valid_false_objections(test_scored, split_logits["test"], "test")

threshold_sweep = {
    "validation": threshold_sweep_diagnostics(valid_scored, "validation"),
    "test": threshold_sweep_diagnostics(test_scored, "test"),
}
confidence_margin_report = {
    "validation": confidence_margin_diagnostics(valid_scored, "validation"),
    "test": confidence_margin_diagnostics(test_scored, "test"),
}


def deployment_rule_sweep(scored: pd.DataFrame, logits: np.ndarray, split_name: str) -> Dict[str, Any]:
    sweep_rows = []
    for threshold in [0.90, 0.95, 0.97, 0.98, 0.99]:
        for margin in [0.0, 0.25, 0.5, 0.75, 1.0]:
            for valid_bias in [0.0, 0.1, 0.2, 0.3]:
                preds = guarded_predictions(
                    logits,
                    nonvalid_conf_threshold=threshold,
                    min_nonvalid_valid_margin=margin,
                    valid_logit_bias=valid_bias,
                )
                tmp = scored.copy()
                tmp["sweep_pred_label"] = [id2label[int(x)] for x in preds]
                metrics = split_gate_metrics_for_predictions(
                    tmp,
                    prediction_col="sweep_pred_label",
                    confidence_col="confidence",
                    false_objection_threshold=None,
                    false_objection_key="valid_false_objection_deployment",
                )
                false_rate = metrics["valid_false_objection_deployment"]["rate"]
                valid_recall = metrics["valid_recall"]
                wrong_tool_precision = metrics["wrong_tool_semantic_precision"]
                promotable = bool(
                    valid_recall is not None
                    and valid_recall >= VALID_RECALL_GATE
                    and false_rate is not None
                    and false_rate <= VALID_FALSE_OBJECTION_DEPLOYMENT_GATE
                    and wrong_tool_precision is not None
                    and wrong_tool_precision >= WRONG_TOOL_PRECISION_GATE
                )
                sweep_rows.append({
                    "split": split_name,
                    "threshold": float(threshold),
                    "margin": float(margin),
                    "valid_bias": float(valid_bias),
                    "promotable": promotable,
                    "valid_recall": valid_recall,
                    "valid_false_objection_rate": false_rate,
                    "valid_false_objections": metrics["valid_false_objection_deployment"]["false_objections"],
                    "valid_rows": metrics["valid_false_objection_deployment"]["valid_rows"],
                    "wrong_tool_semantic_precision": wrong_tool_precision,
                    "wrong_tool_semantic_predicted_rows": metrics["wrong_tool_semantic_predicted_rows"],
                })
    sweep_df = pd.DataFrame(sweep_rows).sort_values(
        ["promotable", "valid_false_objection_rate", "valid_recall"],
        ascending=[False, True, False],
    )
    print(f"Deployment guarded-objection sweep ({split_name}):")
    display(sweep_df.head(30))
    return {
        "split": split_name,
        "rows": sweep_df.to_dict(orient="records"),
        "selected_rule": {
            "thresholds": PROMOTION_DEPLOYMENT_THRESHOLDS,
            "min_margin_over_valid": PROMOTION_MIN_MARGIN_OVER_VALID,
            "valid_logit_bias": PROMOTION_VALID_LOGIT_BIAS,
        },
    }


deployment_rule_sweep_report = {
    "validation": deployment_rule_sweep(valid_scored, split_logits["validation"], "validation"),
    "test": deployment_rule_sweep(test_scored, split_logits["test"], "test"),
}
per_source_diagnostics = {
    "validation": per_source_diagnostic_gates(valid_scored, "validation"),
    "test": per_source_diagnostic_gates(test_scored, "test"),
}
# Backward-compat alias consumed by training_metrics.json / training_run_summary.json.
valid_false_block_rates = {
    threshold_key: dict(entry["valid_false_block"])
    for threshold_key, entry in threshold_sweep["test"]["thresholds"].items()
    if entry["valid_false_block"]["rate"] is not None
}
for threshold_key, block in valid_false_block_rates.items():
    print(f"valid-call false block rate @ {float(threshold_key):.2f}: {block['false_blocks']}/{block['valid_rows']} = {block['rate']:.4f}")

valid_protection_slice_report = {
    "schema_version": "toolcall-verifier-valid-protection-slices/v1",
    "validation": valid_protection_slice_metrics(valid_scored),
    "test": valid_protection_slice_metrics(test_scored),
    "deployment_validation": valid_protection_slice_metrics(
        valid_scored,
        prediction_col="deployment_pred_label",
        confidence_col="deployment_confidence",
        false_objection_threshold=None,
        false_objection_key="valid_false_objection_deployment",
    ),
    "deployment_test": valid_protection_slice_metrics(
        test_scored,
        prediction_col="deployment_pred_label",
        confidence_col="deployment_confidence",
        false_objection_threshold=None,
        false_objection_key="valid_false_objection_deployment",
    ),
}
classifier_diagnostic_report = {
    "schema_version": "toolcall-verifier-classifier-diagnostics/v1",
    "latest_failed_t4_fast_baseline": {
        "test_macro_f1": 0.3716486037399564,
        "test_valid_recall": 0.5279547062986554,
        "test_wrong_tool_semantic_precision": 0.3870967741935484,
    },
    "validation_gate_metrics": split_gate_metrics(valid_scored),
    "test_gate_metrics": split_gate_metrics(test_scored),
    "validation_deployment_gate_metrics": deployment_gate_metrics(valid_scored),
    "test_deployment_gate_metrics": deployment_gate_metrics(test_scored),
    "validation_confusion_pairs": confusion_pair_diagnostics(valid_scored),
    "test_confusion_pairs": confusion_pair_diagnostics(test_scored),
}
promotion_gate_report = {
    "schema_version": "toolcall-verifier-promotion-gates/v1",
    "gate_config": {
        "block_on_raw_0_90_gate": BLOCK_ON_RAW_0_90_GATE,
        "block_on_deployment_gate": BLOCK_ON_DEPLOYMENT_GATE,
        "valid_recall_min": VALID_RECALL_GATE,
        "diagnostic_valid_false_objection_threshold": PROMOTION_DIAGNOSTIC_THRESHOLD,
        "valid_false_objection_0_90_max": VALID_FALSE_OBJECTION_DIAGNOSTIC_GATE,
        "deployment_valid_false_objection_max": VALID_FALSE_OBJECTION_DEPLOYMENT_GATE,
        "deployment_thresholds": PROMOTION_DEPLOYMENT_THRESHOLDS,
        "deployment_min_margin_over_valid": PROMOTION_MIN_MARGIN_OVER_VALID,
        "deployment_valid_logit_bias": PROMOTION_VALID_LOGIT_BIAS,
        "protected_slice_false_objection_max_count": PROTECTED_SLICE_FALSE_OBJECTION_MAX_COUNT,
        "protected_slice_rate_gate_min_valid_rows": PROTECTED_SLICE_RATE_GATE_MIN_VALID_ROWS,
        "wrong_tool_semantic_precision_min": WRONG_TOOL_PRECISION_GATE,
        "needs_clarification_min_support": NEEDS_CLARIFICATION_MIN_SUPPORT,
        "slice_min_valid_rows": VALID_PROTECTION_SLICE_MIN_VALID_ROWS,
    },
    "validation": evaluate_promotion_gates(valid_scored, "validation"),
    "test": evaluate_promotion_gates(test_scored, "test"),
    "threshold_sweep": threshold_sweep,
    "confidence_margin_diagnostics": confidence_margin_report,
    "deployment_rule_sweep": deployment_rule_sweep_report,
    "per_source_diagnostics": per_source_diagnostics,
}
promotion_status, blocked_reasons, artifact_promotable = derive_promotion_status(promotion_gate_report)
promotion_gate_report["promotion_status"] = promotion_status
promotion_gate_report["blocked_reasons"] = blocked_reasons
promotion_gate_report["artifact_promotable"] = artifact_promotable
promotion_gate_report["artifact_status"] = promotion_gate_report.get("artifact_status", promotion_status)

# Authoritative promotability comes only from promotion_gate_report (derived above);
# eval_checkpoint_constrained_promotable is a checkpoint-selection signal, not artifact status.
test_metrics.pop("eval_constrained_promotable", None)
test_metrics["promotion_status"] = promotion_status
test_metrics["artifact_status"] = promotion_gate_report.get("artifact_status", promotion_status)
test_metrics["raw_gate_status"] = promotion_gate_report.get("raw_gate_status")
test_metrics["deployment_gate_status"] = promotion_gate_report.get("deployment_gate_status")
test_metrics["enforcement_eligible_status"] = promotion_gate_report.get("enforcement_eligible_status")
test_metrics["blocked_reasons"] = blocked_reasons
test_metrics["artifact_promotable"] = artifact_promotable


def assert_promotion_consistency(claim_label: str, claimed_promotable: Any) -> None:
    if bool(claimed_promotable) and not artifact_promotable:
        raise RuntimeError(
            f"Promotion consistency violation: {claim_label} claims promotable=True while "
            f"promotion_gate_report is blocked: {json.dumps(blocked_reasons)}"
        )


assert_promotion_consistency("test_metrics.artifact_promotable", test_metrics["artifact_promotable"])
if "eval_constrained_promotable" in test_metrics:
    raise RuntimeError("Ambiguous eval_constrained_promotable key must not be exported")
(DATA_DIR / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
(DATA_DIR / "valid_protection_slice_metrics.json").write_text(json.dumps(valid_protection_slice_report, indent=2))
(DATA_DIR / "classifier_diagnostics.json").write_text(json.dumps(classifier_diagnostic_report, indent=2, ensure_ascii=False))
(DATA_DIR / "promotion_gate_report.json").write_text(json.dumps(promotion_gate_report, indent=2))
calibration_scored.to_json(DATA_DIR / "calibration_scored.jsonl", orient="records", lines=True, force_ascii=False)
valid_scored.to_json(DATA_DIR / "validation_scored.jsonl", orient="records", lines=True, force_ascii=False)
test_scored.to_json(DATA_DIR / "test_scored.jsonl", orient="records", lines=True, force_ascii=False)

combined_training_metrics = {
    "training_run_summary": training_run_summary if "training_run_summary" in globals() else None,
    "test_metrics": test_metrics,
    "valid_false_block_rates": valid_false_block_rates,
    "valid_protection_slice_metrics": valid_protection_slice_report,
    "classifier_diagnostics": classifier_diagnostic_report,
    "split_balance_report": split_balance_report if "split_balance_report" in globals() else None,
    "tokenization_diagnostics": tokenization_diagnostics if "tokenization_diagnostics" in globals() else None,
    "promotion_gate_report": promotion_gate_report,
}
(DATA_DIR / "training_metrics.json").write_text(json.dumps(combined_training_metrics, indent=2))

if "training_run_summary" in globals():
    training_run_summary["test_metrics"] = test_metrics
    training_run_summary["valid_false_block_rates"] = valid_false_block_rates
    training_run_summary["valid_protection_slice_metrics"] = valid_protection_slice_report
    training_run_summary["classifier_diagnostics"] = classifier_diagnostic_report
    training_run_summary["split_balance_summary"] = split_balance_report if "split_balance_report" in globals() else None
    training_run_summary["tokenization_diagnostics_summary"] = tokenization_diagnostics if "tokenization_diagnostics" in globals() else None
    training_run_summary["promotion_gate_report"] = promotion_gate_report
    training_run_summary["promotion_status"] = promotion_status
    training_run_summary["blocked_reasons"] = blocked_reasons
    training_run_summary["artifact_promotable"] = artifact_promotable
    (DATA_DIR / "training_run_summary.json").write_text(json.dumps(training_run_summary, indent=2))

print("Promotion gate report:")
print(json.dumps(promotion_gate_report, indent=2)[:6000])
failed_splits = [name for name in ["validation", "test"] if not promotion_gate_report[name]["passed"]]
if failed_splits:
    failure_message = f"Tool-call verifier promotion gates failed for: {', '.join(failed_splits)}"
    if ENFORCE_TOOLCALL_PROMOTION_GATES:
        raise RuntimeError(failure_message)
    print("WARNING:", failure_message)
    print("Artifact export may continue for shadow/advisory diagnostics only. Do not promote this artifact.")

## 9. Threshold calibration

Use validation data to create conservative per-label thresholds. The exported default mode remains `shadow`; enforcement should be enabled only after repo-specific eval proof.

In [ ]:
BLOCKING_LABELS = [label for label in LABELS if label not in ("valid", "deterministic_invalid")]


def expected_calibration_error(probs: np.ndarray, labels_arr: np.ndarray, n_bins: int = 15) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = predictions == labels_arr
    ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins, endpoint=False), np.linspace(1 / n_bins, 1, n_bins)):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.any():
            ece += mask.mean() * abs(float(accuracies[mask].mean()) - float(confidences[mask].mean()))
    return float(ece)


def fit_temperature(logits: np.ndarray, labels_arr: np.ndarray) -> float:
    logits_t = torch.tensor(logits, dtype=torch.float32)
    labels_t = torch.tensor(labels_arr, dtype=torch.long)
    temperature = torch.nn.Parameter(torch.ones(()))
    optimizer = torch.optim.LBFGS([temperature], lr=0.01, max_iter=50)
    loss_fn = nn.CrossEntropyLoss()
    def closure():
        optimizer.zero_grad()
        scaled = logits_t / temperature.clamp_min(0.05)
        loss = loss_fn(scaled, labels_t)
        loss.backward()
        return loss
    optimizer.step(closure)
    return float(temperature.detach().clamp(0.05, 10.0).item())


calibration_logits = split_logits.get("calibration")
calibration_labels = split_label_ids.get("calibration")
if calibration_logits is not None and len(calibration_logits):
    TEMPERATURE = fit_temperature(calibration_logits, calibration_labels)
else:
    TEMPERATURE = 1.0
print("Temperature:", TEMPERATURE)


def add_calibrated_probs(scored: pd.DataFrame, split_name: str) -> pd.DataFrame:
    logits = split_logits[split_name]
    labels_arr = split_label_ids[split_name]
    uncal = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    cal = torch.softmax(torch.tensor(logits / TEMPERATURE), dim=-1).numpy()
    scored = scored.copy()
    scored["uncalibrated_ece"] = expected_calibration_error(uncal, labels_arr)
    scored["calibrated_ece"] = expected_calibration_error(cal, labels_arr)
    scored["calibrated_confidence"] = cal.max(axis=-1)
    scored["calibrated_pred_label"] = [id2label[int(i)] for i in cal.argmax(axis=-1)]
    for idx, label in id2label.items():
        scored[f"cal_prob_{label}"] = cal[:, int(idx)]
    return scored


calibration_scored = add_calibrated_probs(calibration_scored, "calibration")
valid_scored = add_calibrated_probs(valid_scored, "validation")
test_scored = add_calibrated_probs(test_scored, "test")

reliability_rows = []
for label in LABELS:
    col = f"cal_prob_{label}"
    if col not in valid_scored:
        continue
    for lo, hi in zip(np.linspace(0, 1, 10, endpoint=False), np.linspace(0.1, 1, 10)):
        mask = (valid_scored[col] > lo) & (valid_scored[col] <= hi)
        if mask.any():
            reliability_rows.append({
                "label": label,
                "bin_low": float(lo),
                "bin_high": float(hi),
                "rows": int(mask.sum()),
                "avg_confidence": float(valid_scored.loc[mask, col].mean()),
                "precision": float((valid_scored.loc[mask, "true_label"] == label).mean()),
            })
reliability = pd.DataFrame(reliability_rows)
reliability.to_json(DATA_DIR / "reliability_curves.jsonl", orient="records", lines=True)


def choose_threshold_for_label(scored: pd.DataFrame, label: str, target_precision: float, min_threshold: float, max_threshold: float = 0.995) -> float:
    col = f"cal_prob_{label}"
    if col not in scored:
        return 1.01
    candidates = np.linspace(min_threshold, max_threshold, 80)
    best = 1.01
    for th in candidates:
        selected = scored[scored[col] >= th]
        if selected.empty:
            continue
        precision = (selected["true_label"] == label).mean()
        valid_false_blocks = selected[(selected["true_label"] == "valid") & (label != "valid")]
        if precision >= target_precision and len(valid_false_blocks) == 0:
            best = float(th)
            break
    return best


DEPLOYMENT_RULE = {
    "nonvalid_confidence_thresholds": PROMOTION_DEPLOYMENT_THRESHOLDS,
    "min_margin_over_valid": PROMOTION_MIN_MARGIN_OVER_VALID,
    "valid_logit_bias": PROMOTION_VALID_LOGIT_BIAS,
    "valid_false_objection_gate": VALID_FALSE_OBJECTION_DEPLOYMENT_GATE,
}
RECOMMENDED_ACTION_POLICY = {
    "mode": "advisory",
    "deployment_thresholds": PROMOTION_DEPLOYMENT_THRESHOLDS,
    "min_margin_over_valid": PROMOTION_MIN_MARGIN_OVER_VALID,
    "valid_logit_bias": PROMOTION_VALID_LOGIT_BIAS,
    "raw_0_90_false_objection": "diagnostic_only",
    "enforce": False,
}

candidate_thresholds = {
    "schema_version": "toolcall-verifier-candidate-thresholds/v1",
    "source": "validation_calibration_diagnostics_only",
    "temperature": TEMPERATURE,
    "deployment_rule": DEPLOYMENT_RULE,
    "recommended_action_policy": RECOMMENDED_ACTION_POLICY,
    "notes": [
        "Candidate thresholds are diagnostics only.",
        "The active artifact mode remains shadow; the recommended next mode is advisory after replay.",
        "Promotion is evaluated against the guarded deployment rule, not raw top-label confidence alone.",
    ],
    "labels": {},
}

thresholds = {
    "schema_version": "toolcall-verifier-thresholds/v1",
    "mode": "shadow",
    "default_action": "allow",
    "temperature": TEMPERATURE,
    "deployment_rule": DEPLOYMENT_RULE,
    "recommended_action_policy": RECOMMENDED_ACTION_POLICY,
    "notes": [
        "Deterministic guardrails remain authoritative.",
        "The current artifact family must start in shadow mode.",
        "Advisory is the next candidate action mode after replay; enforcement remains disabled.",
        "deterministic_invalid is never enforced by ML.",
        "Candidate calibrated thresholds are written to candidate_thresholds.json for diagnostics only.",
        "A deployment objection requires both label confidence and margin over valid.",
    ],
    "labels": {},
}

for label in LABELS:
    if label == "valid":
        thresholds["labels"][label] = {"action": "allow", "advisory_min_confidence": 0.0, "enforce_min_confidence": 1.01}
        candidate_thresholds["labels"][label] = {"action": "allow", "advisory_min_confidence": 0.0, "enforce_min_confidence": 1.01}
    elif label == "deterministic_invalid":
        thresholds["labels"][label] = {"action": "deterministic_only", "advisory_min_confidence": 1.01, "enforce_min_confidence": 1.01}
        candidate_thresholds["labels"][label] = {"action": "deterministic_only", "advisory_min_confidence": 1.01, "enforce_min_confidence": 1.01}
    else:
        advisory = choose_threshold_for_label(valid_scored, label, target_precision=0.90, min_threshold=0.80)
        enforce = choose_threshold_for_label(valid_scored, label, target_precision=0.98, min_threshold=0.95)
        deployment_confidence = float(PROMOTION_DEPLOYMENT_THRESHOLDS.get(label, 1.01))
        thresholds["labels"][label] = {
            "action": "shadow_only_until_eval_proven",
            "advisory_min_confidence": 1.01,
            "enforce_min_confidence": 1.01,
            "deployment_candidate_min_confidence": deployment_confidence,
            "deployment_min_margin_over_valid": float(PROMOTION_MIN_MARGIN_OVER_VALID),
        }
        candidate_thresholds["labels"][label] = {
            "action": "diagnostic_only",
            "advisory_min_confidence": advisory,
            "enforce_min_confidence": enforce,
            "deployment_candidate_min_confidence": deployment_confidence,
            "deployment_min_margin_over_valid": float(PROMOTION_MIN_MARGIN_OVER_VALID),
            "deployment_valid_logit_bias": float(PROMOTION_VALID_LOGIT_BIAS),
        }

if "promotion_gate_report" in globals():
    for _threshold_doc in (thresholds, candidate_thresholds):
        _threshold_doc["promotion_status"] = promotion_gate_report.get("promotion_status")
        _threshold_doc["blocked_reasons"] = promotion_gate_report.get("blocked_reasons")
        _threshold_doc["artifact_promotable"] = bool(promotion_gate_report.get("artifact_promotable", False))

calibration_report = {
    "schema_version": "toolcall-verifier-calibration/v1",
    "temperature": TEMPERATURE,
    "calibration_ece_before": float(calibration_scored["uncalibrated_ece"].iloc[0]) if len(calibration_scored) else None,
    "calibration_ece_after": float(calibration_scored["calibrated_ece"].iloc[0]) if len(calibration_scored) else None,
    "validation_ece_before": float(valid_scored["uncalibrated_ece"].iloc[0]) if len(valid_scored) else None,
    "validation_ece_after": float(valid_scored["calibrated_ece"].iloc[0]) if len(valid_scored) else None,
    "deployment_rule": DEPLOYMENT_RULE,
    "recommended_action_policy": RECOMMENDED_ACTION_POLICY,
    "thresholds": thresholds,
    "candidate_thresholds": candidate_thresholds,
}
(DATA_DIR / "thresholds.json").write_text(json.dumps(thresholds, indent=2))
(DATA_DIR / "candidate_thresholds.json").write_text(json.dumps(candidate_thresholds, indent=2))
(DATA_DIR / "calibration_report.json").write_text(json.dumps(calibration_report, indent=2))
print(json.dumps(thresholds, indent=2))

## 10. Save final model and run quick inference

In [ ]:
#@title Save model and tokenizer
final_model_dir = MODEL_DIR / "final"
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

# Copy schema/metadata alongside the model for Hub upload.
model_sidecar_files = [
    "labels.json",
    "input_schema.json",
    "thresholds.json",
    "candidate_thresholds.json",
    "serializer_fixture.json",
    "training_metrics.json",
    "training_run_summary.json",
    "test_metrics.json",
    "calibration_report.json",
    "reliability_curves.jsonl",
    "valid_protection_slice_metrics.json",
    "promotion_gate_report.json",
    "dataset_audit_report.json",
    "source_quality_report.json",
    "train_rebalance_report.json",
    "private_mix_report.json",
    "split_balance_report.json",
    "tokenization_diagnostics.json",
    "classifier_diagnostics.json",
    "telemetry_diagnostics.json",
    "input_schema_v1.json",
    "input_schema_v2.json",
    "serializer_fixture_v1.json",
    "serializer_fixture_v2.json",
    "final_response_input_schema.json",
]
for src_name in model_sidecar_files:
    src = DATA_DIR / src_name
    if src.exists():
        shutil.copy(src, final_model_dir / src_name)

print("Saved to:", final_model_dir)
print(sorted(p.name for p in final_model_dir.iterdir()))


In [ ]:
#@title Quick inference helper
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model=str(final_model_dir),
    tokenizer=str(final_model_dir),
    device=0 if torch.cuda.is_available() else -1,
    top_k=None,
)

def score_candidate(
    user_request: str,
    tools: List[Dict[str, Any]],
    candidate: Any,
    required_steps=None,
    completed_steps=None,
    pending_steps=None,
    terminal_tools=None,
    recent_errors=None,
):
    text = serialize_state(
        user_request=user_request,
        tools=tools,
        candidate=candidate,
        required_steps=required_steps,
        completed_steps=completed_steps,
        pending_steps=pending_steps,
        terminal_tools=terminal_tools,
        recent_errors=recent_errors,
    )
    start = time.perf_counter()
    scores = clf(text, truncation=True, max_length=MAX_LENGTH)[0]
    elapsed_ms = (time.perf_counter() - start) * 1000
    scores = sorted(scores, key=lambda x: x["score"], reverse=True)
    return scores[:5], elapsed_ms, text

sample = FORGE_WORKFLOWS[0]
scores, elapsed_ms, text = score_candidate(
    user_request=sample["user_request"],
    tools=sample["tools"],
    candidate={"name": "report", "arguments": {"summary": "Done."}},
    required_steps=sample["required_steps"],
    completed_steps=[],
    pending_steps=sample["required_steps"],
    terminal_tools=sample["terminal_tools"],
)
print("Latency ms:", round(elapsed_ms, 2))
print(scores)
print(text[:2000])

## 11. ONNX export, quantization, manifest, and parity checks

The Rust side should load:

- `model.onnx` or `model_quantized.onnx`
- tokenizer files
- `labels.json`
- `thresholds.json`
- `artifact_manifest.json`
- `input_schema.json`
- `serializer_fixture.json`
- `input_schema_v1.json` / `input_schema_v2.json`
- `serializer_fixture_v1.json` / `serializer_fixture_v2.json`

In [ ]:
#@title Export to ONNX with Optimum
!python -m optimum.exporters.onnx   --model "{final_model_dir}"   --task text-classification   "{ONNX_DIR}"

# Normalize ONNX filename for Rust config.
onnx_files = sorted(ONNX_DIR.glob("*.onnx"))
if not onnx_files:
    raise RuntimeError("No ONNX file exported")
main_onnx = ONNX_DIR / "model.onnx"
if onnx_files[0].name != "model.onnx":
    shutil.copy(onnx_files[0], main_onnx)

# Copy deployment metadata.
onnx_sidecar_files = [
    "labels.json",
    "input_schema.json",
    "thresholds.json",
    "candidate_thresholds.json",
    "serializer_fixture.json",
    "input_schema_v1.json",
    "input_schema_v2.json",
    "serializer_fixture_v1.json",
    "serializer_fixture_v2.json",
    "training_metrics.json",
    "training_run_summary.json",
    "test_metrics.json",
    "valid_protection_slice_metrics.json",
    "promotion_gate_report.json",
    "dataset_audit_report.json",
    "source_quality_report.json",
    "train_rebalance_report.json",
    "private_mix_report.json",
    "split_balance_report.json",
    "tokenization_diagnostics.json",
    "classifier_diagnostics.json",
    "telemetry_diagnostics.json",
]
for src_name in onnx_sidecar_files:
    src = DATA_DIR / src_name
    if src.exists():
        shutil.copy(src, ONNX_DIR / src_name)

# Copy tokenizer/config files from final_model_dir if Optimum did not copy all of them.
for p in final_model_dir.iterdir():
    if p.name in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "spm.model", "config.json"]:
        shutil.copy(p, ONNX_DIR / p.name)

print("ONNX artifacts:")
print(sorted(p.name for p in ONNX_DIR.iterdir()))


In [ ]:
#@title Optional dynamic quantization
from onnxruntime.quantization import quantize_dynamic, QuantType

src = ONNX_DIR / "model.onnx"
dst = ONNX_DIR / "model_quantized.onnx"
if src.exists():
    quantize_dynamic(str(src), str(dst), weight_type=QuantType.QInt8)
    print("Quantized:", dst)
else:
    print("No model.onnx found.")

In [ ]:
#@title Build artifact manifest
quantized_path = ONNX_DIR / "model_quantized.onnx"
artifact_manifest = {
    "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
    "model_kind": "text-classification-cross-encoder",
    "base_model": BASE_MODEL,
    "label_mode": LABEL_MODE,
    "input_schema_version": INPUT_SCHEMA_VERSION,
    "serializer": SERIALIZER_VERSION,
    "max_length": MAX_LENGTH,
    "requested_gpu_profile": GPU_PROFILE,
    "run_profile": RUN_PROFILE,
    "profile_config": CFG,
    "labels_file": "labels.json",
    "thresholds_file": "thresholds.json",
    "candidate_thresholds_file": "candidate_thresholds.json",
    "input_schema_file": "input_schema.json",
    "serializer_fixture_file": "serializer_fixture.json",
    "training_run_summary_file": "training_run_summary.json",
    "test_metrics_file": "test_metrics.json",
    "valid_protection_slice_metrics_file": "valid_protection_slice_metrics.json",
    "promotion_gate_report_file": "promotion_gate_report.json",
    "dataset_audit_report_file": "dataset_audit_report.json",
    "source_quality_report_file": "source_quality_report.json",
    "train_rebalance_report_file": "train_rebalance_report.json",
    "private_mix_report_file": "private_mix_report.json",
    "split_balance_report_file": "split_balance_report.json",
    "tokenization_diagnostics_file": "tokenization_diagnostics.json",
    "classifier_diagnostics_file": "classifier_diagnostics.json",
    "telemetry_diagnostics_file": "telemetry_diagnostics.json",
    "onnx_parity_report_file": "onnx_parity_report.json",
    "onnx_file": "model.onnx",
    "quantized_onnx_file": "model_quantized.onnx" if quantized_path.exists() else None,
    "quantized_active_status": "pending_parity" if quantized_path.exists() else "not_present",
    "quantized_active_allowed": False,
    "tokenizer_required_files": [
        "tokenizer_config.json",
        "special_tokens_map.json",
        "spm.model",
    ],
    "labels": LABELS,
    "created_unix": int(time.time()),
    "precision_flags": {"fp16": USE_FP16, "bf16": USE_BF16, "tf32": USE_TF32},
    "best_model_checkpoint": getattr(trainer.state, "best_model_checkpoint", None),
    "best_metric": getattr(trainer.state, "best_metric", None),
    "best_metric_name": training_args.metric_for_best_model,
    "t4_proven_baseline": T4_PROVEN_BASELINE,
    "deployment_default": "shadow",
    "shadow_first_reason": "Experimental verifier artifacts require eval replay before advisory or enforcement promotion.",
    "supports_legacy_five_labels": False,
}
if "training_run_summary" in globals():
    artifact_manifest["training_run_summary"] = training_run_summary
if "dataset_audit_report" in globals():
    artifact_manifest["dataset_audit_summary"] = {
        "rows": int(dataset_audit_report.get("rows", 0)),
        "suspicious_valid_hard_negative_rows": int(dataset_audit_report.get("suspicious_valid_hard_negative_rows", 0)),
        "valid_fraction": float(dataset_audit_report.get("valid_fraction", 0.0)),
        "source_quality_report": dataset_audit_report.get("source_quality_report"),
    }
if "train_rebalance_report" in globals():
    artifact_manifest["train_rebalance_summary"] = {
        "enabled": bool(train_rebalance_report.get("enabled", False)),
        "valid_fraction_before": train_rebalance_report.get("valid_fraction_before"),
        "valid_fraction_after": train_rebalance_report.get("valid_fraction_after"),
        "train_rows_before": train_rebalance_report.get("train_rows_before"),
        "train_rows_after": train_rebalance_report.get("train_rows_after"),
    }
if "promotion_gate_report" in globals():
    artifact_manifest["promotion_status"] = promotion_gate_report.get("promotion_status")
    artifact_manifest["artifact_status"] = promotion_gate_report.get("artifact_status")
    artifact_manifest["raw_gate_status"] = promotion_gate_report.get("raw_gate_status")
    artifact_manifest["deployment_gate_status"] = promotion_gate_report.get("deployment_gate_status")
    artifact_manifest["enforcement_eligible"] = bool(promotion_gate_report.get("enforcement_eligible", False))
    artifact_manifest["enforcement_eligible_status"] = promotion_gate_report.get("enforcement_eligible_status")
    artifact_manifest["enforcement_replay_requirements"] = promotion_gate_report.get("enforcement_replay_requirements")
    artifact_manifest["recommended_action_policy"] = globals().get("RECOMMENDED_ACTION_POLICY")
    artifact_manifest["blocked_reasons"] = promotion_gate_report.get("blocked_reasons")
    artifact_manifest["artifact_promotable"] = bool(promotion_gate_report.get("artifact_promotable", False))
    artifact_manifest["high_confidence_mistakes_file"] = "high_confidence_mistakes.jsonl"
    artifact_manifest["validation_valid_false_objections_file"] = "validation_valid_false_objections.jsonl"
    artifact_manifest["test_valid_false_objections_file"] = "test_valid_false_objections.jsonl"
    if "assert_promotion_consistency" in globals():
        assert_promotion_consistency("artifact_manifest.artifact_promotable", artifact_manifest["artifact_promotable"])
        _embedded_summary = artifact_manifest.get("training_run_summary") or {}
        assert_promotion_consistency(
            "artifact_manifest.training_run_summary.artifact_promotable",
            _embedded_summary.get("artifact_promotable"),
        )
(ONNX_DIR / "artifact_manifest.json").write_text(json.dumps(artifact_manifest, indent=2))
(final_model_dir / "artifact_manifest.json").write_text(json.dumps(artifact_manifest, indent=2))
print(json.dumps(artifact_manifest, indent=2))

In [ ]:
#@title PyTorch vs ONNX parity and quantization drift checks
import onnxruntime as ort

ONNX_PARITY_BATCH_SIZE = max(1, min(8, EVAL_BATCH_SIZE))
ONNX_PARITY_ROWS_PER_SPLIT = 512  #@param {type:"integer"}
# Quantized parity failure marks the int8 artifact failed_shadow_only instead of killing
# the run: fp32 ONNX (gated separately, and fatally) remains the deployable artifact.
QUANTIZED_PARITY_HARD_FAIL = False  #@param {type:"boolean"}
PT_FP32_TOP_LABEL_AGREEMENT_GATE = 0.995
FP32_QUANTIZED_TOP_LABEL_AGREEMENT_GATE = 0.98

fixture = json.loads((DATA_DIR / "serializer_fixture.json").read_text())
probe_text = fixture["serialized"]


def parity_records_from_scored(scored: pd.DataFrame, split_name: str) -> List[Dict[str, Any]]:
    if scored.empty:
        return []
    n = len(scored) if ONNX_PARITY_ROWS_PER_SPLIT <= 0 else min(int(ONNX_PARITY_ROWS_PER_SPLIT), len(scored))
    sampled = scored.sample(n=n, random_state=SEED).reset_index(drop=True)
    return [
        {
            "split": split_name,
            "true_label": row.get("true_label"),
            "input_text": row.get("input_text"),
        }
        for row in sampled.to_dict(orient="records")
    ]


parity_records = [{"split": "serializer_fixture", "true_label": None, "input_text": probe_text}]
parity_records.extend(parity_records_from_scored(valid_scored, "validation"))
parity_records.extend(parity_records_from_scored(test_scored, "test"))

seen_texts = set()
deduped_records = []
for record in parity_records:
    text = str(record["input_text"])
    if text in seen_texts:
        continue
    seen_texts.add(text)
    deduped_records.append(record)
parity_records = deduped_records
parity_texts = [record["input_text"] for record in parity_records]

# The parity pass uses fresh CPU model copies. Release training-time GPU objects first.
cleanup_runtime_objects(
    ["clf", "trainer", "model", "train_result"],
    "before ONNX parity checks",
)


def make_ort_session(path: Path):
    options = ort.SessionOptions()
    options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    options.intra_op_num_threads = max(1, min(4, os.cpu_count() or 1))
    options.inter_op_num_threads = 1
    options.enable_mem_pattern = False
    options.enable_cpu_mem_arena = False
    return ort.InferenceSession(str(path), sess_options=options, providers=["CPUExecutionProvider"])


ort_session = make_ort_session(ONNX_DIR / "model.onnx")
pt_model = AutoModelForSequenceClassification.from_pretrained(str(final_model_dir)).to("cpu").eval()
pt_tokenizer = AutoTokenizer.from_pretrained(str(final_model_dir), use_fast=False)


def logits_from_pt(texts: List[str]) -> np.ndarray:
    inputs = pt_tokenizer(texts, return_tensors="pt", truncation=True, max_length=MAX_LENGTH, padding=True)
    with torch.inference_mode():
        return pt_model(**inputs).logits.detach().cpu().numpy()


def logits_from_onnx(session, texts: List[str]) -> np.ndarray:
    inputs = pt_tokenizer(texts, return_tensors="pt", truncation=True, max_length=MAX_LENGTH, padding=True)
    input_names = {i.name for i in session.get_inputs()}
    ort_inputs = {
        key: np.ascontiguousarray(value.cpu().numpy())
        for key, value in inputs.items()
        if key in input_names
    }
    return session.run(None, ort_inputs)[0]


def logits_in_batches(logits_fn, texts: List[str], batch_size: int, desc: str) -> np.ndarray:
    chunks = []
    for start in tqdm(range(0, len(texts), batch_size), desc=desc):
        chunks.append(logits_fn(texts[start:start + batch_size]))
        gc.collect()
    return np.concatenate(chunks, axis=0)


def top_label_agreement_by_split(left_top: np.ndarray, right_top: np.ndarray) -> Dict[str, float]:
    by_split = {}
    splits = [record["split"] for record in parity_records]
    for split in sorted(set(splits)):
        idx = np.array([name == split for name in splits], dtype=bool)
        by_split[split] = float((left_top[idx] == right_top[idx]).mean()) if idx.any() else None
    return by_split


def disagreement_examples(left_top: np.ndarray, right_top: np.ndarray, left_name: str, right_name: str, limit: int = 20) -> List[Dict[str, Any]]:
    examples = []
    for idx, (left, right) in enumerate(zip(left_top, right_top)):
        if int(left) == int(right):
            continue
        record = parity_records[idx]
        examples.append({
            "split": record["split"],
            "true_label": record.get("true_label"),
            left_name: id2label[int(left)],
            right_name: id2label[int(right)],
            "input_text_prefix": str(record["input_text"])[:500],
        })
        if len(examples) >= limit:
            break
    return examples


def write_onnx_parity_report(report: Dict[str, Any]) -> None:
    (ONNX_DIR / "onnx_parity_report.json").write_text(json.dumps(report, indent=2))
    (final_model_dir / "onnx_parity_report.json").write_text(json.dumps(report, indent=2))
    manifest_summary_keys = [
        "rows",
        "rows_by_split",
        "pt_fp32_top_label_agreement",
        "pt_fp32_gate_passed",
        "quantized_present",
        "fp32_quantized_top_label_agreement",
        "fp32_quantized_gate_passed",
        "quantized_active_status",
        "quantized_active_allowed",
        "deployment_recommendation",
    ]
    manifest_summary = {key: report.get(key) for key in manifest_summary_keys if key in report}
    for manifest_path in [ONNX_DIR / "artifact_manifest.json", final_model_dir / "artifact_manifest.json"]:
        if not manifest_path.exists():
            continue
        manifest = json.loads(manifest_path.read_text())
        manifest["onnx_parity_report_file"] = "onnx_parity_report.json"
        manifest["onnx_parity_summary"] = manifest_summary
        manifest["quantized_active_status"] = report.get("quantized_active_status")
        manifest["quantized_active_allowed"] = bool(report.get("quantized_active_allowed", False))
        manifest_path.write_text(json.dumps(manifest, indent=2))


pt_logits = logits_from_pt([probe_text])
ort_logits = logits_from_onnx(ort_session, [probe_text])
pt_top = int(pt_logits.argmax(axis=-1)[0])
ort_top = int(ort_logits.argmax(axis=-1)[0])
max_abs_diff = float(np.max(np.abs(pt_logits - ort_logits)))
print("pt_top:", pt_top, id2label[pt_top])
print("ort_top:", ort_top, id2label[ort_top])
print("max_abs_diff:", max_abs_diff)
assert pt_top == ort_top, "ONNX top label does not match PyTorch top label on serializer fixture"

print("ONNX parity rows:", len(parity_texts), "batch_size:", ONNX_PARITY_BATCH_SIZE)
pt_batch = logits_in_batches(logits_from_pt, parity_texts, ONNX_PARITY_BATCH_SIZE, "PyTorch parity")
fp32_batch = logits_in_batches(lambda texts: logits_from_onnx(ort_session, texts), parity_texts, ONNX_PARITY_BATCH_SIZE, "ONNX fp32 parity")
pt_top_batch = pt_batch.argmax(axis=-1)
fp32_top_batch = fp32_batch.argmax(axis=-1)
pt_fp32_agreement = float((pt_top_batch == fp32_top_batch).mean())
pt_fp32_gate_passed = pt_fp32_agreement >= PT_FP32_TOP_LABEL_AGREEMENT_GATE

onnx_parity_report = {
    "schema_version": "toolcall-verifier-onnx-parity/v1",
    "rows": len(parity_texts),
    "rows_by_split": {split: int(sum(1 for record in parity_records if record["split"] == split)) for split in sorted({record["split"] for record in parity_records})},
    "pt_fp32_top_label_agreement": pt_fp32_agreement,
    "pt_fp32_top_label_agreement_by_split": top_label_agreement_by_split(pt_top_batch, fp32_top_batch),
    "pt_fp32_max_abs_diff": float(np.max(np.abs(pt_batch - fp32_batch))),
    "pt_fp32_gate": {"min_top_label_agreement": PT_FP32_TOP_LABEL_AGREEMENT_GATE},
    "pt_fp32_gate_passed": bool(pt_fp32_gate_passed),
    "pt_fp32_disagreements": int((pt_top_batch != fp32_top_batch).sum()),
    "pt_fp32_disagreement_examples": disagreement_examples(pt_top_batch, fp32_top_batch, "pytorch", "fp32_onnx"),
    "quantized_present": False,
    "quantized_active_status": "not_present",
    "quantized_active_allowed": False,
    "deployment_recommendation": "fp32_onnx_only_until_quantized_parity_passes",
}

q_path = ONNX_DIR / "model_quantized.onnx"
if q_path.exists():
    q_sess = make_ort_session(q_path)
    q_logits = logits_in_batches(lambda texts: logits_from_onnx(q_sess, texts), parity_texts, ONNX_PARITY_BATCH_SIZE, "ONNX quantized parity")
    q_top = q_logits.argmax(axis=-1)
    fp32_q_disagreements = int((fp32_top_batch != q_top).sum())
    fp32_quantized_agreement = float((fp32_top_batch == q_top).mean())
    fp32_quantized_gate_passed = fp32_quantized_agreement >= FP32_QUANTIZED_TOP_LABEL_AGREEMENT_GATE
    onnx_parity_report.update({
        "quantized_present": True,
        "fp32_quantized_top_label_agreement": fp32_quantized_agreement,
        "fp32_quantized_top_label_agreement_by_split": top_label_agreement_by_split(fp32_top_batch, q_top),
        "fp32_quantized_gate": {"min_top_label_agreement": FP32_QUANTIZED_TOP_LABEL_AGREEMENT_GATE},
        "fp32_quantized_gate_passed": bool(fp32_quantized_gate_passed),
        "fp32_quantized_disagreements": fp32_q_disagreements,
        "fp32_quantized_max_abs_diff": float(np.max(np.abs(fp32_batch - q_logits))),
        "fp32_quantized_disagreement_examples": disagreement_examples(fp32_top_batch, q_top, "fp32_onnx", "quantized_onnx"),
        "quantized_active_status": "passed_active_candidate" if fp32_quantized_gate_passed else "failed_shadow_only",
        "quantized_active_allowed": bool(fp32_quantized_gate_passed),
        "deployment_recommendation": "quantized_shadow_only_until_release_replay" if fp32_quantized_gate_passed else "use_fp32_onnx_for_replay_quantized_shadow_only",
    })
    print("quantized disagreements:", fp32_q_disagreements, "/", len(parity_texts))

write_onnx_parity_report(onnx_parity_report)
print(json.dumps(onnx_parity_report, indent=2)[:6000])

if not pt_fp32_gate_passed:
    raise RuntimeError(
        "PyTorch/FP32 ONNX top-label agreement "
        f"{pt_fp32_agreement:.4f} is below gate {PT_FP32_TOP_LABEL_AGREEMENT_GATE:.4f}"
    )
if onnx_parity_report.get("quantized_present") and not onnx_parity_report.get("fp32_quantized_gate_passed"):
    _quant_parity_msg = (
        "FP32/quantized ONNX top-label agreement "
        f"{onnx_parity_report['fp32_quantized_top_label_agreement']:.4f} is below gate "
        f"{FP32_QUANTIZED_TOP_LABEL_AGREEMENT_GATE:.4f}; use FP32 ONNX for replay and keep quantized shadow-only."
    )
    if QUANTIZED_PARITY_HARD_FAIL:
        raise RuntimeError(_quant_parity_msg)
    print("WARNING:", _quant_parity_msg)
    print(
        "Continuing: the quantized artifact is marked failed_shadow_only in "
        "onnx_parity_report.json and must not be deployed; fp32 ONNX passed parity."
    )

## 11b. Final-response verifier pipeline

This is enabled by default for production Colab runs. It trains and exports a separate terminal-answer verifier after the tool-call ONNX parity checks, with explicit cleanup between the two model lifecycles.


In [ ]:
#@title Build final-response verifier dataset and artifacts
FINAL_RESPONSE_MODEL_DIR = None
FINAL_RESPONSE_ONNX_DIR = None
FINAL_RESPONSE_ARTIFACT_DIR = None
FINAL_RESPONSE_DATASET_PATH = None
FINAL_RESPONSE_MEMORY_PROFILE = None
FINAL_RESPONSE_TEST_METRICS = None

if ENABLE_FINAL_RESPONSE_VERIFIER:
    TOOLCALL_RELEASE_NAMES = [
        "trainer", "model", "pt_model", "ort_session", "q_sess", "clf",
        "tokenizer", "pt_tokenizer", "tokenized", "ds", "data_collator", "class_weights",
        "sample_batch", "collated", "split_logits", "split_label_ids",
        "calibration_scored", "valid_scored", "test_scored",
        "pt_logits", "ort_logits", "pt_batch", "fp32_batch", "q_logits",
        "df", "balanced", "train_df", "valid_full_df", "valid_df", "test_df", "calibration_df",
        "lengths", "sample_texts", "public_rows", "all_rows", "normalized_examples",
    ]
    STALE_FINAL_RESPONSE_RELEASE_NAMES = [
        "fr_model", "fr_trainer", "fr_tokenizer", "fr_tokenized", "fr_ds", "fr_data_collator",
        "fr_class_weights", "fr_training_args", "fr_train_result", "fr_test_metrics",
        "fr_pt_model", "fr_pt_tokenizer", "fr_ort_session", "fr_q_session",
        "fr_pt_logits", "fr_ort_logits", "fr_q_logits",
    ]
    cuda_memory_snapshot("before final-response cleanup")
    cleanup_runtime_objects(TOOLCALL_RELEASE_NAMES, "after releasing tool-call training objects")
    cleanup_runtime_objects(STALE_FINAL_RESPONSE_RELEASE_NAMES, "after releasing stale final-response objects")

    FR_MAX_LENGTH = FINAL_RESPONSE_MAX_LENGTH_OVERRIDE or min(MAX_LENGTH, 768)
    FR_NUM_EPOCHS = FINAL_RESPONSE_NUM_EPOCHS_OVERRIDE or NUM_EPOCHS
    FR_TRAIN_BATCH_SIZE = FINAL_RESPONSE_TRAIN_BATCH_SIZE_OVERRIDE or min(TRAIN_BATCH_SIZE, 16)
    FR_EVAL_BATCH_SIZE = FINAL_RESPONSE_EVAL_BATCH_SIZE_OVERRIDE or min(EVAL_BATCH_SIZE, 32)
    FR_GRAD_ACCUM = FINAL_RESPONSE_GRAD_ACCUM_OVERRIDE or max(
        1, math.ceil((TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) / FR_TRAIN_BATCH_SIZE)
    )
    FR_MAX_PER_LABEL = min(FINAL_RESPONSE_MAX_PER_LABEL, 500) if RUN_PROFILE == "debug_smoke" else FINAL_RESPONSE_MAX_PER_LABEL
    FR_BALANCE_LABELS = bool(globals().get("FINAL_RESPONSE_BALANCE_LABELS", True))
    FR_MIN_PER_LABEL = int(globals().get("FINAL_RESPONSE_MIN_PER_LABEL", 0))
    if RUN_PROFILE == "debug_smoke":
        FR_MIN_PER_LABEL = min(FR_MIN_PER_LABEL, 24)
    FINAL_RESPONSE_MEMORY_PROFILE = {
        "max_length": FR_MAX_LENGTH,
        "epochs": FR_NUM_EPOCHS,
        "train_batch_size": FR_TRAIN_BATCH_SIZE,
        "eval_batch_size": FR_EVAL_BATCH_SIZE,
        "grad_accum": FR_GRAD_ACCUM,
        "max_per_label": FR_MAX_PER_LABEL,
        "balance_labels": FR_BALANCE_LABELS,
        "min_per_label": FR_MIN_PER_LABEL,
        "force_retrain": FINAL_RESPONSE_FORCE_RETRAIN,
        "export_cpu_only": FINAL_RESPONSE_EXPORT_CPU_ONLY,
    }
    print("Final-response memory profile:", json.dumps(FINAL_RESPONSE_MEMORY_PROFILE, indent=2))

    def build_final_response_synthetic_rows(repeats: int = 6) -> List[FinalResponseVerifierRow]:
        scenarios = [
            {
                "scenario_family": "grounded_synthesis",
                "user_request": GROUNDED_SYNTHESIS_USER,
                "workflow_state": {"required_steps": ["get_open_role", "get_candidate_pool", "get_compatibility_check"], "completed_steps": ["get_open_role", "get_candidate_pool", "get_compatibility_check"], "pending_steps": [], "terminal_tools": ["submit_hiring_decision"], "recent_errors": []},
                "required_facts": GROUNDED_SYNTHESIS_FACTS,
                "tool_trace": ["get_open_role", "get_candidate_pool", "get_compatibility_check"],
                "tool_results": [{"tool_name": "get_compatibility_check", "content": "Sarah is blocked by a non-compete. James requires visa sponsorship. Diana is mismatched for the IC role track. Aisha can start immediately and has relevant payments experience."}],
                "responses": [
                    ("valid_final_response", GROUNDED_SYNTHESIS_POSITIVE, 1.0),
                    ("missing_tool_fact", "Selected Aisha because she is available and has payments experience.", 0.10),
                    ("contradicts_tool_result", "Select Sarah Chen because she can start immediately and has no restrictions.", 0.05),
                    ("unsupported_claim", "Aisha Nakamura has already accepted the offer and passed background checks.", 0.05),
                    ("failed_to_acknowledge_data_gap", "The missing compensation history proves Aisha is the lowest-risk hire.", 0.05),
                ],
            },
            {
                "scenario_family": "sequential_3step",
                "user_request": "Generate a sales report from the Q4 2024 dataset.",
                "workflow_state": {"required_steps": ["fetch_sales_data", "analyze_sales"], "completed_steps": ["fetch_sales_data", "analyze_sales"], "pending_steps": [], "terminal_tools": ["report"], "recent_errors": []},
                "required_facts": ["23% YoY growth", "Widget Pro", "APAC"],
                "tool_trace": ["fetch_sales_data", "analyze_sales", "report"],
                "tool_results": [{"tool_name": "analyze_sales", "content": "Revenue grew 23% YoY. Top product: Widget Pro. Weakest region: APAC."}],
                "responses": [
                    ("valid_final_response", "Revenue grew 23% YoY. Top product was Widget Pro, and APAC was the weakest region.", 1.0),
                    ("missing_tool_fact", "Revenue improved and Widget Pro performed well.", 0.10),
                    ("contradicts_tool_result", "Revenue declined 23% YoY and EMEA was the weakest region.", 0.05),
                    ("unsupported_claim", "Revenue grew 23% because a new pricing campaign launched in December.", 0.05),
                    ("failed_to_acknowledge_data_gap", "The unavailable regional margin data confirms APAC should be deprioritized.", 0.05),
                ],
            },
            {
                "scenario_family": "data_gap_recovery",
                "user_request": "Summarize the vendor audit and call out any missing required data.",
                "workflow_state": {"required_steps": ["list_transactions", "get_approved_vendors"], "completed_steps": ["list_transactions", "get_approved_vendors"], "pending_steps": [], "terminal_tools": ["submit_audit_report"], "recent_errors": ["vendor risk score was unavailable"]},
                "required_facts": ["TX-1001", "vendor risk score unavailable", "approved vendor list checked"],
                "tool_trace": ["list_transactions", "get_approved_vendors", "submit_audit_report"],
                "tool_results": [{"tool_name": "list_transactions", "content": "TX-1001 exceeded the threshold. Vendor risk score was unavailable."}],
                "responses": [
                    ("valid_final_response", "TX-1001 exceeded the threshold. The approved vendor list was checked, and the vendor risk score was unavailable.", 1.0),
                    ("missing_tool_fact", "TX-1001 exceeded the threshold and the vendor list was checked.", 0.10),
                    ("contradicts_tool_result", "No transactions exceeded the threshold.", 0.05),
                    ("unsupported_claim", "TX-1001 exceeded the threshold because the vendor was sanctioned.", 0.05),
                    ("failed_to_acknowledge_data_gap", "TX-1001 exceeded the threshold and the vendor risk score was high.", 0.05),
                ],
            },
        ]
        rows: List[FinalResponseVerifierRow] = []
        for repeat in range(repeats):
            for scenario in scenarios:
                for label, response, rank in scenario["responses"]:
                    group_id = stable_id("forge_final_synthetic", scenario["scenario_family"], repeat)
                    rows.append(make_final_response_row(
                        "forge_final_synthetic",
                        label,
                        scenario["user_request"],
                        response,
                        rank,
                        workflow_state=scenario["workflow_state"],
                        required_facts=scenario["required_facts"],
                        tool_trace=scenario["tool_trace"],
                        tool_results=scenario["tool_results"],
                        metadata={"generator": "forge_final_synthetic", "scenario_family": scenario["scenario_family"], "example_group_id": group_id},
                        group_id=group_id,
                    ))
        return rows

    def final_response_checkpoint_exists(path: Path) -> bool:
        weight_files = ["model.safetensors", "pytorch_model.bin"]
        return (path / "config.json").exists() and any((path / name).exists() for name in weight_files)

    def read_json_if_exists(path: Path, default: Any) -> Any:
        if not path.exists():
            return default
        try:
            return json.loads(path.read_text())
        except Exception as exc:
            print(f"WARNING: could not read {path}: {exc}")
            return default

    FINAL_RESPONSE_DEPLOYMENT_FILE_NAMES = [
        "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json",
        "spm.model", "config.json", "labels.json", "input_schema.json", "thresholds.json",
        "artifact_manifest.json", "training_provenance.json", "onnx_parity_report.json",
    ]

    def copy_existing_final_response_deployment_files(src_dir: Path, dst_dir: Path) -> None:
        dst_dir.mkdir(parents=True, exist_ok=True)
        for name in FINAL_RESPONSE_DEPLOYMENT_FILE_NAMES:
            src = src_dir / name
            if src.exists():
                shutil.copy(src, dst_dir / name)

    def ensure_final_response_tokenizer_json(model_dir: Path, onnx_dir: Path) -> None:
        tokenizer_json = onnx_dir / "tokenizer.json"
        if tokenizer_json.exists():
            return
        try:
            fast_tokenizer = AutoTokenizer.from_pretrained(str(model_dir), use_fast=True)
            fast_tokenizer.save_pretrained(str(onnx_dir))
        except Exception as exc:
            raise RuntimeError(
                "Final-response ONNX deployment is missing tokenizer.json, and fast tokenizer export failed. "
                "Rust ONNX scoring requires tokenizer.json."
            ) from exc
        if not tokenizer_json.exists():
            raise RuntimeError(
                "Final-response ONNX deployment is missing tokenizer.json after fast tokenizer export."
            )

    def write_final_response_json_artifacts(target: Path) -> None:
        target.mkdir(parents=True, exist_ok=True)
        (target / "labels.json").write_text(json.dumps(final_labels_doc, indent=2))
        (target / "input_schema.json").write_text(json.dumps(FINAL_RESPONSE_INPUT_SCHEMA, indent=2))
        (target / "thresholds.json").write_text(json.dumps(final_thresholds, indent=2))
        (target / "artifact_manifest.json").write_text(json.dumps(final_manifest, indent=2))
        (target / "training_provenance.json").write_text(json.dumps(final_training_provenance, indent=2))

    def export_final_response_onnx(model_dir: Path, onnx_dir: Path) -> Path:
        onnx_dir.mkdir(parents=True, exist_ok=True)
        export_tmp_dir = WORKDIR / "final_response_onnx_export_tmp"
        if export_tmp_dir.exists():
            shutil.rmtree(export_tmp_dir)
        export_tmp_dir.mkdir(parents=True, exist_ok=True)

        export_env = os.environ.copy()
        if FINAL_RESPONSE_EXPORT_CPU_ONLY:
            export_env["CUDA_VISIBLE_DEVICES"] = ""
            export_env["ORT_DISABLE_GPU"] = "1"
            print("Final-response ONNX export will run with CUDA hidden from the child process.")

        subprocess.run([
            "python", "-m", "optimum.exporters.onnx",
            "--model", str(model_dir),
            "--task", "text-classification",
            str(export_tmp_dir),
        ], check=True, env=export_env)

        exported = sorted(export_tmp_dir.glob("*.onnx"))
        if not exported:
            raise RuntimeError("No final-response ONNX file exported")
        main_onnx = onnx_dir / "model.onnx"
        source_onnx = export_tmp_dir / "model.onnx" if (export_tmp_dir / "model.onnx").exists() else exported[0]
        shutil.copy(source_onnx, main_onnx)
        copy_existing_final_response_deployment_files(export_tmp_dir, onnx_dir)
        shutil.rmtree(export_tmp_dir)
        return main_onnx

    def copy_final_response_deployment_files(model_dir: Path, onnx_dir: Path) -> None:
        copy_existing_final_response_deployment_files(model_dir, onnx_dir)
        ensure_final_response_tokenizer_json(model_dir, onnx_dir)

    def logits_from_final_response_pt(model_obj, tokenizer_obj, texts: List[str]) -> np.ndarray:
        inputs = tokenizer_obj(texts, return_tensors="pt", truncation=True, max_length=FR_MAX_LENGTH, padding=True)
        with torch.no_grad():
            return model_obj(**inputs).logits.detach().cpu().numpy()

    def logits_from_final_response_onnx(session_obj, tokenizer_obj, texts: List[str]) -> np.ndarray:
        inputs = tokenizer_obj(texts, return_tensors="pt", truncation=True, max_length=FR_MAX_LENGTH, padding=True)
        input_names = {i.name for i in session_obj.get_inputs()}
        ort_inputs = {key: value.cpu().numpy() for key, value in inputs.items() if key in input_names}
        return session_obj.run(None, ort_inputs)[0]

    def run_final_response_onnx_parity(model_dir: Path, onnx_dir: Path, texts: List[str]) -> Dict[str, Any]:
        import onnxruntime as ort

        if not texts:
            raise RuntimeError("No final-response parity texts available")
        parity_texts = texts[: min(64, len(texts))]
        fr_pt_model = AutoModelForSequenceClassification.from_pretrained(str(model_dir)).to("cpu").eval()
        fr_pt_tokenizer = AutoTokenizer.from_pretrained(str(model_dir), use_fast=False)
        fr_ort_session = ort.InferenceSession(str(onnx_dir / "model.onnx"), providers=["CPUExecutionProvider"])

        fr_pt_logits = logits_from_final_response_pt(fr_pt_model, fr_pt_tokenizer, parity_texts)
        fr_ort_logits = logits_from_final_response_onnx(fr_ort_session, fr_pt_tokenizer, parity_texts)
        pt_top = fr_pt_logits.argmax(axis=-1)
        ort_top = fr_ort_logits.argmax(axis=-1)
        report = {
            "schema_version": "final-response-verifier-onnx-parity/v1",
            "rows": len(parity_texts),
            "pt_fp32_top_label_agreement": float((pt_top == ort_top).mean()),
            "pt_fp32_max_abs_diff": float(np.max(np.abs(fr_pt_logits - fr_ort_logits))),
            "quantized_present": False,
        }

        q_path = onnx_dir / "model_quantized.onnx"
        if q_path.exists():
            fr_q_session = ort.InferenceSession(str(q_path), providers=["CPUExecutionProvider"])
            fr_q_logits = logits_from_final_response_onnx(fr_q_session, fr_pt_tokenizer, parity_texts)
            q_top = fr_q_logits.argmax(axis=-1)
            report.update({
                "quantized_present": True,
                "fp32_quantized_top_label_agreement": float((ort_top == q_top).mean()),
                "fp32_quantized_disagreements": int((ort_top != q_top).sum()),
                "fp32_quantized_max_abs_diff": float(np.max(np.abs(fr_ort_logits - fr_q_logits))),
            })

        (onnx_dir / "onnx_parity_report.json").write_text(json.dumps(report, indent=2))
        (model_dir / "onnx_parity_report.json").write_text(json.dumps(report, indent=2))
        print("Final-response ONNX parity:")
        print(json.dumps(report, indent=2))
        return report

    synthetic_final_response_rows = build_final_response_synthetic_rows()
    base_final_response_rows = list(globals().get("final_response_rows") or [])
    combined_final_response_rows = base_final_response_rows + synthetic_final_response_rows
    if not combined_final_response_rows:
        raise RuntimeError("No final-response training rows were created.")

    final_df = pd.DataFrame([asdict(r) for r in combined_final_response_rows])
    final_df["example_group_id"] = final_df["metadata"].map(lambda m: m.get("example_group_id") if isinstance(m, dict) else None)
    if "input" in final_df.columns:
        final_df["input_text"] = final_df["input"].map(serialize_final_response_state_v1)
    elif "input_text" not in final_df.columns:
        raise ValueError("Final-response rows must contain input_text or input")
    final_df["labels"] = final_df["label"].map(final_response_label2id)
    if final_df["labels"].isna().any():
        bad = sorted(final_df.loc[final_df["labels"].isna(), "label"].unique())
        raise ValueError(f"Unknown final-response labels: {bad}")
    final_df["labels"] = final_df["labels"].astype(int)

    final_pieces = []
    final_raw_counts = final_df["label"].value_counts(dropna=False)
    final_floor = min(FR_MIN_PER_LABEL, FR_MAX_PER_LABEL) if FR_BALANCE_LABELS and FR_MIN_PER_LABEL > 0 else 0
    for label in FINAL_RESPONSE_LABELS:
        group = final_df[final_df["label"] == label]
        if group.empty:
            continue
        sampled = group.sample(n=min(len(group), FR_MAX_PER_LABEL), random_state=SEED)
        if final_floor and len(sampled) < final_floor:
            extra = group.sample(
                n=final_floor - len(sampled),
                replace=True,
                random_state=SEED + final_response_label2id[label] + 101,
            )
            sampled = pd.concat([sampled, extra], ignore_index=True)
        final_pieces.append(sampled)
    final_balanced = pd.concat(final_pieces, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    print("Final-response raw labels:")
    print(final_raw_counts)
    print("Final-response capped/balanced labels:")
    print(final_balanced["label"].value_counts(dropna=False))

    fr_train_df, fr_valid_df, fr_test_df = group_split(final_balanced)
    for name, split_df in [("final_response_train", fr_train_df), ("final_response_validation", fr_valid_df), ("final_response_test", fr_test_df)]:
        path = DATA_DIR / f"{name}.jsonl"
        with path.open("w") as f:
            for row in split_df.to_dict(orient="records"):
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        print("Final-response split", name, len(split_df), "groups", split_df["example_group_id"].nunique(), path)

    final_dataset_path = DATA_DIR / "final_response_verifier_dataset.jsonl"
    with final_dataset_path.open("w") as f:
        for row in final_balanced.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    FINAL_RESPONSE_DATASET_PATH = final_dataset_path

    fr_parity_texts = fr_test_df["input_text"].sample(n=min(64, len(fr_test_df)), random_state=SEED).astype(str).tolist()
    if not fr_parity_texts and len(final_balanced):
        fr_parity_texts = [str(final_balanced["input_text"].iloc[0])]

    fr_model_dir = MODEL_DIR / "final_response"
    fr_final_model_dir = fr_model_dir / "final"
    existing_checkpoint = final_response_checkpoint_exists(fr_final_model_dir)
    retrain_final_response = FINAL_RESPONSE_FORCE_RETRAIN or not existing_checkpoint
    if existing_checkpoint and not FINAL_RESPONSE_FORCE_RETRAIN:
        print("Reusing saved final-response model checkpoint:", fr_final_model_dir)
    else:
        print("Training final-response model. Existing checkpoint:", existing_checkpoint, "force retrain:", FINAL_RESPONSE_FORCE_RETRAIN)

    fr_train_metrics: Dict[str, Any] = {}
    fr_test_metrics: Dict[str, Any] = {}
    if retrain_final_response:
        fr_ds = DatasetDict({
            "train": Dataset.from_pandas(fr_train_df.reset_index(drop=True), preserve_index=False),
            "validation": Dataset.from_pandas(fr_valid_df.reset_index(drop=True), preserve_index=False),
            "test": Dataset.from_pandas(fr_test_df.reset_index(drop=True), preserve_index=False),
        })
        fr_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

        def tokenize_final_response_batch(batch):
            encoded = fr_tokenizer(
                batch["input_text"],
                truncation=True,
                max_length=FR_MAX_LENGTH,
                padding=False,
            )
            encoded["labels"] = [int(x) for x in batch["labels"]]
            return encoded

        fr_tokenized = fr_ds.map(
            tokenize_final_response_batch,
            batched=True,
            remove_columns=fr_ds["train"].column_names,
            desc="Tokenizing final-response verifier rows",
        )
        fr_tokenized = fr_tokenized.cast_column("labels", Value("int64"))
        fr_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
        fr_data_collator = DataCollatorWithPadding(tokenizer=fr_tokenizer)

        cuda_memory_snapshot("before final-response model load")
        fr_model = AutoModelForSequenceClassification.from_pretrained(
            BASE_MODEL,
            num_labels=len(FINAL_RESPONSE_LABELS),
            id2label={int(k): v for k, v in final_response_id2label.items()},
            label2id=final_response_label2id,
        )
        if fr_tokenizer.pad_token_id is not None:
            fr_model.config.pad_token_id = fr_tokenizer.pad_token_id
        if fr_tokenizer.eos_token_id is not None:
            fr_model.config.eos_token_id = fr_tokenizer.eos_token_id
        if fr_tokenizer.bos_token_id is not None:
            fr_model.config.bos_token_id = fr_tokenizer.bos_token_id

        FR_ALL_LABEL_IDS = list(range(len(FINAL_RESPONSE_LABELS)))

        def compute_final_response_metrics(eval_pred):
            logits, labels = eval_pred
            preds = np.argmax(logits, axis=-1)
            present_precision, present_recall, present_f1, _ = precision_recall_fscore_support(
                labels, preds, average="macro", zero_division=0
            )
            full_precision, full_recall, full_f1, _ = precision_recall_fscore_support(
                labels, preds, labels=FR_ALL_LABEL_IDS, average="macro", zero_division=0
            )
            return {
                "accuracy": accuracy_score(labels, preds),
                "macro_precision": present_precision,
                "macro_recall": present_recall,
                "macro_f1": present_f1,
                "macro_precision_all_labels": full_precision,
                "macro_recall_all_labels": full_recall,
                "macro_f1_all_labels": full_f1,
            }

        def build_final_response_class_weights(train_frame: pd.DataFrame) -> Optional[torch.Tensor]:
            if not USE_CLASS_WEIGHTS:
                return None
            counts = train_frame["label"].value_counts()
            weights = []
            for label in FINAL_RESPONSE_LABELS:
                count = counts.get(label, 0)
                if count <= 0:
                    weights.append(0.0)
                else:
                    raw = len(train_frame) / (len(FINAL_RESPONSE_LABELS) * count)
                    weights.append(min(float(raw), MAX_CLASS_WEIGHT))
            weights = np.array(weights, dtype=np.float32)
            nonzero = weights[weights > 0]
            if len(nonzero):
                weights[weights > 0] = weights[weights > 0] / nonzero.mean()
            return torch.tensor(weights, dtype=torch.float)

        fr_training_args = TrainingArguments(
            output_dir=str(fr_model_dir),
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=FR_TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=FR_EVAL_BATCH_SIZE,
            gradient_accumulation_steps=FR_GRAD_ACCUM,
            num_train_epochs=FR_NUM_EPOCHS,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            fp16=USE_FP16,
            bf16=USE_BF16,
            tf32=USE_TF32,
            optim=OPTIMIZER,
            gradient_checkpointing=GRADIENT_CHECKPOINTING,
            auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
            dataloader_num_workers=DATALOADER_NUM_WORKERS if USE_CUDA else 0,
            dataloader_pin_memory=USE_CUDA,
            group_by_length=GROUP_BY_LENGTH,
            save_total_limit=2,
            report_to="none",
            seed=SEED,
        )

        fr_class_weights = build_final_response_class_weights(fr_train_df)
        fr_trainer = WeightedTrainer(
            class_weights=fr_class_weights,
            model=fr_model,
            args=fr_training_args,
            train_dataset=fr_tokenized["train"],
            eval_dataset=fr_tokenized["validation"],
            processing_class=fr_tokenizer,
            data_collator=fr_data_collator,
            compute_metrics=compute_final_response_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
        )
        fr_train_result = fr_trainer.train()
        fr_trainer.save_state()
        cuda_memory_snapshot("after final-response training")
        fr_test_metrics = dict(fr_trainer.evaluate(fr_tokenized["test"]))
        fr_train_metrics = dict(fr_train_result.metrics)
        FINAL_RESPONSE_TEST_METRICS = dict(fr_test_metrics)

        fr_final_model_dir.mkdir(parents=True, exist_ok=True)
        fr_trainer.save_model(str(fr_final_model_dir))
        fr_tokenizer.save_pretrained(str(fr_final_model_dir))
    else:
        previous_provenance = read_json_if_exists(fr_final_model_dir / "training_provenance.json", {})
        fr_train_metrics = dict(previous_provenance.get("train_metrics") or {})
        fr_test_metrics = dict(previous_provenance.get("test_metrics") or {})
        FINAL_RESPONSE_TEST_METRICS = dict(fr_test_metrics) if fr_test_metrics else None

    final_labels_doc = {
        "label_mode": "production",
        "labels": FINAL_RESPONSE_LABELS,
        "label2id": final_response_label2id,
        "id2label": final_response_id2label,
    }
    final_thresholds = {
        "schema_version": FINAL_RESPONSE_THRESHOLDS_SCHEMA_VERSION,
        "mode": "shadow",
        "default_action": "allow",
        "labels": {
            "valid_final_response": {"action": "allow", "advisory_min_confidence": 0.0, "enforce_min_confidence": 1.01},
            "missing_tool_fact": {"action": "advisory_then_enforce_after_eval", "advisory_min_confidence": 0.90, "enforce_min_confidence": 0.995},
            "contradicts_tool_result": {"action": "advisory_then_enforce_after_eval", "advisory_min_confidence": 0.90, "enforce_min_confidence": 0.995},
            "unsupported_claim": {"action": "advisory_then_enforce_after_eval", "advisory_min_confidence": 0.90, "enforce_min_confidence": 0.995},
            "failed_to_acknowledge_data_gap": {"action": "advisory_then_enforce_after_eval", "advisory_min_confidence": 0.90, "enforce_min_confidence": 0.995},
        },
    }
    final_manifest = {
        "artifact_schema_version": FINAL_RESPONSE_ARTIFACT_SCHEMA_VERSION,
        "model_kind": "text-classification-cross-encoder",
        "base_model": BASE_MODEL,
        "label_mode": "production",
        "input_schema_version": FINAL_RESPONSE_INPUT_SCHEMA_VERSION,
        "serializer": FINAL_RESPONSE_SERIALIZER_VERSION,
        "max_length": FR_MAX_LENGTH,
        "requested_gpu_profile": GPU_PROFILE,
        "run_profile": RUN_PROFILE,
        "memory_profile": FINAL_RESPONSE_MEMORY_PROFILE,
        "gpu_info": GPU_INFO,
        "precision_flags": {"fp16": USE_FP16, "bf16": USE_BF16, "tf32": USE_TF32},
        "onnx_file": "model.onnx",
        "quantized_onnx_file": "model_quantized.onnx",
        "tokenizer_required_files": [
            "tokenizer.json",
            "tokenizer_config.json",
            "special_tokens_map.json",
            "spm.model",
        ],
        "labels": FINAL_RESPONSE_LABELS,
        "deployment_default": "shadow",
        "shadow_first_reason": "experimental final-response verifier; promote only after eval replay",
        "created_unix": int(time.time()),
    }
    final_training_provenance = {
        "schema_version": "final-response-verifier-training-provenance/v1",
        "base_model": BASE_MODEL,
        "run_profile": RUN_PROFILE,
        "gpu_info": GPU_INFO,
        "memory_profile": FINAL_RESPONSE_MEMORY_PROFILE,
        "rows": int(len(final_balanced)),
        "train_rows": int(len(fr_train_df)),
        "validation_rows": int(len(fr_valid_df)),
        "test_rows": int(len(fr_test_df)),
        "label_counts": {str(k): int(v) for k, v in final_balanced["label"].value_counts().to_dict().items()},
        "resumed_from_checkpoint": bool(existing_checkpoint and not retrain_final_response),
        "train_metrics": fr_train_metrics,
        "test_metrics": fr_test_metrics,
    }

    write_final_response_json_artifacts(fr_final_model_dir)

    FINAL_RESPONSE_TRAINING_RELEASE_NAMES = [
        "fr_model", "fr_trainer", "fr_tokenizer", "fr_tokenized", "fr_ds", "fr_data_collator",
        "fr_class_weights", "fr_training_args", "fr_train_result",
    ]
    cleanup_runtime_objects(FINAL_RESPONSE_TRAINING_RELEASE_NAMES, "before final-response ONNX export")

    fr_onnx_dir = WORKDIR / "final_response_onnx"
    fr_main_onnx = export_final_response_onnx(fr_final_model_dir, fr_onnx_dir)

    from onnxruntime.quantization import quantize_dynamic, QuantType
    fr_quantized_onnx = fr_onnx_dir / "model_quantized.onnx"
    if fr_quantized_onnx.exists():
        fr_quantized_onnx.unlink()
    quantize_dynamic(str(fr_main_onnx), str(fr_quantized_onnx), weight_type=QuantType.QInt8)
    if not fr_quantized_onnx.exists():
        raise RuntimeError("Final-response ONNX quantization did not write model_quantized.onnx")

    final_manifest["quantized_onnx_file"] = "model_quantized.onnx"
    write_final_response_json_artifacts(fr_final_model_dir)
    copy_final_response_deployment_files(fr_final_model_dir, fr_onnx_dir)
    final_response_onnx_parity_report = run_final_response_onnx_parity(fr_final_model_dir, fr_onnx_dir, fr_parity_texts)
    copy_final_response_deployment_files(fr_final_model_dir, fr_onnx_dir)
    print("Final-response ONNX artifacts:")
    print(sorted(p.name for p in fr_onnx_dir.iterdir()))

    FINAL_RESPONSE_MODEL_DIR = fr_final_model_dir
    FINAL_RESPONSE_ONNX_DIR = fr_onnx_dir
    FINAL_RESPONSE_ARTIFACT_DIR = fr_onnx_dir
    print("Final-response dataset:", final_dataset_path, "rows", len(final_balanced))
    print("Final-response model:", FINAL_RESPONSE_MODEL_DIR)
    print("Final-response ONNX artifact:", FINAL_RESPONSE_ONNX_DIR)

    FINAL_RESPONSE_RELEASE_NAMES = [
        "fr_train_df", "fr_valid_df", "fr_test_df", "final_df", "final_balanced",
        "final_pieces", "combined_final_response_rows", "synthetic_final_response_rows",
        "base_final_response_rows", "fr_test_metrics", "fr_train_metrics", "fr_parity_texts",
        "final_response_onnx_parity_report",
    ]
    cleanup_runtime_objects(FINAL_RESPONSE_RELEASE_NAMES, "after final-response export cleanup")
else:
    print("Final-response verifier pipeline skipped. Set ENABLE_FINAL_RESPONSE_VERIFIER=True in Colab to train/export artifacts.")


## 12. Package and upload artifacts

Upload is enabled by default and private by default. Keep it private until public dataset licenses and any proprietary Forge traces are verified.

In [ ]:
#@title Optional: zip dataset, model, and ONNX artifacts
ZIP_ARTIFACTS = False  #@param {type:"boolean"}

zip_path = None
if ZIP_ARTIFACTS:
    zip_path = shutil.make_archive(
        base_name=str(ARTIFACT_DIR / "toolcall_verifier_artifacts"),
        format="zip",
        root_dir=str(WORKDIR),
        base_dir=".",
    )
    print("Wrote:", zip_path)
else:
    print("Skipping zip archive. Hugging Face upload uses folders directly.")


In [ ]:
#@title Upload dataset/model/ONNX artifacts to Hugging Face Hub
if UPLOAD_TO_HUB:
    from huggingface_hub import HfApi, create_repo, upload_folder
    api = HfApi()
    create_repo(HF_DATASET_REPO, repo_type="dataset", private=PRIVATE, exist_ok=True)
    upload_folder(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        folder_path=str(DATA_DIR),
        commit_message="Add verifier datasets and schemas",
    )
    create_repo(HF_MODEL_REPO, repo_type="model", private=PRIVATE, exist_ok=True)
    upload_folder(
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        folder_path=str(final_model_dir),
        path_in_repo="hf_model",
        commit_message="Add fine-tuned classifier checkpoint",
    )
    upload_folder(
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        folder_path=str(ONNX_DIR),
        path_in_repo="onnx",
        commit_message="Add ONNX deployment artifacts",
    )
    packaged_zip_path = globals().get("zip_path")
    if packaged_zip_path:
        from huggingface_hub import upload_file
        upload_file(
            repo_id=HF_MODEL_REPO,
            repo_type="model",
            path_or_fileobj=packaged_zip_path,
            path_in_repo="artifacts/toolcall_verifier_artifacts.zip",
            commit_message="Add packaged tool-call verifier artifacts",
        )
    else:
        print("No zip archive to upload.")
    if ENABLE_FINAL_RESPONSE_VERIFIER and FINAL_RESPONSE_MODEL_DIR and FINAL_RESPONSE_ONNX_DIR:
        create_repo(HF_FINAL_RESPONSE_MODEL_REPO, repo_type="model", private=PRIVATE, exist_ok=True)
        upload_folder(
            repo_id=HF_FINAL_RESPONSE_MODEL_REPO,
            repo_type="model",
            folder_path=str(FINAL_RESPONSE_MODEL_DIR),
            path_in_repo="hf_model",
            commit_message="Add final-response verifier checkpoint",
        )
        upload_folder(
            repo_id=HF_FINAL_RESPONSE_MODEL_REPO,
            repo_type="model",
            folder_path=str(FINAL_RESPONSE_ONNX_DIR),
            path_in_repo="onnx",
            commit_message="Add final-response verifier ONNX artifacts",
        )
    print("Uploaded dataset and model artifacts.")
else:
    print("Upload disabled.")


## 13. Recommended ablation matrix

The notebook defaults to the full production artifact path: `GPU_PROFILE="auto"`, `ENABLE_FORGE_AUGMENTATION=True`, `ENABLE_FINAL_RESPONSE_VERIFIER=True`, `LABEL_MODE="production"`, `UPLOAD_TO_HUB=True`, and `PRIVATE=True`. On a T4, `auto` selects `t4_proven`; `t4_fast` is only for cheap data-path diagnostics.

Current T4 `serialize_state_v3` run - **candidate-v6-shadow-advisory**:

```text
Latest test results:
  accuracy:                          0.9562
  present-label macro F1:            0.9096
  valid_recall:                      0.9775  (gate: >= 0.94 - pass)
  valid_false_objection @ 0.90:      10/1602 = 0.00624  (raw diagnostic gate: fail)
  wrong_tool_semantic precision:     0.9892  (gate: >= 0.90 - pass)
  wrong_tool_semantic recall:        0.9198
  wrong_arguments_semantic precision:0.9745
  wrong_arguments_semantic recall:   0.9758

Tokenization diagnostics:
  candidate_marker_truncated_rate:   0.0
  schema_marker_truncated_rate:      0.0
  serializer:                        serialize_state_v3
  max_length:                        1024
```

Interpretation:

```text
raw_gate_status:          blocked
  Raw diagnostic valid false objection at 0.90 is still slightly above the ceiling.

deployment_gate_status:   passed
  The guarded deployment rule passes with confidence near 0.98 and margin 0.25.

artifact_status:          promotable_pending_replay, when configured blocking gates pass
  This is still not enforcement-ready.
```

Production action policy under evaluation:

```json
{
  "mode": "advisory",
  "deployment_thresholds": {
    "wrong_tool_semantic": 0.98,
    "wrong_arguments_semantic": 0.98,
    "tool_not_needed": 0.98,
    "needs_clarification": 0.99
  },
  "min_margin_over_valid": 0.25,
  "valid_logit_bias": 0.0,
  "raw_0_90_false_objection": "diagnostic_only"
}
```

Replay gates before any enforcement claim:

```text
ONNX parity passes
quantized ONNX parity passes
shadow replay passes
advisory replay passes
no regression vs no-classifier baseline
```

Replay matrix to run after freezing this artifact:

```text
no_classifier
classifier_fp32_onnx_shadow
classifier_quantized_onnx_shadow
classifier_fp32_onnx_advisory
classifier_quantized_onnx_advisory
```

Protected slice policy:

```python
slice_passes = (
    valid_rows >= 25
    and valid_recall >= 0.94
    and (
        false_objection_rate <= 0.005
        or false_objections <= 1
    )
)
```

For slices under 100 valid examples, treat the count rule as primary. A single false objection in 29 rows should trigger review, not automatically invalidate an otherwise passing deployment-gated artifact.

Do not chase raw `@0.90` perfection unless production will enforce at `0.90`. The safer deployment point is `0.98`, and the model behaves well there.

Targeted data only:

```text
valid -> wrong_arguments_semantic
corrected_error_recovery_positive valid calls
fixed_width_numeric_string valid calls
needs_clarification vs valid boundary
Team-ACE wrong_tool_semantic -> valid misses
Salesforce wrong_arguments_semantic -> valid misses
Salesforce wrong_tool_semantic -> valid misses
```

Do not broadly duplicate valid rows again. The next addendum should add distinct reviewed contrast pairs, not more copies of the same protected positives.


# 14. Rust implementation appendix

## Integration rule

The classifier must run after deterministic checks, never before them.

```text
1. Parse provider response.
2. Validate format, known tool names, and JSON-schema arguments.
3. Enforce required steps, prerequisites, terminal rules, and unsafe batches.
4. If the call is still valid-looking, run the classifier.
5. Shadow mode: log classifier verdict only.
6. Advisory mode: use classifier verdict to choose better nudges.
7. Enforce mode: block only semantic labels that clear the configured confidence threshold and margin over `valid` after eval proof.
```

Your codebase already has the right seams:

- `guardrails/guardrails.rs`: deterministic check pipeline
- `guardrails/policy.rs`: `GuardrailState` and reserved semantic-scoring violation shape
- `guardrails/history.rs`: bounded recent failure memory
- `proxy/handler.rs`: parses OpenAI tools and builds guarded workflow requests
- `bin/forge-eval/report.rs`: good place to add classifier telemetry

## Add modules

```text
guardrails/scoring_context.rs
guardrails/scorer.rs
guardrails/model_scorer.rs        # feature = "onnx-classifier"
guardrails/classifier_artifact.rs
```

## Suggested Rust API

```rust
use crate::clients::base::ToolCall;
use crate::core::tool_spec::ToolSpec;
use crate::guardrails::policy::GuardrailState;

#[derive(Debug, Clone, serde::Serialize, serde::Deserialize)]
pub struct ScoringContext {
    pub schema_version: String,
    pub user_request: String,
    pub required_steps: Vec<String>,
    pub completed_steps: Vec<String>,
    pub pending_steps: Vec<String>,
    pub terminal_tools: Vec<String>,
    pub available_tools: Vec<ToolSpec>,
    pub recent_errors: Vec<String>,
    pub deterministic_state: GuardrailState,
}

#[derive(Debug, Clone, serde::Serialize, serde::Deserialize)]
pub enum ToolCallClass {
    Valid,
    WrongToolSemantic,
    WrongArgumentsSemantic,
    ToolNotNeeded,
    NeedsClarification,
    DeterministicInvalid,
}

#[derive(Debug, Clone)]
pub enum ClassifierAction {
    Allow,
    ShadowLog,
    AdvisoryNudge,
    EnforceBlock,
}

#[derive(Debug, Clone)]
pub struct ToolCallScore {
    pub label: ToolCallClass,
    pub confidence: f32,
    pub rank_score: f32,
    pub action: ClassifierAction,
    pub latency_ms: f32,
}

pub trait ToolCallScorer: Send + Sync {
    fn score(&self, ctx: &ScoringContext, candidate: &ToolCall) -> anyhow::Result<ToolCallScore>;
}
```

## Artifact loader

`classifier_artifact.rs` should validate:

```text
artifact_manifest.json exists and includes training_run_summary/test_metrics provenance
artifact_schema_version == "toolcall-verifier-artifact/v1"
input_schema_version matches the artifact manifest, currently "toolcall-verifier-input/v2" for replacement runs
serializer matches the artifact manifest, currently "serialize_state_v2" for replacement runs
labels.json labels match model config
thresholds.json has every deployed label
tokenizer files exist
ONNX file exists
```

Fail closed for loading errors, but fail open for scoring errors in `shadow` and `advisory` mode.

## Serializer parity test

Use `serializer_fixture.json` from this notebook. Add a Rust unit test that constructs the same `ScoringContext` and `ToolCall`, serializes it, and byte-compares against the fixture's `serialized` string. This catches silent train/inference drift.

## CLI/config flags

Add these to the proxy binary:

```text
--classifier-dir <path>
--classifier-mode off|shadow|advisory|enforce
--classifier-max-latency-ms <n>
FORGE_CLASSIFIER_DIR
FORGE_CLASSIFIER_MODE
FORGE_CLASSIFIER_MAX_LATENCY_MS
```

Default should be `off` unless `--classifier-dir` is passed. First production rollout should use `shadow`.

## Cargo feature sketch

```toml
[features]
default = []
onnx-classifier = ["dep:ort", "dep:tokenizers"]

[dependencies]
ort = { version = "2", optional = true }
tokenizers = { version = "0.20", optional = true }
serde = { version = "1", features = ["derive"] }
serde_json = "1"
anyhow = "1"
```

Pin exact `ort` and `tokenizers` versions after local compile. The notebook writes `spm.model`; if you want Rust inference through `tokenizers`, verify whether a `tokenizer.json` is available. If not, either export a fast tokenizer where safe or use a sidecar scorer process until tokenizer parity is solved.

## Telemetry fields for `forge-eval`

Add these to each report row:

```json
{
  "classifier_enabled": true,
  "classifier_mode": "shadow",
  "classifier_label": "wrong_tool_semantic",
  "classifier_confidence": 0.93,
  "classifier_action": "shadow_log",
  "classifier_model_version": "toolcall-verifier-production-v1",
  "classifier_latency_ms": 4.2
}
```

## Enforcement invariant

The classifier may block only semantic labels:

```text
wrong_tool_semantic
wrong_arguments_semantic
tool_not_needed
needs_clarification
```

It must not override deterministic validation, accept malformed tool calls, execute tools, or rewrite arguments.
